# CREB5 in human AD astrocytes — Notebook 02

GSE157827 target-insulated confirmation and robustness analyses.

This public-release notebook condenses the executed analysis history into the final reproducible workflow. The original audit notebook is retained separately. The donor is the biological replicate throughout differential expression.


## Setup

`RUN_PIPELINE = False` is the fast validation mode for the existing project folder: it does not redownload or recompute the analysis and instead checks the frozen outputs at the end. Set it to `True` only for a clean reconstruction from the source data. `INSTALL_PACKAGES` is separate so repeated tests do not reinstall the environment.


In [ ]:

from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")

ROOT = Path("/content/drive/MyDrive/AD_Astrocyte_Paper_01")
PROJECT = ROOT / "GSE157827_confirmatory_validation"

RUN_PIPELINE = False
INSTALL_PACKAGES = False

PROJECT.mkdir(parents=True, exist_ok=True)

print("Project:", PROJECT)
print("Run full pipeline:", RUN_PIPELINE)


In [ ]:

if INSTALL_PACKAGES:
    import subprocess, sys

    packages = [
        "numpy==2.1.3",
        "pandas==2.2.3",
        "scipy==1.16.3",
        "scikit-learn==1.6.1",
        "anndata==0.12.6",
        "scanpy==1.11.5",
        "harmonypy==2.0.0",
        "igraph==1.0.0",
        "leidenalg==0.12.0",
        "scikit-misc==0.5.2",
        "scrublet==0.2.3",
        "pydeseq2==0.5.4",
        "formulaic==1.2.2",
        "formulaic-contrasts==1.0.0",
        "openpyxl",
        "requests",
    ]
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *packages])


## Confirmatory protocol

The confirmatory target, donor-level pseudobulk analysis unit, expression filter, statistical model rule, contrast, and success criterion were frozen before GSE157827 expression results were examined.


In [ ]:

import json
from datetime import datetime, timezone

protocol_path = ROOT / "ANALYSIS_PROTOCOL_LOCK_v1.json"

protocol = {
    "protocol_version": "1.0",
    "frozen_at_utc": datetime.now(timezone.utc).isoformat(),
    "provisional_study_title": "Cross-cohort transcriptomic analysis of astrocyte responses in Alzheimer's disease",
    "dataset_roles": {
        "SEA_AD_MTG": "primary discovery",
        "SEA_AD_DLPFC": "within-cohort regional replication",
        "GSE160936_EC_SSC": "external regional validation",
        "GSE157827_PFC": "untouched confirmatory validation",
        "GSE268599": "future progression and regional analysis",
    },
    "primary_confirmatory_gene": "CREB5",
    "secondary_locked_genes": ["CSRP1", "SHC1", "KCNH2", "MRGPRF", "AJAP1", "CCDC3"],
    "primary_comparison": "Alzheimer's disease versus non-disease control",
    "primary_analysis_unit": "donor-level astrocyte pseudobulk",
    "counts_requirement": "raw integer counts summed across astrocyte nuclei",
    "expression_filter": "CPM >= 1 in at least the size of the smaller diagnostic group; filtering must not use effect direction",
    "primary_model_rule": "Use disease plus age and sex when metadata are complete and the design is full rank. Missing or unavailable covariates will not be imputed. Simpler models will be reported as sensitivity analyses.",
    "primary_contrast": "AD relative to non-disease control; positive log2FC means higher expression in AD",
    "CREB5_success_rule": "CREB5 must be present, have positive log2FC, and have a two-sided nominal p-value below 0.05 in GSE157827",
    "secondary_gene_rule": "The secondary locked genes will be corrected together for multiple testing",
    "transcriptome_wide_rule": "Genome-wide significance requires FDR below 0.05",
    "prohibited_analysis": "Individual nuclei must not be treated as independent biological replicates for differential expression",
    "interpretation_rule": "Same-direction nonsignificant results are directional support, not statistically significant replication",
}

if RUN_PIPELINE and not protocol_path.exists():
    protocol_path.write_text(json.dumps(protocol, indent=2), encoding="utf-8")

if not protocol_path.exists():
    raise FileNotFoundError(f"Missing protocol lock: {protocol_path}")

print("Protocol lock:", protocol_path)


## Metadata, design lock, and source-file audit

GEO metadata are acquired before expression analysis. The donor/sample design is frozen before raw count inspection.


In [ ]:
if RUN_PIPELINE:
    from pathlib import Path

    import gzip

    import re

    import requests

    import numpy as np

    import pandas as pd

    ROOT = Path('/content/drive/MyDrive/AD_Astrocyte_Paper_01')

    PROJECT = ROOT / 'GSE157827_confirmatory_validation'

    PROJECT.mkdir(parents=True, exist_ok=True)

    protocol_path = ROOT / 'ANALYSIS_PROTOCOL_LOCK_v1.json'

    if not protocol_path.exists():
        raise FileNotFoundError('The locked analysis protocol is missing. Stop.')

    soft_url = 'https://ftp.ncbi.nlm.nih.gov/geo/series/GSE157nnn/GSE157827/soft/GSE157827_family.soft.gz'

    soft_path = PROJECT / 'GSE157827_family.soft.gz'

    temporary_path = Path(str(soft_path) + '.part')

    manifest_path = PROJECT / 'GSE157827_sample_metadata_raw.csv'

    if not soft_path.exists():
        print('Downloading GSE157827 GEO metadata...')
        with requests.get(soft_url, stream=True, timeout=120) as response:
            response.raise_for_status()
            with open(temporary_path, 'wb') as handle:
                for chunk in response.iter_content(chunk_size=1024 * 1024):
                    if chunk:
                        handle.write(chunk)
        temporary_path.replace(soft_path)
        print('Metadata download complete.')
    else:
        print('Existing GEO metadata file found; it will not be downloaded again.')

    print('Compressed metadata size:', round(soft_path.stat().st_size / 1024 ** 2, 3), 'MB')

    samples = {}

    current_gsm = None

    with gzip.open(soft_path, mode='rt', encoding='utf-8', errors='replace') as handle:
        for raw_line in handle:
            line = raw_line.rstrip('\n')
            if line.startswith('^SAMPLE = '):
                current_gsm = line.split(' = ', 1)[1].strip()
                samples[current_gsm] = {}
            elif current_gsm is not None and line.startswith('!Sample_') and (' = ' in line):
                key, value = line.split(' = ', 1)
                key = key.replace('!Sample_', '', 1)
                samples[current_gsm].setdefault(key, []).append(value.strip())

    print('\nSamples parsed:', len(samples))

    if len(samples) != 21:
        raise RuntimeError(f'Expected 21 samples, found {len(samples)}.')

    def clean_column_name(text):
        text = text.strip().lower()
        text = re.sub('[^a-z0-9]+', '_', text)
        return text.strip('_')

    def unique_join(values):
        result = []
        for value in values:
            if value not in result:
                result.append(value)
        return ' | '.join(result)

    rows = []


In [ ]:
if RUN_PIPELINE:
    for gsm, fields in samples.items():
        row = {'GSM': gsm}
        for field in ['title', 'source_name_ch1', 'organism_ch1', 'molecule_ch1', 'extract_protocol_ch1', 'data_processing', 'platform_id', 'relation', 'supplementary_file']:
            if field in fields:
                row[field] = unique_join(fields[field])
        characteristic_values = []
        for key, values in fields.items():
            if key.startswith('characteristics_ch1'):
                characteristic_values.extend(values)
        row['all_characteristics_raw'] = unique_join(characteristic_values)
        for item in characteristic_values:
            if ':' in item:
                label, value = item.split(':', 1)
                column = 'characteristic__' + clean_column_name(label)
                value = value.strip()
            else:
                column = 'characteristic__unparsed'
                value = item.strip()
            if column in row:
                existing = str(row[column]).split(' | ')
                if value not in existing:
                    row[column] = str(row[column]) + ' | ' + value
            else:
                row[column] = value
        rows.append(row)

    manifest = pd.DataFrame(rows).sort_values('GSM').reset_index(drop=True)

    titles = manifest['title'].astype(str).str.strip()

    manifest['diagnosis_inferred_from_title'] = np.select([titles.str.fullmatch('AD\\d+', case=False), titles.str.fullmatch('NC\\d+', case=False)], ['AD', 'Control'], default='Unresolved')

    if manifest['GSM'].duplicated().any():
        raise RuntimeError('Duplicate GSM accessions found.')

    if manifest['title'].duplicated().any():
        raise RuntimeError('Duplicate sample titles found.')

    diagnosis_counts = manifest['diagnosis_inferred_from_title'].value_counts().to_dict()

    expected_counts = {'AD': 12, 'Control': 9}

    if diagnosis_counts != expected_counts:
        raise RuntimeError(f'Unexpected diagnosis counts: {diagnosis_counts}')

    manifest.to_csv(manifest_path, index=False)

    print('\n==========================================')

    print('GSE157827 SAMPLE MANIFEST')

    print('==========================================')

    display_columns = ['GSM', 'title', 'diagnosis_inferred_from_title']

    characteristic_columns = [column for column in manifest.columns if column.startswith('characteristic__')]

    display(manifest[display_columns + characteristic_columns])

    print('\nDiagnosis counts:')

    print(manifest['diagnosis_inferred_from_title'].value_counts().to_string())

    print('\nAll parsed metadata columns:')

    for column in manifest.columns:
        print(column)

    print('\n==========================================')

    print('CHARACTERISTIC VALUE SUMMARY')

    print('==========================================')

    for column in characteristic_columns:
        values = manifest[column].dropna().astype(str).value_counts()
        print(f'\n{column}')
        print(values.to_string())


In [ ]:
if RUN_PIPELINE:
    print('\nMetadata manifest saved:')

    print(manifest_path)

    print('\nGSE157827 METADATA AUDIT COMPLETE')

    print('No expression counts were downloaded or examined.')


In [ ]:
if RUN_PIPELINE:
    from pathlib import Path

    from datetime import datetime, timezone

    import hashlib

    import json

    import numpy as np

    import pandas as pd

    ROOT = Path('/content/drive/MyDrive/AD_Astrocyte_Paper_01')

    PROJECT = ROOT / 'GSE157827_confirmatory_validation'

    metadata_path = PROJECT / 'GSE157827_sample_metadata_raw.csv'

    design_manifest_path = PROJECT / 'GSE157827_confirmatory_design_manifest.csv'

    design_lock_path = PROJECT / 'GSE157827_DESIGN_LOCK_v1.json'

    protocol_path = ROOT / 'ANALYSIS_PROTOCOL_LOCK_v1.json'

    for required_path in [metadata_path, protocol_path]:
        if not required_path.exists():
            raise FileNotFoundError(f'Required file missing: {required_path}')

    metadata = pd.read_csv(metadata_path, dtype={'GSM': str, 'title': str})

    print('Metadata shape:', metadata.shape)

    if len(metadata) != 21:
        raise RuntimeError(f'Expected 21 samples, found {len(metadata)}.')

    if metadata['GSM'].nunique() != 21:
        raise RuntimeError('GSM accessions are not unique.')

    required_columns = ['GSM', 'title', 'characteristic__tissue', 'characteristic__diagnosis', 'diagnosis_inferred_from_title']

    missing_columns = [column for column in required_columns if column not in metadata.columns]

    if missing_columns:
        raise RuntimeError(f'Missing metadata columns: {missing_columns}')

    diagnosis_map = {'healthy control': 'Control', 'AD': 'AD'}

    metadata['diagnosis'] = metadata['characteristic__diagnosis'].map(diagnosis_map)

    if metadata['diagnosis'].isna().any():
        unresolved = metadata.loc[metadata['diagnosis'].isna(), ['GSM', 'title', 'characteristic__diagnosis']]
        print(unresolved)
        raise RuntimeError('At least one GEO diagnosis could not be resolved.')

    agreement = metadata['diagnosis'] == metadata['diagnosis_inferred_from_title']

    print('Title/characteristic diagnosis agreement:', int(agreement.sum()), '/', len(agreement))

    if not agreement.all():
        raise RuntimeError('Diagnosis disagreement detected.')

    tissues = metadata['characteristic__tissue'].dropna().astype(str).str.lower().unique()

    if len(tissues) != 1 or tissues[0] != 'prefrontal cortex':
        raise RuntimeError(f'Unexpected tissue values: {tissues}')

    design_manifest = metadata[['GSM', 'title', 'diagnosis', 'characteristic__tissue', 'organism_ch1', 'molecule_ch1', 'platform_id']].copy()

    design_manifest = design_manifest.sort_values('GSM').reset_index(drop=True)

    group_counts = design_manifest['diagnosis'].value_counts().to_dict()

    expected_counts = {'AD': 12, 'Control': 9}

    if group_counts != expected_counts:
        raise RuntimeError(f'Unexpected group counts: {group_counts}')

    diagnosis_binary = design_manifest['diagnosis'].eq('AD').astype(float).to_numpy()

    design_matrix = np.column_stack([np.ones(len(design_manifest), dtype=float), diagnosis_binary])

    design_rank = np.linalg.matrix_rank(design_matrix)

    n_parameters = design_matrix.shape[1]

    residual_df = design_matrix.shape[0] - design_rank

    condition_number = np.linalg.cond(design_matrix)

    if design_rank != n_parameters:
        raise RuntimeError('The planned design matrix is not full rank.')

    manifest_text = design_manifest.to_csv(index=False)

    manifest_hash = hashlib.sha256(manifest_text.encode('utf-8')).hexdigest()


In [ ]:
if RUN_PIPELINE:
    if design_manifest_path.exists():
        existing_text = design_manifest_path.read_text(encoding='utf-8')
        if existing_text != manifest_text:
            raise RuntimeError('An existing design manifest differs from the newly constructed manifest. Stop.')
        print('Existing design manifest verified; it was not overwritten.')
    else:
        design_manifest_path.write_text(manifest_text, encoding='utf-8')
        print('Design manifest saved.')

    design_lock = {'lock_version': '1.0', 'locked_at_utc': datetime.now(timezone.utc).isoformat(), 'dataset': 'GSE157827', 'tissue': 'prefrontal cortex', 'n_donors': 21, 'group_counts': {'AD': 12, 'Control': 9}, 'biological_replicate': 'individual human donor', 'cell_type': 'astrocyte', 'pseudobulk_definition': 'Raw integer counts summed across all accepted astrocyte nuclei separately for each donor', 'primary_design': '~ diagnosis', 'contrast': ['diagnosis', 'AD', 'Control'], 'effect_direction': 'Positive log2FoldChange means higher expression in AD', 'covariates': [], 'covariate_reason': 'GEO sample metadata provide diagnosis and tissue only; age, sex, RIN, PMI, and brain pH are unavailable and will not be imputed', 'expression_filter': 'CPM >= 1 in at least 9 of 21 donor pseudobulk samples', 'mitochondrial_gene_rule': 'Mitochondrial genes will be excluded from the primary cross-cohort gene universe and pathway ranking', 'primary_confirmatory_gene': 'CREB5', 'CREB5_success_rule': 'CREB5 must be present, have positive log2FoldChange, and have a two-sided nominal p-value below 0.05', 'secondary_locked_genes': ['CSRP1', 'SHC1', 'KCNH2', 'MRGPRF', 'AJAP1', 'CCDC3'], 'secondary_testing_rule': 'Benjamini-Hochberg correction across the six secondary locked genes', 'genome_wide_testing_rule': 'Transcriptome-wide significance requires FDR < 0.05', 'cell_level_testing_prohibited': True, 'cell_annotation_rule': 'Astrocyte identification must be performed without using CREB5 expression or disease differential-expression results. Author-provided labels will be preferred if available; otherwise broad marker-based annotation will be defined before pseudobulk differential expression.', 'design_manifest': str(design_manifest_path), 'design_manifest_sha256': manifest_hash, 'design_rank': int(design_rank), 'n_parameters': int(n_parameters), 'residual_degrees_of_freedom': int(residual_df), 'condition_number': float(condition_number)}

    if design_lock_path.exists():
        print('A GSE157827 design lock already exists and was NOT overwritten:')
        print(design_lock_path)
    else:
        design_lock_path.write_text(json.dumps(design_lock, indent=2), encoding='utf-8')
        print('GSE157827 design lock saved:')
        print(design_lock_path)

    print('\n==========================================')

    print('LOCKED GSE157827 DESIGN')

    print('==========================================')

    print(design_manifest[['GSM', 'title', 'diagnosis']].to_string(index=False))

    print('\nGroup counts:', group_counts)

    print('Design: ~ diagnosis')

    print('Design rank:', design_rank, '/', n_parameters)

    print('Residual degrees of freedom:', residual_df)

    print('Condition number:', round(condition_number, 4))

    print('Manifest SHA-256:', manifest_hash)

    print('\nGSE157827 STATISTICAL DESIGN LOCKED')

    print('Expression data have still not been downloaded or examined.')


In [ ]:
if RUN_PIPELINE:
    from pathlib import Path

    from urllib.parse import urljoin, urlparse, unquote

    import html

    import re

    import shutil

    import requests

    import pandas as pd

    ROOT = Path('/content/drive/MyDrive/AD_Astrocyte_Paper_01')

    PROJECT = ROOT / 'GSE157827_confirmatory_validation'

    design_lock_path = PROJECT / 'GSE157827_DESIGN_LOCK_v1.json'

    inventory_path = PROJECT / 'GSE157827_remote_file_inventory.csv'

    if not design_lock_path.exists():
        raise FileNotFoundError('GSE157827 design lock is missing. Stop.')

    directories = {'supplementary': 'https://ftp.ncbi.nlm.nih.gov/geo/series/GSE157nnn/GSE157827/suppl/', 'series_matrix': 'https://ftp.ncbi.nlm.nih.gov/geo/series/GSE157nnn/GSE157827/matrix/'}

    headers = {'User-Agent': 'Academic research metadata audit for public GEO dataset GSE157827'}

    def list_remote_directory(directory_name, directory_url):
        response = requests.get(directory_url, headers=headers, timeout=60)
        response.raise_for_status()
        hrefs = re.findall('href=["\']([^"\']+)["\']', response.text, flags=re.IGNORECASE)
        rows = []
        seen_urls = set()
        for href in hrefs:
            href = html.unescape(href)
            full_url = urljoin(directory_url, href)
            parsed = urlparse(full_url)
            filename = unquote(Path(parsed.path).name)
            if not filename or href.endswith('/') or filename in {'.', '..'} or (full_url in seen_urls):
                continue
            seen_urls.add(full_url)
            size_bytes = None
            content_type = None
            head_status = None
            try:
                head = requests.head(full_url, headers=headers, allow_redirects=True, timeout=60)
                head_status = head.status_code
                content_type = head.headers.get('Content-Type')
                content_length = head.headers.get('Content-Length')
                if content_length is not None:
                    size_bytes = int(content_length)
            except requests.RequestException:
                head_status = 'HEAD request unavailable'
            lower_name = filename.lower()
            annotation_candidate = any((keyword in lower_name for keyword in ['annot', 'cluster', 'celltype', 'cell_type', 'metadata', 'meta_data']))
            if filename == 'GSE157827_RAW.tar':
                file_role = 'Raw supplementary archive'
            elif 'series_matrix' in lower_name:
                file_role = 'GEO series matrix'
            elif annotation_candidate:
                file_role = 'Possible annotation/metadata file'
            else:
                file_role = 'Other'
            rows.append({'source_directory': directory_name, 'filename': filename, 'file_role': file_role, 'annotation_candidate': annotation_candidate, 'size_bytes': size_bytes, 'size_MB': round(size_bytes / 1024 ** 2, 3) if size_bytes is not None else None, 'size_GB': round(size_bytes / 1024 ** 3, 3) if size_bytes is not None else None, 'head_status': head_status, 'content_type': content_type, 'url': full_url})
        return rows

    inventory_rows = []


In [ ]:
if RUN_PIPELINE:
    for directory_name, directory_url in directories.items():
        print(f'Inspecting {directory_name} directory...')
        inventory_rows.extend(list_remote_directory(directory_name, directory_url))

    inventory = pd.DataFrame(inventory_rows)

    if len(inventory) == 0:
        raise RuntimeError('No remote GEO files were detected.')

    inventory = inventory.sort_values(['source_directory', 'filename']).reset_index(drop=True)

    raw_archive = inventory[inventory['filename'] == 'GSE157827_RAW.tar']

    if len(raw_archive) != 1:
        raise RuntimeError(f'Expected exactly one GSE157827_RAW.tar entry, found {len(raw_archive)}.')

    raw_size = raw_archive['size_bytes'].iloc[0]

    if pd.notna(raw_size) and raw_size < 500 * 1024 ** 2:
        raise RuntimeError('The reported raw archive is unexpectedly small.')

    inventory.to_csv(inventory_path, index=False)

    local_usage = shutil.disk_usage('/content')

    drive_usage = shutil.disk_usage('/content/drive/MyDrive')

    local_free_GB = local_usage.free / 1024 ** 3

    drive_free_GB = drive_usage.free / 1024 ** 3

    print('\n==========================================')

    print('GSE157827 REMOTE FILE INVENTORY')

    print('==========================================')

    display(inventory[['source_directory', 'filename', 'file_role', 'annotation_candidate', 'size_MB', 'size_GB', 'head_status']])

    print('\nPossible author-provided annotation files:')

    annotation_files = inventory[inventory['annotation_candidate']]

    if len(annotation_files) == 0:
        print('None identified from the remote filenames.')
    else:
        print(annotation_files[['filename', 'url']].to_string(index=False))

    print('\nStorage:')

    print('Colab local free space:', round(local_free_GB, 2), 'GB')

    print('Google Drive reported free space:', round(drive_free_GB, 2), 'GB')

    print('\nInventory saved:')

    print(inventory_path)

    print('\nGSE157827 REMOTE INVENTORY COMPLETE')

    print('No expression matrices were downloaded.')


## Raw archive and structural audit

The GEO raw archive is downloaded once and cached on Drive. Matrix files are extracted to local Colab storage only during a full rebuild, then checked for dimensions and donor linkage.


In [ ]:
if RUN_PIPELINE:
    from pathlib import Path, PurePosixPath

    from datetime import datetime, timezone

    import hashlib

    import json

    import re

    import tarfile

    import requests

    import pandas as pd

    ROOT = Path('/content/drive/MyDrive/AD_Astrocyte_Paper_01')

    PROJECT = ROOT / 'GSE157827_confirmatory_validation'

    design_lock_path = PROJECT / 'GSE157827_DESIGN_LOCK_v1.json'

    design_manifest_path = PROJECT / 'GSE157827_confirmatory_design_manifest.csv'

    archive_path = PROJECT / 'GSE157827_RAW.tar'

    partial_path = Path(str(archive_path) + '.part')

    filelist_path = PROJECT / 'GSE157827_GEO_filelist.txt'

    integrity_path = PROJECT / 'GSE157827_RAW_integrity.json'

    member_inventory_path = PROJECT / 'GSE157827_RAW_member_inventory.csv'

    archive_url = 'https://ftp.ncbi.nlm.nih.gov/geo/series/GSE157nnn/GSE157827/suppl/GSE157827_RAW.tar'

    filelist_url = 'https://ftp.ncbi.nlm.nih.gov/geo/series/GSE157nnn/GSE157827/suppl/filelist.txt'

    headers = {'User-Agent': 'Academic research download for public GEO dataset GSE157827'}

    for required_path in [design_lock_path, design_manifest_path]:
        if not required_path.exists():
            raise FileNotFoundError(f'Required locked file missing: {required_path}')

    filelist_response = requests.get(filelist_url, headers=headers, timeout=60)

    filelist_response.raise_for_status()

    filelist_text = filelist_response.text

    if 'GSE157827_RAW.tar' not in filelist_text:
        raise RuntimeError('GEO file list does not contain GSE157827_RAW.tar.')

    if filelist_path.exists():
        existing_filelist = filelist_path.read_text(encoding='utf-8', errors='replace')
        if existing_filelist != filelist_text:
            raise RuntimeError('The remote GEO file list differs from the previously saved file list. Stop.')
    else:
        filelist_path.write_text(filelist_text, encoding='utf-8')

    print('GEO FILE LIST')

    print(filelist_text)

    head = requests.head(archive_url, headers=headers, allow_redirects=True, timeout=60)

    head.raise_for_status()

    remote_size_text = head.headers.get('Content-Length')

    if remote_size_text is None:
        raise RuntimeError('The remote server did not report archive size.')

    remote_size = int(remote_size_text)

    print('\nExpected archive size:', f'{remote_size:,}', 'bytes')

    print('Expected archive size:', round(remote_size / 1024 ** 3, 3), 'GB')


In [ ]:
if RUN_PIPELINE:
    if archive_path.exists():
        existing_size = archive_path.stat().st_size
        if existing_size != remote_size:
            raise RuntimeError(f'A final archive exists but its size is wrong:\nObserved: {existing_size:,}\nExpected: {remote_size:,}\nIt was not deleted or overwritten.')
        print('\nComplete archive already exists; download skipped.')
    else:
        starting_byte = partial_path.stat().st_size if partial_path.exists() else 0
        if starting_byte > remote_size:
            raise RuntimeError('Partial download is larger than the remote archive. Stop.')
        if starting_byte == remote_size:
            partial_path.replace(archive_path)
            print('Completed partial file promoted to final archive.')
        else:
            request_headers = headers.copy()
            if starting_byte > 0:
                request_headers['Range'] = f'bytes={starting_byte}-'
                print('\nResuming download from byte:', f'{starting_byte:,}')
            else:
                print('\nStarting archive download...')
            with requests.get(archive_url, headers=request_headers, stream=True, timeout=(60, 300)) as response:
                response.raise_for_status()
                if starting_byte > 0 and response.status_code == 206:
                    write_mode = 'ab'
                    downloaded = starting_byte
                elif starting_byte > 0 and response.status_code == 200:
                    print('Server did not honor resume request; restarting the partial download.')
                    write_mode = 'wb'
                    downloaded = 0
                else:
                    write_mode = 'wb'
                    downloaded = 0
                report_interval = 100 * 1024 ** 2
                next_report = downloaded + report_interval
                with open(partial_path, write_mode) as output:
                    for chunk in response.iter_content(chunk_size=8 * 1024 ** 2):
                        if not chunk:
                            continue
                        output.write(chunk)
                        downloaded += len(chunk)
                        if downloaded >= next_report:
                            print(f'Downloaded {downloaded / 1024 ** 3:.2f} / {remote_size / 1024 ** 3:.2f} GB')
                            next_report += report_interval
            observed_partial_size = partial_path.stat().st_size
            if observed_partial_size != remote_size:
                raise RuntimeError(f'Download is incomplete.\nDownloaded: {observed_partial_size:,} bytes\nExpected: {remote_size:,} bytes\nRerun this cell to resume.')
            partial_path.replace(archive_path)
            print('\nArchive download complete.')

    final_size = archive_path.stat().st_size

    if final_size != remote_size:
        raise RuntimeError('Final archive size does not match the remote size.')

    print('Final archive size verified:', f'{final_size:,}', 'bytes')

    print('\nCalculating SHA-256...')

    sha256 = hashlib.sha256()


In [ ]:
if RUN_PIPELINE:
    with open(archive_path, 'rb') as handle:
        while True:
            block = handle.read(8 * 1024 ** 2)
            if not block:
                break
            sha256.update(block)

    archive_sha256 = sha256.hexdigest()

    print('SHA-256:', archive_sha256)

    print('\nInspecting TAR member headers...')

    member_rows = []

    with tarfile.open(archive_path, mode='r') as archive:
        members = archive.getmembers()
        for member in members:
            member_path = PurePosixPath(member.name)
            if member_path.is_absolute() or '..' in member_path.parts:
                raise RuntimeError(f'Unsafe archive member path: {member.name}')
            filename = member_path.name
            lower_name = filename.lower()
            gsm_match = re.search('(GSM\\d+)', filename, flags=re.IGNORECASE)
            gsm = gsm_match.group(1).upper() if gsm_match else None
            if '.mtx' in lower_name or 'matrix.mtx' in lower_name:
                file_role = 'matrix'
            elif 'barcode' in lower_name:
                file_role = 'barcodes'
            elif ('feature' in lower_name or 'gene' in lower_name) and '.tsv' in lower_name:
                file_role = 'features_or_genes'
            else:
                file_role = 'other'
            member_rows.append({'member_name': member.name, 'filename': filename, 'GSM': gsm, 'file_role': file_role, 'is_file': member.isfile(), 'size_bytes': int(member.size), 'size_MB': round(member.size / 1024 ** 2, 3)})

    member_inventory = pd.DataFrame(member_rows)

    if len(member_inventory) == 0:
        raise RuntimeError('The TAR archive contains no members.')

    design_manifest = pd.read_csv(design_manifest_path, dtype={'GSM': str})

    expected_gsms = set(design_manifest['GSM'].astype(str))

    observed_gsms = set(member_inventory['GSM'].dropna().astype(str))

    missing_gsms = sorted(expected_gsms - observed_gsms)

    unexpected_gsms = sorted(observed_gsms - expected_gsms)

    if missing_gsms:
        raise RuntimeError('Locked samples missing from archive:\n' + '\n'.join(missing_gsms))

    if unexpected_gsms:
        raise RuntimeError('Unexpected GSM samples in archive:\n' + '\n'.join(unexpected_gsms))

    role_summary = member_inventory[member_inventory['GSM'].notna() & member_inventory['is_file']].groupby(['GSM', 'file_role'], observed=True).size().unstack(fill_value=0).reindex(sorted(expected_gsms))

    for required_role in ['matrix', 'barcodes', 'features_or_genes']:
        if required_role not in role_summary.columns:
            raise RuntimeError(f'No {required_role} files found in the archive.')
        if not (role_summary[required_role] >= 1).all():
            bad_samples = role_summary.index[role_summary[required_role] < 1].tolist()
            raise RuntimeError(f'Samples missing {required_role} files: {bad_samples}')

    member_inventory.to_csv(member_inventory_path, index=False)

    integrity_record = {'dataset': 'GSE157827', 'download_url': archive_url, 'downloaded_at_utc': datetime.now(timezone.utc).isoformat(), 'size_bytes': int(final_size), 'sha256': archive_sha256, 'tar_member_count': int(len(member_inventory)), 'observed_GSM_count': int(len(observed_gsms)), 'archive_extracted': False}

    new_integrity_text = json.dumps(integrity_record, indent=2)


In [ ]:
if RUN_PIPELINE:
    if integrity_path.exists():
        previous = json.loads(integrity_path.read_text(encoding='utf-8'))
        if previous.get('size_bytes') != integrity_record['size_bytes'] or previous.get('sha256') != integrity_record['sha256']:
            raise RuntimeError('Existing integrity record does not match the downloaded archive.')
        print('Existing integrity record verified.')
    else:
        integrity_path.write_text(new_integrity_text, encoding='utf-8')
        print('Integrity record saved.')

    print('\n==========================================')

    print('RAW ARCHIVE MEMBER SUMMARY')

    print('==========================================')

    print('TAR members:', len(member_inventory))

    print('GSM samples:', len(observed_gsms))

    print('\nFiles by sample and role:')

    display(role_summary)

    print('\nFirst 30 archive members:')

    display(member_inventory.head(30))

    print('\nMember inventory saved:')

    print(member_inventory_path)

    print('\nIntegrity record saved:')

    print(integrity_path)

    print('\nGSE157827 RAW ARCHIVE VERIFIED')

    print('No files were extracted and no expression values were examined.')


In [ ]:
if RUN_PIPELINE:
    from pathlib import Path, PurePosixPath

    from datetime import datetime, timezone

    from collections import Counter

    import pandas as pd

    import numpy as np

    import tarfile

    import gzip

    import hashlib

    import shutil

    import re

    import json

    import os

    try:
        from tqdm.auto import tqdm
    except ImportError:

        def tqdm(x, **kwargs):
            return x

    PROJECT_DIR = Path('/content/drive/MyDrive/AD_Astrocyte_Paper_01')

    VALIDATION_DIR = PROJECT_DIR / 'GSE157827_confirmatory_validation'

    ARCHIVE_PATH = VALIDATION_DIR / 'GSE157827_RAW.tar'

    INTEGRITY_PATH = VALIDATION_DIR / 'GSE157827_RAW_integrity.json'

    METADATA_PATH = VALIDATION_DIR / 'GSE157827_sample_metadata_raw.csv'

    LOCAL_EXTRACT_DIR = Path('/content/GSE157827_raw_10x')

    EXTRACTION_MANIFEST_PATH = VALIDATION_DIR / 'GSE157827_local_extraction_manifest.csv'

    DIMENSION_AUDIT_PATH = VALIDATION_DIR / 'GSE157827_raw_matrix_dimension_audit.csv'

    STRUCTURAL_LOCK_PATH = VALIDATION_DIR / 'GSE157827_STRUCTURAL_AUDIT_LOCK_v1.json'

    EXPECTED_ARCHIVE_BYTES = 1313280000

    EXPECTED_ARCHIVE_SHA256 = 'dff1b5358289a44fb3f0db1422aa0351ab573c201e592813653a6072120c9682'

    for required_path in [ARCHIVE_PATH, INTEGRITY_PATH, METADATA_PATH]:
        if not required_path.exists():
            raise FileNotFoundError(f'Required file is missing: {required_path}')

    if ARCHIVE_PATH.stat().st_size != EXPECTED_ARCHIVE_BYTES:
        raise RuntimeError(f'The archive size no longer matches the verified archive.\nObserved: {ARCHIVE_PATH.stat().st_size:,} bytes\nExpected: {EXPECTED_ARCHIVE_BYTES:,} bytes')

    integrity_text = INTEGRITY_PATH.read_text(encoding='utf-8').lower()

    if EXPECTED_ARCHIVE_SHA256 not in integrity_text:
        raise RuntimeError('The saved integrity record does not contain the expected SHA-256.')

    print('Verified archive identity recovered from the integrity record.')

    print(f'Archive size: {EXPECTED_ARCHIVE_BYTES:,} bytes')

    print(f'SHA-256: {EXPECTED_ARCHIVE_SHA256}')

    metadata = pd.read_csv(METADATA_PATH, dtype=str).fillna('')

    normalized_columns = {re.sub('[^a-z0-9]+', '_', str(column).lower()).strip('_'): column for column in metadata.columns}

    gsm_column = None

    for candidate in ['gsm', 'geo_accession', 'sample_accession']:
        if candidate in normalized_columns:
            gsm_column = normalized_columns[candidate]
            break

    if gsm_column is None:
        gsm_scores = {}
        for column in metadata.columns:
            gsm_scores[column] = metadata[column].astype(str).str.contains('\\bGSM\\d+\\b', regex=True).sum()
        gsm_column = max(gsm_scores, key=gsm_scores.get)
        if gsm_scores[gsm_column] == 0:
            raise RuntimeError('Could not identify a GSM column in the metadata.')

    metadata['_GSM'] = metadata[gsm_column].astype(str).str.extract('(GSM\\d+)', expand=False)


In [ ]:
if RUN_PIPELINE:
    if metadata['_GSM'].isna().any():
        raise RuntimeError('At least one metadata row lacks a valid GSM accession.')

    if metadata['_GSM'].duplicated().any():
        duplicated = metadata.loc[metadata['_GSM'].duplicated(keep=False), '_GSM'].tolist()
        raise RuntimeError(f'Duplicated GSM accessions in metadata: {duplicated}')

    expected_gsms = sorted(metadata['_GSM'].unique())

    if len(expected_gsms) != 21:
        raise RuntimeError(f'Expected 21 locked GSM samples, found {len(expected_gsms)}.')

    title_column = normalized_columns.get('title')

    diagnosis_column = normalized_columns.get('diagnosis')

    metadata_by_gsm = metadata.set_index('_GSM')

    print(f'Locked GSM samples recovered: {len(expected_gsms)}')

    suffix_to_role = {'barcodes.tsv.gz': 'barcodes', 'features.tsv.gz': 'features_or_genes', 'genes.tsv.gz': 'features_or_genes', 'matrix.mtx.gz': 'matrix'}

    filename_pattern = re.compile('^(GSM\\d+)_(.+)_(barcodes\\.tsv\\.gz|features\\.tsv\\.gz|genes\\.tsv\\.gz|matrix\\.mtx\\.gz)$')

    archive_records = []

    with tarfile.open(ARCHIVE_PATH, mode='r:') as tar:
        file_members = [member for member in tar.getmembers() if member.isfile()]
        if len(file_members) != 63:
            raise RuntimeError(f'Expected 63 regular files, found {len(file_members)}.')
        for member in file_members:
            posix_path = PurePosixPath(member.name)
            unsafe = posix_path.is_absolute() or '..' in posix_path.parts or len(posix_path.parts) != 1 or ('\\' in member.name) or (':' in member.name)
            if unsafe:
                raise RuntimeError(f'Unsafe archive member: {member.name}')
            filename = posix_path.name
            match = filename_pattern.fullmatch(filename)
            if match is None:
                raise RuntimeError(f'Unexpected archive filename: {filename}')
            gsm, sample_label, suffix = match.groups()
            archive_records.append({'GSM': gsm, 'sample_label_from_filename': sample_label, 'file_role': suffix_to_role[suffix], 'filename': filename, 'size_bytes': int(member.size)})

    archive_inventory = pd.DataFrame(archive_records)

    observed_gsms = sorted(archive_inventory['GSM'].unique())

    if observed_gsms != expected_gsms:
        missing = sorted(set(expected_gsms) - set(observed_gsms))
        unexpected = sorted(set(observed_gsms) - set(expected_gsms))
        raise RuntimeError(f'Archive/metadata GSM mismatch.\nMissing: {missing}\nUnexpected: {unexpected}')

    role_counts = archive_inventory.groupby(['GSM', 'file_role']).size().unstack(fill_value=0).reindex(index=expected_gsms, columns=['barcodes', 'features_or_genes', 'matrix'], fill_value=0)

    if not (role_counts == 1).all().all():
        raise RuntimeError('At least one sample does not have exactly one file of each role:\n' + role_counts.to_string())

    print('Archive member names and sample coverage verified.')

    compressed_member_bytes = int(archive_inventory['size_bytes'].sum())

    local_free_bytes = shutil.disk_usage('/content').free

    required_free_bytes = compressed_member_bytes + 1 * 1024 ** 3

    print(f'Compressed files to extract: {compressed_member_bytes / 1024 ** 3:.3f} GB')

    print(f'Available Colab local space: {local_free_bytes / 1024 ** 3:.2f} GB')

    if local_free_bytes < required_free_bytes:
        raise RuntimeError('Insufficient Colab local storage for safe extraction.')

    LOCAL_EXTRACT_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
if RUN_PIPELINE:
    def calculate_sha256(path, chunk_size=8 * 1024 * 1024):
        digest = hashlib.sha256()
        with open(path, 'rb') as handle:
            while True:
                chunk = handle.read(chunk_size)
                if not chunk:
                    break
                digest.update(chunk)
        return digest.hexdigest()

    extraction_rows = []

    with tarfile.open(ARCHIVE_PATH, mode='r:') as tar:
        members_by_name = {member.name: member for member in tar.getmembers() if member.isfile()}
        for record in tqdm(archive_records, total=len(archive_records), desc='Extracting compressed 10x files'):
            filename = record['filename']
            member = members_by_name[filename]
            destination = LOCAL_EXTRACT_DIR / filename
            destination_resolved = destination.resolve()
            local_root_resolved = LOCAL_EXTRACT_DIR.resolve()
            if destination_resolved.parent != local_root_resolved or destination_resolved.name != filename:
                raise RuntimeError(f'Unsafe destination: {destination}')
            if destination.exists() and destination.stat().st_size == member.size:
                extraction_status = 'reused_existing_local_file'
            else:
                temporary_destination = Path(str(destination) + '.part')
                source = tar.extractfile(member)
                if source is None:
                    raise RuntimeError(f'Could not read TAR member: {filename}')
                digest_while_writing = hashlib.sha256()
                with source, open(temporary_destination, 'wb') as output:
                    while True:
                        chunk = source.read(8 * 1024 * 1024)
                        if not chunk:
                            break
                        output.write(chunk)
                        digest_while_writing.update(chunk)
                if temporary_destination.stat().st_size != member.size:
                    raise RuntimeError(f'Extracted size mismatch for {filename}.')
                os.replace(temporary_destination, destination)
                extraction_status = 'extracted'
            if destination.stat().st_size != member.size:
                raise RuntimeError(f'Final size mismatch for {filename}.')
            file_sha256 = calculate_sha256(destination)
            extraction_rows.append({**record, 'compressed_sha256': file_sha256, 'extraction_status': extraction_status, 'storage': 'Colab local ephemeral storage'})

    extraction_manifest = pd.DataFrame(extraction_rows).sort_values(['GSM', 'file_role'])

    extraction_manifest.to_csv(EXTRACTION_MANIFEST_PATH, index=False)

    print(f'Safely extracted or verified {len(extraction_manifest)} files.')

    print(f'Extraction manifest saved: {EXTRACTION_MANIFEST_PATH}')


In [ ]:
if RUN_PIPELINE:
    def audit_tsv_gzip(path, capture_feature_types=False):
        """
        Count non-empty rows, check uniqueness of the first column,
        and hash the uncompressed contents.
        """
        row_count = 0
        first_column_values = set()
        column_counts = Counter()
        feature_types = set()
        uncompressed_digest = hashlib.sha256()
        with gzip.open(path, 'rb') as handle:
            for raw_line in handle:
                uncompressed_digest.update(raw_line)
                stripped = raw_line.rstrip(b'\r\n')
                if not stripped:
                    continue
                fields = stripped.split(b'\t')
                row_count += 1
                column_counts[len(fields)] += 1
                first_column_values.add(fields[0])
                if capture_feature_types and len(fields) >= 3:
                    feature_types.add(fields[2].decode('utf-8', errors='replace'))
        return {'row_count': row_count, 'unique_first_column': len(first_column_values), 'column_counts': dict(column_counts), 'feature_types': sorted(feature_types), 'uncompressed_sha256': uncompressed_digest.hexdigest()}

    def read_matrix_market_header(path):
        """
        Read only the Matrix Market banner and dimension line.
        No expression-coordinate entries are read.
        """
        with gzip.open(path, 'rt', encoding='utf-8') as handle:
            banner = handle.readline().strip()
            if not banner.lower().startswith('%%matrixmarket matrix coordinate'):
                raise RuntimeError(f'Unexpected Matrix Market banner in {path.name}: {banner}')
            dimensions = None
            for line in handle:
                stripped = line.strip()
                if not stripped or stripped.startswith('%'):
                    continue
                parts = stripped.split()
                if len(parts) != 3:
                    raise RuntimeError(f'Invalid Matrix Market dimension line in {path.name}')
                dimensions = tuple(map(int, parts))
                break
        if dimensions is None:
            raise RuntimeError(f'No dimension line found in {path.name}')
        n_rows, n_columns, n_nonzero = dimensions
        return {'matrix_banner': banner, 'matrix_rows': n_rows, 'matrix_columns': n_columns, 'matrix_nonzero_entries': n_nonzero}

    files_by_sample = {}

    for record in archive_records:
        files_by_sample.setdefault(record['GSM'], {})[record['file_role']] = LOCAL_EXTRACT_DIR / record['filename']

    sample_audits = []


In [ ]:
if RUN_PIPELINE:
    for gsm in tqdm(expected_gsms, desc='Auditing matrix structures'):
        sample_files = files_by_sample[gsm]
        barcode_audit = audit_tsv_gzip(sample_files['barcodes'], capture_feature_types=False)
        feature_audit = audit_tsv_gzip(sample_files['features_or_genes'], capture_feature_types=True)
        matrix_audit = read_matrix_market_header(sample_files['matrix'])
        feature_count = feature_audit['row_count']
        barcode_count = barcode_audit['row_count']
        if matrix_audit['matrix_rows'] == feature_count and matrix_audit['matrix_columns'] == barcode_count:
            orientation = 'features_x_barcodes'
        elif matrix_audit['matrix_rows'] == barcode_count and matrix_audit['matrix_columns'] == feature_count:
            orientation = 'barcodes_x_features'
        else:
            orientation = 'dimension_mismatch'
        filename_label = archive_inventory.loc[archive_inventory['GSM'] == gsm, 'sample_label_from_filename'].iloc[0]
        metadata_title = str(metadata_by_gsm.at[gsm, title_column]) if title_column is not None else filename_label
        diagnosis = str(metadata_by_gsm.at[gsm, diagnosis_column]) if diagnosis_column is not None else ''
        sample_audits.append({'GSM': gsm, 'sample_label': filename_label, 'metadata_title': metadata_title, 'diagnosis': diagnosis, 'barcode_count': barcode_count, 'unique_barcodes': barcode_audit['unique_first_column'], 'feature_count': feature_count, 'unique_feature_ids': feature_audit['unique_first_column'], 'feature_column_structure': json.dumps(feature_audit['column_counts'], sort_keys=True), 'feature_types': '; '.join(feature_audit['feature_types']), 'feature_reference_uncompressed_sha256': feature_audit['uncompressed_sha256'], **matrix_audit, 'orientation': orientation})

    dimension_audit = pd.DataFrame(sample_audits)

    if not (dimension_audit['orientation'] == 'features_x_barcodes').all():
        bad = dimension_audit.loc[dimension_audit['orientation'] != 'features_x_barcodes']
        raise RuntimeError('At least one matrix has incompatible dimensions:\n' + bad.to_string(index=False))

    if not (dimension_audit['barcode_count'] == dimension_audit['unique_barcodes']).all():
        bad = dimension_audit.loc[dimension_audit['barcode_count'] != dimension_audit['unique_barcodes']]
        raise RuntimeError('Duplicated barcodes detected within at least one sample:\n' + bad.to_string(index=False))

    if not (dimension_audit['feature_count'] == dimension_audit['unique_feature_ids']).all():
        bad = dimension_audit.loc[dimension_audit['feature_count'] != dimension_audit['unique_feature_ids']]
        raise RuntimeError('Duplicated feature IDs detected:\n' + bad.to_string(index=False))

    unique_feature_references = dimension_audit['feature_reference_uncompressed_sha256'].nunique()

    if unique_feature_references != 1:
        raise RuntimeError(f'Samples do not all use an identical feature reference. Observed references: {unique_feature_references}')

    if (dimension_audit['matrix_nonzero_entries'] <= 0).any():
        raise RuntimeError('At least one matrix contains no nonzero entries.')

    dimension_audit.to_csv(DIMENSION_AUDIT_PATH, index=False)

    structural_lock = {'dataset': 'GSE157827', 'audit_version': 'v1', 'created_utc': datetime.now(timezone.utc).isoformat(), 'archive_size_bytes': EXPECTED_ARCHIVE_BYTES, 'archive_sha256': EXPECTED_ARCHIVE_SHA256, 'n_locked_samples': int(len(expected_gsms)), 'n_archive_files': int(len(archive_records)), 'files_per_sample': {'barcodes': 1, 'features_or_genes': 1, 'matrix': 1}, 'total_input_barcodes': int(dimension_audit['barcode_count'].sum()), 'feature_count': int(dimension_audit['feature_count'].iloc[0]), 'unique_feature_references': int(unique_feature_references), 'matrix_orientation': 'features_x_barcodes', 'expression_coordinate_entries_read': False, 'differential_expression_performed': False, 'creb5_examined': False, 'local_extraction_directory': str(LOCAL_EXTRACT_DIR), 'local_files_are_ephemeral': True, 'dimension_audit_csv': str(DIMENSION_AUDIT_PATH), 'extraction_manifest_csv': str(EXTRACTION_MANIFEST_PATH)}

    with open(STRUCTURAL_LOCK_PATH, 'w', encoding='utf-8') as handle:
        json.dump(structural_lock, handle, indent=2)

    print('\n' + '=' * 76)

    print('GSE157827 STRUCTURAL MATRIX AUDIT')

    print('=' * 76)

    report_columns = ['GSM', 'sample_label', 'diagnosis', 'barcode_count', 'feature_count', 'matrix_nonzero_entries', 'orientation']

    print(dimension_audit[report_columns].to_string(index=False, justify='left'))

    print('\n' + '-' * 76)

    print(f'Samples audited: {len(dimension_audit)}')

    print(f"Total input barcodes: {dimension_audit['barcode_count'].sum():,}")

    print(f"Features per sample: {dimension_audit['feature_count'].iloc[0]:,}")

    print(f'Unique feature references: {unique_feature_references}')

    print(f"All matrices oriented as features × barcodes: {(dimension_audit['orientation'] == 'features_x_barcodes').all()}")

    print(f'\nDimension audit saved:\n{DIMENSION_AUDIT_PATH}')

    print(f'\nStructural lock saved:\n{STRUCTURAL_LOCK_PATH}')

    print('\nGSE157827 STRUCTURAL AUDIT COMPLETE')

    print('Expression-coordinate entries were not read.')

    print('CREB5 remains unexamined.')


In [ ]:
if RUN_PIPELINE:
    from pathlib import Path

    from datetime import datetime, timezone

    import pandas as pd

    import json

    PROJECT_DIR = Path('/content/drive/MyDrive/AD_Astrocyte_Paper_01')

    VALIDATION_DIR = PROJECT_DIR / 'GSE157827_confirmatory_validation'

    METADATA_PATH = VALIDATION_DIR / 'GSE157827_sample_metadata_raw.csv'

    ORIGINAL_AUDIT_PATH = VALIDATION_DIR / 'GSE157827_raw_matrix_dimension_audit.csv'

    ORIGINAL_LOCK_PATH = VALIDATION_DIR / 'GSE157827_STRUCTURAL_AUDIT_LOCK_v1.json'

    CORRECTED_AUDIT_PATH = VALIDATION_DIR / 'GSE157827_raw_matrix_dimension_audit_v1_1.csv'

    LINKAGE_AUDIT_PATH = VALIDATION_DIR / 'GSE157827_metadata_linkage_audit_v1.csv'

    CORRECTED_LOCK_PATH = VALIDATION_DIR / 'GSE157827_STRUCTURAL_AUDIT_LOCK_v1_1.json'

    for required_path in [METADATA_PATH, ORIGINAL_AUDIT_PATH, ORIGINAL_LOCK_PATH]:
        if not required_path.exists():
            raise FileNotFoundError(f'Missing required file: {required_path}')

    metadata = pd.read_csv(METADATA_PATH, dtype=str).fillna('')

    dimension_audit = pd.read_csv(ORIGINAL_AUDIT_PATH, dtype=str).fillna('')

    required_metadata_columns = ['GSM', 'title', 'diagnosis_inferred_from_title', 'characteristic__diagnosis']

    missing_columns = [column for column in required_metadata_columns if column not in metadata.columns]

    if missing_columns:
        raise RuntimeError(f'Required metadata columns are missing: {missing_columns}')

    metadata['GSM'] = metadata['GSM'].str.strip()

    metadata['title'] = metadata['title'].str.strip()

    metadata['diagnosis_from_title'] = metadata['diagnosis_inferred_from_title'].str.strip()

    characteristic_mapping = {'ad': 'AD', 'healthy control': 'Control', 'control': 'Control', 'normal control': 'Control', 'nc': 'Control'}

    metadata['diagnosis_from_characteristic'] = metadata['characteristic__diagnosis'].str.strip().str.lower().map(characteristic_mapping)

    if metadata['diagnosis_from_characteristic'].isna().any():
        bad = metadata.loc[metadata['diagnosis_from_characteristic'].isna(), ['GSM', 'title', 'characteristic__diagnosis']]
        raise RuntimeError('Unrecognized diagnosis characteristic:\n' + bad.to_string(index=False))

    allowed_diagnoses = {'AD', 'Control'}

    if not set(metadata['diagnosis_from_title']).issubset(allowed_diagnoses):
        raise RuntimeError('Unexpected diagnosis inferred from one or more titles.')

    metadata['diagnosis_agreement'] = metadata['diagnosis_from_title'] == metadata['diagnosis_from_characteristic']

    if not metadata['diagnosis_agreement'].all():
        disagreement = metadata.loc[~metadata['diagnosis_agreement'], ['GSM', 'title', 'diagnosis_from_title', 'characteristic__diagnosis', 'diagnosis_from_characteristic']]
        raise RuntimeError('Title and GEO characteristic diagnoses disagree:\n' + disagreement.to_string(index=False))

    diagnosis_counts = metadata['diagnosis_from_title'].value_counts().to_dict()

    expected_counts = {'AD': 12, 'Control': 9}

    if diagnosis_counts != expected_counts:
        raise RuntimeError(f'Diagnosis counts changed.\nObserved: {diagnosis_counts}\nExpected: {expected_counts}')

    if metadata['GSM'].duplicated().any():
        raise RuntimeError('Duplicated GSM accessions in metadata.')

    if set(metadata['GSM']) != set(dimension_audit['GSM']):
        raise RuntimeError('The structural audit and metadata contain different GSM samples.')

    metadata_by_gsm = metadata.set_index('GSM')

    dimension_audit['metadata_title'] = dimension_audit['GSM'].map(metadata_by_gsm['title'])

    dimension_audit['diagnosis'] = dimension_audit['GSM'].map(metadata_by_gsm['diagnosis_from_title'])

    dimension_audit['diagnosis_characteristic_raw'] = dimension_audit['GSM'].map(metadata_by_gsm['characteristic__diagnosis'])

    dimension_audit['diagnosis_agreement'] = dimension_audit['GSM'].map(metadata_by_gsm['diagnosis_agreement'])

    if dimension_audit[['metadata_title', 'diagnosis', 'diagnosis_characteristic_raw']].isna().any().any():
        raise RuntimeError('Metadata linkage produced missing values.')

    corrected_counts = dimension_audit['diagnosis'].value_counts().to_dict()

    if corrected_counts != expected_counts:
        raise RuntimeError(f'Corrected audit has unexpected counts: {corrected_counts}')

    dimension_audit.to_csv(CORRECTED_AUDIT_PATH, index=False)


In [ ]:
if RUN_PIPELINE:
    linkage_audit = metadata[['GSM', 'title', 'diagnosis_from_title', 'characteristic__diagnosis', 'diagnosis_from_characteristic', 'diagnosis_agreement']].copy()

    linkage_audit = linkage_audit.rename(columns={'diagnosis_from_title': 'diagnosis'})

    linkage_audit.to_csv(LINKAGE_AUDIT_PATH, index=False)

    with open(ORIGINAL_LOCK_PATH, 'r', encoding='utf-8') as handle:
        corrected_lock = json.load(handle)

    corrected_lock.update({'audit_version': 'v1.1', 'metadata_linkage_corrected_utc': datetime.now(timezone.utc).isoformat(), 'diagnosis_primary_source': 'diagnosis_inferred_from_title', 'diagnosis_confirmation_source': 'characteristic__diagnosis', 'title_characteristic_agreement': True, 'diagnosis_counts': expected_counts, 'dimension_audit_csv': str(CORRECTED_AUDIT_PATH), 'metadata_linkage_audit_csv': str(LINKAGE_AUDIT_PATH), 'expression_coordinate_entries_read': False, 'differential_expression_performed': False, 'creb5_examined': False})

    with open(CORRECTED_LOCK_PATH, 'w', encoding='utf-8') as handle:
        json.dump(corrected_lock, handle, indent=2)

    print('=' * 72)

    print('GSE157827 CORRECTED METADATA LINKAGE')

    print('=' * 72)

    print(linkage_audit.to_string(index=False, justify='left'))

    print('\nDiagnosis counts:')

    print(linkage_audit['diagnosis'].value_counts().reindex(['AD', 'Control']).to_string())

    print(f"\nTitle/characteristic agreement: {linkage_audit['diagnosis_agreement'].sum()} / {len(linkage_audit)}")

    print(f'\nCorrected dimension audit:\n{CORRECTED_AUDIT_PATH}')

    print(f'\nMetadata-linkage audit:\n{LINKAGE_AUDIT_PATH}')

    print(f'\nCorrected structural lock:\n{CORRECTED_LOCK_PATH}')

    print('\nGSE157827 METADATA LINKAGE CORRECTED AND LOCKED')

    print('No expression values were read.')

    print('CREB5 remains unexamined.')


## Diagnosis-blind nucleus QC and doublet removal

The published nucleus-QC rules are reproduced without diagnosis or target-gene inspection. Scrublet is run separately by donor. The one prespecified threshold rescue uses the median threshold from the nonflagged libraries.


In [ ]:
if RUN_PIPELINE:
    from pathlib import Path

    from datetime import datetime, timezone

    import pandas as pd

    import numpy as np

    import scipy

    from scipy.io import mmread

    import gzip

    import json

    import gc

    import re

    try:
        from tqdm.auto import tqdm
    except ImportError:

        def tqdm(x, **kwargs):
            return x

    PROJECT_DIR = Path('/content/drive/MyDrive/AD_Astrocyte_Paper_01')

    VALIDATION_DIR = PROJECT_DIR / 'GSE157827_confirmatory_validation'

    LOCAL_EXTRACT_DIR = Path('/content/GSE157827_raw_10x')

    STRUCTURAL_AUDIT_PATH = VALIDATION_DIR / 'GSE157827_raw_matrix_dimension_audit_v1_1.csv'

    STRUCTURAL_LOCK_PATH = VALIDATION_DIR / 'GSE157827_STRUCTURAL_AUDIT_LOCK_v1_1.json'

    QC_POLICY_LOCK_PATH = VALIDATION_DIR / 'GSE157827_QC_POLICY_LOCK_v1.json'

    NUCLEUS_QC_PATH = VALIDATION_DIR / 'GSE157827_nucleus_QC_metrics_author_rules_v1.csv.gz'

    QC_PASS_BARCODES_PATH = VALIDATION_DIR / 'GSE157827_author_QC_pass_barcodes_v1.csv.gz'

    SAMPLE_QC_SUMMARY_PATH = VALIDATION_DIR / 'GSE157827_sample_QC_summary_author_rules_v1.csv'

    QC_RUN_AUDIT_PATH = VALIDATION_DIR / 'GSE157827_QC_RUN_AUDIT_v1.json'

    for required_path in [STRUCTURAL_AUDIT_PATH, STRUCTURAL_LOCK_PATH]:
        if not required_path.exists():
            raise FileNotFoundError(f'Missing required file: {required_path}')

    if not LOCAL_EXTRACT_DIR.exists():
        raise FileNotFoundError('The local extracted files are missing. The Colab runtime may have restarted. Rerun Step 5.')

    qc_policy_core = {'dataset': 'GSE157827', 'policy_version': 'v1', 'method': "Reproduction of the original study's second QC stage", 'primary_reference': {'citation': 'Lau et al., PNAS, 2020', 'doi': '10.1073/pnas.2008762117', 'pmid': '32989152'}, 'retain_if': {'detected_genes': '> 200', 'total_UMI_counts': '< 20000', 'mitochondrial_percentage': '< 20'}, 'expected_published_final_nuclei': 169496, 'diagnosis_used_for_QC': False, 'target_genes_queried': False, 'doublet_detection_performed_at_this_stage': False, 'normalization_performed': False, 'clustering_performed': False, 'differential_expression_performed': False}

    if QC_POLICY_LOCK_PATH.exists():
        with open(QC_POLICY_LOCK_PATH, 'r', encoding='utf-8') as handle:
            existing_policy = json.load(handle)
        keys_to_verify = ['retain_if', 'expected_published_final_nuclei', 'diagnosis_used_for_QC', 'target_genes_queried']
        for key in keys_to_verify:
            if existing_policy.get(key) != qc_policy_core.get(key):
                raise RuntimeError(f'Existing QC policy differs for field: {key}')
        print('Existing QC policy lock verified.')
    else:
        qc_policy_to_save = {**qc_policy_core, 'created_utc': datetime.now(timezone.utc).isoformat()}
        with open(QC_POLICY_LOCK_PATH, 'w', encoding='utf-8') as handle:
            json.dump(qc_policy_to_save, handle, indent=2)
        print('QC policy locked before expression-aware processing.')

    print(f'QC policy: {QC_POLICY_LOCK_PATH}')

    sample_table = pd.read_csv(STRUCTURAL_AUDIT_PATH, usecols=['GSM', 'sample_label', 'barcode_count', 'feature_count', 'matrix_nonzero_entries', 'orientation'])

    sample_table['GSM'] = sample_table['GSM'].astype(str)

    sample_table['sample_label'] = sample_table['sample_label'].astype(str)

    numeric_columns = ['barcode_count', 'feature_count', 'matrix_nonzero_entries']

    for column in numeric_columns:
        sample_table[column] = pd.to_numeric(sample_table[column], errors='raise').astype(np.int64)

    if len(sample_table) != 21:
        raise RuntimeError(f'Expected 21 samples, found {len(sample_table)}.')


In [ ]:
if RUN_PIPELINE:
    if not (sample_table['orientation'] == 'features_x_barcodes').all():
        raise RuntimeError('Unexpected matrix orientation.')

    if sample_table['GSM'].duplicated().any():
        raise RuntimeError('Duplicated GSM accessions in sample table.')

    if 'diagnosis' in sample_table.columns:
        raise RuntimeError('Diagnosis must not be present during QC.')

    print(f'Diagnosis-blind samples prepared: {len(sample_table)}')

    def locate_sample_file(gsm, suffix):
        matches = sorted(LOCAL_EXTRACT_DIR.glob(f'{gsm}_*_{suffix}'))
        if len(matches) != 1:
            raise RuntimeError(f'Expected one {suffix} file for {gsm}; found {len(matches)}.')
        return matches[0]

    first_gsm = sample_table.iloc[0]['GSM']

    reference_feature_path = locate_sample_file(first_gsm, 'features.tsv.gz')

    features = pd.read_csv(reference_feature_path, sep='\t', header=None, compression='gzip', dtype=str)

    if features.shape[1] < 2:
        raise RuntimeError('The feature reference has fewer than two columns.')

    features = features.rename(columns={0: 'gene_id', 1: 'gene_symbol', 2: 'feature_type'})

    gene_symbols_upper = features['gene_symbol'].fillna('').astype(str).str.upper()

    mitochondrial_mask = gene_symbols_upper.str.startswith('MT-').to_numpy()

    ribosomal_mask = gene_symbols_upper.str.match('^RP[SL][0-9]').to_numpy()

    hemoglobin_mask = gene_symbols_upper.str.match('^HB[ABDEGQMZ][0-9]').to_numpy()

    if mitochondrial_mask.sum() == 0:
        raise RuntimeError('No mitochondrial features were detected using the MT- prefix.')

    expected_feature_count = int(sample_table['feature_count'].iloc[0])

    if len(features) != expected_feature_count:
        raise RuntimeError(f'Feature-reference length mismatch: {len(features):,} versus {expected_feature_count:,}')

    print(f'Feature-reference genes: {len(features):,}')

    print(f'Mitochondrial features: {mitochondrial_mask.sum():,}')

    print(f'Ribosomal features: {ribosomal_mask.sum():,}')

    print(f'Hemoglobin features: {hemoglobin_mask.sum():,}')

    def feature_set_counts(matrix_coo, feature_mask, n_barcodes):
        """
        Calculate counts from a selected feature set for every barcode.
        """
        selected_entries = feature_mask[matrix_coo.row]
        result = np.bincount(matrix_coo.col[selected_entries], weights=matrix_coo.data[selected_entries], minlength=n_barcodes)
        del selected_entries
        return result

    def percentage(numerator, denominator):
        result = np.zeros(len(denominator), dtype=np.float64)
        np.divide(numerator, denominator, out=result, where=denominator > 0)
        return result * 100.0

    def quantile(values, probability):
        return float(np.quantile(values, probability))

    all_nucleus_metrics = []

    sample_summaries = []


In [ ]:
if RUN_PIPELINE:
    for sample in tqdm(sample_table.itertuples(index=False), total=len(sample_table), desc='Calculating diagnosis-blind nucleus QC'):
        gsm = sample.GSM
        sample_label = sample.sample_label
        matrix_path = locate_sample_file(gsm, 'matrix.mtx.gz')
        barcode_path = locate_sample_file(gsm, 'barcodes.tsv.gz')
        barcodes = pd.read_csv(barcode_path, sep='\t', header=None, compression='gzip', dtype=str)[0].astype(str).to_numpy()
        if len(barcodes) != sample.barcode_count:
            raise RuntimeError(f'Barcode-count mismatch for {gsm}: {len(barcodes):,} versus {sample.barcode_count:,}')
        if len(np.unique(barcodes)) != len(barcodes):
            raise RuntimeError(f'Duplicated barcodes detected in {gsm}.')
        with gzip.open(matrix_path, 'rb') as matrix_handle:
            matrix = mmread(matrix_handle)
        matrix = matrix.tocoo(copy=False)
        if matrix.shape != (sample.feature_count, sample.barcode_count):
            raise RuntimeError(f'Matrix-shape mismatch for {gsm}: {matrix.shape}')
        raw_coordinate_entries = int(matrix.nnz)
        if raw_coordinate_entries != sample.matrix_nonzero_entries:
            raise RuntimeError(f'Matrix nonzero-entry mismatch for {gsm}: {raw_coordinate_entries:,} versus {sample.matrix_nonzero_entries:,}')
        if matrix.nnz > 0 and np.min(matrix.data) < 0:
            raise RuntimeError(f'Negative counts detected in {gsm}.')
        matrix.sum_duplicates()
        matrix.eliminate_zeros()
        n_barcodes = matrix.shape[1]
        total_counts = np.bincount(matrix.col, weights=matrix.data, minlength=n_barcodes)
        total_counts = np.rint(total_counts).astype(np.int64)
        detected_genes = np.bincount(matrix.col, minlength=n_barcodes).astype(np.int32)
        mitochondrial_counts = feature_set_counts(matrix, mitochondrial_mask, n_barcodes)
        ribosomal_counts = feature_set_counts(matrix, ribosomal_mask, n_barcodes)
        hemoglobin_counts = feature_set_counts(matrix, hemoglobin_mask, n_barcodes)
        pct_mitochondrial = percentage(mitochondrial_counts, total_counts)
        pct_ribosomal = percentage(ribosomal_counts, total_counts)
        pct_hemoglobin = percentage(hemoglobin_counts, total_counts)
        fail_low_genes = detected_genes <= 200
        fail_high_counts = total_counts >= 20000
        fail_high_mitochondrial = pct_mitochondrial >= 20.0
        qc_pass_author = ~(fail_low_genes | fail_high_counts | fail_high_mitochondrial)
        nucleus_ids = np.char.add(np.char.add(gsm, ':'), barcodes.astype(str))
        nucleus_metrics = pd.DataFrame({'nucleus_id': nucleus_ids, 'GSM': gsm, 'sample_label': sample_label, 'barcode': barcodes, 'total_UMI_counts': total_counts, 'detected_genes': detected_genes, 'pct_mitochondrial': pct_mitochondrial.astype(np.float32), 'pct_ribosomal': pct_ribosomal.astype(np.float32), 'pct_hemoglobin': pct_hemoglobin.astype(np.float32), 'fail_detected_genes_le_200': fail_low_genes, 'fail_total_UMI_ge_20000': fail_high_counts, 'fail_pct_mitochondrial_ge_20': fail_high_mitochondrial, 'qc_pass_author_rules': qc_pass_author})
        all_nucleus_metrics.append(nucleus_metrics)
        sample_summaries.append({'GSM': gsm, 'sample_label': sample_label, 'input_barcodes': int(n_barcodes), 'author_QC_pass': int(qc_pass_author.sum()), 'author_QC_removed': int((~qc_pass_author).sum()), 'author_QC_pass_percent': float(qc_pass_author.mean() * 100), 'fail_low_genes': int(fail_low_genes.sum()), 'fail_high_UMI': int(fail_high_counts.sum()), 'fail_high_mitochondrial': int(fail_high_mitochondrial.sum()), 'median_UMI_input': quantile(total_counts, 0.5), 'median_genes_input': quantile(detected_genes, 0.5), 'median_pct_mitochondrial_input': quantile(pct_mitochondrial, 0.5), 'p01_genes_input': quantile(detected_genes, 0.01), 'p99_genes_input': quantile(detected_genes, 0.99), 'p99_UMI_input': quantile(total_counts, 0.99), 'p99_pct_mitochondrial_input': quantile(pct_mitochondrial, 0.99)})
        del matrix
        del nucleus_metrics
        del total_counts
        del detected_genes
        del mitochondrial_counts
        del ribosomal_counts
        del hemoglobin_counts
        del pct_mitochondrial
        del pct_ribosomal
        del pct_hemoglobin
        del qc_pass_author
        gc.collect()

    nucleus_qc = pd.concat(all_nucleus_metrics, ignore_index=True)

    sample_qc_summary = pd.DataFrame(sample_summaries)

    expected_input_barcodes = int(sample_table['barcode_count'].sum())


In [ ]:
if RUN_PIPELINE:
    if len(nucleus_qc) != expected_input_barcodes:
        raise RuntimeError(f'Combined nucleus count mismatch: {len(nucleus_qc):,} versus {expected_input_barcodes:,}')

    if nucleus_qc['nucleus_id'].duplicated().any():
        raise RuntimeError('Duplicated global nucleus IDs detected.')

    observed_pass_nuclei = int(nucleus_qc['qc_pass_author_rules'].sum())

    observed_removed_nuclei = int((~nucleus_qc['qc_pass_author_rules']).sum())

    expected_published_nuclei = 169496

    matches_published_count = observed_pass_nuclei == expected_published_nuclei

    nucleus_qc.to_csv(NUCLEUS_QC_PATH, index=False, compression={'method': 'gzip', 'compresslevel': 6})

    qc_pass_barcodes = nucleus_qc.loc[nucleus_qc['qc_pass_author_rules'], ['nucleus_id', 'GSM', 'sample_label', 'barcode']].copy()

    qc_pass_barcodes.to_csv(QC_PASS_BARCODES_PATH, index=False, compression={'method': 'gzip', 'compresslevel': 6})

    sample_qc_summary.to_csv(SAMPLE_QC_SUMMARY_PATH, index=False)

    qc_run_audit = {'dataset': 'GSE157827', 'run_version': 'v1', 'completed_utc': datetime.now(timezone.utc).isoformat(), 'scipy_version': scipy.__version__, 'n_samples': int(len(sample_table)), 'input_barcodes': expected_input_barcodes, 'author_QC_pass_nuclei': observed_pass_nuclei, 'author_QC_removed_nuclei': observed_removed_nuclei, 'expected_published_final_nuclei': expected_published_nuclei, 'matches_published_final_nucleus_count': bool(matches_published_count), 'diagnosis_used_for_QC': False, 'target_genes_queried': False, 'normalization_performed': False, 'clustering_performed': False, 'differential_expression_performed': False, 'cell_level_metrics_path': str(NUCLEUS_QC_PATH), 'QC_pass_barcodes_path': str(QC_PASS_BARCODES_PATH), 'sample_summary_path': str(SAMPLE_QC_SUMMARY_PATH)}

    with open(QC_RUN_AUDIT_PATH, 'w', encoding='utf-8') as handle:
        json.dump(qc_run_audit, handle, indent=2)

    print('\n' + '=' * 92)

    print('GSE157827 DIAGNOSIS-BLIND AUTHOR QC REPRODUCTION')

    print('=' * 92)

    report_columns = ['GSM', 'sample_label', 'input_barcodes', 'author_QC_pass', 'author_QC_removed', 'author_QC_pass_percent', 'median_UMI_input', 'median_genes_input', 'median_pct_mitochondrial_input']

    report = sample_qc_summary[report_columns].copy()

    for column in ['author_QC_pass_percent', 'median_UMI_input', 'median_genes_input', 'median_pct_mitochondrial_input']:
        report[column] = report[column].round(2)

    print(report.to_string(index=False, justify='left'))

    print('\n' + '-' * 92)

    print(f'Input barcodes: {expected_input_barcodes:,}')

    print(f'QC-pass nuclei: {observed_pass_nuclei:,}')

    print(f'QC-removed nuclei: {observed_removed_nuclei:,}')

    print(f'Published final nucleus count: {expected_published_nuclei:,}')

    print(f'Exact reproduction of published count: {matches_published_count}')

    print('\nSaved artifacts:')

    print(f'Cell-level QC metrics:\n{NUCLEUS_QC_PATH}')

    print(f'\nQC-pass barcode manifest:\n{QC_PASS_BARCODES_PATH}')

    print(f'\nSample QC summary:\n{SAMPLE_QC_SUMMARY_PATH}')

    print(f'\nQC run audit:\n{QC_RUN_AUDIT_PATH}')

    print('\nGSE157827 AUTHOR QC REPRODUCTION COMPLETE')

    print('Diagnosis was not used.')

    print('No target genes were queried.')

    print('No differential-expression testing was performed.')


In [ ]:
if RUN_PIPELINE:
    from pathlib import Path

    from datetime import datetime, timezone

    import pandas as pd

    import numpy as np

    import json

    PROJECT_DIR = Path('/content/drive/MyDrive/AD_Astrocyte_Paper_01')

    VALIDATION_DIR = PROJECT_DIR / 'GSE157827_confirmatory_validation'

    METADATA_PATH = VALIDATION_DIR / 'GSE157827_sample_metadata_raw.csv'

    NUCLEUS_QC_PATH = VALIDATION_DIR / 'GSE157827_nucleus_QC_metrics_author_rules_v1.csv.gz'

    SAMPLE_QC_PATH = VALIDATION_DIR / 'GSE157827_sample_QC_summary_author_rules_v1.csv'

    GROUP_RECONCILIATION_PATH = VALIDATION_DIR / 'GSE157827_QC_group_reconciliation_v1.csv'

    DONOR_REVIEW_PATH = VALIDATION_DIR / 'GSE157827_donor_QC_review_v1.csv'

    RECONCILIATION_LOCK_PATH = VALIDATION_DIR / 'GSE157827_QC_RECONCILIATION_LOCK_v1.json'

    for required_path in [METADATA_PATH, NUCLEUS_QC_PATH, SAMPLE_QC_PATH]:
        if not required_path.exists():
            raise FileNotFoundError(f'Missing file: {required_path}')

    metadata = pd.read_csv(METADATA_PATH, dtype=str).fillna('')

    nucleus_qc = pd.read_csv(NUCLEUS_QC_PATH)

    sample_qc = pd.read_csv(SAMPLE_QC_PATH)

    required_metadata = ['GSM', 'diagnosis_inferred_from_title', 'characteristic__diagnosis']

    missing = [column for column in required_metadata if column not in metadata.columns]

    if missing:
        raise RuntimeError(f'Missing metadata columns: {missing}')

    diagnosis_map = metadata.set_index('GSM')['diagnosis_inferred_from_title'].to_dict()

    nucleus_qc['diagnosis'] = nucleus_qc['GSM'].map(diagnosis_map)

    sample_qc['diagnosis'] = sample_qc['GSM'].map(diagnosis_map)

    if nucleus_qc['diagnosis'].isna().any():
        raise RuntimeError('Some nuclei could not be linked to diagnosis.')

    if sample_qc['diagnosis'].isna().any():
        raise RuntimeError('Some samples could not be linked to diagnosis.')

    if nucleus_qc['qc_pass_author_rules'].dtype != bool:
        nucleus_qc['qc_pass_author_rules'] = nucleus_qc['qc_pass_author_rules'].astype(str).str.lower().map({'true': True, 'false': False})

    if nucleus_qc['qc_pass_author_rules'].isna().any():
        raise RuntimeError('Could not parse the saved QC-pass field.')

    group_reconciliation = nucleus_qc.groupby('diagnosis', observed=True).agg(input_barcodes=('nucleus_id', 'size'), reproduced_QC_pass=('qc_pass_author_rules', 'sum')).reset_index()

    published_group_counts = {'AD': 90713, 'Control': 78783}

    group_reconciliation['published_final_nuclei'] = group_reconciliation['diagnosis'].map(published_group_counts).astype(int)

    group_reconciliation['difference'] = group_reconciliation['reproduced_QC_pass'] - group_reconciliation['published_final_nuclei']

    group_reconciliation['absolute_percent_difference'] = group_reconciliation['difference'].abs() / group_reconciliation['published_final_nuclei'] * 100

    group_reconciliation.to_csv(GROUP_RECONCILIATION_PATH, index=False)

    passed = nucleus_qc['qc_pass_author_rules']

    boundary_audit = {'all_nuclei_detected_genes_eq_200': int((nucleus_qc['detected_genes'] == 200).sum()), 'pass_nuclei_detected_genes_eq_201': int((passed & (nucleus_qc['detected_genes'] == 201)).sum()), 'all_nuclei_total_UMI_eq_20000': int((nucleus_qc['total_UMI_counts'] == 20000).sum()), 'pass_nuclei_total_UMI_eq_19999': int((passed & (nucleus_qc['total_UMI_counts'] == 19999)).sum()), 'all_nuclei_pct_mito_exactly_20': int(np.isclose(nucleus_qc['pct_mitochondrial'], 20.0, rtol=0, atol=1e-10).sum()), 'pass_nuclei_pct_mito_19_99_to_below_20': int((passed & (nucleus_qc['pct_mitochondrial'] >= 19.99) & (nucleus_qc['pct_mitochondrial'] < 20.0)).sum())}

    SAMPLE_QC_PASS_PERCENT_SENSITIVITY_THRESHOLD = 80.0

    sample_qc['severe_QC_loss_sensitivity_flag'] = sample_qc['author_QC_pass_percent'] < SAMPLE_QC_PASS_PERCENT_SENSITIVITY_THRESHOLD

    sample_qc['primary_analysis_status'] = 'retain'

    sample_qc['sensitivity_analysis_status'] = np.where(sample_qc['severe_QC_loss_sensitivity_flag'], 'exclude_in_severe_QC_loss_sensitivity', 'retain')

    sample_qc.to_csv(DONOR_REVIEW_PATH, index=False)

    flagged_donors = sample_qc.loc[sample_qc['severe_QC_loss_sensitivity_flag'], 'GSM'].astype(str).tolist()

    observed_total = int(nucleus_qc['qc_pass_author_rules'].sum())

    published_total = 169496

    total_difference = observed_total - published_total

    reconciliation_lock = {'dataset': 'GSE157827', 'version': 'v1', 'created_utc': datetime.now(timezone.utc).isoformat(), 'documented_author_QC_rules': {'detected_genes': '> 200', 'total_UMI_counts': '< 20000', 'mitochondrial_percentage': '< 20'}, 'input_barcodes': int(len(nucleus_qc)), 'reproduced_QC_pass_nuclei': observed_total, 'published_final_nuclei': published_total, 'difference_nuclei': total_difference, 'absolute_percent_difference': float(abs(total_difference) / published_total * 100), 'group_reconciliation': group_reconciliation.to_dict(orient='records'), 'boundary_audit': boundary_audit, 'primary_QC_population_decision': 'Retain all 169506 nuclei satisfying the explicitly documented author QC rules. Do not remove ten arbitrary nuclei merely to force agreement with the reported total.', 'primary_donor_rule_after_astrocyte_annotation': 'Retain a donor if at least 20 high-confidence singlet astrocyte nuclei are available.', 'sensitivity_donor_rules': ['Repeat the confirmatory pseudobulk analysis after excluding donors with less than 80 percent nucleus retention under the author QC rules.', 'Repeat the analysis using a stricter minimum of 50 high-confidence singlet astrocyte nuclei per donor.'], 'sample_QC_retention_threshold_percent': SAMPLE_QC_PASS_PERCENT_SENSITIVITY_THRESHOLD, 'sample_QC_sensitivity_flagged_GSMs': flagged_donors, 'all_flagged_donors_retained_in_primary_analysis': True, 'diagnosis_was_used_to_define_cell_QC': False, 'target_genes_queried': False, 'differential_expression_performed': False}

    with open(RECONCILIATION_LOCK_PATH, 'w', encoding='utf-8') as handle:
        json.dump(reconciliation_lock, handle, indent=2)

    print('=' * 78)


In [ ]:
if RUN_PIPELINE:
    print('GSE157827 QC RECONCILIATION')

    print('=' * 78)

    print(group_reconciliation.round({'absolute_percent_difference': 5}).to_string(index=False, justify='left'))

    print('\nThreshold-boundary audit:')

    for label, value in boundary_audit.items():
        print(f'{label}: {value:,}')

    print('\nSevere sample-level QC-loss sensitivity flags:')

    flag_report = sample_qc[['GSM', 'sample_label', 'diagnosis', 'input_barcodes', 'author_QC_pass', 'author_QC_pass_percent', 'severe_QC_loss_sensitivity_flag']].copy()

    flag_report['author_QC_pass_percent'] = flag_report['author_QC_pass_percent'].round(2)

    print(flag_report.to_string(index=False, justify='left'))

    print('\nPrimary QC population:')

    print(f'{observed_total:,} nuclei')

    print('\nPublished count difference:')

    print(f'{observed_total:,} - {published_total:,} = {total_difference:,}')

    print('\nPrimary decision:')

    print('Keep all nuclei satisfying the documented QC rules.')

    print('Do not tune thresholds to remove an arbitrary ten nuclei.')

    print(f'\nGroup reconciliation saved:\n{GROUP_RECONCILIATION_PATH}')

    print(f'\nDonor QC review saved:\n{DONOR_REVIEW_PATH}')

    print(f'\nDecision lock saved:\n{RECONCILIATION_LOCK_PATH}')

    print('\nGSE157827 QC RECONCILIATION LOCKED')

    print('No target genes were queried.')

    print('No differential-expression testing was performed.')


In [ ]:
if RUN_PIPELINE:
    from pathlib import Path

    from datetime import datetime, timezone

    import pandas as pd

    import numpy as np

    import scipy

    from scipy.io import mmread

    import matplotlib.pyplot as plt

    import gzip

    import json

    import gc

    import sys

    import subprocess

    import importlib

    import importlib.metadata

    try:
        from tqdm.auto import tqdm
    except ImportError:

        def tqdm(x, **kwargs):
            return x

    SCRUBLET_VERSION = '0.2.3'

    try:
        installed_scrublet_version = importlib.metadata.version('scrublet')
    except importlib.metadata.PackageNotFoundError:
        installed_scrublet_version = None

    if installed_scrublet_version != SCRUBLET_VERSION:
        print(f'Installing scrublet=={SCRUBLET_VERSION}...')
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet', '--no-input', f'scrublet=={SCRUBLET_VERSION}'])
        importlib.invalidate_caches()

    import scrublet as scr

    installed_scrublet_version = importlib.metadata.version('scrublet')

    if installed_scrublet_version != SCRUBLET_VERSION:
        raise RuntimeError(f'Unexpected Scrublet version: {installed_scrublet_version}')

    print(f'Scrublet version: {installed_scrublet_version}')

    PROJECT_DIR = Path('/content/drive/MyDrive/AD_Astrocyte_Paper_01')

    VALIDATION_DIR = PROJECT_DIR / 'GSE157827_confirmatory_validation'

    LOCAL_EXTRACT_DIR = Path('/content/GSE157827_raw_10x')

    STRUCTURAL_AUDIT_PATH = VALIDATION_DIR / 'GSE157827_raw_matrix_dimension_audit_v1_1.csv'

    NUCLEUS_QC_PATH = VALIDATION_DIR / 'GSE157827_nucleus_QC_metrics_author_rules_v1.csv.gz'

    RECONCILIATION_LOCK_PATH = VALIDATION_DIR / 'GSE157827_QC_RECONCILIATION_LOCK_v1.json'

    DOUBLET_POLICY_LOCK_PATH = VALIDATION_DIR / 'GSE157827_DOUBLET_POLICY_LOCK_v1.json'

    DOUBLET_SCORE_PATH = VALIDATION_DIR / 'GSE157827_scrublet_scores_per_donor_v1.csv.gz'

    SINGLET_BARCODE_PATH = VALIDATION_DIR / 'GSE157827_authorQC_scrublet_singlet_barcodes_v1.csv.gz'

    DOUBLET_SUMMARY_PATH = VALIDATION_DIR / 'GSE157827_scrublet_sample_summary_v1.csv'

    DOUBLET_RUN_AUDIT_PATH = VALIDATION_DIR / 'GSE157827_DOUBLET_RUN_AUDIT_v1.json'

    HISTOGRAM_PNG_PATH = VALIDATION_DIR / 'GSE157827_scrublet_score_histograms_v1.png'

    HISTOGRAM_PDF_PATH = VALIDATION_DIR / 'GSE157827_scrublet_score_histograms_v1.pdf'

    for required_path in [STRUCTURAL_AUDIT_PATH, NUCLEUS_QC_PATH, RECONCILIATION_LOCK_PATH]:
        if not required_path.exists():
            raise FileNotFoundError(f'Missing file: {required_path}')

    if not LOCAL_EXTRACT_DIR.exists():
        raise FileNotFoundError('The local 10x files are missing. If the Colab runtime restarted, rerun Step 5.')

    doublet_policy_core = {'dataset': 'GSE157827', 'policy_version': 'v1', 'method': 'Scrublet', 'scrublet_version': SCRUBLET_VERSION, 'processing_unit': 'Each GSM library independently', 'input_population': 'Nuclei passing the locked original-study QC rules', 'expected_doublet_rate': 0.1, 'sim_doublet_ratio': 2.0, 'min_counts': 3, 'min_cells': 3, 'min_gene_variability_percentile': 85, 'number_of_principal_components': 30, 'automatic_threshold': True, 'approximate_neighbors': True, 'random_seed': 0, 'diagnosis_used': False, 'target_genes_queried': False, 'primary_decision': 'Remove automatically predicted doublets only when automatic threshold detection succeeds for the donor.'}


In [ ]:
if RUN_PIPELINE:
    if DOUBLET_POLICY_LOCK_PATH.exists():
        with open(DOUBLET_POLICY_LOCK_PATH, 'r', encoding='utf-8') as handle:
            existing_policy = json.load(handle)
        fields_to_verify = ['method', 'scrublet_version', 'processing_unit', 'expected_doublet_rate', 'sim_doublet_ratio', 'min_counts', 'min_cells', 'min_gene_variability_percentile', 'number_of_principal_components', 'automatic_threshold', 'random_seed', 'diagnosis_used', 'target_genes_queried']
        for field in fields_to_verify:
            if existing_policy.get(field) != doublet_policy_core.get(field):
                raise RuntimeError(f'Existing doublet policy differs for: {field}')
        print('Existing doublet policy lock verified.')
    else:
        policy_to_save = {**doublet_policy_core, 'created_utc': datetime.now(timezone.utc).isoformat()}
        with open(DOUBLET_POLICY_LOCK_PATH, 'w', encoding='utf-8') as handle:
            json.dump(policy_to_save, handle, indent=2)
        print('Doublet policy locked before matrix processing.')

    print(f'Doublet policy: {DOUBLET_POLICY_LOCK_PATH}')

    sample_table = pd.read_csv(STRUCTURAL_AUDIT_PATH, usecols=['GSM', 'sample_label', 'barcode_count', 'feature_count', 'matrix_nonzero_entries', 'orientation'])

    nucleus_qc = pd.read_csv(NUCLEUS_QC_PATH, usecols=['nucleus_id', 'GSM', 'sample_label', 'barcode', 'qc_pass_author_rules'], dtype={'nucleus_id': str, 'GSM': str, 'sample_label': str, 'barcode': str})

    if nucleus_qc['qc_pass_author_rules'].dtype != bool:
        nucleus_qc['qc_pass_author_rules'] = nucleus_qc['qc_pass_author_rules'].astype(str).str.lower().map({'true': True, 'false': False})

    if nucleus_qc['qc_pass_author_rules'].isna().any():
        raise RuntimeError('Could not parse author-QC decisions.')

    if int(nucleus_qc['qc_pass_author_rules'].sum()) != 169506:
        raise RuntimeError('The locked author-QC population is not 169,506 nuclei.')

    if len(sample_table) != 21:
        raise RuntimeError('Expected 21 sample libraries.')

    if 'diagnosis' in sample_table.columns:
        raise RuntimeError('Diagnosis must not be present.')

    print('Diagnosis-blind author-QC population verified: 169,506')

    def locate_sample_file(gsm, suffix):
        matches = sorted(LOCAL_EXTRACT_DIR.glob(f'{gsm}_*_{suffix}'))
        if len(matches) != 1:
            raise RuntimeError(f'Expected one {suffix} file for {gsm}; found {len(matches)}.')
        return matches[0]

    figure, axes = plt.subplots(7, 3, figsize=(15, 24), sharex=True)

    axes = axes.flatten()

    histogram_bins = np.linspace(0, 1, 51)

    doublet_score_frames = []

    sample_summaries = []

    failed_samples = []


In [ ]:
if RUN_PIPELINE:
    for plot_index, sample in enumerate(tqdm(sample_table.itertuples(index=False), total=len(sample_table), desc='Per-donor Scrublet')):
        gsm = str(sample.GSM)
        sample_label = str(sample.sample_label)
        axis = axes[plot_index]
        matrix_path = locate_sample_file(gsm, 'matrix.mtx.gz')
        barcode_path = locate_sample_file(gsm, 'barcodes.tsv.gz')
        barcodes = pd.read_csv(barcode_path, sep='\t', header=None, compression='gzip', dtype=str)[0].astype(str).to_numpy()
        if len(barcodes) != int(sample.barcode_count):
            raise RuntimeError(f'Barcode-count mismatch for {gsm}.')
        sample_qc = nucleus_qc.loc[nucleus_qc['GSM'] == gsm].set_index('barcode').reindex(barcodes)
        if sample_qc['nucleus_id'].isna().any():
            raise RuntimeError(f'QC/barcode alignment failed for {gsm}.')
        pass_mask = sample_qc['qc_pass_author_rules'].astype(bool).to_numpy()
        pass_barcodes = barcodes[pass_mask]
        if len(pass_barcodes) < 100:
            raise RuntimeError(f'Too few QC-pass nuclei for Scrublet in {gsm}: {len(pass_barcodes)}')
        with gzip.open(matrix_path, 'rb') as matrix_handle:
            matrix = mmread(matrix_handle)
        matrix = matrix.tocsc(copy=False)
        expected_shape = (int(sample.feature_count), int(sample.barcode_count))
        if matrix.shape != expected_shape:
            raise RuntimeError(f'Matrix-shape mismatch for {gsm}: {matrix.shape}')
        if matrix.nnz != int(sample.matrix_nonzero_entries):
            raise RuntimeError(f'Matrix coordinate-count mismatch for {gsm}.')
        matrix.sum_duplicates()
        matrix.eliminate_zeros()
        counts_nuclei_by_genes = matrix[:, pass_mask].transpose().tocsc()
        del matrix
        gc.collect()
        try:
            scrub = scr.Scrublet(counts_nuclei_by_genes, expected_doublet_rate=0.1, sim_doublet_ratio=2.0, random_state=0)
            doublet_scores, predicted_doublets = scrub.scrub_doublets(min_counts=3, min_cells=3, min_gene_variability_pctl=85, log_transform=False, mean_center=True, normalize_variance=True, n_prin_comps=30, use_approx_neighbors=True, verbose=False)
            threshold = getattr(scrub, 'threshold_', np.nan)
            threshold_success = predicted_doublets is not None and np.isfinite(threshold) and (len(predicted_doublets) == len(pass_barcodes))
            if not threshold_success:
                failed_samples.append(gsm)
                sample_summaries.append({'GSM': gsm, 'sample_label': sample_label, 'author_QC_nuclei': len(pass_barcodes), 'predicted_doublets': np.nan, 'detected_doublet_percent': np.nan, 'retained_singlets': np.nan, 'automatic_threshold': np.nan, 'detectable_doublet_fraction': np.nan, 'estimated_overall_doublet_rate': np.nan, 'status': 'automatic_threshold_failed', 'review_flag': True})
                axis.hist(doublet_scores, bins=histogram_bins, color='grey', alpha=0.8)
                axis.set_title(f'{sample_label} ({gsm})\nthreshold failed', color='red', fontsize=9)
            else:
                predicted_doublets = np.asarray(predicted_doublets, dtype=bool)
                detected_doublet_percent = float(predicted_doublets.mean() * 100)
                retained_singlets = int((~predicted_doublets).sum())
                detectable_fraction = float(getattr(scrub, 'detectable_doublet_fraction_', np.nan))
                estimated_overall_rate = float(getattr(scrub, 'overall_doublet_rate_', np.nan))
                review_flag = bool(detected_doublet_percent < 0.5 or detected_doublet_percent > 20.0)
                nucleus_ids = np.char.add(np.char.add(gsm, ':'), pass_barcodes.astype(str))
                doublet_score_frames.append(pd.DataFrame({'nucleus_id': nucleus_ids, 'GSM': gsm, 'sample_label': sample_label, 'barcode': pass_barcodes, 'scrublet_score': np.asarray(doublet_scores, dtype=np.float32), 'scrublet_threshold': float(threshold), 'scrublet_predicted_doublet': predicted_doublets, 'analysis_keep_singlet': ~predicted_doublets}))
                sample_summaries.append({'GSM': gsm, 'sample_label': sample_label, 'author_QC_nuclei': len(pass_barcodes), 'predicted_doublets': int(predicted_doublets.sum()), 'detected_doublet_percent': detected_doublet_percent, 'retained_singlets': retained_singlets, 'automatic_threshold': float(threshold), 'detectable_doublet_fraction': detectable_fraction, 'estimated_overall_doublet_rate': estimated_overall_rate, 'status': 'success', 'review_flag': review_flag})
                axis.hist(scrub.doublet_scores_sim_, bins=histogram_bins, density=True, histtype='step', linewidth=1.3, color='#D55E00', label='Simulated')
                axis.hist(doublet_scores, bins=histogram_bins, density=True, histtype='step', linewidth=1.3, color='#0072B2', label='Observed')
                axis.axvline(float(threshold), color='black', linestyle='--', linewidth=1)
                title_color = 'darkorange' if review_flag else 'black'
                axis.set_title(f'{sample_label} ({gsm})\ndoublets={detected_doublet_percent:.2f}%, T={threshold:.3f}', color=title_color, fontsize=9)
        except Exception as error:
            failed_samples.append(gsm)
            sample_summaries.append({'GSM': gsm, 'sample_label': sample_label, 'author_QC_nuclei': len(pass_barcodes), 'predicted_doublets': np.nan, 'detected_doublet_percent': np.nan, 'retained_singlets': np.nan, 'automatic_threshold': np.nan, 'detectable_doublet_fraction': np.nan, 'estimated_overall_doublet_rate': np.nan, 'status': f'error: {type(error).__name__}: {error}', 'review_flag': True})
            axis.text(0.5, 0.5, f'{sample_label}\n{type(error).__name__}', ha='center', va='center', color='red', transform=axis.transAxes)
            axis.set_title(f'{gsm}: ERROR', color='red')
        axis.set_xlim(0, 1)
        axis.set_xlabel('Scrublet score', fontsize=8)
        axis.set_ylabel('Density', fontsize=8)
        axis.tick_params(labelsize=7)
        del counts_nuclei_by_genes
        gc.collect()


In [ ]:
if RUN_PIPELINE:
    handles, labels = axes[0].get_legend_handles_labels()

    if handles:
        figure.legend(handles, labels, loc='upper center', ncol=2, frameon=False)

    figure.suptitle('GSE157827 per-donor Scrublet score distributions\nDiagnosis-blind; author-QC nuclei only', fontsize=15, y=1.002)

    figure.tight_layout()

    figure.savefig(HISTOGRAM_PNG_PATH, dpi=220, bbox_inches='tight')

    figure.savefig(HISTOGRAM_PDF_PATH, bbox_inches='tight')

    plt.close(figure)

    doublet_summary = pd.DataFrame(sample_summaries)

    doublet_summary.to_csv(DOUBLET_SUMMARY_PATH, index=False)

    all_samples_successful = len(failed_samples) == 0

    if all_samples_successful:
        doublet_scores_table = pd.concat(doublet_score_frames, ignore_index=True)
        if len(doublet_scores_table) != 169506:
            raise RuntimeError('The combined Scrublet score table does not contain all 169,506 author-QC nuclei.')
        if doublet_scores_table['nucleus_id'].duplicated().any():
            raise RuntimeError('Duplicated nucleus IDs after Scrublet.')
        doublet_scores_table.to_csv(DOUBLET_SCORE_PATH, index=False, compression={'method': 'gzip', 'compresslevel': 6})
        singlet_barcodes = doublet_scores_table.loc[doublet_scores_table['analysis_keep_singlet'], ['nucleus_id', 'GSM', 'sample_label', 'barcode']].copy()
        singlet_barcodes.to_csv(SINGLET_BARCODE_PATH, index=False, compression={'method': 'gzip', 'compresslevel': 6})
        total_predicted_doublets = int(doublet_scores_table['scrublet_predicted_doublet'].sum())
        total_singlets = int(doublet_scores_table['analysis_keep_singlet'].sum())
    else:
        total_predicted_doublets = None
        total_singlets = None

    doublet_run_audit = {'dataset': 'GSE157827', 'run_version': 'v1', 'completed_utc': datetime.now(timezone.utc).isoformat(), 'scrublet_version': installed_scrublet_version, 'scipy_version': scipy.__version__, 'n_samples': int(len(sample_table)), 'author_QC_input_nuclei': 169506, 'all_samples_successful': bool(all_samples_successful), 'failed_samples': failed_samples, 'total_predicted_doublets': total_predicted_doublets, 'total_retained_singlets': total_singlets, 'diagnosis_used': False, 'target_genes_queried': False, 'normalization_for_cell_type_annotation_performed': False, 'clustering_performed': False, 'differential_expression_performed': False, 'sample_summary_path': str(DOUBLET_SUMMARY_PATH), 'histogram_png_path': str(HISTOGRAM_PNG_PATH), 'histogram_pdf_path': str(HISTOGRAM_PDF_PATH), 'score_table_path': str(DOUBLET_SCORE_PATH) if all_samples_successful else None, 'singlet_barcode_path': str(SINGLET_BARCODE_PATH) if all_samples_successful else None}

    with open(DOUBLET_RUN_AUDIT_PATH, 'w', encoding='utf-8') as handle:
        json.dump(doublet_run_audit, handle, indent=2)

    print('\n' + '=' * 104)

    print('GSE157827 PER-DONOR SCRUBLET SUMMARY')

    print('=' * 104)

    report = doublet_summary[['GSM', 'sample_label', 'author_QC_nuclei', 'predicted_doublets', 'detected_doublet_percent', 'retained_singlets', 'automatic_threshold', 'estimated_overall_doublet_rate', 'status', 'review_flag']].copy()

    for column in ['detected_doublet_percent', 'automatic_threshold', 'estimated_overall_doublet_rate']:
        report[column] = pd.to_numeric(report[column], errors='coerce').round(4)

    print(report.to_string(index=False, justify='left'))

    print('\n' + '-' * 104)

    print(f'All 21 samples successful: {all_samples_successful}')

    if all_samples_successful:
        print(f'Author-QC input nuclei: {169506:,}')
        print(f'Predicted doublets: {total_predicted_doublets:,}')
        print(f'Retained singlets: {total_singlets:,}')
        review_samples = doublet_summary.loc[doublet_summary['review_flag'], 'GSM'].tolist()
        print(f'Samples flagged for review: {review_samples}')
        print(f'\nDoublet scores:\n{DOUBLET_SCORE_PATH}')
        print(f'\nSinglet barcode manifest:\n{SINGLET_BARCODE_PATH}')
    else:
        print(f'Failed samples: {failed_samples}')
        print('Do not proceed to annotation yet.')

    print(f'\nSample summary:\n{DOUBLET_SUMMARY_PATH}')

    print(f'\nDiagnostic histogram:\n{HISTOGRAM_PNG_PATH}')

    print(f'\nRun audit:\n{DOUBLET_RUN_AUDIT_PATH}')

    print('\nGSE157827 PER-DONOR DOUBLET ASSESSMENT COMPLETE')

    print('Diagnosis was not used.')

    print('No target genes were queried.')

    print('No differential-expression testing was performed.')


In [ ]:
if RUN_PIPELINE:
    from pathlib import Path

    from datetime import datetime, timezone

    import pandas as pd

    import numpy as np

    import json

    PROJECT_DIR = Path('/content/drive/MyDrive/AD_Astrocyte_Paper_01')

    VALIDATION_DIR = PROJECT_DIR / 'GSE157827_confirmatory_validation'

    ORIGINAL_SCORE_PATH = VALIDATION_DIR / 'GSE157827_scrublet_scores_per_donor_v1.csv.gz'

    ORIGINAL_SUMMARY_PATH = VALIDATION_DIR / 'GSE157827_scrublet_sample_summary_v1.csv'

    CORRECTED_SCORE_PATH = VALIDATION_DIR / 'GSE157827_scrublet_scores_per_donor_v1_1.csv.gz'

    CORRECTED_SINGLET_PATH = VALIDATION_DIR / 'GSE157827_authorQC_scrublet_singlet_barcodes_v1_1.csv.gz'

    CORRECTED_SUMMARY_PATH = VALIDATION_DIR / 'GSE157827_scrublet_sample_summary_v1_1.csv'

    RESCUE_LOCK_PATH = VALIDATION_DIR / 'GSE157827_DOUBLET_RESCUE_LOCK_v1_1.json'

    for required_path in [ORIGINAL_SCORE_PATH, ORIGINAL_SUMMARY_PATH]:
        if not required_path.exists():
            raise FileNotFoundError(f'Missing file: {required_path}')

    scores = pd.read_csv(ORIGINAL_SCORE_PATH, dtype={'nucleus_id': str, 'GSM': str, 'sample_label': str, 'barcode': str})

    summary = pd.read_csv(ORIGINAL_SUMMARY_PATH, dtype={'GSM': str, 'sample_label': str, 'status': str})

    def parse_boolean(series):
        if series.dtype == bool:
            return series
        parsed = series.astype(str).str.strip().str.lower().map({'true': True, 'false': False})
        if parsed.isna().any():
            raise RuntimeError(f'Could not parse Boolean field: {series.name}')
        return parsed.astype(bool)

    scores['scrublet_predicted_doublet'] = parse_boolean(scores['scrublet_predicted_doublet'])

    scores['analysis_keep_singlet'] = parse_boolean(scores['analysis_keep_singlet'])

    summary['review_flag'] = parse_boolean(summary['review_flag'])

    if len(scores) != 169506:
        raise RuntimeError(f'Expected 169,506 score rows, found {len(scores):,}.')

    if scores['nucleus_id'].duplicated().any():
        raise RuntimeError('Duplicated nucleus IDs detected.')

    flagged = summary.loc[summary['review_flag'], 'GSM'].tolist()

    expected_flagged = ['GSM4775563']

    if flagged != expected_flagged:
        raise RuntimeError(f'Expected flagged library {expected_flagged}; observed {flagged}.')

    reference_libraries = summary.loc[(summary['status'] == 'success') & ~summary['review_flag']].copy()

    if len(reference_libraries) != 20:
        raise RuntimeError(f'Expected 20 reference libraries, found {len(reference_libraries)}.')

    reference_thresholds = pd.to_numeric(reference_libraries['automatic_threshold'], errors='raise')

    consensus_threshold = float(np.median(reference_thresholds))

    if not np.isfinite(consensus_threshold):
        raise RuntimeError('Consensus threshold is not finite.')

    RESCUE_GSM = 'GSM4775563'

    rescue_mask = scores['GSM'] == RESCUE_GSM

    if rescue_mask.sum() == 0:
        raise RuntimeError(f'No scores found for {RESCUE_GSM}.')

    scores['scrublet_threshold_original'] = scores['scrublet_threshold']

    scores['scrublet_predicted_doublet_original'] = scores['scrublet_predicted_doublet']

    scores['final_scrublet_threshold'] = scores['scrublet_threshold']

    scores['final_scrublet_predicted_doublet'] = scores['scrublet_predicted_doublet']

    scores['threshold_method'] = 'automatic_per_donor'

    rescued_scores = pd.to_numeric(scores.loc[rescue_mask, 'scrublet_score'], errors='raise').to_numpy()

    rescued_predictions = rescued_scores > consensus_threshold

    rescued_rate_percent = float(rescued_predictions.mean() * 100)


In [ ]:
if RUN_PIPELINE:
    if not 0.5 <= rescued_rate_percent <= 20.0:
        raise RuntimeError(f'Consensus-threshold rescue still produced an implausible doublet rate.\nThreshold: {consensus_threshold:.6f}\nRate: {rescued_rate_percent:.4f}%')

    scores.loc[rescue_mask, 'final_scrublet_threshold'] = consensus_threshold

    scores.loc[rescue_mask, 'final_scrublet_predicted_doublet'] = rescued_predictions

    scores.loc[rescue_mask, 'threshold_method'] = 'median_of_20_nonflagged_library_thresholds'

    scores['final_scrublet_predicted_doublet'] = scores['final_scrublet_predicted_doublet'].astype(bool)

    scores['analysis_keep_singlet'] = ~scores['final_scrublet_predicted_doublet']

    summary['original_predicted_doublets'] = summary['predicted_doublets']

    summary['original_detected_doublet_percent'] = summary['detected_doublet_percent']

    summary['original_threshold'] = summary['automatic_threshold']

    summary['final_threshold'] = summary['automatic_threshold']

    summary['final_predicted_doublets'] = summary['predicted_doublets']

    summary['final_detected_doublet_percent'] = summary['detected_doublet_percent']

    summary['final_retained_singlets'] = summary['retained_singlets']

    summary['threshold_method'] = 'automatic_per_donor'

    summary['review_resolution'] = 'not_required'

    rescue_summary_mask = summary['GSM'] == RESCUE_GSM

    original_rescue_calls = int(scores.loc[rescue_mask, 'scrublet_predicted_doublet_original'].sum())

    final_rescue_calls = int(rescued_predictions.sum())

    final_rescue_singlets = int(len(rescued_predictions) - final_rescue_calls)

    summary.loc[rescue_summary_mask, 'final_threshold'] = consensus_threshold

    summary.loc[rescue_summary_mask, 'final_predicted_doublets'] = final_rescue_calls

    summary.loc[rescue_summary_mask, 'final_detected_doublet_percent'] = rescued_rate_percent

    summary.loc[rescue_summary_mask, 'final_retained_singlets'] = final_rescue_singlets

    summary.loc[rescue_summary_mask, 'threshold_method'] = 'median_of_20_nonflagged_library_thresholds'

    summary.loc[rescue_summary_mask, 'review_resolution'] = 'resolved_by_locked_consensus_threshold'

    summary['remaining_review_flag'] = False

    final_doublet_count = int(scores['final_scrublet_predicted_doublet'].sum())

    final_singlet_count = int(scores['analysis_keep_singlet'].sum())

    if final_doublet_count + final_singlet_count != 169506:
        raise RuntimeError('Final doublet/singlet counts do not sum.')

    scores.to_csv(CORRECTED_SCORE_PATH, index=False, compression={'method': 'gzip', 'compresslevel': 6})

    singlets = scores.loc[scores['analysis_keep_singlet'], ['nucleus_id', 'GSM', 'sample_label', 'barcode', 'final_scrublet_threshold', 'threshold_method']].copy()

    if len(singlets) != final_singlet_count:
        raise RuntimeError('Singlet manifest count mismatch.')

    singlets.to_csv(CORRECTED_SINGLET_PATH, index=False, compression={'method': 'gzip', 'compresslevel': 6})

    summary.to_csv(CORRECTED_SUMMARY_PATH, index=False)

    rescue_lock = {'dataset': 'GSE157827', 'version': 'v1.1', 'created_utc': datetime.now(timezone.utc).isoformat(), 'rescue_GSM': RESCUE_GSM, 'rescue_trigger': 'Automatic detected-doublet rate below the predefined 0.5 percent review threshold.', 'original_automatic_threshold': float(summary.loc[rescue_summary_mask, 'original_threshold'].iloc[0]), 'original_predicted_doublets': original_rescue_calls, 'original_detected_doublet_percent': float(summary.loc[rescue_summary_mask, 'original_detected_doublet_percent'].iloc[0]), 'reference_library_count': int(len(reference_libraries)), 'reference_library_GSMs': reference_libraries['GSM'].tolist(), 'reference_automatic_thresholds': reference_thresholds.astype(float).tolist(), 'consensus_method': 'Median automatic threshold of the 20 non-flagged libraries', 'consensus_threshold': consensus_threshold, 'rescued_predicted_doublets': final_rescue_calls, 'rescued_detected_doublet_percent': rescued_rate_percent, 'final_total_predicted_doublets': final_doublet_count, 'final_total_retained_singlets': final_singlet_count, 'diagnosis_used_for_rescue': False, 'target_genes_queried': False, 'expression_matrices_reloaded': False, 'sensitivity_analysis_plan': ['Repeat confirmatory testing using the original GSM4775563 automatic calls.', 'Repeat confirmatory testing after excluding GSM4775563 entirely.'], 'corrected_score_path': str(CORRECTED_SCORE_PATH), 'corrected_singlet_path': str(CORRECTED_SINGLET_PATH), 'corrected_summary_path': str(CORRECTED_SUMMARY_PATH)}

    with open(RESCUE_LOCK_PATH, 'w', encoding='utf-8') as handle:
        json.dump(rescue_lock, handle, indent=2)

    print('=' * 78)

    print('GSE157827 SCRUBLET THRESHOLD RESCUE')

    print('=' * 78)

    print(f'Flagged library: {RESCUE_GSM}')

    print(f"Original automatic threshold: {rescue_lock['original_automatic_threshold']:.6f}")

    print(f'Median of 20 reference thresholds: {consensus_threshold:.6f}')

    print(f'Original predicted doublets: {original_rescue_calls:,}')

    print(f'Rescued predicted doublets: {final_rescue_calls:,}')

    print(f'Rescued detected-doublet rate: {rescued_rate_percent:.4f}%')

    print('\nFinal cohort counts:')

    print(f'Author-QC nuclei: {169506:,}')

    print(f'Final predicted doublets: {final_doublet_count:,}')

    print(f'Final retained singlets: {final_singlet_count:,}')

    print('\nCorrected sample summary:')

    print(summary[['GSM', 'sample_label', 'author_QC_nuclei', 'original_predicted_doublets', 'final_predicted_doublets', 'final_detected_doublet_percent', 'final_retained_singlets', 'threshold_method', 'remaining_review_flag']].to_string(index=False, justify='left'))


In [ ]:
if RUN_PIPELINE:
    print(f'\nCorrected score table:\n{CORRECTED_SCORE_PATH}')

    print(f'\nCorrected singlet manifest:\n{CORRECTED_SINGLET_PATH}')

    print(f'\nCorrected summary:\n{CORRECTED_SUMMARY_PATH}')

    print(f'\nRescue lock:\n{RESCUE_LOCK_PATH}')

    print('\nGSE157827 DOUBLET DECISIONS FINALIZED')

    print('Original v1 outputs were preserved.')

    print('Diagnosis was not used to choose the rescue threshold.')

    print('No target genes or differential expression were examined.')


## Frozen singlet raw-count partitions

The final 159,144 singlet nuclei are reconstructed as donor-partitioned sparse raw-count matrices. The implementation below is the final no-AnnData reconstruction used after the earlier memory/package experiments.


In [ ]:
if RUN_PIPELINE:
    from pathlib import Path

    from datetime import datetime, timezone

    import pandas as pd

    import numpy as np

    import scipy

    import scipy.sparse as sp

    from scipy.io import mmread

    import gzip

    import json

    import gc

    PROJECT_DIR = Path('/content/drive/MyDrive/AD_Astrocyte_Paper_01')

    VALIDATION_DIR = PROJECT_DIR / 'GSE157827_confirmatory_validation'

    LOCAL_EXTRACT_DIR = Path('/content/GSE157827_raw_10x')

    SINGLET_PATH = VALIDATION_DIR / 'GSE157827_authorQC_scrublet_singlet_barcodes_v1_1.csv.gz'

    OUTPUT_DIR = VALIDATION_DIR / 'GSE157827_STEP8_raw_singlet_matrix_v1_1'

    OBS_PATH = VALIDATION_DIR / 'GSE157827_STEP8_singlet_obs_v1_1.csv.gz'

    VAR_PATH = VALIDATION_DIR / 'GSE157827_STEP8_feature_metadata_v1_1.csv.gz'

    PARTITION_MANIFEST_PATH = VALIDATION_DIR / 'GSE157827_STEP8_matrix_partition_manifest_v1_1.csv'

    STEP8_LOCK_PATH = VALIDATION_DIR / 'GSE157827_STEP8_MATRIX_LOCK_v1_1.json'

    EXPECTED_SINGLETS = 159144

    EXPECTED_DONORS = 21

    EXPECTED_FEATURES = 33538

    if not LOCAL_EXTRACT_DIR.exists():
        raise FileNotFoundError(f'Raw 10x directory is missing:\n{LOCAL_EXTRACT_DIR}\nRerun STEP 5.')

    if not SINGLET_PATH.exists():
        raise FileNotFoundError(f'Locked corrected singlet manifest is missing:\n{SINGLET_PATH}')

    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    print('STEP 8 started.')

    print('scipy:', scipy.__version__)

    print('Raw 10x directory: FOUND')

    print('Locked singlet manifest: FOUND')

    singlets = pd.read_csv(SINGLET_PATH, dtype={'nucleus_id': str, 'GSM': str, 'sample_label': str, 'barcode': str})

    required_columns = {'nucleus_id', 'GSM', 'sample_label', 'barcode'}

    missing_columns = required_columns - set(singlets.columns)

    if missing_columns:
        raise RuntimeError(f'Missing required columns from locked singlet manifest: {sorted(missing_columns)}')

    if len(singlets) != EXPECTED_SINGLETS:
        raise RuntimeError(f'Expected {EXPECTED_SINGLETS:,} singlets, found {len(singlets):,}.')

    if singlets['GSM'].nunique() != EXPECTED_DONORS:
        raise RuntimeError(f"Expected {EXPECTED_DONORS} GSM donors, found {singlets['GSM'].nunique()}.")

    if singlets['nucleus_id'].duplicated().any():
        raise RuntimeError('Duplicate nucleus IDs detected.')

    expected_ids = singlets['GSM'].astype(str) + ':' + singlets['barcode'].astype(str)

    if not np.array_equal(expected_ids.to_numpy(), singlets['nucleus_id'].to_numpy()):
        raise RuntimeError('nucleus_id does not equal GSM:barcode.')

    print(f"Locked population verified: {len(singlets):,} nuclei across {singlets['GSM'].nunique()} donors")

    def locate_file(gsm, suffix):
        matches = sorted(LOCAL_EXTRACT_DIR.glob(f'{gsm}_*_{suffix}'))
        if len(matches) != 1:
            raise RuntimeError(f'{gsm}: expected exactly one {suffix}; found {len(matches)}')
        return matches[0]

    ordered_gsms = list(singlets['GSM'].drop_duplicates())

    first_gsm = ordered_gsms[0]

    reference_feature_path = locate_file(first_gsm, 'features.tsv.gz')

    reference_features = pd.read_csv(reference_feature_path, sep='\t', header=None, compression='gzip', dtype=str)


In [ ]:
if RUN_PIPELINE:
    if len(reference_features) != EXPECTED_FEATURES:
        raise RuntimeError(f'Expected {EXPECTED_FEATURES:,} features; found {len(reference_features):,}.')

    if reference_features.shape[1] == 3:
        reference_features.columns = ['gene_id', 'gene_symbol', 'feature_type']
    elif reference_features.shape[1] == 2:
        reference_features.columns = ['gene_id', 'gene_symbol']
    else:
        raise RuntimeError(f'Unexpected number of feature-file columns: {reference_features.shape[1]}')

    if reference_features['gene_id'].duplicated().any():
        raise RuntimeError('Duplicate gene IDs in reference feature file.')

    reference_features.to_csv(VAR_PATH, index=False, compression={'method': 'gzip', 'compresslevel': 6})

    print(f'Common feature reference established: {len(reference_features):,} features')

    partition_records = []

    obs_frames = []

    global_row_start = 0


In [ ]:
if RUN_PIPELINE:
    for donor_number, gsm in enumerate(ordered_gsms, start=1):
        donor_singlets = singlets.loc[singlets['GSM'] == gsm, ['nucleus_id', 'GSM', 'sample_label', 'barcode']].copy().reset_index(drop=True)
        sample_labels = donor_singlets['sample_label'].dropna().unique()
        if len(sample_labels) != 1:
            raise RuntimeError(f'{gsm}: expected one sample label, found {sample_labels}')
        sample_label = str(sample_labels[0])
        print('\n' + '=' * 68)
        print(f'[{donor_number:02d}/21] {gsm} / {sample_label}')
        print(f'Locked singlets: {len(donor_singlets):,}')
        barcode_path = locate_file(gsm, 'barcodes.tsv.gz')
        feature_path = locate_file(gsm, 'features.tsv.gz')
        matrix_path = locate_file(gsm, 'matrix.mtx.gz')
        donor_features = pd.read_csv(feature_path, sep='\t', header=None, compression='gzip', dtype=str)
        if donor_features.shape != reference_features.shape:
            raise RuntimeError(f'{gsm}: feature-table shape mismatch.')
        donor_features.columns = reference_features.columns
        if not np.array_equal(donor_features.to_numpy(), reference_features.to_numpy()):
            raise RuntimeError(f'{gsm}: feature identity/order mismatch.')
        del donor_features
        raw_barcodes = pd.read_csv(barcode_path, sep='\t', header=None, compression='gzip', dtype=str).iloc[:, 0].astype(str)
        if raw_barcodes.duplicated().any():
            raise RuntimeError(f'{gsm}: duplicate raw barcodes.')
        barcode_index = pd.Index(raw_barcodes.to_numpy())
        selected_positions = barcode_index.get_indexer(donor_singlets['barcode'].to_numpy())
        if np.any(selected_positions < 0):
            missing = donor_singlets.loc[selected_positions < 0, 'barcode'].tolist()
            raise RuntimeError(f'{gsm}: locked barcodes missing from raw matrix. Examples: {missing[:5]}')
        if len(np.unique(selected_positions)) != len(selected_positions):
            raise RuntimeError(f'{gsm}: duplicate selected barcode positions.')
        print('Reading raw matrix...')
        with gzip.open(matrix_path, 'rb') as handle:
            raw_matrix = mmread(handle)
        expected_shape = (EXPECTED_FEATURES, len(raw_barcodes))
        if raw_matrix.shape != expected_shape:
            raise RuntimeError(f'{gsm}: raw matrix shape {raw_matrix.shape} != {expected_shape}')
        if raw_matrix.nnz == 0:
            raise RuntimeError(f'{gsm}: matrix contains no nonzero counts.')
        raw_matrix = raw_matrix.tocsc()
        if np.any(raw_matrix.data < 0):
            raise RuntimeError(f'{gsm}: negative raw counts detected.')
        if not np.all(np.equal(raw_matrix.data, np.rint(raw_matrix.data))):
            raise RuntimeError(f'{gsm}: non-integer raw counts detected.')
        donor_matrix = raw_matrix[:, selected_positions].transpose().tocsr()
        if donor_matrix.shape != (len(donor_singlets), EXPECTED_FEATURES):
            raise RuntimeError(f'{gsm}: selected matrix shape mismatch: {donor_matrix.shape}')
        max_count = int(donor_matrix.data.max()) if donor_matrix.nnz > 0 else 0
        if max_count > np.iinfo(np.int32).max:
            raise RuntimeError(f'{gsm}: count exceeds int32 range.')
        donor_matrix.data = donor_matrix.data.astype(np.int32, copy=False)
        partition_path = OUTPUT_DIR / f'{gsm}_{sample_label}_singlet_raw_counts_v1_1.npz'
        print('Saving sparse donor partition...')
        sp.save_npz(partition_path, donor_matrix, compressed=True)
        check_matrix = sp.load_npz(partition_path)
        if check_matrix.shape != donor_matrix.shape:
            raise RuntimeError(f'{gsm}: saved matrix shape verification failed.')
        if check_matrix.nnz != donor_matrix.nnz:
            raise RuntimeError(f'{gsm}: saved matrix NNZ verification failed.')
        if check_matrix.dtype != np.int32:
            raise RuntimeError(f'{gsm}: saved matrix dtype verification failed.')
        global_row_end = global_row_start + len(donor_singlets)
        donor_obs = donor_singlets[['nucleus_id', 'GSM', 'barcode']].copy()
        donor_obs['matrix_partition'] = partition_path.name
        donor_obs['partition_row'] = np.arange(len(donor_obs), dtype=np.int64)
        donor_obs['global_row'] = np.arange(global_row_start, global_row_end, dtype=np.int64)
        obs_frames.append(donor_obs)
        partition_records.append({'GSM': gsm, 'sample_label': sample_label, 'raw_barcodes': int(len(raw_barcodes)), 'retained_singlets': int(donor_matrix.shape[0]), 'features': int(donor_matrix.shape[1]), 'nonzero_entries': int(donor_matrix.nnz), 'total_UMI_counts': int(donor_matrix.sum()), 'matrix_dtype': str(donor_matrix.dtype), 'partition_file': str(partition_path), 'global_row_start': int(global_row_start), 'global_row_end_exclusive': int(global_row_end)})
        print(f'Saved: {partition_path.name}')
        print(f'Shape: {donor_matrix.shape}')
        print(f'Nonzero entries: {donor_matrix.nnz:,}')
        global_row_start = global_row_end
        del raw_matrix
        del donor_matrix
        del check_matrix
        del raw_barcodes
        del barcode_index
        del selected_positions
        gc.collect()


In [ ]:
if RUN_PIPELINE:
    obs = pd.concat(obs_frames, ignore_index=True)

    if len(obs) != EXPECTED_SINGLETS:
        raise RuntimeError(f'Global observation count = {len(obs):,}; expected {EXPECTED_SINGLETS:,}.')

    if obs['nucleus_id'].duplicated().any():
        raise RuntimeError('Duplicate nucleus IDs in final observation table.')

    if not np.array_equal(obs['global_row'].to_numpy(), np.arange(EXPECTED_SINGLETS, dtype=np.int64)):
        raise RuntimeError('Global row indexing is not contiguous.')

    obs.to_csv(OBS_PATH, index=False, compression={'method': 'gzip', 'compresslevel': 6})

    partition_manifest = pd.DataFrame(partition_records)

    if len(partition_manifest) != EXPECTED_DONORS:
        raise RuntimeError('Expected 21 matrix partitions.')

    if partition_manifest['retained_singlets'].sum() != EXPECTED_SINGLETS:
        raise RuntimeError('Partition singlet counts do not sum to 159,144.')

    if not (partition_manifest['features'] == EXPECTED_FEATURES).all():
        raise RuntimeError('Feature count differs across partitions.')

    partition_manifest.to_csv(PARTITION_MANIFEST_PATH, index=False)

    saved_npz_files = sorted(OUTPUT_DIR.glob('*_singlet_raw_counts_v1_1.npz'))

    if len(saved_npz_files) != EXPECTED_DONORS:
        raise RuntimeError(f'Expected 21 NPZ matrix partitions, found {len(saved_npz_files)}.')

    step8_lock = {'dataset': 'GSE157827', 'step': 8, 'version': 'v1.1', 'created_utc': datetime.now(timezone.utc).isoformat(), 'input_locked_singlet_manifest': str(SINGLET_PATH), 'number_of_donors': int(EXPECTED_DONORS), 'number_of_singlet_nuclei': int(EXPECTED_SINGLETS), 'number_of_features': int(EXPECTED_FEATURES), 'matrix_storage': '21 donor-partitioned scipy CSR NPZ matrices; rows=nuclei, columns=genes', 'feature_order_file': str(VAR_PATH), 'observation_order_file': str(OBS_PATH), 'partition_manifest_file': str(PARTITION_MANIFEST_PATH), 'matrix_partition_directory': str(OUTPUT_DIR), 'raw_integer_counts_preserved': True, 'diagnosis_used_for_matrix_construction': False, 'target_genes_queried': False, 'CREB5_queried': False, 'normalization_performed': False, 'dimension_reduction_performed': False, 'clustering_performed': False, 'cell_type_annotation_performed': False, 'differential_expression_performed': False}

    with open(STEP8_LOCK_PATH, 'w', encoding='utf-8') as handle:
        json.dump(step8_lock, handle, indent=2)

    print('\n' + '=' * 72)

    print('GSE157827 STEP 8 COMPLETE')

    print('=' * 72)

    print(f'Donors:             {len(partition_manifest)}')

    print(f'Locked singlets:    {len(obs):,}')

    print(f'Features:           {len(reference_features):,}')

    print(f'Matrix partitions:  {len(saved_npz_files)}')

    print(f"Total nonzero entries: {partition_manifest['nonzero_entries'].sum():,}")

    print(f"Total raw UMI counts: {partition_manifest['total_UMI_counts'].sum():,}")

    print('\nRaw integer counts preserved: YES')

    print('Diagnosis used for reconstruction: NO')

    print('CREB5 queried: NO')

    print('Normalization performed: NO')

    print('Clustering performed: NO')

    print('Differential expression performed: NO')

    print('\nSaved matrix partitions:')

    print(OUTPUT_DIR)

    print('\nObservation order:')

    print(OBS_PATH)

    print('\nFeature order:')

    print(VAR_PATH)

    print('\nPartition manifest:')

    print(PARTITION_MANIFEST_PATH)

    print('\nSTEP 8 lock:')

    print(STEP8_LOCK_PATH)


## Primary astrocyte definition

Marker panels and operational acceptance rules are locked before marker-expression inspection. The primary workflow uses 1,000 diagnosis-blind variable genes, 50 latent components, the first 20 for clustering, and 43 MiniBatchKMeans clusters with seed 20260912.


In [ ]:
if RUN_PIPELINE:
    from pathlib import Path

    from datetime import datetime, timezone

    import json

    PROJECT_DIR = Path('/content/drive/MyDrive/AD_Astrocyte_Paper_01')

    VALIDATION_DIR = PROJECT_DIR / 'GSE157827_confirmatory_validation'

    STEP8_LOCK_PATH = VALIDATION_DIR / 'GSE157827_STEP8_MATRIX_LOCK_v1_1.json'

    ANNOTATION_LOCK_PATH = VALIDATION_DIR / 'GSE157827_STEP9A_CELLTYPE_ANNOTATION_LOCK_v1.json'

    if not STEP8_LOCK_PATH.exists():
        raise FileNotFoundError('STEP 8 matrix lock is missing. Stop.')

    with open(STEP8_LOCK_PATH, 'r', encoding='utf-8') as handle:
        step8_lock = json.load(handle)

    if step8_lock.get('number_of_singlet_nuclei') != 159144:
        raise RuntimeError('STEP 8 singlet count is not 159,144.')

    if not step8_lock.get('raw_integer_counts_preserved', False):
        raise RuntimeError('STEP 8 does not confirm preservation of raw integer counts.')

    marker_panels = {'Astrocyte': ['AQP4', 'SLC1A2', 'SLC1A3', 'GLUL', 'GJA1', 'ALDH1L1', 'SOX9'], 'Endothelial': ['CLDN5', 'FLT1', 'PECAM1', 'VWF', 'KDR', 'EMCN'], 'Excitatory_neuron': ['CAMK2A', 'SLC17A7', 'NRGN', 'SATB2'], 'Inhibitory_neuron': ['GAD1', 'GAD2', 'SLC32A1', 'SLC6A1'], 'Microglia': ['CSF1R', 'CX3CR1', 'P2RY12', 'AIF1', 'TYROBP', 'C1QA'], 'Oligodendrocyte': ['MBP', 'MOG', 'PLP1', 'MOBP', 'MAG', 'CLDN11']}

    annotation_rules = {'expression_transform': 'For annotation only: library-size normalize each nucleus to 10,000 counts followed by log1p.', 'lineage_score': 'Mean log1p(CP10K) expression across all available marker genes in that lineage panel.', 'marker_detection': 'A marker is detected when its raw UMI count is greater than zero in that nucleus.', 'broad_label_rule': 'The broad cell-type label is the lineage with the largest marker-panel score. A nucleus lacking adequate marker evidence is labelled Unknown rather than forced into a lineage.', 'minimum_markers_for_non_unknown': 2, 'astrocyte_primary_rule': 'A high-confidence astrocyte must satisfy ALL of: (1) Astrocyte is the highest lineage score; (2) at least two astrocyte panel genes have raw UMI > 0; (3) no competing lineage has a greater marker-detection count than the astrocyte panel.', 'mixed_identity_rule': 'Nuclei with strong conflicting lineage evidence will be retained as Unknown/Mixed and excluded from the primary astrocyte pseudobulk.', 'disease_blind': True, 'diagnosis_not_loaded_during_annotation': True, 'CREB5_prohibited_for_annotation': True, 'target_genes_prohibited_for_annotation': True, 'DE_results_prohibited_for_annotation': True, 'astrocyte_subclustering_required_for_primary_DE': False, 'annotation_level': 'Broad major cortical cell type only'}

    donor_rules = {'primary_minimum_high_confidence_astrocytes': 20, 'sensitivity_minimum_high_confidence_astrocytes': 50, 'biological_replicate': 'human donor', 'pseudobulk_rule': 'Raw integer counts will be summed across accepted astrocyte nuclei separately for each donor.'}

    all_markers = [gene for panel in marker_panels.values() for gene in panel]

    if len(all_markers) != len(set(all_markers)):
        raise RuntimeError('A marker occurs in more than one decision panel.')

    prohibited = {'CREB5', 'CSRP1', 'SHC1', 'KCNH2', 'MRGPRF', 'AJAP1', 'CCDC3'}

    overlap = prohibited.intersection(set(all_markers))

    if overlap:
        raise RuntimeError(f'Locked target genes accidentally present in annotation panels: {sorted(overlap)}')

    annotation_lock = {'dataset': 'GSE157827', 'step': '9A', 'version': 'v1', 'created_utc': datetime.now(timezone.utc).isoformat(), 'purpose': 'Freeze diagnosis-blind broad cell-type annotation before examining marker expression.', 'input_population': '159,144 author-QC + Scrublet singlet nuclei', 'marker_panels': marker_panels, 'annotation_rules': annotation_rules, 'donor_rules': donor_rules, 'author_reference_scheme': 'Six broad major cell types used by Lau et al. for GSE157827: astrocytes, endothelial cells, excitatory neurons, inhibitory neurons, microglia, oligodendrocytes.', 'expression_examined_during_this_step': False, 'diagnosis_used': False, 'CREB5_queried': False, 'differential_expression_performed': False}

    with open(ANNOTATION_LOCK_PATH, 'w', encoding='utf-8') as handle:
        json.dump(annotation_lock, handle, indent=2)

    print('=' * 72)

    print('GSE157827 STEP 9A ANNOTATION RULES LOCKED')

    print('=' * 72)

    print(f"Input population: {step8_lock['number_of_singlet_nuclei']:,} singlets")

    print(f'Broad lineages locked: {len(marker_panels)}')

    for cell_type, genes in marker_panels.items():
        print(f'{cell_type:20s}: ' + ', '.join(genes))

    print('\nPrimary astrocyte minimum markers: 2')

    print('Primary donor minimum astrocytes: 20')

    print('Sensitivity donor minimum astrocytes: 50')

    print('\nExpression examined: NO')

    print('Diagnosis used: NO')

    print('CREB5 queried: NO')

    print('Differential expression performed: NO')

    print(f'\nAnnotation lock saved:\n{ANNOTATION_LOCK_PATH}')


In [ ]:
if RUN_PIPELINE:
    from pathlib import Path

    from datetime import datetime, timezone

    import json

    PROJECT_DIR = Path('/content/drive/MyDrive/AD_Astrocyte_Paper_01')

    VALIDATION_DIR = PROJECT_DIR / 'GSE157827_confirmatory_validation'

    ANNOTATION_LOCK_PATH = VALIDATION_DIR / 'GSE157827_STEP9A_CELLTYPE_ANNOTATION_LOCK_v1.json'

    OPERATIONAL_LOCK_PATH = VALIDATION_DIR / 'GSE157827_STEP9A1_OPERATIONAL_ANNOTATION_LOCK_v1.json'

    if not ANNOTATION_LOCK_PATH.exists():
        raise FileNotFoundError('STEP 9A annotation lock is missing.')

    with open(ANNOTATION_LOCK_PATH, 'r', encoding='utf-8') as handle:
        annotation_lock = json.load(handle)

    operational_rules = {'normalization': 'For annotation only, divide each marker raw UMI count by total raw UMI counts for that nucleus, multiply by 10000, then apply log1p.', 'gene_symbol_duplicate_rule': 'If multiple feature rows share a marker gene symbol, their raw counts are summed before normalization.', 'lineage_score': 'Arithmetic mean of log1p(CP10K) values across all available genes in that lineage marker panel.', 'marker_detected': 'Marker gene has summed raw UMI count > 0.', 'minimum_detected_markers': 2, 'top_lineage_rule': 'Lineage must have the uniquely highest lineage score. Exact score ties are classified as Unknown.', 'conflicting_lineage_rule': 'A competing lineage constitutes conflicting evidence when its number of detected marker genes is strictly greater than the detected-marker count of the top-scoring lineage.', 'broad_label_rule': 'Assign the uniquely top-scoring lineage only when at least two of its markers are detected and no competing lineage has a strictly greater detected-marker count. Otherwise label Unknown.', 'high_confidence_astrocyte_rule': 'Astrocyte must be the uniquely highest lineage score, at least two astrocyte markers must be detected, and no competing lineage may have a strictly greater detected-marker count.', 'diagnosis_used': False, 'target_genes_used': False, 'CREB5_used': False, 'differential_expression_used': False}

    lock = {'dataset': 'GSE157827', 'step': '9A.1', 'created_utc': datetime.now(timezone.utc).isoformat(), 'parent_annotation_lock': str(ANNOTATION_LOCK_PATH), 'rules': operational_rules, 'expression_examined': False, 'diagnosis_loaded': False, 'CREB5_queried': False, 'differential_expression_performed': False}

    with open(OPERATIONAL_LOCK_PATH, 'w', encoding='utf-8') as handle:
        json.dump(lock, handle, indent=2)

    print('=' * 72)

    print('GSE157827 STEP 9A.1 OPERATIONAL RULES LOCKED')

    print('=' * 72)

    print('Minimum detected markers: 2')

    print('Conflicting lineage: competitor has strictly greater marker-detection count')

    print('Exact top-score ties: Unknown')

    print('High-confidence astrocytes use the same predetermined rule.')

    print('\nExpression examined: NO')

    print('Diagnosis loaded: NO')

    print('CREB5 queried: NO')

    print('Differential expression performed: NO')

    print(f'\nOperational lock saved:\n{OPERATIONAL_LOCK_PATH}')


In [ ]:
if RUN_PIPELINE:
    from pathlib import Path

    from datetime import datetime, timezone

    import json

    PROJECT_DIR = Path('/content/drive/MyDrive/AD_Astrocyte_Paper_01')

    VALIDATION_DIR = PROJECT_DIR / 'GSE157827_confirmatory_validation'

    STEP8_LOCK_PATH = VALIDATION_DIR / 'GSE157827_STEP8_MATRIX_LOCK_v1_1.json'

    STEP9C_LOCK_PATH = VALIDATION_DIR / 'GSE157827_STEP9C_ANNOTATION_QC_LOCK_v1.json'

    STEP9D_LOCK_PATH = VALIDATION_DIR / 'GSE157827_STEP9D_CLUSTERING_PLAN_LOCK_v1.json'

    for path in [STEP8_LOCK_PATH, STEP9C_LOCK_PATH]:
        if not path.exists():
            raise FileNotFoundError(f'Required lock missing:\n{path}')

    with open(STEP8_LOCK_PATH, 'r', encoding='utf-8') as handle:
        step8 = json.load(handle)

    with open(STEP9C_LOCK_PATH, 'r', encoding='utf-8') as handle:
        step9c = json.load(handle)

    if step8['number_of_singlet_nuclei'] != 159144:
        raise RuntimeError('Unexpected STEP 8 nucleus count.')

    if step9c.get('step9b_accepted_for_primary_pseudobulk', True):
        raise RuntimeError('STEP 9C does not document rejection of the STEP 9B pilot annotation.')

    prohibited_target_genes = ['CREB5', 'CSRP1', 'SHC1', 'KCNH2', 'MRGPRF', 'AJAP1', 'CCDC3']

    clustering_specification = {'input_nuclei': 159144, 'input_counts': 'STEP 8 raw integer singlet counts', 'diagnosis_used': False, 'normalization': 'library-size normalization to 10,000 counts per nucleus followed by log1p', 'hvg_number': 1000, 'hvg_selection': 'Top 1000 genes by variance of log1p(CP10K) across all 159,144 nuclei, calculated without disease labels', 'target_gene_exclusion': 'CREB5 and the six secondary locked genes are removed from the eligible HVG universe before HVG ranking', 'mitochondrial_gene_exclusion_from_HVGs': True, 'number_of_components_computed': 50, 'components_used_for_clustering': 20, 'cluster_count': 43, 'random_seed': 20260912, 'python_dimension_reduction': 'StandardScaler(with_mean=False) followed by TruncatedSVD(n_components=50)', 'python_clustering': 'MiniBatchKMeans(n_clusters=43, n_init=20, random_state=20260912)', 'reason_for_43_clusters': 'Lau et al. reported 43 initial transcriptome-wide clusters before assigning six major cell types', 'cluster_annotation_stage': 'Cluster identities will be assigned only AFTER unsupervised clustering, using predetermined cell-lineage marker panels.', 'published_cell_types': ['Astrocyte', 'Endothelial', 'Excitatory_neuron', 'Inhibitory_neuron', 'Microglia', 'Oligodendrocyte'], 'step9b_status': 'Rejected as primary annotation; retained as diagnosis-blind pilot/sensitivity annotation'}

    cluster_marker_panels = {'Astrocyte': ['AQP4', 'SLC1A2', 'SLC1A3', 'GFAP', 'ALDH1L1', 'ADGRV1', 'GPC5', 'RYR3'], 'Endothelial': ['CLDN5', 'FLT1', 'PECAM1', 'VWF', 'ABCB1', 'EBF1'], 'Excitatory_neuron': ['CAMK2A', 'SLC17A7', 'NRGN', 'SATB2', 'CBLN2', 'LDB2'], 'Inhibitory_neuron': ['GAD1', 'GAD2', 'SLC32A1', 'LHFPL3', 'PCDH15'], 'Microglia': ['CSF1R', 'CX3CR1', 'P2RY12', 'AIF1', 'C3', 'LRMDA', 'DOCK8'], 'Oligodendrocyte': ['MBP', 'MOG', 'PLP1', 'MOBP', 'MAG', 'ST18']}

    marker_genes = {gene for genes in cluster_marker_panels.values() for gene in genes}

    overlap = set(prohibited_target_genes) & marker_genes

    if overlap:
        raise RuntimeError(f'Confirmatory target genes accidentally present in marker panels: {sorted(overlap)}')

    annotation_specification = {'unit_of_annotation': 'whole unsupervised cluster, not individual nucleus', 'marker_summary': 'Mean log1p(CP10K) marker expression calculated for each of the 43 clusters', 'marker_standardization': 'For each marker gene, cluster mean expression will be standardized across the 43 clusters before lineage-panel averaging', 'cluster_score': 'Mean standardized expression of available marker genes belonging to each lineage', 'provisional_cluster_label': 'Lineage having the highest marker-panel score', 'ambiguous_cluster_rule': 'Clusters with conflicting major-lineage marker patterns will remain Unknown rather than being forced into Astrocyte or another lineage', 'astrocyte_identity_rule': 'Only complete clusters assigned Astrocyte will enter the primary astrocyte pseudobulk', 'published_composition_not_used_to_assign_labels': True, 'step9b_labels_not_used_to_assign_clusters': True, 'diagnosis_not_used': True, 'CREB5_not_used': True, 'DE_results_not_used': True}

    lock = {'dataset': 'GSE157827', 'step': '9D', 'version': 'v1', 'created_utc': datetime.now(timezone.utc).isoformat(), 'reason': 'STEP 9B per-nucleus marker classification failed annotation-composition QC; move to transcriptome-wide cluster-based annotation.', 'clustering_specification': clustering_specification, 'cluster_marker_panels': cluster_marker_panels, 'annotation_specification': annotation_specification, 'prohibited_target_genes': prohibited_target_genes, 'expression_examined_in_this_step': False, 'diagnosis_used': False, 'CREB5_queried': False, 'differential_expression_performed': False}

    with open(STEP9D_LOCK_PATH, 'w', encoding='utf-8') as handle:
        json.dump(lock, handle, indent=2)

    print('=' * 76)

    print('GSE157827 STEP 9D TRANSCRIPTOME-WIDE CLUSTERING PLAN LOCKED')

    print('=' * 76)

    print('\nInput nuclei: 159,144')

    print('HVGs: 1,000')

    print('Components computed: 50')

    print('Components used for clustering: 20')

    print('Initial clusters: 43')

    print('Random seed: 20260912')

    print('\nConfirmatory targets excluded from clustering:')

    for gene in prohibited_target_genes:
        print(' -', gene)

    print('\nCluster-level lineages:')

    for lineage, markers in cluster_marker_panels.items():
        print(f'{lineage:20s}: ' + ', '.join(markers))

    print('\nSTEP 9B used for primary annotation: NO')

    print('Published percentages used to force labels: NO')

    print('Expression examined in STEP 9D: NO')

    print('Diagnosis used: NO')

    print('CREB5 queried: NO')

    print('Differential expression performed: NO')

    print(f'\nClustering-plan lock saved:\n{STEP9D_LOCK_PATH}')


In [ ]:
if RUN_PIPELINE:
    from pathlib import Path

    from datetime import datetime, timezone

    import json

    PROJECT_DIR = Path('/content/drive/MyDrive/AD_Astrocyte_Paper_01')

    VALIDATION_DIR = PROJECT_DIR / 'GSE157827_confirmatory_validation'

    STEP9D_LOCK_PATH = VALIDATION_DIR / 'GSE157827_STEP9D_CLUSTERING_PLAN_LOCK_v1.json'

    STEP9D1_LOCK_PATH = VALIDATION_DIR / 'GSE157827_STEP9D1_DONOR_CORRECTION_LOCK_v1.json'

    if not STEP9D_LOCK_PATH.exists():
        raise FileNotFoundError('STEP 9D clustering lock is missing.')

    with open(STEP9D_LOCK_PATH, 'r', encoding='utf-8') as handle:
        step9d = json.load(handle)

    donor_correction = {'purpose': 'Reduce donor/library effects during cell-type clustering without using diagnosis.', 'stage': 'After 50-dimensional SVD representation and before clustering', 'method': "For each donor, subtract that donor's mean 50-dimensional latent vector and add the global latent-space mean vector.", 'components_corrected': 50, 'components_used_for_clustering_after_correction': 20, 'diagnosis_used': False, 'disease_labels_used': False, 'target_genes_used': False, 'CREB5_used': False, 'reason': 'The published workflow integrated samples before clustering; latent donor-centering provides a diagnosis-blind Python approximation to reduce sample-specific clustering.', 'important_limitation': 'This is not an exact reproduction of Seurat anchor integration.'}

    lock = {'dataset': 'GSE157827', 'step': '9D.1', 'version': 'v1', 'created_utc': datetime.now(timezone.utc).isoformat(), 'parent_clustering_lock': str(STEP9D_LOCK_PATH), 'donor_correction': donor_correction, 'expression_examined': False, 'diagnosis_used': False, 'CREB5_queried': False, 'differential_expression_performed': False}

    with open(STEP9D1_LOCK_PATH, 'w', encoding='utf-8') as handle:
        json.dump(lock, handle, indent=2)

    print('=' * 76)

    print('GSE157827 STEP 9D.1 DONOR-CORRECTION RULE LOCKED')

    print('=' * 76)

    print('Correction stage: latent 50-dimensional space')

    print('Correction method: donor-centering + global mean restoration')

    print('Components used for clustering afterward: 20')

    print('\nDiagnosis used: NO')

    print('CREB5 queried: NO')

    print('Expression examined: NO')

    print('Differential expression performed: NO')

    print(f'\nLock saved:\n{STEP9D1_LOCK_PATH}')


In [ ]:
if RUN_PIPELINE:
    from pathlib import Path

    from datetime import datetime, timezone

    import pandas as pd

    import numpy as np

    import scipy.sparse as sp

    from sklearn.preprocessing import StandardScaler

    from sklearn.decomposition import TruncatedSVD

    from sklearn.cluster import MiniBatchKMeans

    from sklearn.utils.sparsefuncs import inplace_csr_row_scale

    import sklearn

    import json

    import gc

    PROJECT_DIR = Path('/content/drive/MyDrive/AD_Astrocyte_Paper_01')

    VALIDATION_DIR = PROJECT_DIR / 'GSE157827_confirmatory_validation'

    STEP8_DIR = VALIDATION_DIR / 'GSE157827_STEP8_raw_singlet_matrix_v1_1'

    OBS_PATH = VALIDATION_DIR / 'GSE157827_STEP8_singlet_obs_v1_1.csv.gz'

    VAR_PATH = VALIDATION_DIR / 'GSE157827_STEP8_feature_metadata_v1_1.csv.gz'

    PARTITION_MANIFEST_PATH = VALIDATION_DIR / 'GSE157827_STEP8_matrix_partition_manifest_v1_1.csv'

    STEP9D_LOCK_PATH = VALIDATION_DIR / 'GSE157827_STEP9D_CLUSTERING_PLAN_LOCK_v1.json'

    STEP9D1_LOCK_PATH = VALIDATION_DIR / 'GSE157827_STEP9D1_DONOR_CORRECTION_LOCK_v1.json'

    HVG_PATH = VALIDATION_DIR / 'GSE157827_STEP9E_HVG1000_v1.csv'

    CLUSTER_PATH = VALIDATION_DIR / 'GSE157827_STEP9E_43cluster_assignments_v1.csv.gz'

    CLUSTER_SUMMARY_PATH = VALIDATION_DIR / 'GSE157827_STEP9E_cluster_sizes_v1.csv'

    LATENT_PATH = VALIDATION_DIR / 'GSE157827_STEP9E_corrected_latent50_v1.npy'

    SVD_VARIANCE_PATH = VALIDATION_DIR / 'GSE157827_STEP9E_SVD_explained_variance_v1.csv'

    STEP9E_LOCK_PATH = VALIDATION_DIR / 'GSE157827_STEP9E_CLUSTERING_LOCK_v1.json'

    EXPECTED_NUCLEI = 159144

    EXPECTED_FEATURES = 33538

    EXPECTED_DONORS = 21

    N_HVG = 1000

    N_COMPONENTS = 50

    N_CLUSTER_COMPONENTS = 20

    N_CLUSTERS = 43

    RANDOM_SEED = 20260912

    for path in [OBS_PATH, VAR_PATH, PARTITION_MANIFEST_PATH, STEP9D_LOCK_PATH, STEP9D1_LOCK_PATH]:
        if not path.exists():
            raise FileNotFoundError(f'Required input missing:\n{path}')

    with open(STEP9D_LOCK_PATH, 'r', encoding='utf-8') as handle:
        step9d = json.load(handle)

    with open(STEP9D1_LOCK_PATH, 'r', encoding='utf-8') as handle:
        step9d1 = json.load(handle)

    prohibited_target_genes = set(step9d['prohibited_target_genes'])

    obs = pd.read_csv(OBS_PATH, dtype={'nucleus_id': str, 'GSM': str, 'barcode': str, 'matrix_partition': str})

    var = pd.read_csv(VAR_PATH, dtype=str)

    partitions = pd.read_csv(PARTITION_MANIFEST_PATH, dtype={'GSM': str, 'partition_file': str})

    if len(obs) != EXPECTED_NUCLEI:
        raise RuntimeError(f'Expected {EXPECTED_NUCLEI:,} nuclei; found {len(obs):,}.')

    if len(var) != EXPECTED_FEATURES:
        raise RuntimeError(f'Expected {EXPECTED_FEATURES:,} features; found {len(var):,}.')

    if len(partitions) != EXPECTED_DONORS:
        raise RuntimeError(f'Expected {EXPECTED_DONORS} donor partitions; found {len(partitions)}.')

    if 'diagnosis' in obs.columns:
        raise RuntimeError('Diagnosis unexpectedly present in clustering input.')

    if 'gene_symbol' not in var.columns:
        raise RuntimeError('gene_symbol missing from feature metadata.')


In [ ]:
if RUN_PIPELINE:
    print('=' * 72)

    print('GSE157827 STEP 9E STARTED')

    print('=' * 72)

    print('sklearn:', sklearn.__version__)

    print(f'Nuclei: {len(obs):,}')

    print(f'Features: {len(var):,}')

    print(f'Donors: {len(partitions)}')

    print('\nDiagnosis loaded: NO')

    print('CREB5 eligible for HVG selection: NO')

    gene_symbols = var['gene_symbol'].fillna('').astype(str)

    mitochondrial_mask = gene_symbols.str.upper().str.startswith('MT-').to_numpy()

    target_mask = gene_symbols.isin(prohibited_target_genes).to_numpy()

    eligible_mask = ~mitochondrial_mask & ~target_mask

    eligible_indices = np.flatnonzero(eligible_mask)

    if len(eligible_indices) <= N_HVG:
        raise RuntimeError('Too few eligible genes for 1000-HVG selection.')

    if target_mask.sum() == 0:
        raise RuntimeError('None of the prohibited target symbols were found in the feature table. Stop and inspect feature naming.')

    print(f'\nMitochondrial features excluded: {int(mitochondrial_mask.sum())}')

    print(f'Locked target features excluded: {int(target_mask.sum())}')

    print(f'Eligible HVG universe: {len(eligible_indices):,}')

    n_eligible = len(eligible_indices)

    gene_sum = np.zeros(n_eligible, dtype=np.float64)

    gene_sum_sq = np.zeros(n_eligible, dtype=np.float64)

    observed_nuclei = 0

    print('\nPASS 1: calculating diagnosis-blind log(CP10K) gene variances...')

    for donor_number, partition in enumerate(partitions.itertuples(index=False), start=1):
        gsm = str(partition.GSM)
        matrix_path = Path(partition.partition_file)
        if not matrix_path.exists():
            raise FileNotFoundError(f'Missing partition:\n{matrix_path}')
        X = sp.load_npz(matrix_path).tocsr()
        if X.shape[1] != EXPECTED_FEATURES:
            raise RuntimeError(f'{gsm}: unexpected feature count.')
        donor_obs = obs.loc[obs['GSM'] == gsm]
        if X.shape[0] != len(donor_obs):
            raise RuntimeError(f'{gsm}: matrix/observation row mismatch.')
        library_size = np.asarray(X.sum(axis=1)).ravel().astype(np.float32)
        if np.any(library_size <= 0):
            raise RuntimeError(f'{gsm}: zero-count nucleus found.')
        Xe = X[:, eligible_indices].astype(np.float32).tocsr()
        row_scale = (10000.0 / library_size).astype(np.float32)
        inplace_csr_row_scale(Xe, row_scale)
        np.log1p(Xe.data, out=Xe.data)
        gene_sum += np.bincount(Xe.indices, weights=Xe.data, minlength=n_eligible)
        gene_sum_sq += np.bincount(Xe.indices, weights=Xe.data * Xe.data, minlength=n_eligible)
        observed_nuclei += X.shape[0]
        print(f'[{donor_number:02d}/21] {gsm}: {X.shape[0]:,} nuclei')
        del X
        del Xe
        del library_size
        del row_scale
        gc.collect()

    if observed_nuclei != EXPECTED_NUCLEI:
        raise RuntimeError(f'Variance pass saw {observed_nuclei:,} nuclei instead of {EXPECTED_NUCLEI:,}.')


In [ ]:
if RUN_PIPELINE:
    gene_mean = gene_sum / EXPECTED_NUCLEI

    gene_variance = gene_sum_sq / EXPECTED_NUCLEI - gene_mean ** 2

    gene_variance = np.maximum(gene_variance, 0.0)

    top_local_indices = np.argpartition(gene_variance, -N_HVG)[-N_HVG:]

    top_local_indices = top_local_indices[np.argsort(gene_variance[top_local_indices])[::-1]]

    hvg_feature_indices = eligible_indices[top_local_indices]

    hvg_table = var.iloc[hvg_feature_indices].copy()

    hvg_table.insert(0, 'feature_index', hvg_feature_indices)

    hvg_table.insert(1, 'HVG_rank', np.arange(1, N_HVG + 1))

    hvg_table['mean_logCP10K'] = gene_mean[top_local_indices]

    hvg_table['variance_logCP10K'] = gene_variance[top_local_indices]

    if hvg_table['gene_symbol'].isin(prohibited_target_genes).any():
        raise RuntimeError('Locked target gene entered HVG set.')

    if hvg_table['gene_symbol'].fillna('').str.upper().str.startswith('MT-').any():
        raise RuntimeError('Mitochondrial gene entered HVG set.')

    hvg_table.to_csv(HVG_PATH, index=False)

    print('\nHVG selection complete.')

    print(f'Selected features: {len(hvg_table):,}')

    print('Locked target genes in HVGs: 0')

    print('\nPASS 2: building 159,144 x 1,000 sparse HVG matrix...')

    hvg_chunks = []

    row_gsms = []

    for donor_number, partition in enumerate(partitions.itertuples(index=False), start=1):
        gsm = str(partition.GSM)
        X = sp.load_npz(Path(partition.partition_file)).tocsr()
        library_size = np.asarray(X.sum(axis=1)).ravel().astype(np.float32)
        Xh = X[:, hvg_feature_indices].astype(np.float32).tocsr()
        row_scale = (10000.0 / library_size).astype(np.float32)
        inplace_csr_row_scale(Xh, row_scale)
        np.log1p(Xh.data, out=Xh.data)
        hvg_chunks.append(Xh)
        row_gsms.extend([gsm] * Xh.shape[0])
        print(f'[{donor_number:02d}/21] {gsm}: {Xh.shape[0]:,} nuclei')
        del X
        del library_size
        del row_scale
        gc.collect()

    X_hvg = sp.vstack(hvg_chunks, format='csr')

    del hvg_chunks

    gc.collect()

    if X_hvg.shape != (EXPECTED_NUCLEI, N_HVG):
        raise RuntimeError(f'Unexpected HVG matrix shape: {X_hvg.shape}')

    row_gsms = np.asarray(row_gsms, dtype=object)

    if not np.array_equal(row_gsms, obs['GSM'].to_numpy()):
        raise RuntimeError('HVG matrix donor row order does not match STEP 8 observation table.')

    print(f'\nHVG matrix shape: {X_hvg.shape}')

    print(f'HVG matrix nonzeros: {X_hvg.nnz:,}')

    print('\nSparse feature scaling...')

    scaler = StandardScaler(with_mean=False, with_std=True, copy=False)

    X_hvg = scaler.fit_transform(X_hvg)

    print('\nCalculating 50-dimensional latent representation...')

    svd = TruncatedSVD(n_components=N_COMPONENTS, algorithm='randomized', n_iter=7, random_state=RANDOM_SEED)

    latent = svd.fit_transform(X_hvg).astype(np.float32)

    if latent.shape != (EXPECTED_NUCLEI, N_COMPONENTS):
        raise RuntimeError(f'Unexpected latent shape: {latent.shape}')


In [ ]:
if RUN_PIPELINE:
    variance_table = pd.DataFrame({'component': np.arange(1, N_COMPONENTS + 1), 'explained_variance_ratio': svd.explained_variance_ratio_})

    variance_table.to_csv(SVD_VARIANCE_PATH, index=False)

    print(f'Total explained variance ratio (50 components): {svd.explained_variance_ratio_.sum():.4f}')

    del X_hvg

    gc.collect()

    print('\nApplying diagnosis-blind donor correction in latent space...')

    global_mean = latent.mean(axis=0, dtype=np.float64).astype(np.float32)

    corrected_latent = latent.copy()

    for gsm in partitions['GSM'].astype(str):
        mask = row_gsms == gsm
        donor_mean = latent[mask].mean(axis=0, dtype=np.float64).astype(np.float32)
        corrected_latent[mask] = latent[mask] - donor_mean + global_mean

    del latent

    gc.collect()

    np.save(LATENT_PATH, corrected_latent, allow_pickle=False)

    print('\nClustering 159,144 nuclei into 43 unsupervised clusters...')

    cluster_input = corrected_latent[:, :N_CLUSTER_COMPONENTS]

    kmeans = MiniBatchKMeans(n_clusters=N_CLUSTERS, random_state=RANDOM_SEED, n_init=20, batch_size=4096, max_iter=300, reassignment_ratio=0.01)

    cluster_labels = kmeans.fit_predict(cluster_input)

    if len(np.unique(cluster_labels)) != N_CLUSTERS:
        raise RuntimeError('Clustering did not produce exactly 43 clusters.')

    cluster_assignments = obs[['nucleus_id', 'GSM', 'barcode', 'matrix_partition', 'partition_row', 'global_row']].copy()

    cluster_assignments['cluster_id'] = cluster_labels.astype(np.int16)

    cluster_assignments.to_csv(CLUSTER_PATH, index=False, compression={'method': 'gzip', 'compresslevel': 6})

    cluster_summary = cluster_assignments['cluster_id'].value_counts().sort_index().rename_axis('cluster_id').reset_index(name='nuclei')

    cluster_summary['percent_of_all_nuclei'] = 100.0 * cluster_summary['nuclei'] / EXPECTED_NUCLEI

    cluster_summary.to_csv(CLUSTER_SUMMARY_PATH, index=False)

    if len(cluster_assignments) != EXPECTED_NUCLEI:
        raise RuntimeError('Cluster assignment count mismatch.')

    if cluster_assignments['nucleus_id'].duplicated().any():
        raise RuntimeError('Duplicate nucleus IDs in clustering output.')

    if cluster_summary['nuclei'].sum() != EXPECTED_NUCLEI:
        raise RuntimeError('Cluster sizes do not sum to 159,144.')

    step9e_lock = {'dataset': 'GSE157827', 'step': '9E', 'version': 'v1', 'created_utc': datetime.now(timezone.utc).isoformat(), 'input_nuclei': EXPECTED_NUCLEI, 'input_features': EXPECTED_FEATURES, 'eligible_HVG_features': int(len(eligible_indices)), 'selected_HVGs': N_HVG, 'components_computed': N_COMPONENTS, 'components_used_for_clustering': N_CLUSTER_COMPONENTS, 'clusters': N_CLUSTERS, 'random_seed': RANDOM_SEED, 'normalization': 'log1p(CP10K)', 'feature_scaling': 'StandardScaler(with_mean=False)', 'dimension_reduction': 'TruncatedSVD', 'donor_correction': 'Subtract donor latent centroid and restore global latent centroid', 'clustering': 'MiniBatchKMeans', 'HVG_file': str(HVG_PATH), 'latent_file': str(LATENT_PATH), 'cluster_assignment_file': str(CLUSTER_PATH), 'cluster_summary_file': str(CLUSTER_SUMMARY_PATH), 'prohibited_target_genes': sorted(prohibited_target_genes), 'diagnosis_loaded': False, 'CREB5_queried': False, 'target_gene_expression_statistics_computed': False, 'cell_type_annotation_performed': False, 'differential_expression_performed': False}

    with open(STEP9E_LOCK_PATH, 'w', encoding='utf-8') as handle:
        json.dump(step9e_lock, handle, indent=2)

    print('\n' + '=' * 76)

    print('GSE157827 STEP 9E COMPLETE')

    print('=' * 76)

    print(f'Nuclei clustered: {len(cluster_assignments):,}')

    print(f'HVGs: {N_HVG:,}')

    print(f'Components computed: {N_COMPONENTS}')

    print(f'Components used for clustering: {N_CLUSTER_COMPONENTS}')

    print(f'Clusters: {cluster_summary.shape[0]}')

    print('\nCluster sizes:')

    print(cluster_summary.to_string(index=False, float_format=lambda x: f'{x:.3f}'))

    print('\nDiagnosis loaded: NO')

    print('CREB5 queried: NO')

    print('Locked target-gene expression statistics computed: NO')

    print('Cell-type labels assigned: NO')

    print('Differential expression performed: NO')

    print(f'\nHVG table:\n{HVG_PATH}')

    print(f'\nCluster assignments:\n{CLUSTER_PATH}')

    print(f'\nCorrected latent matrix:\n{LATENT_PATH}')

    print(f'\nClustering lock:\n{STEP9E_LOCK_PATH}')


In [ ]:
if RUN_PIPELINE:
    from pathlib import Path

    from datetime import datetime, timezone

    import pandas as pd

    import numpy as np

    import scipy.sparse as sp

    import json

    import gc

    PROJECT_DIR = Path('/content/drive/MyDrive/AD_Astrocyte_Paper_01')

    VALIDATION_DIR = PROJECT_DIR / 'GSE157827_confirmatory_validation'

    VAR_PATH = VALIDATION_DIR / 'GSE157827_STEP8_feature_metadata_v1_1.csv.gz'

    PARTITION_MANIFEST_PATH = VALIDATION_DIR / 'GSE157827_STEP8_matrix_partition_manifest_v1_1.csv'

    CLUSTER_PATH = VALIDATION_DIR / 'GSE157827_STEP9E_43cluster_assignments_v1.csv.gz'

    STEP9D_LOCK_PATH = VALIDATION_DIR / 'GSE157827_STEP9D_CLUSTERING_PLAN_LOCK_v1.json'

    STEP9E_LOCK_PATH = VALIDATION_DIR / 'GSE157827_STEP9E_CLUSTERING_LOCK_v1.json'

    CLUSTER_MARKER_PATH = VALIDATION_DIR / 'GSE157827_STEP9F_cluster_marker_profiles_v1.csv'

    CLUSTER_LABEL_PATH = VALIDATION_DIR / 'GSE157827_STEP9F_cluster_celltype_labels_v1.csv'

    NUCLEUS_LABEL_PATH = VALIDATION_DIR / 'GSE157827_STEP9F_cluster_based_nucleus_labels_v1.csv.gz'

    ASTROCYTE_PATH = VALIDATION_DIR / 'GSE157827_STEP9F_primary_astrocyte_manifest_v1.csv.gz'

    ASTRO_DONOR_PATH = VALIDATION_DIR / 'GSE157827_STEP9F_primary_astrocyte_counts_per_donor_v1.csv'

    STEP9F_LOCK_PATH = VALIDATION_DIR / 'GSE157827_STEP9F_ASTROCYTE_IDENTITY_LOCK_v1.json'

    EXPECTED_NUCLEI = 159144

    EXPECTED_CLUSTERS = 43

    EXPECTED_DONORS = 21

    for path in [VAR_PATH, PARTITION_MANIFEST_PATH, CLUSTER_PATH, STEP9D_LOCK_PATH, STEP9E_LOCK_PATH]:
        if not path.exists():
            raise FileNotFoundError(f'Required file missing:\n{path}')

    with open(STEP9D_LOCK_PATH, 'r', encoding='utf-8') as handle:
        step9d = json.load(handle)

    marker_panels = step9d['cluster_marker_panels']

    prohibited_targets = set(step9d['prohibited_target_genes'])

    annotation_rule = {'marker_expression': 'mean nucleus-level log1p(CP10K) within each cluster', 'marker_standardization': 'z-score each marker gene across the 43 cluster means', 'lineage_score': 'arithmetic mean of marker z-scores within lineage', 'assigned_lineage': 'lineage with uniquely highest lineage score', 'minimum_top_lineage_score': 0.0, 'minimum_positive_z_markers': 2, 'unknown_rule': 'Assign Unknown if the top lineage score is <= 0, if fewer than two markers in the top lineage have positive z-score, or if the highest lineage score is tied.', 'composition_used_for_assignment': False, 'step9b_labels_used': False, 'diagnosis_used': False, 'CREB5_used': False}

    all_marker_genes = {gene for genes in marker_panels.values() for gene in genes}

    target_overlap = all_marker_genes & prohibited_targets

    if target_overlap:
        raise RuntimeError(f'Locked target gene found in annotation markers: {sorted(target_overlap)}')

    clusters = pd.read_csv(CLUSTER_PATH, dtype={'nucleus_id': str, 'GSM': str, 'barcode': str, 'matrix_partition': str})

    var = pd.read_csv(VAR_PATH, dtype=str)

    partitions = pd.read_csv(PARTITION_MANIFEST_PATH, dtype={'GSM': str, 'partition_file': str})

    if len(clusters) != EXPECTED_NUCLEI:
        raise RuntimeError(f'Expected {EXPECTED_NUCLEI:,} nuclei; found {len(clusters):,}.')

    if clusters['cluster_id'].nunique() != EXPECTED_CLUSTERS:
        raise RuntimeError('STEP 9E does not contain exactly 43 clusters.')

    if len(partitions) != EXPECTED_DONORS:
        raise RuntimeError('Partition manifest does not contain 21 donors.')

    if 'diagnosis' in clusters.columns:
        raise RuntimeError('Diagnosis unexpectedly present.')

    print('=' * 72)

    print('GSE157827 STEP 9F STARTED')

    print('=' * 72)

    print(f'Nuclei: {len(clusters):,}')

    print(f"Clusters: {clusters['cluster_id'].nunique()}")

    print(f'Donors: {len(partitions)}')

    print('Diagnosis loaded: NO')

    print('CREB5 queried: NO')


In [ ]:
if RUN_PIPELINE:
    if 'gene_symbol' not in var.columns:
        raise RuntimeError('gene_symbol is absent from feature metadata.')

    symbol_to_indices = {}

    for idx, symbol in enumerate(var['gene_symbol'].fillna('').astype(str)):
        if symbol:
            symbol_to_indices.setdefault(symbol, []).append(idx)

    marker_genes = sorted(all_marker_genes)

    missing_markers = [gene for gene in marker_genes if gene not in symbol_to_indices]

    if missing_markers:
        print('\nMarkers absent from feature table:')
        for gene in missing_markers:
            print(' -', gene)

    cluster_ids = np.arange(EXPECTED_CLUSTERS, dtype=np.int16)

    cluster_n = np.zeros(EXPECTED_CLUSTERS, dtype=np.int64)

    marker_log_sums = {gene: np.zeros(EXPECTED_CLUSTERS, dtype=np.float64) for gene in marker_genes}

    for donor_number, partition in enumerate(partitions.itertuples(index=False), start=1):
        gsm = str(partition.GSM)
        matrix_path = Path(partition.partition_file)
        if not matrix_path.exists():
            raise FileNotFoundError(f'Missing matrix partition:\n{matrix_path}')
        donor_clusters = clusters.loc[clusters['GSM'] == gsm].sort_values('partition_row').reset_index(drop=True)
        X = sp.load_npz(matrix_path).tocsr()
        if X.shape[0] != len(donor_clusters):
            raise RuntimeError(f'{gsm}: matrix/cluster row mismatch.')
        expected_partition_rows = np.arange(X.shape[0], dtype=np.int64)
        if not np.array_equal(donor_clusters['partition_row'].to_numpy(), expected_partition_rows):
            raise RuntimeError(f'{gsm}: partition-row order mismatch.')
        donor_cluster_ids = donor_clusters['cluster_id'].to_numpy(dtype=np.int16)
        cluster_n += np.bincount(donor_cluster_ids, minlength=EXPECTED_CLUSTERS)
        library_size = np.asarray(X.sum(axis=1)).ravel().astype(np.float64)
        if np.any(library_size <= 0):
            raise RuntimeError(f'{gsm}: zero-count nucleus found.')
        for gene in marker_genes:
            indices = symbol_to_indices.get(gene, [])
            if len(indices) == 0:
                continue
            if len(indices) == 1:
                raw = X[:, indices[0]].toarray().ravel()
            else:
                raw = np.asarray(X[:, indices].sum(axis=1)).ravel()
            raw = raw.astype(np.float64, copy=False)
            log_cp10k = np.log1p(raw / library_size * 10000.0)
            marker_log_sums[gene] += np.bincount(donor_cluster_ids, weights=log_cp10k, minlength=EXPECTED_CLUSTERS)
        print(f'[{donor_number:02d}/21] {gsm}: {X.shape[0]:,} nuclei')
        del X
        del library_size
        del donor_clusters
        del donor_cluster_ids
        gc.collect()

    if cluster_n.sum() != EXPECTED_NUCLEI:
        raise RuntimeError('Cluster-cell accumulator does not sum to 159,144.')

    if np.any(cluster_n == 0):
        raise RuntimeError('At least one cluster contains zero nuclei.')

    marker_mean = {}


In [ ]:
if RUN_PIPELINE:
    for gene in marker_genes:
        marker_mean[gene] = marker_log_sums[gene] / cluster_n

    marker_z = {}

    for gene in marker_genes:
        values = marker_mean[gene]
        sd = values.std(ddof=0)
        if sd == 0:
            marker_z[gene] = np.zeros(EXPECTED_CLUSTERS, dtype=np.float64)
        else:
            marker_z[gene] = (values - values.mean()) / sd

    lineages = list(marker_panels.keys())

    lineage_scores = np.zeros((EXPECTED_CLUSTERS, len(lineages)), dtype=np.float64)

    lineage_positive_markers = np.zeros((EXPECTED_CLUSTERS, len(lineages)), dtype=np.int16)

    for lineage_idx, lineage in enumerate(lineages):
        available = [gene for gene in marker_panels[lineage] if gene in marker_z]
        if len(available) < 2:
            raise RuntimeError(f'Fewer than 2 markers available for {lineage}.')
        Z = np.column_stack([marker_z[gene] for gene in available])
        lineage_scores[:, lineage_idx] = Z.mean(axis=1)
        lineage_positive_markers[:, lineage_idx] = (Z > 0).sum(axis=1)

    top_index = np.argmax(lineage_scores, axis=1)

    top_score = lineage_scores[np.arange(EXPECTED_CLUSTERS), top_index]

    sorted_scores = np.sort(lineage_scores, axis=1)

    second_score = sorted_scores[:, -2]

    score_margin = top_score - second_score

    number_at_top = np.sum(np.isclose(lineage_scores, top_score[:, None], rtol=0, atol=1e-12), axis=1)

    unique_top = number_at_top == 1

    top_positive_markers = lineage_positive_markers[np.arange(EXPECTED_CLUSTERS), top_index]

    accepted = unique_top & (top_score > 0) & (top_positive_markers >= 2)

    lineage_array = np.asarray(lineages, dtype=object)

    provisional_label = lineage_array[top_index]

    cluster_label = np.where(accepted, provisional_label, 'Unknown')

    cluster_table = pd.DataFrame({'cluster_id': cluster_ids, 'nuclei': cluster_n, 'percent_of_dataset': 100.0 * cluster_n / EXPECTED_NUCLEI, 'assigned_cell_type': cluster_label, 'top_lineage_score': top_score, 'second_lineage_score': second_score, 'score_margin': score_margin, 'top_lineage_positive_markers': top_positive_markers})

    for lineage_idx, lineage in enumerate(lineages):
        safe = lineage.lower().replace(' ', '_')
        cluster_table[f'score_{safe}'] = lineage_scores[:, lineage_idx]
        cluster_table[f'positive_markers_{safe}'] = lineage_positive_markers[:, lineage_idx]

    cluster_table.to_csv(CLUSTER_LABEL_PATH, index=False)

    profile = pd.DataFrame({'cluster_id': cluster_ids, 'nuclei': cluster_n})

    for gene in marker_genes:
        profile[f'mean_{gene}'] = marker_mean[gene]
        profile[f'z_{gene}'] = marker_z[gene]

    profile.to_csv(CLUSTER_MARKER_PATH, index=False)

    label_map = dict(zip(cluster_table['cluster_id'], cluster_table['assigned_cell_type']))

    nucleus_labels = clusters.copy()

    nucleus_labels['cluster_cell_type'] = nucleus_labels['cluster_id'].map(label_map)

    if nucleus_labels['cluster_cell_type'].isna().any():
        raise RuntimeError('Missing propagated cluster label.')

    nucleus_labels.to_csv(NUCLEUS_LABEL_PATH, index=False, compression={'method': 'gzip', 'compresslevel': 6})

    astrocytes = nucleus_labels.loc[nucleus_labels['cluster_cell_type'] == 'Astrocyte', ['nucleus_id', 'GSM', 'barcode', 'matrix_partition', 'partition_row', 'global_row', 'cluster_id']].copy().reset_index(drop=True)

    astrocytes.to_csv(ASTROCYTE_PATH, index=False, compression={'method': 'gzip', 'compresslevel': 6})

    total_by_donor = clusters.groupby('GSM').size().rename('total_locked_singlets')

    astro_by_donor = astrocytes.groupby('GSM').size().rename('cluster_based_astrocytes')

    donor_summary = total_by_donor.to_frame().join(astro_by_donor, how='left').fillna({'cluster_based_astrocytes': 0}).reset_index()

    donor_summary['cluster_based_astrocytes'] = donor_summary['cluster_based_astrocytes'].astype(int)


In [ ]:
if RUN_PIPELINE:
    donor_summary['astrocyte_percent'] = 100.0 * donor_summary['cluster_based_astrocytes'] / donor_summary['total_locked_singlets']

    donor_summary['primary_eligible_ge20'] = donor_summary['cluster_based_astrocytes'] >= 20

    donor_summary['sensitivity_eligible_ge50'] = donor_summary['cluster_based_astrocytes'] >= 50

    donor_summary.to_csv(ASTRO_DONOR_PATH, index=False)

    composition = nucleus_labels['cluster_cell_type'].value_counts().rename_axis('cell_type').reset_index(name='nuclei')

    composition['percent'] = 100.0 * composition['nuclei'] / EXPECTED_NUCLEI

    step9f_lock = {'dataset': 'GSE157827', 'step': '9F', 'created_utc': datetime.now(timezone.utc).isoformat(), 'annotation_unit': 'whole STEP 9E cluster', 'annotation_rule': annotation_rule, 'clusters_total': EXPECTED_CLUSTERS, 'astrocyte_clusters': [int(x) for x in cluster_table.loc[cluster_table['assigned_cell_type'] == 'Astrocyte', 'cluster_id'].tolist()], 'primary_astrocyte_nuclei': int(len(astrocytes)), 'donors_primary_eligible_ge20': int(donor_summary['primary_eligible_ge20'].sum()), 'donors_sensitivity_eligible_ge50': int(donor_summary['sensitivity_eligible_ge50'].sum()), 'step9b_labels_used': False, 'published_composition_used_for_assignment': False, 'diagnosis_loaded': False, 'CREB5_queried': False, 'locked_target_genes_queried': False, 'differential_expression_performed': False, 'primary_astrocyte_manifest': str(ASTROCYTE_PATH), 'cluster_annotation_table': str(CLUSTER_LABEL_PATH)}

    with open(STEP9F_LOCK_PATH, 'w', encoding='utf-8') as handle:
        json.dump(step9f_lock, handle, indent=2)

    print('\n' + '=' * 78)

    print('GSE157827 STEP 9F COMPLETE')

    print('=' * 78)

    print('\nCluster-level assignments:')

    print(cluster_table[['cluster_id', 'nuclei', 'assigned_cell_type', 'top_lineage_score', 'second_lineage_score', 'score_margin', 'top_lineage_positive_markers']].to_string(index=False, float_format=lambda x: f'{x:.3f}'))

    print('\nBroad cluster-based composition:')

    print(composition.to_string(index=False, float_format=lambda x: f'{x:.2f}'))

    astro_clusters = cluster_table.loc[cluster_table['assigned_cell_type'] == 'Astrocyte', 'cluster_id'].tolist()

    print('\nAstrocyte clusters:', astro_clusters)

    print(f'Primary astrocyte nuclei: {len(astrocytes):,}')

    print('\nAstrocytes per donor:')

    print(donor_summary.to_string(index=False, float_format=lambda x: f'{x:.2f}'))

    print(f"\nDonors >=20 astrocytes: {int(donor_summary['primary_eligible_ge20'].sum())}/{EXPECTED_DONORS}")

    print(f"Donors >=50 astrocytes: {int(donor_summary['sensitivity_eligible_ge50'].sum())}/{EXPECTED_DONORS}")

    print('\nDiagnosis loaded: NO')

    print('CREB5 queried: NO')

    print('Locked targets queried: NO')

    print('Differential expression performed: NO')

    print(f'\nCluster labels:\n{CLUSTER_LABEL_PATH}')

    print(f'\nMarker-profile audit:\n{CLUSTER_MARKER_PATH}')

    print(f'\nPrimary astrocyte manifest:\n{ASTROCYTE_PATH}')

    print(f'\nSTEP 9F identity lock:\n{STEP9F_LOCK_PATH}')


In [ ]:
if RUN_PIPELINE:
    from pathlib import Path

    from datetime import datetime, timezone

    import pandas as pd

    import json

    PROJECT_DIR = Path('/content/drive/MyDrive/AD_Astrocyte_Paper_01')

    VALIDATION_DIR = PROJECT_DIR / 'GSE157827_confirmatory_validation'

    CLUSTER_LABEL_PATH = VALIDATION_DIR / 'GSE157827_STEP9F_cluster_celltype_labels_v1.csv'

    ASTROCYTE_PATH = VALIDATION_DIR / 'GSE157827_STEP9F_primary_astrocyte_manifest_v1.csv.gz'

    ASTRO_DONOR_PATH = VALIDATION_DIR / 'GSE157827_STEP9F_primary_astrocyte_counts_per_donor_v1.csv'

    STEP9F_LOCK_PATH = VALIDATION_DIR / 'GSE157827_STEP9F_ASTROCYTE_IDENTITY_LOCK_v1.json'

    STEP9G_LOCK_PATH = VALIDATION_DIR / 'GSE157827_STEP9G_PRIMARY_ASTROCYTE_ACCEPTANCE_LOCK_v1.json'

    for path in [CLUSTER_LABEL_PATH, ASTROCYTE_PATH, ASTRO_DONOR_PATH, STEP9F_LOCK_PATH]:
        if not path.exists():
            raise FileNotFoundError(f'Missing required STEP 9F file:\n{path}')

    cluster_table = pd.read_csv(CLUSTER_LABEL_PATH)

    astrocytes = pd.read_csv(ASTROCYTE_PATH, dtype={'nucleus_id': str, 'GSM': str, 'barcode': str})

    donor_summary = pd.read_csv(ASTRO_DONOR_PATH, dtype={'GSM': str})

    astro_clusters = cluster_table.loc[cluster_table['assigned_cell_type'] == 'Astrocyte'].sort_values('cluster_id').copy()

    expected_clusters = [3, 10, 17, 28, 29]

    observed_clusters = astro_clusters['cluster_id'].astype(int).tolist()

    if observed_clusters != expected_clusters:
        raise RuntimeError(f'Unexpected astrocyte clusters: {observed_clusters}')

    if len(astrocytes) != 17067:
        raise RuntimeError(f'Expected 17,067 astrocytes; found {len(astrocytes):,}.')

    if donor_summary['GSM'].nunique() != 21:
        raise RuntimeError('Expected astrocytes from all 21 donors.')

    if donor_summary['cluster_based_astrocytes'].min() < 50:
        raise RuntimeError('At least one donor has fewer than 50 accepted astrocytes.')

    if not (astro_clusters['top_lineage_positive_markers'] == 8).all():
        raise RuntimeError('Not all accepted astrocyte clusters have all 8 frozen markers positive.')

    minimum_astrocyte_score_margin = float(astro_clusters['score_margin'].min())

    minimum_astrocyte_top_score = float(astro_clusters['top_lineage_score'].min())

    acceptance_lock = {'dataset': 'GSE157827', 'step': '9G', 'version': 'v1', 'created_utc': datetime.now(timezone.utc).isoformat(), 'decision': 'Accept STEP 9F cluster-defined astrocyte population for primary donor-level pseudobulk.', 'accepted_astrocyte_clusters': observed_clusters, 'accepted_astrocyte_nuclei': int(len(astrocytes)), 'donors represented': int(donor_summary['GSM'].nunique()), 'minimum_astrocytes_per_donor': int(donor_summary['cluster_based_astrocytes'].min()), 'all_accepted_clusters_have_all_8_astrocyte_markers_positive': True, 'minimum_observed_astrocyte_cluster_score_margin': minimum_astrocyte_score_margin, 'minimum_observed_astrocyte_top_lineage_score': minimum_astrocyte_top_score, 'step9b_per_nucleus_annotation_used': False, 'non_astrocyte_labels_claimed_as_exact_reproduction_of_Lau': False, 'published_cell_proportions_used_to_assign_astrocytes': False, 'astrocyte_manifest': str(ASTROCYTE_PATH), 'diagnosis_used_for_annotation': False, 'CREB5_queried': False, 'differential_expression_performed': False, 'primary_astrocyte_identity_now_frozen': True}

    with open(STEP9G_LOCK_PATH, 'w', encoding='utf-8') as handle:
        json.dump(acceptance_lock, handle, indent=2)

    print('=' * 76)

    print('GSE157827 STEP 9G PRIMARY ASTROCYTES ACCEPTED')

    print('=' * 76)

    print('\nAccepted clusters:', observed_clusters)

    print(f'Accepted astrocyte nuclei: {len(astrocytes):,}')

    print(f"Donors represented: {donor_summary['GSM'].nunique()}/21")

    print('Minimum astrocytes in any donor:', int(donor_summary['cluster_based_astrocytes'].min()))

    print('All astrocyte clusters have 8/8 positive frozen markers: YES')

    print('Minimum astrocyte cluster score margin:', f'{minimum_astrocyte_score_margin:.3f}')

    print('\nSTEP 9B used: NO')

    print('Published proportions used to assign cells: NO')

    print('Diagnosis used for annotation: NO')

    print('CREB5 queried: NO')

    print('Differential expression performed: NO')

    print('\nPRIMARY ASTROCYTE IDENTITY FROZEN: YES')

    print(f'\nAcceptance lock:\n{STEP9G_LOCK_PATH}')


### Discarded pilot annotation

The original audit notebook also evaluated a preliminary per-nucleus marker classifier. It failed the prespecified composition-QC check and was not used for primary pseudobulk. The public workflow therefore proceeds directly to the frozen transcriptome-wide cluster-based annotation used in the paper; the discarded pilot remains documented in the untouched audit notebook.


## Primary donor-level astrocyte pseudobulk

Raw integer counts from the accepted astrocytes are summed donor by donor. No normalization or differential expression is performed at this stage.


In [ ]:
if RUN_PIPELINE:
    from pathlib import Path

    from datetime import datetime, timezone

    import pandas as pd

    import numpy as np

    import scipy.sparse as sp

    import json

    import gc

    PROJECT_DIR = Path('/content/drive/MyDrive/AD_Astrocyte_Paper_01')

    VALIDATION_DIR = PROJECT_DIR / 'GSE157827_confirmatory_validation'

    ASTROCYTE_PATH = VALIDATION_DIR / 'GSE157827_STEP9F_primary_astrocyte_manifest_v1.csv.gz'

    STEP9G_LOCK_PATH = VALIDATION_DIR / 'GSE157827_STEP9G_PRIMARY_ASTROCYTE_ACCEPTANCE_LOCK_v1.json'

    PARTITION_MANIFEST_PATH = VALIDATION_DIR / 'GSE157827_STEP8_matrix_partition_manifest_v1_1.csv'

    VAR_PATH = VALIDATION_DIR / 'GSE157827_STEP8_feature_metadata_v1_1.csv.gz'

    PSEUDOBULK_NPZ_PATH = VALIDATION_DIR / 'GSE157827_STEP10_astrocyte_pseudobulk_raw_counts_v1.npz'

    PSEUDOBULK_CSV_PATH = VALIDATION_DIR / 'GSE157827_STEP10_astrocyte_pseudobulk_raw_counts_v1.csv.gz'

    DONOR_MANIFEST_PATH = VALIDATION_DIR / 'GSE157827_STEP10_pseudobulk_donor_manifest_v1.csv'

    STEP10_LOCK_PATH = VALIDATION_DIR / 'GSE157827_STEP10_PSEUDOBULK_LOCK_v1.json'

    EXPECTED_ASTROCYTES = 17067

    EXPECTED_DONORS = 21

    EXPECTED_FEATURES = 33538

    for path in [ASTROCYTE_PATH, STEP9G_LOCK_PATH, PARTITION_MANIFEST_PATH, VAR_PATH]:
        if not path.exists():
            raise FileNotFoundError(f'Required input missing:\n{path}')

    with open(STEP9G_LOCK_PATH, 'r', encoding='utf-8') as handle:
        step9g = json.load(handle)

    if not step9g.get('primary_astrocyte_identity_now_frozen', False):
        raise RuntimeError('STEP 9G does not confirm frozen astrocyte identity.')

    astrocytes = pd.read_csv(ASTROCYTE_PATH, dtype={'nucleus_id': str, 'GSM': str, 'barcode': str, 'matrix_partition': str})

    if len(astrocytes) != EXPECTED_ASTROCYTES:
        raise RuntimeError(f'Expected {EXPECTED_ASTROCYTES:,} astrocytes; found {len(astrocytes):,}.')

    if astrocytes['nucleus_id'].duplicated().any():
        raise RuntimeError('Duplicate astrocyte nucleus IDs detected.')

    if astrocytes['GSM'].nunique() != EXPECTED_DONORS:
        raise RuntimeError('Frozen astrocytes do not represent all 21 donors.')

    print('=' * 72)

    print('GSE157827 STEP 10 STARTED')

    print('=' * 72)

    print(f'Frozen astrocytes: {len(astrocytes):,}')

    print(f"Donors represented: {astrocytes['GSM'].nunique()}")

    print('\nDiagnosis loaded: NO')

    print('CREB5 queried: NO')

    partitions = pd.read_csv(PARTITION_MANIFEST_PATH, dtype={'GSM': str, 'sample_label': str, 'partition_file': str})

    var = pd.read_csv(VAR_PATH, dtype=str)

    if len(partitions) != EXPECTED_DONORS:
        raise RuntimeError(f'Expected {EXPECTED_DONORS} donor partitions; found {len(partitions)}.')

    if len(var) != EXPECTED_FEATURES:
        raise RuntimeError(f'Expected {EXPECTED_FEATURES:,} features; found {len(var):,}.')

    if 'gene_id' not in var.columns:
        raise RuntimeError('gene_id missing from feature metadata.')

    if 'gene_symbol' not in var.columns:
        raise RuntimeError('gene_symbol missing from feature metadata.')

    if var['gene_id'].duplicated().any():
        raise RuntimeError('gene_id is not unique.')

    donor_order = partitions['GSM'].astype(str).tolist()


In [ ]:
if RUN_PIPELINE:
    if len(donor_order) != len(set(donor_order)):
        raise RuntimeError('Duplicate GSMs in partition manifest.')

    pseudobulk = np.zeros((EXPECTED_DONORS, EXPECTED_FEATURES), dtype=np.int64)

    donor_audit_rows = []

    for donor_index, partition in enumerate(partitions.itertuples(index=False)):
        gsm = str(partition.GSM)
        sample_label = str(partition.sample_label)
        matrix_path = Path(partition.partition_file)
        if not matrix_path.exists():
            raise FileNotFoundError(f'Raw-count partition missing:\n{matrix_path}')
        donor_astro = astrocytes.loc[astrocytes['GSM'] == gsm].sort_values('partition_row').copy()
        n_astro = len(donor_astro)
        if n_astro < 20:
            raise RuntimeError(f'{gsm}: only {n_astro} astrocytes; fails frozen primary donor rule.')
        X = sp.load_npz(matrix_path).tocsr()
        if X.shape[1] != EXPECTED_FEATURES:
            raise RuntimeError(f'{gsm}: feature count mismatch.')
        rows = donor_astro['partition_row'].to_numpy(dtype=np.int64)
        if np.any(rows < 0) or np.any(rows >= X.shape[0]):
            raise RuntimeError(f'{gsm}: invalid partition_row in frozen astrocyte manifest.')
        if len(np.unique(rows)) != len(rows):
            raise RuntimeError(f'{gsm}: duplicated astrocyte matrix rows.')
        donor_counts = np.asarray(X[rows, :].sum(axis=0)).ravel()
        if np.any(donor_counts < 0):
            raise RuntimeError(f'{gsm}: negative pseudobulk count detected.')
        if not np.all(donor_counts == np.rint(donor_counts)):
            raise RuntimeError(f'{gsm}: non-integer pseudobulk counts detected.')
        donor_counts = donor_counts.astype(np.int64, copy=False)
        pseudobulk[donor_index, :] = donor_counts
        total_umi = int(donor_counts.sum())
        detected_features = int(np.count_nonzero(donor_counts))
        donor_audit_rows.append({'matrix_row': int(donor_index), 'GSM': gsm, 'sample_label': sample_label, 'astrocyte_nuclei': int(n_astro), 'total_raw_UMI_counts': total_umi, 'detected_features': detected_features, 'primary_eligible_ge20': bool(n_astro >= 20), 'sensitivity_eligible_ge50': bool(n_astro >= 50)})
        print(f'[{donor_index + 1:02d}/21] {gsm} / {sample_label}: {n_astro:,} astrocytes, {total_umi:,} raw UMIs')
        del X
        del donor_counts
        del rows
        del donor_astro
        gc.collect()

    donor_manifest = pd.DataFrame(donor_audit_rows)

    if donor_manifest['astrocyte_nuclei'].sum() != EXPECTED_ASTROCYTES:
        raise RuntimeError('Pseudobulk donor astrocyte counts do not sum to 17,067.')

    if pseudobulk.shape != (EXPECTED_DONORS, EXPECTED_FEATURES):
        raise RuntimeError(f'Unexpected pseudobulk shape: {pseudobulk.shape}')

    if np.any(pseudobulk < 0):
        raise RuntimeError('Negative count detected in pseudobulk matrix.')

    if not np.issubdtype(pseudobulk.dtype, np.integer):
        raise RuntimeError('Pseudobulk matrix is not integer.')

    if pseudobulk.sum() <= 0:
        raise RuntimeError('Pseudobulk matrix contains no counts.')

    pseudobulk_sparse = sp.csr_matrix(pseudobulk)

    sp.save_npz(PSEUDOBULK_NPZ_PATH, pseudobulk_sparse, compressed=True)

    check = sp.load_npz(PSEUDOBULK_NPZ_PATH)

    if check.shape != pseudobulk.shape:
        raise RuntimeError('Saved NPZ shape verification failed.')


In [ ]:
if RUN_PIPELINE:
    if not np.array_equal(check.toarray(), pseudobulk):
        raise RuntimeError('Saved NPZ count verification failed.')

    del check

    count_table = var[['gene_id', 'gene_symbol']].copy()

    for donor_index, gsm in enumerate(donor_order):
        count_table[gsm] = pseudobulk[donor_index, :]

    count_table.to_csv(PSEUDOBULK_CSV_PATH, index=False, compression={'method': 'gzip', 'compresslevel': 6})

    donor_manifest.to_csv(DONOR_MANIFEST_PATH, index=False)

    step10_lock = {'dataset': 'GSE157827', 'step': 10, 'version': 'v1', 'created_utc': datetime.now(timezone.utc).isoformat(), 'input_astrocyte_manifest': str(ASTROCYTE_PATH), 'accepted_astrocyte_nuclei': EXPECTED_ASTROCYTES, 'biological_replicates': EXPECTED_DONORS, 'features': EXPECTED_FEATURES, 'matrix_shape_donors_by_features': [EXPECTED_DONORS, EXPECTED_FEATURES], 'pseudobulk_definition': 'Original raw integer UMI counts summed across all frozen accepted astrocyte nuclei separately for each human donor.', 'matrix_dtype': str(pseudobulk.dtype), 'total_pseudobulk_raw_UMIs': int(pseudobulk.sum()), 'minimum_astrocytes_per_donor': int(donor_manifest['astrocyte_nuclei'].min()), 'maximum_astrocytes_per_donor': int(donor_manifest['astrocyte_nuclei'].max()), 'all_21_donors_retained': True, 'all_primary_donor_thresholds_passed': bool(donor_manifest['primary_eligible_ge20'].all()), 'all_sensitivity_donor_thresholds_passed': bool(donor_manifest['sensitivity_eligible_ge50'].all()), 'normalization_performed': False, 'gene_filtering_performed': False, 'mitochondrial_genes_removed': False, 'diagnosis_loaded': False, 'CREB5_queried': False, 'locked_target_genes_queried': False, 'differential_expression_performed': False, 'pseudobulk_npz': str(PSEUDOBULK_NPZ_PATH), 'pseudobulk_csv': str(PSEUDOBULK_CSV_PATH), 'donor_manifest': str(DONOR_MANIFEST_PATH)}

    with open(STEP10_LOCK_PATH, 'w', encoding='utf-8') as handle:
        json.dump(step10_lock, handle, indent=2)

    print('\n' + '=' * 76)

    print('GSE157827 STEP 10 COMPLETE')

    print('=' * 76)

    print(f'Pseudobulk matrix shape: {pseudobulk.shape}')

    print(f'Biological replicates: {pseudobulk.shape[0]}')

    print(f'Features: {pseudobulk.shape[1]:,}')

    print(f"Astrocyte nuclei summed: {donor_manifest['astrocyte_nuclei'].sum():,}")

    print(f'Total raw astrocyte UMIs: {pseudobulk.sum():,}')

    print(f"Minimum astrocytes/donor: {donor_manifest['astrocyte_nuclei'].min()}")

    print(f"Maximum astrocytes/donor: {donor_manifest['astrocyte_nuclei'].max()}")

    print('\nDonor pseudobulk audit:')

    print(donor_manifest.to_string(index=False))

    print('\nRaw integer counts preserved: YES')

    print('Normalization performed: NO')

    print('Gene filtering performed: NO')

    print('Diagnosis loaded: NO')

    print('CREB5 queried: NO')

    print('Differential expression performed: NO')

    print(f'\nRaw pseudobulk NPZ:\n{PSEUDOBULK_NPZ_PATH}')

    print(f'\nRaw pseudobulk CSV:\n{PSEUDOBULK_CSV_PATH}')

    print(f'\nDonor manifest:\n{DONOR_MANIFEST_PATH}')

    print(f'\nSTEP 10 lock:\n{STEP10_LOCK_PATH}')


## Author donor metadata and primary model

The author supplementary archive is retrieved from Europe PMC only if the metadata workbook is absent. The analysis then links the `Patient_info` sheet to the 21 donor pseudobulks and freezes the model `~ age + sex + diagnosis`.


In [ ]:

if RUN_PIPELINE:
    import io, zipfile, requests

    supplement_dir = PROJECT / "Lau2020_original_supplementary"
    workbook = supplement_dir / "pnas.2008762117.sd01.xlsx"

    if not workbook.exists():
        supplement_dir.mkdir(parents=True, exist_ok=True)
        url = "https://www.ebi.ac.uk/europepmc/webservices/rest/PMC7568283/supplementaryFiles"
        response = requests.get(url, timeout=180)
        response.raise_for_status()

        with zipfile.ZipFile(io.BytesIO(response.content)) as zf:
            for member in zf.infolist():
                target = Path(member.filename)
                if target.is_absolute() or ".." in target.parts:
                    raise RuntimeError(f"Unsafe supplementary path: {member.filename}")
                zf.extract(member, supplement_dir)

    if not workbook.exists():
        matches = list(supplement_dir.rglob("pnas.2008762117.sd01.xlsx"))
        if len(matches) != 1:
            raise FileNotFoundError("Author metadata workbook was not recovered uniquely.")
        workbook = matches[0]

    print("Author metadata workbook:", workbook)


In [ ]:
if RUN_PIPELINE:
    from pathlib import Path

    from datetime import datetime, timezone

    import pandas as pd

    import numpy as np

    import json

    import hashlib

    PROJECT_DIR = Path('/content/drive/MyDrive/AD_Astrocyte_Paper_01')

    VALIDATION_DIR = PROJECT_DIR / 'GSE157827_confirmatory_validation'

    SUPPLEMENT_DIR = VALIDATION_DIR / 'Lau2020_original_supplementary'

    WORKBOOK_PATH = SUPPLEMENT_DIR / 'pnas.2008762117.sd01.xlsx'

    DONOR_MANIFEST_PATH = VALIDATION_DIR / 'GSE157827_STEP10_pseudobulk_donor_manifest_v1.csv'

    STEP10_LOCK_PATH = VALIDATION_DIR / 'GSE157827_STEP10_PSEUDOBULK_LOCK_v1.json'

    AUTHOR_METADATA_PATH = VALIDATION_DIR / 'GSE157827_STEP11B_authoritative_donor_metadata_v1.csv'

    DESIGN_MANIFEST_PATH = VALIDATION_DIR / 'GSE157827_STEP11B_primary_design_manifest_v1.csv'

    DESIGN_LOCK_PATH = VALIDATION_DIR / 'GSE157827_STEP11B_PRIMARY_DESIGN_LOCK_v1.json'

    EXPECTED_DONORS = 21

    EXPECTED_AD = 12

    EXPECTED_CONTROL = 9

    for path in [WORKBOOK_PATH, DONOR_MANIFEST_PATH, STEP10_LOCK_PATH]:
        if not path.exists():
            raise FileNotFoundError(f'Required input missing:\n{path}')

    with open(STEP10_LOCK_PATH, 'r', encoding='utf-8') as handle:
        step10 = json.load(handle)

    if step10.get('biological_replicates') != EXPECTED_DONORS:
        raise RuntimeError('STEP 10 does not contain 21 biological replicates.')

    sha256 = hashlib.sha256()

    with open(WORKBOOK_PATH, 'rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''):
            sha256.update(chunk)

    workbook_sha256 = sha256.hexdigest()

    patient = pd.read_excel(WORKBOOK_PATH, sheet_name='Patient_info')

    required_author_columns = ['ID', 'CONDITION', 'AGE', 'SEX', 'PMD', 'Braak tangle stage', 'APOE']

    missing_columns = [col for col in required_author_columns if col not in patient.columns]

    if missing_columns:
        raise RuntimeError(f'Missing expected author metadata columns: {missing_columns}')

    patient = patient[required_author_columns].copy()

    patient['ID'] = patient['ID'].astype(str).str.strip()

    patient['CONDITION'] = patient['CONDITION'].astype(str).str.strip().str.upper()

    patient['SEX'] = patient['SEX'].astype(str).str.strip().str.upper()

    patient['AGE'] = pd.to_numeric(patient['AGE'], errors='raise')

    patient['PMD'] = pd.to_numeric(patient['PMD'], errors='raise')

    patient['Braak tangle stage'] = pd.to_numeric(patient['Braak tangle stage'], errors='raise')

    if len(patient) != EXPECTED_DONORS:
        raise RuntimeError(f'Expected 21 author donor rows; found {len(patient)}.')

    if patient['ID'].duplicated().any():
        raise RuntimeError('Duplicate donor IDs in author Patient_info.')

    if not patient['SEX'].isin(['M', 'F']).all():
        raise RuntimeError('Unexpected SEX value in author metadata.')

    if not patient['CONDITION'].isin(['AD', 'NC']).all():
        raise RuntimeError('Unexpected CONDITION value in author metadata.')

    condition_map = {'AD': 'AD', 'NC': 'Control'}

    patient['diagnosis'] = patient['CONDITION'].map(condition_map)

    if patient['diagnosis'].isna().any():
        raise RuntimeError('Diagnosis mapping failed.')

    donors = pd.read_csv(DONOR_MANIFEST_PATH, dtype={'GSM': str, 'sample_label': str})


In [ ]:
if RUN_PIPELINE:
    if len(donors) != EXPECTED_DONORS:
        raise RuntimeError('STEP 10 donor manifest does not contain 21 rows.')

    if donors['GSM'].duplicated().any():
        raise RuntimeError('Duplicate GSM in STEP 10 donor manifest.')

    if donors['sample_label'].duplicated().any():
        raise RuntimeError('Duplicate sample_label in STEP 10 donor manifest.')

    step10_ids = set(donors['sample_label'].astype(str))

    author_ids = set(patient['ID'].astype(str))

    missing_from_author = sorted(step10_ids - author_ids)

    extra_in_author = sorted(author_ids - step10_ids)

    if missing_from_author or extra_in_author:
        raise RuntimeError(f'Author/STEP10 donor IDs do not match exactly.\nMissing from author metadata: {missing_from_author}\nExtra in author metadata: {extra_in_author}')

    metadata = donors[['matrix_row', 'GSM', 'sample_label', 'astrocyte_nuclei', 'total_raw_UMI_counts']].merge(patient, left_on='sample_label', right_on='ID', how='left', validate='one_to_one', indicator=True).sort_values('matrix_row').reset_index(drop=True)

    if not (metadata['_merge'] == 'both').all():
        raise RuntimeError('At least one pseudobulk donor failed metadata linkage.')

    metadata = metadata.drop(columns=['_merge', 'ID'])

    metadata = metadata.rename(columns={'AGE': 'age', 'SEX': 'sex', 'PMD': 'PMD_hours', 'Braak tangle stage': 'Braak_stage', 'APOE': 'APOE', 'CONDITION': 'author_condition'})

    for column in ['age', 'sex', 'diagnosis']:
        if metadata[column].isna().any():
            raise RuntimeError(f'Missing primary covariate: {column}')

    diagnosis_counts = metadata['diagnosis'].value_counts()

    if int(diagnosis_counts.get('AD', 0)) != EXPECTED_AD:
        raise RuntimeError('Expected 12 AD donors.')

    if int(diagnosis_counts.get('Control', 0)) != EXPECTED_CONTROL:
        raise RuntimeError('Expected 9 Control donors.')

    metadata['diagnosis'] = pd.Categorical(metadata['diagnosis'], categories=['Control', 'AD'], ordered=True)

    metadata['sex'] = pd.Categorical(metadata['sex'], categories=['F', 'M'], ordered=True)

    mean_age_all = float(metadata['age'].mean())

    metadata['age_centered'] = metadata['age'] - mean_age_all

    sex_M = metadata['sex'].astype(str).eq('M').astype(float).to_numpy()

    diagnosis_AD = metadata['diagnosis'].astype(str).eq('AD').astype(float).to_numpy()

    design_matrix = np.column_stack([np.ones(EXPECTED_DONORS, dtype=float), metadata['age_centered'].to_numpy(dtype=float), sex_M, diagnosis_AD])

    design_columns = ['Intercept', 'age_centered', 'sex_M', 'diagnosis_AD']

    design_rank = int(np.linalg.matrix_rank(design_matrix))

    n_parameters = int(design_matrix.shape[1])

    residual_df = int(EXPECTED_DONORS - design_rank)

    if design_rank != n_parameters:
        raise RuntimeError('Primary design is NOT full rank. Stop.')

    metadata_for_summary = metadata.copy()

    metadata_for_summary['diagnosis'] = metadata_for_summary['diagnosis'].astype(str)

    metadata_for_summary['sex'] = metadata_for_summary['sex'].astype(str)

    group_summary = metadata_for_summary.groupby('diagnosis', observed=True).agg(donors=('GSM', 'size'), mean_age=('age', 'mean'), sd_age=('age', 'std'), min_age=('age', 'min'), max_age=('age', 'max'), female=('sex', lambda x: int((x == 'F').sum())), male=('sex', lambda x: int((x == 'M').sum())))

    metadata_to_save = metadata.copy()

    metadata_to_save['diagnosis'] = metadata_to_save['diagnosis'].astype(str)

    metadata_to_save['sex'] = metadata_to_save['sex'].astype(str)

    metadata_to_save.to_csv(AUTHOR_METADATA_PATH, index=False)

    design_manifest = metadata_to_save[['matrix_row', 'GSM', 'sample_label', 'diagnosis', 'age', 'sex', 'age_centered', 'astrocyte_nuclei', 'total_raw_UMI_counts']].copy()

    design_manifest.to_csv(DESIGN_MANIFEST_PATH, index=False)

    design_lock = {'dataset': 'GSE157827', 'step': '11B', 'version': 'v1', 'created_utc': datetime.now(timezone.utc).isoformat(), 'metadata_source': 'Lau et al. original supplementary workbook', 'metadata_workbook': str(WORKBOOK_PATH), 'metadata_sheet': 'Patient_info', 'metadata_workbook_sha256': workbook_sha256, 'author_metadata_complete_for_all_21_donors': True, 'age_complete': True, 'sex_complete': True, 'diagnosis_complete': True, 'number_AD': EXPECTED_AD, 'number_Control': EXPECTED_CONTROL, 'primary_design': '~ age + sex + diagnosis', 'primary_design_reference_levels': {'diagnosis': 'Control', 'sex': 'F'}, 'primary_contrast': 'AD versus Control', 'effect_direction': 'Positive diagnosis log2FoldChange means higher expression in AD', 'design_rank': design_rank, 'number_model_parameters': n_parameters, 'residual_degrees_of_freedom': residual_df, 'design_full_rank': True, 'age_centered_only_for_numeric_rank_check': True, 'age_used_as_primary_covariate': True, 'sex_used_as_primary_covariate': True, 'PMD_available_but_not_in_prespecified_primary_model': True, 'Braak_available_but_not_used_as_primary_covariate': True, 'APOE_available_but_not_used_as_primary_covariate': True, 'missing_covariates_imputed': False, 'author_metadata_file': str(AUTHOR_METADATA_PATH), 'primary_design_manifest': str(DESIGN_MANIFEST_PATH), 'expression_examined': False, 'gene_filtering_performed': False, 'CREB5_queried': False, 'locked_target_genes_queried': False, 'differential_expression_performed': False}

    with open(DESIGN_LOCK_PATH, 'w', encoding='utf-8') as handle:
        json.dump(design_lock, handle, indent=2)

    print('=' * 76)

    print('GSE157827 STEP 11B PRIMARY DESIGN LOCKED')

    print('=' * 76)

    print(f'\nAuthor donors linked: {len(metadata)}/21')


In [ ]:
if RUN_PIPELINE:
    print(f'AD donors: {EXPECTED_AD}')

    print(f'Control donors: {EXPECTED_CONTROL}')

    print('\nGroup metadata:')

    print(group_summary.to_string(float_format=lambda x: f'{x:.2f}'))

    print('\nPrimary design:')

    print('    ~ age + sex + diagnosis')

    print('\nReference diagnosis: Control')

    print('Reference sex: F')

    print('Contrast: AD versus Control')

    print('Positive diagnosis coefficient: higher in AD')

    print(f'\nDesign dimensions: {design_matrix.shape}')

    print(f'Design rank: {design_rank}/{n_parameters}')

    print(f'Residual degrees of freedom: {residual_df}')

    print('Design full rank: YES')

    print('\nPMD available: YES (not added to prespecified primary model)')

    print('Braak stage available: YES (not used as adjustment covariate)')

    print('APOE available: YES (not added to prespecified primary model)')

    print('\nExpression examined: NO')

    print('Gene filtering performed: NO')

    print('CREB5 queried: NO')

    print('Differential expression performed: NO')

    print(f'\nAuthoritative metadata:\n{AUTHOR_METADATA_PATH}')

    print(f'\nPrimary design manifest:\n{DESIGN_MANIFEST_PATH}')

    print(f'\nPrimary design lock:\n{DESIGN_LOCK_PATH}')


## Frozen gene universe and primary differential expression

The gene universe is fixed without diagnosis or target inspection. The primary donor-level model uses PyDESeq2 0.5.4 with `~ age + sex + diagnosis`. The full genome-wide table is written before CREB5 is extracted.


In [ ]:
if RUN_PIPELINE:
    from pathlib import Path

    from datetime import datetime, timezone

    import pandas as pd

    import numpy as np

    import scipy.sparse as sp

    import hashlib

    import json

    PROJECT_DIR = Path('/content/drive/MyDrive/AD_Astrocyte_Paper_01')

    VALIDATION_DIR = PROJECT_DIR / 'GSE157827_confirmatory_validation'

    PSEUDOBULK_NPZ_PATH = VALIDATION_DIR / 'GSE157827_STEP10_astrocyte_pseudobulk_raw_counts_v1.npz'

    DONOR_MANIFEST_PATH = VALIDATION_DIR / 'GSE157827_STEP10_pseudobulk_donor_manifest_v1.csv'

    VAR_PATH = VALIDATION_DIR / 'GSE157827_STEP8_feature_metadata_v1_1.csv.gz'

    DESIGN_LOCK_PATH = VALIDATION_DIR / 'GSE157827_STEP11B_PRIMARY_DESIGN_LOCK_v1.json'

    GENE_AUDIT_PATH = VALIDATION_DIR / 'GSE157827_STEP12_gene_filter_audit_v1.csv.gz'

    PRIMARY_FEATURES_PATH = VALIDATION_DIR / 'GSE157827_STEP12_primary_DE_features_v1.csv.gz'

    FILTERED_NPZ_PATH = VALIDATION_DIR / 'GSE157827_STEP12_primary_DE_raw_counts_v1.npz'

    FILTERED_CSV_PATH = VALIDATION_DIR / 'GSE157827_STEP12_primary_DE_raw_counts_genes_x_donors_v1.csv.gz'

    STEP12_LOCK_PATH = VALIDATION_DIR / 'GSE157827_STEP12_GENE_UNIVERSE_LOCK_v1.json'

    EXPECTED_DONORS = 21

    EXPECTED_FEATURES = 33538

    CPM_THRESHOLD = 1.0

    MIN_DONORS_CPM = 9

    for path in [PSEUDOBULK_NPZ_PATH, DONOR_MANIFEST_PATH, VAR_PATH, DESIGN_LOCK_PATH]:
        if not path.exists():
            raise FileNotFoundError(f'Required input missing:\n{path}')

    with open(DESIGN_LOCK_PATH, 'r', encoding='utf-8') as handle:
        design_lock = json.load(handle)

    if not design_lock.get('design_full_rank', False):
        raise RuntimeError('Primary design is not documented as full rank.')

    if design_lock.get('primary_design') != '~ age + sex + diagnosis':
        raise RuntimeError('Unexpected primary design.')

    X = sp.load_npz(PSEUDOBULK_NPZ_PATH).tocsr()

    if X.shape != (EXPECTED_DONORS, EXPECTED_FEATURES):
        raise RuntimeError(f'Unexpected pseudobulk shape: {X.shape}')

    if not np.issubdtype(X.dtype, np.integer):
        raise RuntimeError('STEP 10 pseudobulk matrix is not integer.')

    if X.nnz == 0:
        raise RuntimeError('Pseudobulk matrix contains no nonzero counts.')

    if X.data.min() < 0:
        raise RuntimeError('Negative raw count detected.')

    donors = pd.read_csv(DONOR_MANIFEST_PATH, dtype={'GSM': str, 'sample_label': str}).sort_values('matrix_row').reset_index(drop=True)

    var = pd.read_csv(VAR_PATH, dtype=str)

    if len(donors) != EXPECTED_DONORS:
        raise RuntimeError('Expected 21 donor manifest rows.')

    if len(var) != EXPECTED_FEATURES:
        raise RuntimeError('Expected 33,538 feature rows.')

    if 'gene_id' not in var.columns:
        raise RuntimeError('gene_id missing from feature metadata.')

    if 'gene_symbol' not in var.columns:
        raise RuntimeError('gene_symbol missing from feature metadata.')

    if var['gene_id'].duplicated().any():
        raise RuntimeError('gene_id is not unique.')

    print('=' * 76)

    print('GSE157827 STEP 12 STARTED')

    print('=' * 76)


In [ ]:
if RUN_PIPELINE:
    print(f'\nRaw pseudobulk shape: {X.shape}')

    print(f'Donors: {len(donors)}')

    print(f'Features before filtering: {len(var):,}')

    print('\nFiltering rule:')

    print('    CPM >= 1 in at least 9 of 21 donors')

    print('    AND non-mitochondrial')

    print('\nDiagnosis used for gene filtering: NO')

    print('Target genes queried: NO')

    matrix_library_sizes = np.asarray(X.sum(axis=1)).ravel().astype(np.int64)

    manifest_library_sizes = donors['total_raw_UMI_counts'].to_numpy(dtype=np.int64)

    if not np.array_equal(matrix_library_sizes, manifest_library_sizes):
        raise RuntimeError('Pseudobulk matrix library sizes do not match the STEP 10 donor audit.')

    if np.any(matrix_library_sizes <= 0):
        raise RuntimeError('At least one pseudobulk library has zero counts.')

    print('\nSTEP 10 library-size verification: PASS')

    counts = X.toarray().astype(np.int64, copy=False)

    cpm = counts.astype(np.float64) / matrix_library_sizes[:, None] * 1000000.0

    if not np.isfinite(cpm).all():
        raise RuntimeError('Non-finite CPM value detected.')

    donors_ge_1cpm = (cpm >= CPM_THRESHOLD).sum(axis=0)

    expression_pass = donors_ge_1cpm >= MIN_DONORS_CPM

    gene_symbols = var['gene_symbol'].fillna('').astype(str)

    mitochondrial = gene_symbols.str.upper().str.startswith('MT-').to_numpy()

    non_mitochondrial = ~mitochondrial

    primary_universe = expression_pass & non_mitochondrial

    n_expression_pass = int(expression_pass.sum())

    n_mito_total = int(mitochondrial.sum())

    n_expression_pass_mito = int((expression_pass & mitochondrial).sum())

    n_primary = int(primary_universe.sum())

    if n_primary <= 0:
        raise RuntimeError('Primary gene universe is empty.')

    gene_audit = var[['gene_id', 'gene_symbol']].copy()

    gene_audit['donors_CPM_ge_1'] = donors_ge_1cpm.astype(np.int16)

    gene_audit['expression_filter_pass'] = expression_pass

    gene_audit['mitochondrial'] = mitochondrial

    gene_audit['primary_DE_universe'] = primary_universe

    gene_audit.to_csv(GENE_AUDIT_PATH, index=False, compression={'method': 'gzip', 'compresslevel': 6})

    primary_feature_indices = np.flatnonzero(primary_universe)

    primary_features = var.iloc[primary_feature_indices][['gene_id', 'gene_symbol']].copy()

    primary_features.insert(0, 'original_feature_index', primary_feature_indices)

    primary_features.insert(1, 'DE_universe_index', np.arange(n_primary, dtype=np.int64))

    primary_features.to_csv(PRIMARY_FEATURES_PATH, index=False, compression={'method': 'gzip', 'compresslevel': 6})

    gene_id_payload = ('\n'.join(primary_features['gene_id'].astype(str)) + '\n').encode('utf-8')

    primary_universe_sha256 = hashlib.sha256(gene_id_payload).hexdigest()

    filtered_counts = counts[:, primary_feature_indices]

    if not np.issubdtype(filtered_counts.dtype, np.integer):
        raise RuntimeError('Filtered count matrix lost integer dtype.')

    filtered_sparse = sp.csr_matrix(filtered_counts)

    sp.save_npz(FILTERED_NPZ_PATH, filtered_sparse, compressed=True)

    check = sp.load_npz(FILTERED_NPZ_PATH)

    if check.shape != (EXPECTED_DONORS, n_primary):
        raise RuntimeError('Saved filtered NPZ shape verification failed.')

    if not np.array_equal(check.toarray(), filtered_counts):
        raise RuntimeError('Saved filtered count verification failed.')

    del check


In [ ]:
if RUN_PIPELINE:
    filtered_table = primary_features[['gene_id', 'gene_symbol']].copy()

    donor_order = donors['GSM'].astype(str).tolist()

    for donor_index, gsm in enumerate(donor_order):
        filtered_table[gsm] = filtered_counts[donor_index, :]

    filtered_table.to_csv(FILTERED_CSV_PATH, index=False, compression={'method': 'gzip', 'compresslevel': 6})

    if filtered_counts.shape != (EXPECTED_DONORS, n_primary):
        raise RuntimeError('Final filtered count matrix shape mismatch.')

    if np.any(filtered_counts < 0):
        raise RuntimeError('Negative value in filtered counts.')

    if primary_features['gene_id'].duplicated().any():
        raise RuntimeError('Duplicate gene IDs in primary universe.')

    if mitochondrial[primary_feature_indices].any():
        raise RuntimeError('Mitochondrial feature entered primary universe.')

    if not (donors_ge_1cpm[primary_feature_indices] >= MIN_DONORS_CPM).all():
        raise RuntimeError('A retained gene violates the CPM rule.')

    step12_lock = {'dataset': 'GSE157827', 'step': 12, 'version': 'v1', 'created_utc': datetime.now(timezone.utc).isoformat(), 'input_raw_pseudobulk': str(PSEUDOBULK_NPZ_PATH), 'input_matrix_shape': [EXPECTED_DONORS, EXPECTED_FEATURES], 'expression_filter': 'CPM >= 1 in at least 9 of 21 donors', 'CPM_threshold': CPM_THRESHOLD, 'minimum_donors': MIN_DONORS_CPM, 'CPM_denominator': 'Total raw astrocyte pseudobulk UMI count for each donor across all 33,538 features', 'filter_uses_diagnosis': False, 'filter_uses_effect_direction': False, 'features_before_filtering': EXPECTED_FEATURES, 'features_passing_expression_filter': n_expression_pass, 'mitochondrial_features_in_reference': n_mito_total, 'expression_passing_mitochondrial_features_removed': n_expression_pass_mito, 'primary_DE_features': n_primary, 'mitochondrial_definition': 'gene_symbol starts with MT- (case-insensitive)', 'primary_universe_gene_id_sha256': primary_universe_sha256, 'primary_feature_file': str(PRIMARY_FEATURES_PATH), 'gene_filter_audit': str(GENE_AUDIT_PATH), 'filtered_raw_counts_npz': str(FILTERED_NPZ_PATH), 'filtered_raw_counts_genes_x_donors_csv': str(FILTERED_CSV_PATH), 'primary_design_already_locked': '~ age + sex + diagnosis', 'normalization_model_fitted': False, 'target_genes_queried': False, 'differential_expression_performed': False, 'primary_gene_universe_frozen': True}

    with open(STEP12_LOCK_PATH, 'w', encoding='utf-8') as handle:
        json.dump(step12_lock, handle, indent=2)

    print('\n' + '=' * 76)

    print('GSE157827 STEP 12 COMPLETE')

    print('=' * 76)

    print(f'\nFeatures before filtering: {EXPECTED_FEATURES:,}')

    print(f'Features passing CPM rule: {n_expression_pass:,}')

    print(f'Mitochondrial features in reference: {n_mito_total:,}')

    print(f'Expression-passing mitochondrial features removed: {n_expression_pass_mito:,}')

    print(f'PRIMARY DE GENE UNIVERSE: {n_primary:,} features')

    print('\nFilter:')

    print('    CPM >= 1 in at least 9/21 donors')

    print('    mitochondrial genes excluded')

    print(f'\nFrozen universe SHA-256:\n{primary_universe_sha256}')

    print('\nDiagnosis used for filtering: NO')

    print('Effect direction used for filtering: NO')

    print('Normalization model fitted: NO')

    print('Target genes queried: NO')

    print('Differential expression performed: NO')

    print('\nPRIMARY GENE UNIVERSE FROZEN: YES')

    print(f'\nGene-filter audit:\n{GENE_AUDIT_PATH}')

    print(f'\nFrozen primary features:\n{PRIMARY_FEATURES_PATH}')

    print(f'\nFiltered raw-count NPZ:\n{FILTERED_NPZ_PATH}')

    print(f'\nGenes x donors DESeq2 input:\n{FILTERED_CSV_PATH}')

    print(f'\nSTEP 12 lock:\n{STEP12_LOCK_PATH}')


In [ ]:
if RUN_PIPELINE:
    import sys

    import subprocess

    import importlib.metadata as md

    from pathlib import Path

    from datetime import datetime, timezone

    import json

    REQUIRED = {'numpy': '2.1.3', 'pandas': '2.2.3', 'scipy': '1.16.3', 'scikit-learn': '1.6.1', 'anndata': '0.12.6', 'pydeseq2': '0.5.4', 'formulaic': '1.2.2', 'formulaic-contrasts': '1.0.0'}

    def installed_version(package):
        try:
            return md.version(package)
        except md.PackageNotFoundError:
            return None

    observed_before = {package: installed_version(package) for package in REQUIRED}

    print('=' * 76)

    print('GSE157827 STEP 13A — PYDESEQ2 ENVIRONMENT AUDIT')

    print('=' * 76)

    print('\nRequired / observed versions:')

    environment_ok = True

    for package, required_version in REQUIRED.items():
        observed = observed_before[package]
        status = 'OK' if observed == required_version else 'CHANGE REQUIRED'
        if observed != required_version:
            environment_ok = False
        print(f'{package:22s} required={required_version:10s} observed={str(observed):15s} {status}')

    if not environment_ok:
        print('\nEnvironment differs from the locked Paper-01 PyDESeq2 stack.')
        print('Installing exact known-working versions...')
        packages = [f'{package}=={version}' for package, version in REQUIRED.items()]
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet', '--no-input', *packages])
        print('\nINSTALLATION COMPLETE.')
        print('Restart the Colab runtime ONCE, then rerun this same STEP 13A cell.')
        print('\nDo NOT rerun Steps 5-12.')
        raise SystemExit('Runtime restart required before DE.')

    import numpy as np

    import pandas as pd

    import scipy

    import sklearn

    import anndata

    from pydeseq2.dds import DeseqDataSet

    from pydeseq2.ds import DeseqStats

    observed_after = {package: installed_version(package) for package in REQUIRED}

    for package, expected in REQUIRED.items():
        if observed_after[package] != expected:
            raise RuntimeError(f'{package}: expected {expected}, found {observed_after[package]}')

    PROJECT_DIR = Path('/content/drive/MyDrive/AD_Astrocyte_Paper_01')

    VALIDATION_DIR = PROJECT_DIR / 'GSE157827_confirmatory_validation'

    DESIGN_LOCK_PATH = VALIDATION_DIR / 'GSE157827_STEP11B_PRIMARY_DESIGN_LOCK_v1.json'

    GENE_LOCK_PATH = VALIDATION_DIR / 'GSE157827_STEP12_GENE_UNIVERSE_LOCK_v1.json'

    ENV_LOCK_PATH = VALIDATION_DIR / 'GSE157827_STEP13A_PYDESEQ2_ENVIRONMENT_LOCK_v1.json'

    for path in [DESIGN_LOCK_PATH, GENE_LOCK_PATH]:
        if not path.exists():
            raise FileNotFoundError(f'Required frozen lock missing:\n{path}')

    with open(DESIGN_LOCK_PATH, 'r', encoding='utf-8') as handle:
        design_lock = json.load(handle)


In [ ]:
if RUN_PIPELINE:
    with open(GENE_LOCK_PATH, 'r', encoding='utf-8') as handle:
        gene_lock = json.load(handle)

    if design_lock.get('primary_design') != '~ age + sex + diagnosis':
        raise RuntimeError('Unexpected frozen primary design.')

    if not gene_lock.get('primary_gene_universe_frozen', False):
        raise RuntimeError('STEP 12 gene universe is not frozen.')

    if gene_lock.get('primary_DE_features') != 16871:
        raise RuntimeError('Expected frozen universe of 16,871 features.')

    expected_hash = 'b2cd51ebe6557d79be9d7d4b9f7d514a77778bcc4857c7371252388827e8466a'

    if gene_lock.get('primary_universe_gene_id_sha256') != expected_hash:
        raise RuntimeError('Frozen gene-universe SHA-256 mismatch.')

    environment_lock = {'dataset': 'GSE157827', 'step': '13A', 'created_utc': datetime.now(timezone.utc).isoformat(), 'python': sys.version, 'package_versions': observed_after, 'primary_design': '~ age + sex + diagnosis', 'primary_gene_universe': 16871, 'primary_gene_universe_sha256': expected_hash, 'DE_engine': 'PyDESeq2', 'DE_engine_version': '0.5.4', 'refit_cooks': True, 'cooks_filter': True, 'independent_filter': True, 'alpha': 0.05, 'contrast': ['diagnosis', 'AD', 'Control'], 'expression_loaded': False, 'CREB5_queried': False, 'differential_expression_performed': False}

    with open(ENV_LOCK_PATH, 'w', encoding='utf-8') as handle:
        json.dump(environment_lock, handle, indent=2)

    print('\n' + '=' * 76)

    print('GSE157827 STEP 13A COMPLETE')

    print('=' * 76)

    for package, version in observed_after.items():
        print(f'{package:22s}: {version}')

    print('\nDE engine: PyDESeq2 0.5.4')

    print('Frozen design: ~ age + sex + diagnosis')

    print('Frozen contrast: AD versus Control')

    print('Frozen gene universe: 16,871')

    print('Frozen gene-universe SHA-256:')

    print(expected_hash)

    print('\nExpression loaded: NO')

    print('CREB5 queried: NO')

    print('Differential expression performed: NO')

    print('\nPYDESEQ2 ENVIRONMENT READY: YES')

    print(f'\nEnvironment lock:\n{ENV_LOCK_PATH}')


In [ ]:
if RUN_PIPELINE:
    from pathlib import Path

    from datetime import datetime, timezone

    import json

    import hashlib

    import importlib.metadata as md

    import numpy as np

    import pandas as pd

    import scipy.sparse as sp

    from pydeseq2.dds import DeseqDataSet

    from pydeseq2.ds import DeseqStats

    PROJECT_DIR = Path('/content/drive/MyDrive/AD_Astrocyte_Paper_01')

    VALIDATION_DIR = PROJECT_DIR / 'GSE157827_confirmatory_validation'

    COUNTS_PATH = VALIDATION_DIR / 'GSE157827_STEP12_primary_DE_raw_counts_v1.npz'

    FEATURES_PATH = VALIDATION_DIR / 'GSE157827_STEP12_primary_DE_features_v1.csv.gz'

    DESIGN_PATH = VALIDATION_DIR / 'GSE157827_STEP11B_primary_design_manifest_v1.csv'

    DONOR_MANIFEST_PATH = VALIDATION_DIR / 'GSE157827_STEP10_pseudobulk_donor_manifest_v1.csv'

    STEP12_LOCK_PATH = VALIDATION_DIR / 'GSE157827_STEP12_GENE_UNIVERSE_LOCK_v1.json'

    STEP13A_LOCK_PATH = VALIDATION_DIR / 'GSE157827_STEP13A_PYDESEQ2_ENVIRONMENT_LOCK_v1.json'

    RESULTS_PATH = VALIDATION_DIR / 'GSE157827_STEP13B_DESeq2_primary_age_sex_adjusted_v1.csv.gz'

    SIGNIFICANT_PATH = VALIDATION_DIR / 'GSE157827_STEP13B_DESeq2_primary_FDR05_v1.csv.gz'

    TARGET_PATH = VALIDATION_DIR / 'GSE157827_STEP13B_locked_target_results_v1.csv'

    SIZE_FACTOR_PATH = VALIDATION_DIR / 'GSE157827_STEP13B_DESeq2_size_factors_v1.csv'

    STEP13B_LOCK_PATH = VALIDATION_DIR / 'GSE157827_STEP13B_PRIMARY_CONFIRMATORY_RESULT_LOCK_v1.json'

    EXPECTED_DONORS = 21

    EXPECTED_GENES = 16871

    EXPECTED_UNIVERSE_SHA256 = 'b2cd51ebe6557d79be9d7d4b9f7d514a77778bcc4857c7371252388827e8466a'

    PRIMARY_GENE = 'CREB5'

    SECONDARY_GENES = ['CSRP1', 'SHC1', 'KCNH2', 'MRGPRF', 'AJAP1', 'CCDC3']

    for path in [COUNTS_PATH, FEATURES_PATH, DESIGN_PATH, DONOR_MANIFEST_PATH, STEP12_LOCK_PATH, STEP13A_LOCK_PATH]:
        if not path.exists():
            raise FileNotFoundError(f'Required frozen input missing:\n{path}')

    with open(STEP12_LOCK_PATH, 'r', encoding='utf-8') as handle:
        step12 = json.load(handle)

    with open(STEP13A_LOCK_PATH, 'r', encoding='utf-8') as handle:
        step13a = json.load(handle)

    if not step12.get('primary_gene_universe_frozen', False):
        raise RuntimeError('STEP 12 gene universe is not frozen.')

    if step12.get('primary_DE_features') != EXPECTED_GENES:
        raise RuntimeError('Unexpected frozen gene-universe size.')

    if step12.get('primary_universe_gene_id_sha256') != EXPECTED_UNIVERSE_SHA256:
        raise RuntimeError('Frozen universe hash mismatch.')

    if step13a.get('DE_engine_version') != '0.5.4':
        raise RuntimeError('Unexpected PyDESeq2 version lock.')

    required_versions = {'numpy': '2.1.3', 'pandas': '2.2.3', 'scipy': '1.16.3', 'anndata': '0.12.6', 'pydeseq2': '0.5.4', 'formulaic': '1.2.2', 'formulaic-contrasts': '1.0.0'}

    for pkg, expected in required_versions.items():
        observed = md.version(pkg)
        if observed != expected:
            raise RuntimeError(f'{pkg}: expected {expected}, found {observed}.')

    X = sp.load_npz(COUNTS_PATH).tocsr()

    if X.shape != (EXPECTED_DONORS, EXPECTED_GENES):
        raise RuntimeError(f'Unexpected count matrix shape: {X.shape}')

    if not np.issubdtype(X.dtype, np.integer):
        raise RuntimeError('DESeq2 input counts are not integer.')

    if X.data.size > 0 and X.data.min() < 0:
        raise RuntimeError('Negative raw count detected.')


In [ ]:
if RUN_PIPELINE:
    features = pd.read_csv(FEATURES_PATH, dtype={'gene_id': str, 'gene_symbol': str})

    if len(features) != EXPECTED_GENES:
        raise RuntimeError('Feature-table length does not equal 16,871.')

    if features['gene_id'].duplicated().any():
        raise RuntimeError('Duplicate gene IDs in frozen feature table.')

    feature_ids = features['gene_id'].astype(str).tolist()

    payload = ('\n'.join(feature_ids) + '\n').encode('utf-8')

    observed_hash = hashlib.sha256(payload).hexdigest()

    if observed_hash != EXPECTED_UNIVERSE_SHA256:
        raise RuntimeError('Feature list no longer matches frozen SHA-256.')

    design = pd.read_csv(DESIGN_PATH, dtype={'GSM': str, 'sample_label': str, 'diagnosis': str, 'sex': str}).sort_values('matrix_row').reset_index(drop=True)

    donors = pd.read_csv(DONOR_MANIFEST_PATH, dtype={'GSM': str, 'sample_label': str}).sort_values('matrix_row').reset_index(drop=True)

    if len(design) != EXPECTED_DONORS:
        raise RuntimeError('Design manifest does not contain 21 donors.')

    if not np.array_equal(design['matrix_row'].to_numpy(), np.arange(EXPECTED_DONORS)):
        raise RuntimeError('Design matrix_row is not 0..20.')

    if not np.array_equal(donors['GSM'].to_numpy(), design['GSM'].to_numpy()):
        raise RuntimeError('STEP 10 and STEP 11B donor ordering differs.')

    sample_ids = design['GSM'].astype(str).tolist()

    counts = pd.DataFrame(X.toarray().astype(np.int64, copy=False), index=pd.Index(sample_ids, dtype=object), columns=pd.Index(feature_ids, dtype=object))

    if counts.shape != (EXPECTED_DONORS, EXPECTED_GENES):
        raise RuntimeError('Counts DataFrame shape mismatch.')

    if counts.to_numpy().min() < 0:
        raise RuntimeError('Negative DESeq2 count.')

    metadata = design[['age', 'sex', 'diagnosis']].copy()

    metadata['age'] = pd.to_numeric(metadata['age'], errors='raise')

    metadata['sex'] = pd.Categorical(metadata['sex'].astype(str), categories=['F', 'M'], ordered=True)

    metadata['diagnosis'] = pd.Categorical(metadata['diagnosis'].astype(str), categories=['Control', 'AD'], ordered=True)

    metadata.index = pd.Index(sample_ids, dtype=object)

    metadata.columns = pd.Index(metadata.columns.astype(str).tolist(), dtype=object)

    if not counts.index.equals(metadata.index):
        raise RuntimeError('Count/metadata donor ordering mismatch.')

    if metadata.isna().any().any():
        raise RuntimeError('Missing primary model metadata.')

    if metadata['diagnosis'].astype(str).value_counts().to_dict() != {'AD': 12, 'Control': 9}:
        raise RuntimeError('Unexpected diagnosis counts.')

    print('=' * 78)

    print('GSE157827 STEP 13B PRIMARY CONFIRMATORY DE')

    print('=' * 78)

    print(f'\nBiological replicates: {counts.shape[0]}')

    print(f'Frozen genes: {counts.shape[1]:,}')

    print('Model: ~ age + sex + diagnosis')

    print('Contrast: AD versus Control')

    print('Reference diagnosis: Control')

    print('Positive log2FC: higher expression in AD')

    print('\nFrozen universe hash verified: YES')

    print('Raw integer counts verified: YES')

    print('Donor ordering verified: YES')

    print('\n*** TARGET BLINDING ENDS AFTER MODEL FIT ***')

    dds = DeseqDataSet(counts=counts, metadata=metadata, design='~ age + sex + diagnosis', refit_cooks=True, n_cpus=2, quiet=False)

    design_matrix = dds.obsm['design_matrix']

    design_rank = int(np.linalg.matrix_rank(design_matrix.to_numpy()))

    print('\nPyDESeq2 design columns:')

    print(design_matrix.columns.tolist())

    print('\nPyDESeq2 design rank:', design_rank, '/', design_matrix.shape[1])


In [ ]:
if RUN_PIPELINE:
    if design_rank != design_matrix.shape[1]:
        raise RuntimeError('PyDESeq2 design matrix is not full rank.')

    if design_matrix.shape[1] != 4:
        raise RuntimeError('Expected four model parameters.')

    print('\nRunning primary DESeq2 model...')

    dds.deseq2()

    print('\nDESEQ2 FIT COMPLETE')

    if 'size_factors' not in dds.obs.columns:
        raise RuntimeError('PyDESeq2 size factors were not found.')

    size_factors = dds.obs['size_factors'].to_numpy(dtype=float)

    if len(size_factors) != EXPECTED_DONORS:
        raise RuntimeError('Unexpected number of size factors.')

    if not np.isfinite(size_factors).all():
        raise RuntimeError('Non-finite DESeq2 size factor.')

    if (size_factors <= 0).any():
        raise RuntimeError('Non-positive DESeq2 size factor.')

    size_factor_table = pd.DataFrame({'GSM': sample_ids, 'size_factor': size_factors})

    size_factor_table.to_csv(SIZE_FACTOR_PATH, index=False)

    stats = DeseqStats(dds, contrast=['diagnosis', 'AD', 'Control'], alpha=0.05, cooks_filter=True, independent_filter=True, n_cpus=2, quiet=False)

    stats.summary()

    results = stats.results_df.copy()

    results.index = pd.Index(results.index.astype(str).tolist(), dtype=object)

    results.index.name = 'gene_id'

    results = results.reset_index()

    gene_map = features[['gene_id', 'gene_symbol']].copy()

    results = results.merge(gene_map, on='gene_id', how='left', validate='one_to_one')

    if len(results) != EXPECTED_GENES:
        raise RuntimeError('DE result does not contain all 16,871 genes.')

    required_result_columns = ['baseMean', 'log2FoldChange', 'lfcSE', 'stat', 'pvalue', 'padj']

    missing_result_columns = [column for column in required_result_columns if column not in results.columns]

    if missing_result_columns:
        raise RuntimeError(f'Missing DESeq2 result columns: {missing_result_columns}')

    results = results[['gene_id', 'gene_symbol', 'baseMean', 'log2FoldChange', 'lfcSE', 'stat', 'pvalue', 'padj']]

    results.to_csv(RESULTS_PATH, index=False, compression={'method': 'gzip', 'compresslevel': 6})

    finite_p = np.isfinite(results['pvalue'])

    finite_padj = np.isfinite(results['padj'])

    genome_sig = finite_padj & (results['padj'] < 0.05)

    significant = results.loc[genome_sig].copy().sort_values(['padj', 'pvalue'])

    significant.to_csv(SIGNIFICANT_PATH, index=False, compression={'method': 'gzip', 'compresslevel': 6})

    locked_targets = [PRIMARY_GENE, *SECONDARY_GENES]

    target_rows = []

    for gene in locked_targets:
        matches = results.loc[results['gene_symbol'].astype(str) == gene].copy()
        if len(matches) == 0:
            target_rows.append({'gene': gene, 'status': 'not_in_frozen_primary_universe', 'gene_id': None, 'baseMean': np.nan, 'log2FoldChange': np.nan, 'lfcSE': np.nan, 'stat': np.nan, 'pvalue': np.nan, 'genome_wide_padj': np.nan})
        elif len(matches) == 1:
            row = matches.iloc[0]
            target_rows.append({'gene': gene, 'status': 'tested', 'gene_id': str(row['gene_id']), 'baseMean': float(row['baseMean']), 'log2FoldChange': float(row['log2FoldChange']), 'lfcSE': float(row['lfcSE']), 'stat': float(row['stat']), 'pvalue': float(row['pvalue']), 'genome_wide_padj': float(row['padj']) if np.isfinite(row['padj']) else np.nan})
        else:
            target_rows.append({'gene': gene, 'status': f'ambiguous_multiple_features:{len(matches)}', 'gene_id': None, 'baseMean': np.nan, 'log2FoldChange': np.nan, 'lfcSE': np.nan, 'stat': np.nan, 'pvalue': np.nan, 'genome_wide_padj': np.nan})

    targets = pd.DataFrame(target_rows)

    targets['secondary_BH_qvalue'] = np.nan

    secondary_mask = targets['gene'].isin(SECONDARY_GENES)

    secondary = targets.loc[secondary_mask].copy()

    secondary_family_complete = bool((secondary['status'] == 'tested').all() and np.isfinite(secondary['pvalue']).all())


In [ ]:
if RUN_PIPELINE:
    def benjamini_hochberg(pvalues):
        p = np.asarray(pvalues, dtype=float)
        m = len(p)
        order = np.argsort(p)
        ranked_p = p[order]
        adjusted = ranked_p * m / np.arange(1, m + 1)
        adjusted = np.minimum.accumulate(adjusted[::-1])[::-1]
        adjusted = np.minimum(adjusted, 1.0)
        output = np.empty(m, dtype=float)
        output[order] = adjusted
        return output

    if secondary_family_complete:
        secondary_q = benjamini_hochberg(secondary['pvalue'].to_numpy(dtype=float))
        for idx, qvalue in zip(secondary.index, secondary_q):
            targets.loc[idx, 'secondary_BH_qvalue'] = qvalue

    creb5 = targets.loc[targets['gene'] == PRIMARY_GENE]

    if len(creb5) != 1:
        raise RuntimeError('Unexpected CREB5 target-table structure.')

    creb5 = creb5.iloc[0]

    CREB5_present = creb5['status'] == 'tested'

    CREB5_positive = bool(CREB5_present and np.isfinite(creb5['log2FoldChange']) and (creb5['log2FoldChange'] > 0))

    CREB5_nominal_significant = bool(CREB5_present and np.isfinite(creb5['pvalue']) and (creb5['pvalue'] < 0.05))

    CREB5_confirmatory_success = bool(CREB5_present and CREB5_positive and CREB5_nominal_significant)

    targets.to_csv(TARGET_PATH, index=False)

    n_genome_sig = int(genome_sig.sum())

    n_genome_up = int((genome_sig & (results['log2FoldChange'] > 0)).sum())

    n_genome_down = int((genome_sig & (results['log2FoldChange'] < 0)).sum())

    def json_number(value):
        try:
            value = float(value)
        except (TypeError, ValueError):
            return None
        if not np.isfinite(value):
            return None
        return value

    result_lock = {'dataset': 'GSE157827', 'step': '13B', 'created_utc': datetime.now(timezone.utc).isoformat(), 'biological_analysis_unit': 'donor-level astrocyte pseudobulk', 'number_of_donors': EXPECTED_DONORS, 'AD_donors': 12, 'Control_donors': 9, 'primary_gene_universe': EXPECTED_GENES, 'primary_gene_universe_sha256': EXPECTED_UNIVERSE_SHA256, 'DE_engine': 'PyDESeq2 0.5.4', 'primary_design': '~ age + sex + diagnosis', 'primary_contrast': 'AD versus Control', 'positive_log2FC_meaning': 'higher expression in AD', 'refit_cooks': True, 'cooks_filter': True, 'independent_filter': True, 'genome_wide_alpha': 0.05, 'genome_wide_FDR_significant': n_genome_sig, 'genome_wide_FDR_up_in_AD': n_genome_up, 'genome_wide_FDR_down_in_AD': n_genome_down, 'finite_nominal_pvalues': int(finite_p.sum()), 'finite_genome_wide_adjusted_pvalues': int(finite_padj.sum()), 'CREB5_success_rule': 'present in frozen universe AND log2FoldChange > 0 AND two-sided nominal p < 0.05', 'CREB5_present': bool(CREB5_present), 'CREB5_gene_id': str(creb5['gene_id']) if CREB5_present else None, 'CREB5_baseMean': json_number(creb5['baseMean']), 'CREB5_log2FoldChange': json_number(creb5['log2FoldChange']), 'CREB5_lfcSE': json_number(creb5['lfcSE']), 'CREB5_Wald_stat': json_number(creb5['stat']), 'CREB5_nominal_pvalue': json_number(creb5['pvalue']), 'CREB5_genome_wide_padj': json_number(creb5['genome_wide_padj']), 'CREB5_positive_direction': bool(CREB5_positive), 'CREB5_nominal_p_below_0_05': bool(CREB5_nominal_significant), 'CREB5_CONFIRMATORY_SUCCESS': bool(CREB5_confirmatory_success), 'secondary_locked_genes': SECONDARY_GENES, 'secondary_testing_rule': 'Benjamini-Hochberg correction across the six locked secondary genes', 'secondary_family_complete': bool(secondary_family_complete), 'full_results': str(RESULTS_PATH), 'FDR05_results': str(SIGNIFICANT_PATH), 'locked_target_results': str(TARGET_PATH), 'size_factors': str(SIZE_FACTOR_PATH)}

    with open(STEP13B_LOCK_PATH, 'w', encoding='utf-8') as handle:
        json.dump(result_lock, handle, indent=2)

    print('\n' + '=' * 78)

    print('GSE157827 STEP 13B COMPLETE')

    print('=' * 78)

    print(f'\nGenes tested: {len(results):,}')

    print(f'Finite nominal p-values: {int(finite_p.sum()):,}')

    print(f'Finite genome-wide adjusted p-values: {int(finite_padj.sum()):,}')

    print('\nGenome-wide FDR < 0.05:')

    print(f'    Total: {n_genome_sig}')

    print(f'    Up in AD: {n_genome_up}')

    print(f'    Down in AD: {n_genome_down}')

    print('\nDESeq2 size-factor range:')

    print('    min / median / max =', f'{size_factors.min():.4f}', '/', f'{np.median(size_factors):.4f}', '/', f'{size_factors.max():.4f}')

    print('\n' + '=' * 78)

    print('PRIMARY CONFIRMATORY GENE — CREB5')

    print('=' * 78)


In [ ]:
if RUN_PIPELINE:
    if CREB5_present:
        print('Gene ID:', creb5['gene_id'])
        print('baseMean:', f"{creb5['baseMean']:.6f}")
        print('log2FoldChange (AD vs Control):', f"{creb5['log2FoldChange']:.6f}")
        print('lfcSE:', f"{creb5['lfcSE']:.6f}")
        print('Wald statistic:', f"{creb5['stat']:.6f}")
        print('Nominal two-sided p-value:', f"{creb5['pvalue']:.8g}")
        print('Genome-wide padj:', f"{creb5['genome_wide_padj']:.8g}" if np.isfinite(creb5['genome_wide_padj']) else 'NA')
    else:
        print('CREB5 was NOT present in the frozen primary gene universe.')

    print('\nCREB5 present:', 'YES' if CREB5_present else 'NO')

    print('CREB5 direction positive:', 'YES' if CREB5_positive else 'NO')

    print('CREB5 nominal p < 0.05:', 'YES' if CREB5_nominal_significant else 'NO')

    print('\n' + '=' * 78)

    print('CREB5 CONFIRMATORY SUCCESS:', 'YES' if CREB5_confirmatory_success else 'NO')

    print('=' * 78)

    print('\nSecondary locked genes:')

    secondary_display = targets.loc[targets['gene'].isin(SECONDARY_GENES), ['gene', 'status', 'gene_id', 'baseMean', 'log2FoldChange', 'pvalue', 'secondary_BH_qvalue', 'genome_wide_padj']]

    print(secondary_display.to_string(index=False, float_format=lambda x: f'{x:.6g}'))

    print('\nSecondary six-gene BH family complete:', 'YES' if secondary_family_complete else 'NO')

    print(f'\nFull genome-wide results:\n{RESULTS_PATH}')

    print(f'\nGenome-wide FDR < 0.05 results:\n{SIGNIFICANT_PATH}')

    print(f'\nLocked target results:\n{TARGET_PATH}')

    print(f'\nSize factors:\n{SIZE_FACTOR_PATH}')

    print(f'\nPrimary result lock:\n{STEP13B_LOCK_PATH}')


## Prespecified primary-result sensitivities

Three sensitivity models retain the frozen gene universe: diagnosis only; exclusion of the three severe-QC-loss donors; and exclusion of the Scrublet-rescue donor.


In [ ]:
if RUN_PIPELINE:
    from pathlib import Path

    from datetime import datetime, timezone

    import json

    import hashlib

    import numpy as np

    import pandas as pd

    import scipy.sparse as sp

    from pydeseq2.dds import DeseqDataSet

    from pydeseq2.ds import DeseqStats

    PROJECT_DIR = Path('/content/drive/MyDrive/AD_Astrocyte_Paper_01')

    VALIDATION_DIR = PROJECT_DIR / 'GSE157827_confirmatory_validation'

    COUNTS_PATH = VALIDATION_DIR / 'GSE157827_STEP12_primary_DE_raw_counts_v1.npz'

    FEATURES_PATH = VALIDATION_DIR / 'GSE157827_STEP12_primary_DE_features_v1.csv.gz'

    DESIGN_PATH = VALIDATION_DIR / 'GSE157827_STEP11B_primary_design_manifest_v1.csv'

    PRIMARY_RESULT_LOCK_PATH = VALIDATION_DIR / 'GSE157827_STEP13B_PRIMARY_CONFIRMATORY_RESULT_LOCK_v1.json'

    OUTPUT_PATH = VALIDATION_DIR / 'GSE157827_STEP14A_DESeq2_sensitivity_diagnosis_only_v1.csv.gz'

    CREB5_PATH = VALIDATION_DIR / 'GSE157827_STEP14A_CREB5_sensitivity_diagnosis_only_v1.csv'

    LOCK_PATH = VALIDATION_DIR / 'GSE157827_STEP14A_DIAGNOSIS_ONLY_SENSITIVITY_LOCK_v1.json'

    EXPECTED_DONORS = 21

    EXPECTED_GENES = 16871

    EXPECTED_HASH = 'b2cd51ebe6557d79be9d7d4b9f7d514a77778bcc4857c7371252388827e8466a'

    CREB5_SYMBOL = 'CREB5'

    for path in [COUNTS_PATH, FEATURES_PATH, DESIGN_PATH, PRIMARY_RESULT_LOCK_PATH]:
        if not path.exists():
            raise FileNotFoundError(f'Required file missing:\n{path}')

    with open(PRIMARY_RESULT_LOCK_PATH, 'r', encoding='utf-8') as handle:
        primary_lock = json.load(handle)

    if not primary_lock.get('CREB5_CONFIRMATORY_SUCCESS', False):
        raise RuntimeError('Primary Step 13B result lock does not record CREB5 confirmatory success.')

    X = sp.load_npz(COUNTS_PATH).tocsr()

    if X.shape != (EXPECTED_DONORS, EXPECTED_GENES):
        raise RuntimeError(f'Unexpected count matrix shape: {X.shape}')

    if not np.issubdtype(X.dtype, np.integer):
        raise RuntimeError('Counts are not raw integers.')

    features = pd.read_csv(FEATURES_PATH, dtype={'gene_id': str, 'gene_symbol': str})

    if len(features) != EXPECTED_GENES:
        raise RuntimeError('Frozen feature universe is not 16,871 genes.')

    feature_ids = features['gene_id'].astype(str).tolist()

    payload = ('\n'.join(feature_ids) + '\n').encode('utf-8')

    observed_hash = hashlib.sha256(payload).hexdigest()

    if observed_hash != EXPECTED_HASH:
        raise RuntimeError('Frozen gene-universe SHA-256 mismatch.')

    design = pd.read_csv(DESIGN_PATH, dtype={'GSM': str, 'sample_label': str, 'diagnosis': str}).sort_values('matrix_row').reset_index(drop=True)

    if len(design) != EXPECTED_DONORS:
        raise RuntimeError('Design manifest does not contain 21 donors.')

    if not np.array_equal(design['matrix_row'].to_numpy(), np.arange(EXPECTED_DONORS)):
        raise RuntimeError('matrix_row ordering is not 0..20.')

    sample_ids = design['GSM'].astype(str).tolist()

    counts = pd.DataFrame(X.toarray().astype(np.int64, copy=False), index=pd.Index(sample_ids, dtype=object), columns=pd.Index(feature_ids, dtype=object))

    metadata = pd.DataFrame({'diagnosis': pd.Categorical(design['diagnosis'].astype(str), categories=['Control', 'AD'], ordered=True)}, index=pd.Index(sample_ids, dtype=object))

    metadata.columns = pd.Index(metadata.columns.astype(str).tolist(), dtype=object)

    if not counts.index.equals(metadata.index):
        raise RuntimeError('Counts/metadata donor order mismatch.')

    diagnosis_counts = pd.Series(metadata['diagnosis'].astype(str)).value_counts().to_dict()


In [ ]:
if RUN_PIPELINE:
    if diagnosis_counts != {'AD': 12, 'Control': 9}:
        raise RuntimeError(f'Unexpected diagnosis counts: {diagnosis_counts}')

    print('=' * 76)

    print('GSE157827 STEP 14A — DIAGNOSIS-ONLY SENSITIVITY')

    print('=' * 76)

    print(f'\nDonors: {counts.shape[0]}')

    print(f'Frozen genes: {counts.shape[1]:,}')

    print('Sensitivity model: ~ diagnosis')

    print('Contrast: AD versus Control')

    print('Reference: Control')

    print('\nGene universe changed: NO')

    print('Donors changed: NO')

    print('Astrocyte definition changed: NO')

    dds = DeseqDataSet(counts=counts, metadata=metadata, design='~ diagnosis', refit_cooks=True, n_cpus=2, quiet=False)

    design_matrix = dds.obsm['design_matrix']

    rank = int(np.linalg.matrix_rank(design_matrix.to_numpy()))

    print('\nDesign columns:', design_matrix.columns.tolist())

    print('Design rank:', rank, '/', design_matrix.shape[1])

    if rank != design_matrix.shape[1]:
        raise RuntimeError('Diagnosis-only design is not full rank.')

    print('\nRunning diagnosis-only DESeq2 sensitivity...')

    dds.deseq2()

    print('\nDESEQ2 FIT COMPLETE')

    stats = DeseqStats(dds, contrast=['diagnosis', 'AD', 'Control'], alpha=0.05, cooks_filter=True, independent_filter=True, n_cpus=2, quiet=False)

    stats.summary()

    results = stats.results_df.copy()

    results.index = pd.Index(results.index.astype(str).tolist(), dtype=object)

    results.index.name = 'gene_id'

    results = results.reset_index().merge(features[['gene_id', 'gene_symbol']], on='gene_id', how='left', validate='one_to_one')

    results = results[['gene_id', 'gene_symbol', 'baseMean', 'log2FoldChange', 'lfcSE', 'stat', 'pvalue', 'padj']]

    if len(results) != EXPECTED_GENES:
        raise RuntimeError('Sensitivity results do not contain all frozen genes.')

    results.to_csv(OUTPUT_PATH, index=False, compression={'method': 'gzip', 'compresslevel': 6})

    creb5 = results.loc[results['gene_symbol'].astype(str) == CREB5_SYMBOL].copy()

    if len(creb5) != 1:
        raise RuntimeError(f'Expected exactly one CREB5 feature; found {len(creb5)}.')

    creb5 = creb5.iloc[0]

    creb5_positive = bool(creb5['log2FoldChange'] > 0)

    creb5_p05 = bool(np.isfinite(creb5['pvalue']) and creb5['pvalue'] < 0.05)

    creb5_sensitivity_support = bool(creb5_positive and creb5_p05)

    creb5_table = pd.DataFrame([{'gene': CREB5_SYMBOL, 'gene_id': creb5['gene_id'], 'model': '~ diagnosis', 'log2FoldChange': creb5['log2FoldChange'], 'lfcSE': creb5['lfcSE'], 'stat': creb5['stat'], 'pvalue': creb5['pvalue'], 'genome_wide_padj': creb5['padj'], 'positive_direction': creb5_positive, 'nominal_p_below_0_05': creb5_p05, 'supports_primary_result': creb5_sensitivity_support}])

    creb5_table.to_csv(CREB5_PATH, index=False)

    finite_padj = np.isfinite(results['padj'])

    sig = finite_padj & (results['padj'] < 0.05)

    n_sig = int(sig.sum())

    n_up = int((sig & (results['log2FoldChange'] > 0)).sum())

    n_down = int((sig & (results['log2FoldChange'] < 0)).sum())

    lock = {'dataset': 'GSE157827', 'step': '14A', 'analysis_type': 'diagnosis-only sensitivity', 'created_utc': datetime.now(timezone.utc).isoformat(), 'primary_model': '~ age + sex + diagnosis', 'sensitivity_model': '~ diagnosis', 'contrast': 'AD versus Control', 'donors': 21, 'gene_universe': 16871, 'gene_universe_sha256': EXPECTED_HASH, 'donor_set_changed': False, 'gene_universe_changed': False, 'astrocyte_definition_changed': False, 'CREB5_gene_id': str(creb5['gene_id']), 'CREB5_log2FoldChange': float(creb5['log2FoldChange']), 'CREB5_lfcSE': float(creb5['lfcSE']), 'CREB5_Wald_stat': float(creb5['stat']), 'CREB5_nominal_pvalue': float(creb5['pvalue']), 'CREB5_genome_wide_padj': float(creb5['padj']) if np.isfinite(creb5['padj']) else None, 'CREB5_positive': creb5_positive, 'CREB5_nominal_p_below_0_05': creb5_p05, 'CREB5_supports_primary_result': creb5_sensitivity_support, 'genome_wide_FDR05': n_sig, 'genome_wide_up': n_up, 'genome_wide_down': n_down, 'full_results': str(OUTPUT_PATH), 'CREB5_result': str(CREB5_PATH)}

    with open(LOCK_PATH, 'w', encoding='utf-8') as handle:
        json.dump(lock, handle, indent=2)

    print('\n' + '=' * 76)

    print('GSE157827 STEP 14A COMPLETE')

    print('=' * 76)

    print('\nSensitivity model: ~ diagnosis')

    print('Donors retained: 21/21')


In [ ]:
if RUN_PIPELINE:
    print('Frozen genes retained: 16,871/16,871')

    print('\nGenome-wide FDR < 0.05:')

    print(f'    Total: {n_sig}')

    print(f'    Up in AD: {n_up}')

    print(f'    Down in AD: {n_down}')

    print('\n' + '=' * 76)

    print('CREB5 — DIAGNOSIS-ONLY SENSITIVITY')

    print('=' * 76)

    print('Gene ID:', creb5['gene_id'])

    print('log2FoldChange:', f"{creb5['log2FoldChange']:.6f}")

    print('lfcSE:', f"{creb5['lfcSE']:.6f}")

    print('Wald statistic:', f"{creb5['stat']:.6f}")

    print('Nominal p-value:', f"{creb5['pvalue']:.8g}")

    print('Genome-wide padj:', f"{creb5['padj']:.8g}" if np.isfinite(creb5['padj']) else 'NA')

    print('\nDirection remains positive:', 'YES' if creb5_positive else 'NO')

    print('Nominal p < 0.05:', 'YES' if creb5_p05 else 'NO')

    print('Supports primary CREB5 result:', 'YES' if creb5_sensitivity_support else 'NO')

    print(f'\nFull sensitivity results:\n{OUTPUT_PATH}')

    print(f'\nCREB5 sensitivity result:\n{CREB5_PATH}')

    print(f'\nSensitivity lock:\n{LOCK_PATH}')


In [ ]:
if RUN_PIPELINE:
    from pathlib import Path

    from datetime import datetime, timezone

    import json

    import hashlib

    import numpy as np

    import pandas as pd

    import scipy.sparse as sp

    from pydeseq2.dds import DeseqDataSet

    from pydeseq2.ds import DeseqStats

    PROJECT_DIR = Path('/content/drive/MyDrive/AD_Astrocyte_Paper_01')

    VALIDATION_DIR = PROJECT_DIR / 'GSE157827_confirmatory_validation'

    COUNTS_PATH = VALIDATION_DIR / 'GSE157827_STEP12_primary_DE_raw_counts_v1.npz'

    FEATURES_PATH = VALIDATION_DIR / 'GSE157827_STEP12_primary_DE_features_v1.csv.gz'

    DESIGN_PATH = VALIDATION_DIR / 'GSE157827_STEP11B_primary_design_manifest_v1.csv'

    PRIMARY_RESULT_LOCK_PATH = VALIDATION_DIR / 'GSE157827_STEP13B_PRIMARY_CONFIRMATORY_RESULT_LOCK_v1.json'

    OUTPUT_PATH = VALIDATION_DIR / 'GSE157827_STEP14B_DESeq2_sensitivity_exclude_severe_QC_loss_v1.csv.gz'

    CREB5_PATH = VALIDATION_DIR / 'GSE157827_STEP14B_CREB5_sensitivity_exclude_severe_QC_loss_v1.csv'

    DONOR_PATH = VALIDATION_DIR / 'GSE157827_STEP14B_retained_donors_v1.csv'

    LOCK_PATH = VALIDATION_DIR / 'GSE157827_STEP14B_SEVERE_QC_LOSS_SENSITIVITY_LOCK_v1.json'

    EXPECTED_GENES = 16871

    EXPECTED_RETAINED_DONORS = 18

    EXPECTED_HASH = 'b2cd51ebe6557d79be9d7d4b9f7d514a77778bcc4857c7371252388827e8466a'

    CREB5_SYMBOL = 'CREB5'

    EXCLUDE = {'AD6': 'GSM4775565', 'NC11': 'GSM4775575', 'NC16': 'GSM4775579'}

    for path in [COUNTS_PATH, FEATURES_PATH, DESIGN_PATH, PRIMARY_RESULT_LOCK_PATH]:
        if not path.exists():
            raise FileNotFoundError(f'Required input missing:\n{path}')

    with open(PRIMARY_RESULT_LOCK_PATH, 'r', encoding='utf-8') as handle:
        primary_lock = json.load(handle)

    if not primary_lock.get('CREB5_CONFIRMATORY_SUCCESS', False):
        raise RuntimeError('Step 13B primary CREB5 result lock is missing or does not record confirmatory success.')

    X = sp.load_npz(COUNTS_PATH).tocsr()

    if X.shape != (21, EXPECTED_GENES):
        raise RuntimeError(f'Unexpected original matrix shape: {X.shape}')

    if not np.issubdtype(X.dtype, np.integer):
        raise RuntimeError('Counts are not raw integers.')

    features = pd.read_csv(FEATURES_PATH, dtype={'gene_id': str, 'gene_symbol': str})

    if len(features) != EXPECTED_GENES:
        raise RuntimeError('Frozen feature table is not 16,871 genes.')

    feature_ids = features['gene_id'].astype(str).tolist()

    payload = ('\n'.join(feature_ids) + '\n').encode('utf-8')

    observed_hash = hashlib.sha256(payload).hexdigest()

    if observed_hash != EXPECTED_HASH:
        raise RuntimeError('Frozen gene-universe SHA-256 mismatch.')

    design = pd.read_csv(DESIGN_PATH, dtype={'GSM': str, 'sample_label': str, 'diagnosis': str, 'sex': str}).sort_values('matrix_row').reset_index(drop=True)

    if len(design) != 21:
        raise RuntimeError('Expected 21 original donors.')

    if not np.array_equal(design['matrix_row'].to_numpy(), np.arange(21)):
        raise RuntimeError('Original donor matrix ordering changed.')


In [ ]:
if RUN_PIPELINE:
    for sample_label, expected_gsm in EXCLUDE.items():
        rows = design.loc[design['sample_label'] == sample_label]
        if len(rows) != 1:
            raise RuntimeError(f'Expected exactly one row for {sample_label}.')
        observed_gsm = str(rows.iloc[0]['GSM'])
        if observed_gsm != expected_gsm:
            raise RuntimeError(f'{sample_label}: expected {expected_gsm}, found {observed_gsm}.')

    print('=' * 78)

    print('GSE157827 STEP 14B — SEVERE-QC-LOSS EXCLUSION SENSITIVITY')

    print('=' * 78)

    print('\nFrozen exclusions:')

    for sample_label, gsm in EXCLUDE.items():
        print(f'    {sample_label}: {gsm}')

    exclude_labels = set(EXCLUDE.keys())

    keep_mask = ~design['sample_label'].isin(exclude_labels).to_numpy()

    if int(keep_mask.sum()) != EXPECTED_RETAINED_DONORS:
        raise RuntimeError('Expected 18 donors after exclusion.')

    X_sub = X[keep_mask, :]

    design_sub = design.loc[keep_mask].copy().reset_index(drop=True)

    if X_sub.shape != (EXPECTED_RETAINED_DONORS, EXPECTED_GENES):
        raise RuntimeError(f'Unexpected sensitivity matrix shape: {X_sub.shape}')

    group_counts = design_sub['diagnosis'].value_counts().to_dict()

    if group_counts != {'AD': 11, 'Control': 7}:
        raise RuntimeError(f'Unexpected retained group counts: {group_counts}')

    if set(design_sub['sample_label']) & exclude_labels:
        raise RuntimeError('At least one excluded donor remains.')

    design_sub.to_csv(DONOR_PATH, index=False)

    sample_ids = design_sub['GSM'].astype(str).tolist()

    counts = pd.DataFrame(X_sub.toarray().astype(np.int64, copy=False), index=pd.Index(sample_ids, dtype=object), columns=pd.Index(feature_ids, dtype=object))

    metadata = design_sub[['age', 'sex', 'diagnosis']].copy()

    metadata['age'] = pd.to_numeric(metadata['age'], errors='raise')

    metadata['sex'] = pd.Categorical(metadata['sex'].astype(str), categories=['F', 'M'], ordered=True)

    metadata['diagnosis'] = pd.Categorical(metadata['diagnosis'].astype(str), categories=['Control', 'AD'], ordered=True)

    metadata.index = pd.Index(sample_ids, dtype=object)

    metadata.columns = pd.Index(metadata.columns.astype(str).tolist(), dtype=object)

    if not counts.index.equals(metadata.index):
        raise RuntimeError('Counts/metadata sample order mismatch.')

    if metadata.isna().any().any():
        raise RuntimeError('Missing model metadata after donor exclusion.')

    print(f'\nDonors retained: {len(design_sub)}/21')

    print('AD retained: 11')

    print('Control retained: 7')

    print(f'Frozen genes retained: {counts.shape[1]:,}/16,871')

    print('\nModel remains:')

    print('    ~ age + sex + diagnosis')

    print('\nGene universe refiltered: NO')

    print('Astrocyte definition changed: NO')

    print('Only donor exclusion changed: YES')

    dds = DeseqDataSet(counts=counts, metadata=metadata, design='~ age + sex + diagnosis', refit_cooks=True, n_cpus=2, quiet=False)

    design_matrix = dds.obsm['design_matrix']

    rank = int(np.linalg.matrix_rank(design_matrix.to_numpy()))

    print('\nDesign columns:')

    print(design_matrix.columns.tolist())

    print('Design rank:', rank, '/', design_matrix.shape[1])


In [ ]:
if RUN_PIPELINE:
    if rank != design_matrix.shape[1]:
        raise RuntimeError('Reduced sensitivity design is not full rank.')

    if design_matrix.shape[1] != 4:
        raise RuntimeError('Expected four model parameters.')

    print('\nRunning severe-QC-loss exclusion sensitivity...')

    dds.deseq2()

    print('\nDESEQ2 FIT COMPLETE')

    stats = DeseqStats(dds, contrast=['diagnosis', 'AD', 'Control'], alpha=0.05, cooks_filter=True, independent_filter=True, n_cpus=2, quiet=False)

    stats.summary()

    results = stats.results_df.copy()

    results.index = pd.Index(results.index.astype(str).tolist(), dtype=object)

    results.index.name = 'gene_id'

    results = results.reset_index().merge(features[['gene_id', 'gene_symbol']], on='gene_id', how='left', validate='one_to_one')

    results = results[['gene_id', 'gene_symbol', 'baseMean', 'log2FoldChange', 'lfcSE', 'stat', 'pvalue', 'padj']]

    if len(results) != EXPECTED_GENES:
        raise RuntimeError('Sensitivity results do not contain all 16,871 frozen genes.')

    results.to_csv(OUTPUT_PATH, index=False, compression={'method': 'gzip', 'compresslevel': 6})

    creb5_rows = results.loc[results['gene_symbol'].astype(str) == CREB5_SYMBOL]

    if len(creb5_rows) != 1:
        raise RuntimeError(f'Expected exactly one CREB5 row; found {len(creb5_rows)}.')

    creb5 = creb5_rows.iloc[0]

    positive = bool(np.isfinite(creb5['log2FoldChange']) and creb5['log2FoldChange'] > 0)

    p05 = bool(np.isfinite(creb5['pvalue']) and creb5['pvalue'] < 0.05)

    supports_primary = bool(positive and p05)

    creb5_output = pd.DataFrame([{'gene': 'CREB5', 'gene_id': creb5['gene_id'], 'model': '~ age + sex + diagnosis', 'excluded_donors': 'AD6;NC11;NC16', 'retained_donors': 18, 'log2FoldChange': creb5['log2FoldChange'], 'lfcSE': creb5['lfcSE'], 'stat': creb5['stat'], 'pvalue': creb5['pvalue'], 'genome_wide_padj': creb5['padj'], 'positive_direction': positive, 'nominal_p_below_0_05': p05, 'supports_primary_result': supports_primary}])

    creb5_output.to_csv(CREB5_PATH, index=False)

    finite_padj = np.isfinite(results['padj'])

    sig = finite_padj & (results['padj'] < 0.05)

    n_sig = int(sig.sum())

    n_up = int((sig & (results['log2FoldChange'] > 0)).sum())

    n_down = int((sig & (results['log2FoldChange'] < 0)).sum())

    def safe_number(value):
        value = float(value)
        return value if np.isfinite(value) else None

    lock = {'dataset': 'GSE157827', 'step': '14B', 'analysis_type': 'severe-QC-loss donor-exclusion sensitivity', 'created_utc': datetime.now(timezone.utc).isoformat(), 'exclusion_rule': 'pre-DE author-QC pass percentage < 80%', 'excluded_sample_labels': ['AD6', 'NC11', 'NC16'], 'excluded_GSM': ['GSM4775565', 'GSM4775575', 'GSM4775579'], 'original_donors': 21, 'retained_donors': 18, 'retained_AD': 11, 'retained_Control': 7, 'model': '~ age + sex + diagnosis', 'contrast': 'AD versus Control', 'gene_universe': EXPECTED_GENES, 'gene_universe_sha256': EXPECTED_HASH, 'gene_universe_refiltered': False, 'astrocyte_definition_changed': False, 'only_donor_set_changed': True, 'CREB5_gene_id': str(creb5['gene_id']), 'CREB5_log2FoldChange': safe_number(creb5['log2FoldChange']), 'CREB5_lfcSE': safe_number(creb5['lfcSE']), 'CREB5_Wald_stat': safe_number(creb5['stat']), 'CREB5_nominal_pvalue': safe_number(creb5['pvalue']), 'CREB5_genome_wide_padj': safe_number(creb5['padj']), 'CREB5_positive': positive, 'CREB5_nominal_p_below_0_05': p05, 'CREB5_supports_primary_result': supports_primary, 'genome_wide_FDR05': n_sig, 'genome_wide_up': n_up, 'genome_wide_down': n_down, 'full_results': str(OUTPUT_PATH), 'CREB5_result': str(CREB5_PATH), 'retained_donor_manifest': str(DONOR_PATH)}

    with open(LOCK_PATH, 'w', encoding='utf-8') as handle:
        json.dump(lock, handle, indent=2)

    print('\n' + '=' * 78)

    print('GSE157827 STEP 14B COMPLETE')

    print('=' * 78)

    print('\nExcluded:')

    print('    AD6  / GSM4775565')

    print('    NC11 / GSM4775575')

    print('    NC16 / GSM4775579')

    print('\nDonors retained: 18/21')

    print('AD: 11')

    print('Control: 7')

    print('Frozen genes: 16,871')

    print('Model: ~ age + sex + diagnosis')

    print('\nGenome-wide FDR < 0.05:')

    print(f'    Total: {n_sig}')

    print(f'    Up in AD: {n_up}')

    print(f'    Down in AD: {n_down}')

    print('\n' + '=' * 78)

    print('CREB5 — SEVERE-QC-LOSS EXCLUSION SENSITIVITY')


In [ ]:
if RUN_PIPELINE:
    print('=' * 78)

    print('Gene ID:', creb5['gene_id'])

    print('log2FoldChange:', f"{creb5['log2FoldChange']:.6f}")

    print('lfcSE:', f"{creb5['lfcSE']:.6f}")

    print('Wald statistic:', f"{creb5['stat']:.6f}")

    print('Nominal p-value:', f"{creb5['pvalue']:.8g}")

    print('Genome-wide padj:', f"{creb5['padj']:.8g}" if np.isfinite(creb5['padj']) else 'NA')

    print('\nDirection remains positive:', 'YES' if positive else 'NO')

    print('Nominal p < 0.05:', 'YES' if p05 else 'NO')

    print('Supports primary CREB5 result:', 'YES' if supports_primary else 'NO')

    print(f'\nFull sensitivity results:\n{OUTPUT_PATH}')

    print(f'\nCREB5 sensitivity result:\n{CREB5_PATH}')

    print(f'\nRetained donor manifest:\n{DONOR_PATH}')

    print(f'\nSensitivity lock:\n{LOCK_PATH}')


In [ ]:
if RUN_PIPELINE:
    from pathlib import Path

    from datetime import datetime, timezone

    import hashlib

    import importlib.metadata as md

    import json

    import numpy as np

    import pandas as pd

    import scipy.sparse as sp

    from pydeseq2.dds import DeseqDataSet

    from pydeseq2.ds import DeseqStats

    REQUIRED = {'numpy': '2.1.3', 'pandas': '2.2.3', 'scipy': '1.16.3', 'scikit-learn': '1.6.1', 'anndata': '0.12.6', 'pydeseq2': '0.5.4', 'formulaic': '1.2.2', 'formulaic-contrasts': '1.0.0'}

    observed_versions = {}

    for package, expected in REQUIRED.items():
        try:
            observed = md.version(package)
        except md.PackageNotFoundError:
            observed = None
        observed_versions[package] = observed
        if observed != expected:
            raise RuntimeError(f'Environment changed: {package} expected {expected}, found {observed}. Re-run Step 13A and restart if requested.')

    PROJECT_DIR = Path('/content/drive/MyDrive/AD_Astrocyte_Paper_01')

    VALIDATION_DIR = PROJECT_DIR / 'GSE157827_confirmatory_validation'

    COUNTS_PATH = VALIDATION_DIR / 'GSE157827_STEP12_primary_DE_raw_counts_v1.npz'

    FEATURES_PATH = VALIDATION_DIR / 'GSE157827_STEP12_primary_DE_features_v1.csv.gz'

    DESIGN_PATH = VALIDATION_DIR / 'GSE157827_STEP11B_primary_design_manifest_v1.csv'

    STEP12_LOCK_PATH = VALIDATION_DIR / 'GSE157827_STEP12_GENE_UNIVERSE_LOCK_v1.json'

    STEP13B_LOCK_PATH = VALIDATION_DIR / 'GSE157827_STEP13B_PRIMARY_CONFIRMATORY_RESULT_LOCK_v1.json'

    DOUBLET_RESCUE_LOCK_PATH = VALIDATION_DIR / 'GSE157827_DOUBLET_RESCUE_LOCK_v1_1.json'

    OUTPUT_PATH = VALIDATION_DIR / 'GSE157827_STEP14C_DESeq2_sensitivity_exclude_scrublet_rescue_donor_v1.csv.gz'

    CREB5_PATH = VALIDATION_DIR / 'GSE157827_STEP14C_CREB5_sensitivity_exclude_scrublet_rescue_donor_v1.csv'

    DONOR_PATH = VALIDATION_DIR / 'GSE157827_STEP14C_retained_donors_v1.csv'

    LOCK_PATH = VALIDATION_DIR / 'GSE157827_STEP14C_SCRUBLET_RESCUE_DONOR_EXCLUSION_LOCK_v1.json'

    for path in [COUNTS_PATH, FEATURES_PATH, DESIGN_PATH, STEP12_LOCK_PATH, STEP13B_LOCK_PATH, DOUBLET_RESCUE_LOCK_PATH]:
        if not path.exists():
            raise FileNotFoundError(f'Required frozen input is missing:\n{path}')

    with open(STEP12_LOCK_PATH, 'r', encoding='utf-8') as handle:
        step12_lock = json.load(handle)

    with open(STEP13B_LOCK_PATH, 'r', encoding='utf-8') as handle:
        primary_lock = json.load(handle)

    with open(DOUBLET_RESCUE_LOCK_PATH, 'r', encoding='utf-8') as handle:
        rescue_lock = json.load(handle)

    if not step12_lock.get('primary_gene_universe_frozen', False):
        raise RuntimeError('Step 12 gene universe is not frozen.')

    if rescue_lock.get('rescue_GSM') != 'GSM4775563':
        raise RuntimeError('The frozen Scrublet rescue donor is not GSM4775563.')

    EXPECTED_ORIGINAL_DONORS = 21

    EXPECTED_RETAINED_DONORS = 20

    EXPECTED_GENES = 16871

    EXCLUDED_GSM = 'GSM4775563'

    EXCLUDED_SAMPLE = 'AD4'

    X = sp.load_npz(COUNTS_PATH).tocsr()

    if X.shape != (EXPECTED_ORIGINAL_DONORS, EXPECTED_GENES):
        raise RuntimeError(f'Unexpected frozen count-matrix shape: {X.shape}')

    if X.nnz == 0 or np.any(X.data < 0):
        raise RuntimeError('Frozen count matrix is empty or contains negatives.')


In [ ]:
if RUN_PIPELINE:
    if not np.all(np.equal(X.data, np.rint(X.data))):
        raise RuntimeError('Frozen count matrix is not integer-valued.')

    features = pd.read_csv(FEATURES_PATH, dtype={'gene_id': str, 'gene_symbol': str})

    if len(features) != EXPECTED_GENES:
        raise RuntimeError('Frozen feature table is not 16,871 genes.')

    if features['gene_id'].duplicated().any():
        raise RuntimeError('Duplicated gene IDs in frozen feature table.')

    feature_ids = features['gene_id'].astype(str).tolist()

    gene_id_payload = ('\n'.join(feature_ids) + '\n').encode('utf-8')

    observed_universe_hash = hashlib.sha256(gene_id_payload).hexdigest()

    expected_universe_hash = step12_lock.get('primary_universe_gene_id_sha256')

    if observed_universe_hash != expected_universe_hash:
        raise RuntimeError('Frozen gene-universe SHA-256 mismatch.')

    design = pd.read_csv(DESIGN_PATH, dtype={'GSM': str, 'sample_label': str, 'diagnosis': str, 'sex': str}).sort_values('matrix_row').reset_index(drop=True)

    if len(design) != EXPECTED_ORIGINAL_DONORS:
        raise RuntimeError('Expected 21 original donors.')

    if not np.array_equal(design['matrix_row'].to_numpy(), np.arange(EXPECTED_ORIGINAL_DONORS)):
        raise RuntimeError('Original donor matrix ordering changed.')

    excluded_rows = design.loc[design['GSM'] == EXCLUDED_GSM]

    if len(excluded_rows) != 1:
        raise RuntimeError('Expected exactly one GSM4775563 row.')

    if str(excluded_rows.iloc[0]['sample_label']) != EXCLUDED_SAMPLE:
        raise RuntimeError('GSM4775563 is not linked to AD4 as expected.')

    keep_mask = design['GSM'].ne(EXCLUDED_GSM).to_numpy()

    X_sub = X[keep_mask, :]

    design_sub = design.loc[keep_mask].copy().reset_index(drop=True)

    if X_sub.shape != (EXPECTED_RETAINED_DONORS, EXPECTED_GENES):
        raise RuntimeError(f'Unexpected sensitivity matrix shape: {X_sub.shape}')

    group_counts = design_sub['diagnosis'].value_counts().to_dict()

    if group_counts != {'AD': 11, 'Control': 9}:
        raise RuntimeError(f'Unexpected retained group counts: {group_counts}')

    if EXCLUDED_GSM in set(design_sub['GSM']):
        raise RuntimeError('The rescue donor remains in the sensitivity cohort.')

    design_sub.to_csv(DONOR_PATH, index=False)

    sample_ids = design_sub['GSM'].astype(str).tolist()

    counts = pd.DataFrame(X_sub.toarray().astype(np.int64, copy=False), index=pd.Index(sample_ids, dtype=object), columns=pd.Index(feature_ids, dtype=object))

    metadata = design_sub[['age', 'sex', 'diagnosis']].copy()

    metadata['age'] = pd.to_numeric(metadata['age'], errors='raise')

    metadata['sex'] = pd.Categorical(metadata['sex'].astype(str), categories=['F', 'M'], ordered=True)

    metadata['diagnosis'] = pd.Categorical(metadata['diagnosis'].astype(str), categories=['Control', 'AD'], ordered=True)

    metadata.index = pd.Index(sample_ids, dtype=object)

    metadata.columns = pd.Index(metadata.columns.astype(str), dtype=object)

    if not counts.index.equals(metadata.index):
        raise RuntimeError('Counts/metadata donor order mismatch.')

    if metadata.isna().any().any():
        raise RuntimeError('Missing metadata after donor exclusion.')

    print('=' * 78)

    print('GSE157827 STEP 14C — SCRUBLET-RESCUE DONOR EXCLUSION')

    print('=' * 78)

    print(f'\nExcluded donor: {EXCLUDED_GSM} / {EXCLUDED_SAMPLE}')

    print('Reason: automatic Scrublet threshold required locked rescue')

    print(f'Donors retained: {len(design_sub)}/21')

    print('AD retained: 11')

    print('Control retained: 9')

    print(f'Frozen genes retained: {counts.shape[1]:,}/16,871')


In [ ]:
if RUN_PIPELINE:
    print('Model unchanged: ~ age + sex + diagnosis')

    print('Gene universe refiltered: NO')

    print('Astrocyte definition changed: NO')

    dds = DeseqDataSet(counts=counts, metadata=metadata, design='~ age + sex + diagnosis', refit_cooks=True, n_cpus=2, quiet=False)

    design_matrix = dds.obsm['design_matrix']

    rank = int(np.linalg.matrix_rank(design_matrix.to_numpy()))

    print('\nDesign columns:', design_matrix.columns.tolist())

    print('Design rank:', rank, '/', design_matrix.shape[1])

    if rank != design_matrix.shape[1] or design_matrix.shape[1] != 4:
        raise RuntimeError('Sensitivity design is not full rank with 4 terms.')

    print('\nRunning donor-exclusion DESeq2 sensitivity...')

    dds.deseq2()

    stats = DeseqStats(dds, contrast=['diagnosis', 'AD', 'Control'], alpha=0.05, cooks_filter=True, independent_filter=True, n_cpus=2, quiet=False)

    stats.summary()

    results = stats.results_df.copy()

    results.index = pd.Index(results.index.astype(str), dtype=object)

    results.index.name = 'gene_id'

    results = results.reset_index().merge(features[['gene_id', 'gene_symbol']], on='gene_id', how='left', validate='one_to_one')

    results = results[['gene_id', 'gene_symbol', 'baseMean', 'log2FoldChange', 'lfcSE', 'stat', 'pvalue', 'padj']]

    if len(results) != EXPECTED_GENES:
        raise RuntimeError('Sensitivity results do not contain 16,871 genes.')

    results.to_csv(OUTPUT_PATH, index=False, compression={'method': 'gzip', 'compresslevel': 6})

    creb5_rows = results.loc[results['gene_symbol'].astype(str) == 'CREB5']

    if len(creb5_rows) != 1:
        raise RuntimeError(f'Expected one CREB5 row; found {len(creb5_rows)}.')

    creb5 = creb5_rows.iloc[0]

    positive = bool(np.isfinite(creb5['log2FoldChange']) and float(creb5['log2FoldChange']) > 0)

    nominal_p05 = bool(np.isfinite(creb5['pvalue']) and float(creb5['pvalue']) < 0.05)

    supports_primary = bool(positive and nominal_p05)

    primary_lfc = primary_lock.get('CREB5_log2FoldChange')

    primary_p = primary_lock.get('CREB5_nominal_pvalue')

    creb5_output = pd.DataFrame([{'gene': 'CREB5', 'gene_id': creb5['gene_id'], 'model': '~ age + sex + diagnosis', 'excluded_GSM': EXCLUDED_GSM, 'excluded_sample': EXCLUDED_SAMPLE, 'retained_donors': EXPECTED_RETAINED_DONORS, 'log2FoldChange': creb5['log2FoldChange'], 'lfcSE': creb5['lfcSE'], 'stat': creb5['stat'], 'pvalue': creb5['pvalue'], 'genome_wide_padj': creb5['padj'], 'positive_direction': positive, 'nominal_p_below_0_05': nominal_p05, 'supports_primary_result': supports_primary}])

    creb5_output.to_csv(CREB5_PATH, index=False)

    finite_padj = np.isfinite(results['padj'])

    significant = finite_padj & results['padj'].lt(0.05)

    n_sig = int(significant.sum())

    n_up = int((significant & results['log2FoldChange'].gt(0)).sum())

    n_down = int((significant & results['log2FoldChange'].lt(0)).sum())

    def safe_number(value):
        if value is None:
            return None
        value = float(value)
        return value if np.isfinite(value) else None

    result_lock = {'dataset': 'GSE157827', 'step': '14C', 'analysis_type': 'Scrublet-rescue donor exclusion sensitivity', 'created_utc': datetime.now(timezone.utc).isoformat(), 'rationale': 'Prospectively specified sensitivity from the Step 7B doublet-rescue decision lock.', 'excluded_GSM': EXCLUDED_GSM, 'excluded_sample_label': EXCLUDED_SAMPLE, 'original_donors': EXPECTED_ORIGINAL_DONORS, 'retained_donors': EXPECTED_RETAINED_DONORS, 'retained_AD': 11, 'retained_Control': 9, 'model': '~ age + sex + diagnosis', 'contrast': 'AD versus Control', 'design_rank': rank, 'gene_universe': EXPECTED_GENES, 'gene_universe_sha256': observed_universe_hash, 'gene_universe_refiltered': False, 'astrocyte_definition_changed': False, 'only_donor_set_changed': True, 'software': observed_versions, 'primary_CREB5_log2FoldChange': safe_number(primary_lfc), 'primary_CREB5_nominal_pvalue': safe_number(primary_p), 'sensitivity_CREB5_gene_id': str(creb5['gene_id']), 'sensitivity_CREB5_log2FoldChange': safe_number(creb5['log2FoldChange']), 'sensitivity_CREB5_lfcSE': safe_number(creb5['lfcSE']), 'sensitivity_CREB5_Wald_stat': safe_number(creb5['stat']), 'sensitivity_CREB5_nominal_pvalue': safe_number(creb5['pvalue']), 'sensitivity_CREB5_genome_wide_padj': safe_number(creb5['padj']), 'CREB5_positive': positive, 'CREB5_nominal_p_below_0_05': nominal_p05, 'CREB5_supports_primary_result': supports_primary, 'genome_wide_FDR05': n_sig, 'genome_wide_up': n_up, 'genome_wide_down': n_down, 'full_results': str(OUTPUT_PATH), 'CREB5_result': str(CREB5_PATH), 'retained_donor_manifest': str(DONOR_PATH)}

    with open(LOCK_PATH, 'w', encoding='utf-8') as handle:
        json.dump(result_lock, handle, indent=2)

    print('\n' + '=' * 78)

    print('GSE157827 STEP 14C COMPLETE')

    print('=' * 78)

    print(f'\nExcluded: {EXCLUDED_SAMPLE} / {EXCLUDED_GSM}')

    print('Donors retained: 20/21 (11 AD, 9 Control)')

    print('Frozen genes: 16,871')

    print('Model: ~ age + sex + diagnosis')

    print('\nGenome-wide FDR < 0.05:')

    print(f'    Total: {n_sig}')


In [ ]:
if RUN_PIPELINE:
    print(f'    Up in AD: {n_up}')

    print(f'    Down in AD: {n_down}')

    print('\n' + '=' * 78)

    print('CREB5 — SCRUBLET-RESCUE DONOR EXCLUSION')

    print('=' * 78)

    print('Gene ID:', creb5['gene_id'])

    print('log2FoldChange:', f"{creb5['log2FoldChange']:.6f}")

    print('lfcSE:', f"{creb5['lfcSE']:.6f}")

    print('Wald statistic:', f"{creb5['stat']:.6f}")

    print('Nominal p-value:', f"{creb5['pvalue']:.8g}")

    print('Genome-wide padj:', f"{creb5['padj']:.8g}" if np.isfinite(creb5['padj']) else 'NA')

    print('\nDirection remains positive:', 'YES' if positive else 'NO')

    print('Nominal p < 0.05:', 'YES' if nominal_p05 else 'NO')

    print('Supports primary CREB5 result:', 'YES' if supports_primary else 'NO')

    print(f'\nFull sensitivity results:\n{OUTPUT_PATH}')

    print(f'\nCREB5 sensitivity result:\n{CREB5_PATH}')

    print(f'\nRetained donor manifest:\n{DONOR_PATH}')

    print(f'\nSensitivity lock:\n{LOCK_PATH}')


## Independent astrocyte-annotation sensitivity

This post-result robustness analysis changes the astrocyte-definition route while keeping the frozen 16,871-gene universe and primary DE model. Seven locked targets are excluded from feature selection. The original default LOESS span of 0.30 was numerically unstable; the prespecified stability sequence selects the first successful span, which was 0.40.


In [ ]:
if RUN_PIPELINE:
    from pathlib import Path

    from datetime import datetime, timezone

    import json

    PROJECT_DIR = Path('/content/drive/MyDrive/AD_Astrocyte_Paper_01')

    VALIDATION_DIR = PROJECT_DIR / 'GSE157827_confirmatory_validation'

    STEP8_DIR = VALIDATION_DIR / 'GSE157827_STEP8_raw_singlet_matrix_v1_1'

    OBS_PATH = VALIDATION_DIR / 'GSE157827_STEP8_singlet_obs_v1_1.csv.gz'

    VAR_PATH = VALIDATION_DIR / 'GSE157827_STEP8_feature_metadata_v1_1.csv.gz'

    PARTITION_PATH = VALIDATION_DIR / 'GSE157827_STEP8_matrix_partition_manifest_v1_1.csv'

    OLD_CLUSTER_PLAN_PATH = VALIDATION_DIR / 'GSE157827_STEP9D_CLUSTERING_PLAN_LOCK_v1.json'

    PRIMARY_RESULT_LOCK_PATH = VALIDATION_DIR / 'GSE157827_STEP13B_PRIMARY_CONFIRMATORY_RESULT_LOCK_v1.json'

    SCRUBLET_SENSITIVITY_LOCK_PATH = VALIDATION_DIR / 'GSE157827_STEP14C_SCRUBLET_RESCUE_DONOR_EXCLUSION_LOCK_v1.json'

    PLAN_LOCK_PATH = VALIDATION_DIR / 'GSE157827_STEP15A_INDEPENDENT_ANNOTATION_SENSITIVITY_PLAN_LOCK_v1.json'

    required_paths = [STEP8_DIR, OBS_PATH, VAR_PATH, PARTITION_PATH, OLD_CLUSTER_PLAN_PATH, PRIMARY_RESULT_LOCK_PATH, SCRUBLET_SENSITIVITY_LOCK_PATH]

    for path in required_paths:
        if not path.exists():
            raise FileNotFoundError(f'Required frozen checkpoint missing:\n{path}')

    partition_files = sorted(STEP8_DIR.glob('*_singlet_raw_counts_v1_1.npz'))

    if len(partition_files) != 21:
        raise RuntimeError(f'Expected 21 frozen donor partitions, found {len(partition_files)}.')

    with open(OLD_CLUSTER_PLAN_PATH, 'r', encoding='utf-8') as handle:
        old_cluster_plan = json.load(handle)

    marker_panels = old_cluster_plan.get('cluster_marker_panels')

    if not isinstance(marker_panels, dict) or 'Astrocyte' not in marker_panels:
        raise RuntimeError('Frozen Step 9D marker panels are unavailable.')

    TARGET_PANEL = ['CREB5', 'CSRP1', 'SHC1', 'KCNH2', 'MRGPRF', 'AJAP1', 'CCDC3']

    marker_symbols = {str(gene).upper() for genes in marker_panels.values() for gene in genes}

    target_overlap = marker_symbols.intersection(TARGET_PANEL)

    if target_overlap:
        raise RuntimeError(f'A confirmatory target occurs in the frozen annotation panels: {sorted(target_overlap)}')

    plan_core = {'dataset': 'GSE157827', 'step': '15A', 'version': 'v1', 'analysis_role': 'Post-result robustness analysis of astrocyte identity; not a replacement primary analysis', 'reason': 'Test whether the CREB5 result depends on the original global-variance, donor-centered, forced-43-cluster MiniBatchKMeans annotation workflow', 'primary_analysis_remains_frozen': True, 'input_population': '159,144 final author-QC Scrublet singlets', 'input_donors': 21, 'input_features': 33538, 'software_environment': {'scanpy': '1.12.4', 'pandas': '2.2.3', 'numpy': '2.1.3', 'anndata': '0.12.6'}, 'target_panel_excluded_before_feature_statistics': TARGET_PANEL, 'technical_feature_exclusions': ['mitochondrial genes', 'ribosomal-protein genes', 'hemoglobin genes'], 'hvg_eligibility': 'Gene detected in at least 3 singlet nuclei in at least 3 donors; one deterministic representative per gene symbol', 'hvg_method': 'scanpy Seurat-v3-paper normalized variance', 'hvg_batch_key': 'GSM donor', 'hvg_input': 'raw integer UMI counts', 'number_of_HVGs': 3000, 'normalization_after_HVG_selection': 'log1p(CP10K)', 'PCA_components': 50, 'batch_correction': {'method': 'Scanpy Harmony2', 'batch_key': 'GSM donor', 'diagnosis_used': False, 'components_supplied': 50}, 'neighbor_graph': {'representation': 'Harmony-corrected PCA', 'components': 30, 'neighbors': 15, 'metric': 'euclidean', 'random_seed': 20260912}, 'Leiden': {'primary_sensitivity_resolution': 0.8, 'additional_resolution_audits': [0.5, 1.2], 'random_seed': 20260912, 'cluster_number_forced': False}, 'annotation_marker_panels': marker_panels, 'annotation_rule': {'marker_expression': 'mean nucleus-level log1p(CP10K) by cluster', 'standardization': 'z-score each marker across clusters', 'lineage_score': 'mean marker z-score within each lineage panel', 'assignment': 'unique highest lineage score', 'minimum_top_score': 0.0, 'minimum_positive_z_markers': 2, 'unknown_if_rule_fails': True, 'published_composition_used_for_assignment': False}, 'primary_sensitivity_astrocytes': 'All nuclei in resolution-0.8 Leiden clusters assigned Astrocyte by the frozen rule', 'minimum_astrocytes_per_donor': 20, 'alternative_pseudobulk': 'Sum original raw integer counts for alternative astrocytes separately within each donor', 'DE_gene_universe': 'Reuse the exact frozen Step 12 set of 16,871 genes; do not refilter', 'DE_model': '~ age + sex + diagnosis', 'DE_contrast': 'AD versus Control', 'CREB5_sensitivity_support_rule': 'positive log2FoldChange and nominal two-sided p < 0.05', 'diagnosis_prohibited_until_alternative_astrocytes_are_frozen': True, 'targets_prohibited_from_HVGs_PCA_Harmony_neighbors_Leiden_annotation': True, 'results_known_before_this_sensitivity_was_planned': True, 'tuning_to_match_primary_effect_prohibited': True, 'frozen_inputs': {'matrix_partition_directory': str(STEP8_DIR), 'observation_table': str(OBS_PATH), 'feature_table': str(VAR_PATH), 'partition_manifest': str(PARTITION_PATH), 'original_marker_plan': str(OLD_CLUSTER_PLAN_PATH), 'primary_result_lock': str(PRIMARY_RESULT_LOCK_PATH), 'completed_scrublet_donor_sensitivity': str(SCRUBLET_SENSITIVITY_LOCK_PATH)}}

    if PLAN_LOCK_PATH.exists():
        with open(PLAN_LOCK_PATH, 'r', encoding='utf-8') as handle:
            existing_plan = json.load(handle)
        existing_core = {key: value for key, value in existing_plan.items() if key != 'created_utc'}
        if existing_core != plan_core:
            raise RuntimeError('An existing Step 15A lock differs from this specification. It was not overwritten.')
        status = 'Existing identical lock verified; not overwritten'
    else:
        plan_to_save = {**plan_core, 'created_utc': datetime.now(timezone.utc).isoformat()}
        with open(PLAN_LOCK_PATH, 'w', encoding='utf-8') as handle:
            json.dump(plan_to_save, handle, indent=2)
        status = 'New lock created'

    print('=' * 78)

    print('GSE157827 STEP 15A — INDEPENDENT ANNOTATION SENSITIVITY LOCK')

    print('=' * 78)

    print(f'\nStatus: {status}')

    print('Analysis role: post-result robustness sensitivity')

    print('Primary analysis changed: NO')

    print('Input nuclei: 159,144')

    print('Input donors: 21')

    print('HVGs: 3,000 batch-aware Seurat-v3-paper')

    print('Batch correction: Harmony2 using GSM only')

    print('Clustering: neighbor graph + Leiden; cluster count not forced')

    print('Primary sensitivity resolution: 0.8')


In [ ]:
if RUN_PIPELINE:
    print('Audit resolutions: 0.5 and 1.2')

    print('Annotation panels: inherited unchanged from Step 9D')

    print('Diagnosis available during clustering/annotation: NO')

    print('CREB5 or secondary targets available to clustering/annotation: NO')

    print('Gene universe for later DE will be refiltered: NO')

    print('\nSTEP 15A PLAN FROZEN BEFORE ALTERNATIVE CLUSTERING')

    print(f'\nPlan lock:\n{PLAN_LOCK_PATH}')


In [ ]:
if RUN_PIPELINE:
    from pathlib import Path

    from datetime import datetime, timezone

    from importlib.metadata import version

    import hashlib

    import json

    import sys

    PROJECT_DIR = Path('/content/drive/MyDrive/AD_Astrocyte_Paper_01')

    VALIDATION_DIR = PROJECT_DIR / 'GSE157827_confirmatory_validation'

    PLAN_LOCK_PATH = VALIDATION_DIR / 'GSE157827_STEP15A_INDEPENDENT_ANNOTATION_SENSITIVITY_PLAN_LOCK_v1.json'

    AMENDMENT_PATH = VALIDATION_DIR / 'GSE157827_STEP15A_AMENDMENT01_SOFTWARE_COMPATIBILITY_v1.json'

    if not PLAN_LOCK_PATH.exists():
        raise FileNotFoundError(f'Missing STEP 15A lock:\n{PLAN_LOCK_PATH}')

    with open(PLAN_LOCK_PATH, 'rb') as handle:
        original_bytes = handle.read()

    original_sha256 = hashlib.sha256(original_bytes).hexdigest()

    original = json.loads(original_bytes.decode('utf-8'))

    expected_original = {'scanpy': '1.12.4', 'pandas': '2.2.3', 'numpy': '2.1.3', 'anndata': '0.12.6'}

    if original.get('software_environment') != expected_original:
        raise RuntimeError('Original STEP 15A software environment does not match the frozen plan.')

    actual_environment = {'python': sys.version.split()[0], 'numpy': version('numpy'), 'pandas': version('pandas'), 'anndata': version('anndata'), 'scanpy': version('scanpy'), 'harmonypy': version('harmonypy'), 'igraph': version('igraph'), 'leidenalg': version('leidenalg'), 'scikit-misc': version('scikit-misc')}

    expected_current = {'numpy': '2.1.3', 'pandas': '2.2.3', 'anndata': '0.12.6', 'scanpy': '1.11.5'}

    for package, expected in expected_current.items():
        observed = actual_environment[package]
        if observed != expected:
            raise RuntimeError(f'{package}: expected {expected}, found {observed}')

    amendment = {'dataset': 'GSE157827', 'parent_step': '15A', 'amendment': '15A-AMENDMENT-01', 'version': 'v1', 'created_utc': datetime.now(timezone.utc).isoformat(), 'created_before_STEP15B_results': True, 'reason': 'The frozen STEP 15A environment specified Scanpy 1.12.4 with pandas 2.2.3, an incompatible dependency combination. Scanpy was changed prospectively to 1.11.5 while retaining pandas 2.2.3, numpy 2.1.3 and anndata 0.12.6.', 'original_STEP15A_lock': str(PLAN_LOCK_PATH), 'original_STEP15A_sha256': original_sha256, 'original_software_environment': expected_original, 'amended_runtime_environment': actual_environment, 'changed_field': {'package': 'scanpy', 'from': '1.12.4', 'to': '1.11.5'}, 'scientific_parameters_changed': False, 'HVG_definition_changed': False, 'HVG_number_changed': False, 'batch_key_changed': False, 'normalization_changed': False, 'PCA_parameters_changed': False, 'Harmony_parameters_changed': False, 'neighbor_parameters_changed': False, 'Leiden_parameters_changed': False, 'annotation_rule_changed': False, 'diagnosis_blinding_changed': False, 'target_blinding_changed': False, 'DE_plan_changed': False, 'CREB5_success_rule_changed': False}

    if AMENDMENT_PATH.exists():
        with open(AMENDMENT_PATH, 'r', encoding='utf-8') as handle:
            existing = json.load(handle)
        if existing.get('changed_field') != amendment['changed_field']:
            raise RuntimeError('Conflicting software amendment already exists.')
        print('Existing compatible amendment found.')
    else:
        with open(AMENDMENT_PATH, 'w', encoding='utf-8') as handle:
            json.dump(amendment, handle, indent=2)
        print('STEP 15A-AMENDMENT-01 CREATED')

    print('\nScientific parameters changed: NO')

    print('Diagnosis blinding changed: NO')

    print('Target blinding changed: NO')

    print('CREB5 success rule changed: NO')

    print('\nSoftware change:')

    print('Scanpy 1.12.4 -> 1.11.5')

    print(f'\nAmendment saved:\n{AMENDMENT_PATH}')


### Batch-aware HVG eligibility and frozen LOESS recovery


In [ ]:
if RUN_PIPELINE:
    from pathlib import Path

    from datetime import datetime, timezone

    from importlib.metadata import version

    import gc

    import hashlib

    import json

    import sys

    import numpy as np

    import pandas as pd

    import scipy.sparse as sp

    import scanpy as sc

    import anndata as ad

    PROJECT_DIR = Path('/content/drive/MyDrive/AD_Astrocyte_Paper_01')

    VALIDATION_DIR = PROJECT_DIR / 'GSE157827_confirmatory_validation'

    OBS_PATH = VALIDATION_DIR / 'GSE157827_STEP8_singlet_obs_v1_1.csv.gz'

    VAR_PATH = VALIDATION_DIR / 'GSE157827_STEP8_feature_metadata_v1_1.csv.gz'

    PARTITION_PATH = VALIDATION_DIR / 'GSE157827_STEP8_matrix_partition_manifest_v1_1.csv'

    PLAN_LOCK_PATH = VALIDATION_DIR / 'GSE157827_STEP15A_INDEPENDENT_ANNOTATION_SENSITIVITY_PLAN_LOCK_v1.json'

    AMENDMENT_PATH = VALIDATION_DIR / 'GSE157827_STEP15A_AMENDMENT01_SOFTWARE_COMPATIBILITY_v1.json'

    FEATURE_ELIGIBILITY_PATH = VALIDATION_DIR / 'GSE157827_STEP15B1_feature_eligibility_v1.csv.gz'

    HVG_TABLE_PATH = VALIDATION_DIR / 'GSE157827_STEP15B1_HVG3000_seurat_v3_paper_v1.csv'

    HVG_RAW_MATRIX_PATH = VALIDATION_DIR / 'GSE157827_STEP15B1_HVG3000_raw_counts_v1.npz'

    BLIND_OBS_PATH = VALIDATION_DIR / 'GSE157827_STEP15B1_blind_obs_v1.csv.gz'

    RESULT_LOCK_PATH = VALIDATION_DIR / 'GSE157827_STEP15B1_HVG_SELECTION_LOCK_v1.json'

    EXPECTED_NUCLEI = 159144

    EXPECTED_DONORS = 21

    EXPECTED_FEATURES = 33538

    N_HVG = 3000

    TARGET_PANEL = ['CREB5', 'CSRP1', 'SHC1', 'KCNH2', 'MRGPRF', 'AJAP1', 'CCDC3']

    if RESULT_LOCK_PATH.exists():
        raise RuntimeError(f'STEP 15B.1 already completed.\nExisting lock:\n{RESULT_LOCK_PATH}')

    for path in [OBS_PATH, VAR_PATH, PARTITION_PATH, PLAN_LOCK_PATH, AMENDMENT_PATH]:
        if not path.exists():
            raise FileNotFoundError(f'Required input missing:\n{path}')

    with open(PLAN_LOCK_PATH, 'r', encoding='utf-8') as handle:
        plan = json.load(handle)

    with open(AMENDMENT_PATH, 'r', encoding='utf-8') as handle:
        amendment = json.load(handle)

    if plan.get('dataset') != 'GSE157827':
        raise RuntimeError('Wrong STEP 15A dataset.')

    if plan.get('step') != '15A':
        raise RuntimeError('Wrong STEP 15A lock.')

    if plan.get('input_donors') != EXPECTED_DONORS:
        raise RuntimeError('Frozen donor count changed.')

    if plan.get('input_features') != EXPECTED_FEATURES:
        raise RuntimeError('Frozen feature count changed.')

    if plan.get('number_of_HVGs') != N_HVG:
        raise RuntimeError('Frozen HVG number changed.')

    if plan.get('hvg_method') != 'scanpy Seurat-v3-paper normalized variance':
        raise RuntimeError('Frozen HVG method changed.')

    if plan.get('hvg_batch_key') != 'GSM donor':
        raise RuntimeError('Frozen HVG batch key changed.')

    if plan.get('hvg_input') != 'raw integer UMI counts':
        raise RuntimeError('Frozen HVG input changed.')


In [ ]:
if RUN_PIPELINE:
    if plan.get('normalization_after_HVG_selection') != 'log1p(CP10K)':
        raise RuntimeError('Frozen normalization rule changed.')

    if plan.get('diagnosis_prohibited_until_alternative_astrocytes_are_frozen') is not True:
        raise RuntimeError('Diagnosis blinding rule changed.')

    if plan.get('targets_prohibited_from_HVGs_PCA_Harmony_neighbors_Leiden_annotation') is not True:
        raise RuntimeError('Target blinding rule changed.')

    if plan.get('target_panel_excluded_before_feature_statistics') != TARGET_PANEL:
        raise RuntimeError('Frozen target panel changed.')

    if amendment.get('scientific_parameters_changed') is not False:
        raise RuntimeError('Software amendment changed scientific parameters.')

    runtime_versions = {'python': sys.version.split()[0], 'numpy': version('numpy'), 'pandas': version('pandas'), 'anndata': version('anndata'), 'scanpy': version('scanpy'), 'harmonypy': version('harmonypy'), 'igraph': version('igraph'), 'leidenalg': version('leidenalg'), 'scikit-misc': version('scikit-misc')}

    expected_core = {'numpy': '2.1.3', 'pandas': '2.2.3', 'anndata': '0.12.6', 'scanpy': '1.11.5'}

    for package, expected in expected_core.items():
        observed = runtime_versions[package]
        if observed != expected:
            raise RuntimeError(f'{package}: expected {expected}; found {observed}')

    print('=' * 78)

    print('GSE157827 STEP 15B.1 — DIAGNOSIS-BLIND HVG SELECTION')

    print('=' * 78)

    print('\nSoftware amendment verified.')

    print('Scientific specification unchanged.')

    required_obs_columns = ['nucleus_id', 'GSM', 'barcode', 'matrix_partition', 'partition_row', 'global_row']

    header = pd.read_csv(OBS_PATH, nrows=0).columns.tolist()

    missing = [c for c in required_obs_columns if c not in header]

    if missing:
        raise RuntimeError(f'Missing structural columns: {missing}')

    obs = pd.read_csv(OBS_PATH, usecols=required_obs_columns, dtype={'nucleus_id': str, 'GSM': str, 'barcode': str, 'matrix_partition': str})

    obs['partition_row'] = pd.to_numeric(obs['partition_row'], errors='raise').astype(np.int64)

    obs['global_row'] = pd.to_numeric(obs['global_row'], errors='raise').astype(np.int64)

    obs = obs.sort_values('global_row').reset_index(drop=True)

    if len(obs) != EXPECTED_NUCLEI:
        raise RuntimeError(f'Expected {EXPECTED_NUCLEI:,} nuclei; found {len(obs):,}.')

    if obs['nucleus_id'].duplicated().any():
        raise RuntimeError('Duplicate nucleus_id.')

    if not np.array_equal(obs['global_row'].to_numpy(), np.arange(EXPECTED_NUCLEI, dtype=np.int64)):
        raise RuntimeError('Frozen global-row ordering invalid.')

    if obs['GSM'].nunique() != EXPECTED_DONORS:
        raise RuntimeError('Unexpected number of GSM donors.')

    var = pd.read_csv(VAR_PATH, dtype=str)

    if len(var) != EXPECTED_FEATURES:
        raise RuntimeError(f'Expected {EXPECTED_FEATURES:,} features; found {len(var):,}.')

    for required in ['gene_id', 'gene_symbol']:
        if required not in var.columns:
            raise RuntimeError(f'Missing feature column: {required}')

    if var['gene_id'].duplicated().any():
        raise RuntimeError('Duplicate gene_id values.')

    partitions = pd.read_csv(PARTITION_PATH, dtype={'GSM': str, 'partition_file': str})

    if len(partitions) != EXPECTED_DONORS:
        raise RuntimeError(f'Expected {EXPECTED_DONORS} partitions; found {len(partitions)}.')

    if partitions['GSM'].duplicated().any():
        raise RuntimeError('Duplicate GSM in partition manifest.')

    required_partition_columns = ['GSM', 'partition_file', 'global_row_start', 'global_row_end_exclusive']

    missing = [c for c in required_partition_columns if c not in partitions.columns]

    if missing:
        raise RuntimeError(f'Missing manifest columns: {missing}')


In [ ]:
if RUN_PIPELINE:
    partitions['global_row_start'] = pd.to_numeric(partitions['global_row_start'], errors='raise').astype(np.int64)

    partitions['global_row_end_exclusive'] = pd.to_numeric(partitions['global_row_end_exclusive'], errors='raise').astype(np.int64)

    partitions = partitions.sort_values('global_row_start').reset_index(drop=True)

    if int(partitions.iloc[0]['global_row_start']) != 0:
        raise RuntimeError('First partition does not begin at row 0.')

    if int(partitions.iloc[-1]['global_row_end_exclusive']) != EXPECTED_NUCLEI:
        raise RuntimeError('Final partition endpoint mismatch.')

    if not np.array_equal(partitions['global_row_start'].to_numpy()[1:], partitions['global_row_end_exclusive'].to_numpy()[:-1]):
        raise RuntimeError('Partition ranges are not contiguous.')

    symbols = var['gene_symbol'].fillna('').astype(str).str.strip()

    symbols_upper = symbols.str.upper()

    nonempty = symbols_upper != ''

    representative = nonempty & ~symbols_upper.duplicated(keep='first')

    mitochondrial = symbols_upper.str.startswith('MT-')

    ribosomal = symbols_upper.str.match('^RP[SL][0-9]')

    hemoglobin = symbols_upper.str.match('^HB[ABDEGQMZ][0-9]')

    locked_target = symbols_upper.isin(TARGET_PANEL)

    candidate_mask = representative & ~mitochondrial & ~ribosomal & ~hemoglobin & ~locked_target

    candidate_indices = np.flatnonzero(candidate_mask.to_numpy())

    if len(candidate_indices) <= N_HVG:
        raise RuntimeError('Too few candidate genes.')

    if symbols_upper.iloc[candidate_indices].isin(TARGET_PANEL).any():
        raise RuntimeError('Locked target survived exclusion.')

    candidate_meta = var.iloc[candidate_indices].copy()

    candidate_meta.insert(0, 'original_feature_index', candidate_indices.astype(np.int64))

    candidate_meta['gene_symbol_upper'] = candidate_meta['gene_symbol'].fillna('').astype(str).str.upper()

    print(f'\nInput nuclei: {len(obs):,}')

    print(f"Input donors: {obs['GSM'].nunique()}")

    print(f'Input features: {len(var):,}')

    print(f'Candidate genes before detection rule: {len(candidate_indices):,}')

    print('\nDiagnosis loaded: NO')

    print('sample_label loaded: NO')

    print('Target expression analyzed: NO')

    donor_detection_hits = np.zeros(len(candidate_indices), dtype=np.int16)

    print('\nPASS 1 — donor detection eligibility')


In [ ]:
if RUN_PIPELINE:
    for donor_i, partition in enumerate(partitions.itertuples(index=False), start=1):
        gsm = str(partition.GSM)
        matrix_path = Path(partition.partition_file)
        if not matrix_path.exists():
            raise FileNotFoundError(f'{gsm}: missing matrix:\n{matrix_path}')
        start = int(partition.global_row_start)
        end = int(partition.global_row_end_exclusive)
        donor_obs = obs.iloc[start:end]
        if not (donor_obs['GSM'] == gsm).all():
            raise RuntimeError(f'{gsm}: global-row/GSM mismatch.')
        if not np.array_equal(donor_obs['partition_row'].to_numpy(), np.arange(end - start, dtype=np.int64)):
            raise RuntimeError(f'{gsm}: partition-row mismatch.')
        X = sp.load_npz(matrix_path).tocsr()
        expected_shape = (end - start, EXPECTED_FEATURES)
        if X.shape != expected_shape:
            raise RuntimeError(f'{gsm}: matrix shape {X.shape}; expected {expected_shape}.')
        if X.nnz == 0:
            raise RuntimeError(f'{gsm}: empty matrix.')
        if np.any(X.data < 0):
            raise RuntimeError(f'{gsm}: negative counts detected.')
        if not np.issubdtype(X.data.dtype, np.integer):
            if not np.allclose(X.data, np.rint(X.data)):
                raise RuntimeError(f'{gsm}: non-integer raw counts.')
        Xc = X[:, candidate_indices].tocsr()
        detected = np.asarray((Xc > 0).sum(axis=0)).ravel()
        donor_detection_hits += (detected >= 3).astype(np.int16)
        print(f'  [{donor_i:02d}/{EXPECTED_DONORS}] {gsm}: {X.shape[0]:,} nuclei')
        del X
        del Xc
        del detected
        gc.collect()

    eligible_mask = donor_detection_hits >= 3

    eligible_indices = candidate_indices[eligible_mask]

    candidate_meta['donors_with_at_least_3_detected_nuclei'] = donor_detection_hits

    candidate_meta['eligible_for_HVG_statistics'] = eligible_mask

    candidate_meta.to_csv(FEATURE_ELIGIBILITY_PATH, index=False, compression={'method': 'gzip', 'compresslevel': 6})

    eligible_meta = candidate_meta.loc[eligible_mask].copy().reset_index(drop=True)

    if len(eligible_indices) <= N_HVG:
        raise RuntimeError(f'Only {len(eligible_indices):,} genes passed eligibility.')

    print('\nEligible genes:', f'{len(eligible_indices):,}')

    print('\nPASS 2 — building eligible raw matrix')

    chunks = []

    for donor_i, partition in enumerate(partitions.itertuples(index=False), start=1):
        X = sp.load_npz(Path(partition.partition_file)).tocsr()
        Xe = X[:, eligible_indices].copy().tocsr()
        chunks.append(Xe)
        print(f'  [{donor_i:02d}/{EXPECTED_DONORS}] {partition.GSM}')
        del X
        gc.collect()

    X_eligible = sp.vstack(chunks, format='csr')

    del chunks

    gc.collect()

    if X_eligible.shape != (EXPECTED_NUCLEI, len(eligible_indices)):
        raise RuntimeError(f'Eligible matrix shape mismatch: {X_eligible.shape}')

    print('\nEligible raw matrix:', f'{X_eligible.shape[0]:,} x {X_eligible.shape[1]:,}')


In [ ]:
if RUN_PIPELINE:
    blind_obs = pd.DataFrame({'GSM': obs['GSM'].astype(str).to_numpy()}, index=pd.Index(obs['nucleus_id'].astype(str), name='nucleus_id'))

    obs[['nucleus_id', 'GSM', 'barcode', 'matrix_partition', 'partition_row', 'global_row']].to_csv(BLIND_OBS_PATH, index=False, compression={'method': 'gzip', 'compresslevel': 6})

    adata_var = eligible_meta[['original_feature_index', 'gene_id', 'gene_symbol', 'gene_symbol_upper', 'donors_with_at_least_3_detected_nuclei']].copy()

    adata_var.index = pd.Index(adata_var['gene_symbol_upper'].astype(str), name='gene_symbol')

    if adata_var.index.duplicated().any():
        raise RuntimeError('Duplicate gene symbols entered AnnData.')

    adata = ad.AnnData(X=X_eligible, obs=blind_obs, var=adata_var)

    del X_eligible

    gc.collect()

    if set(adata.obs.columns) != {'GSM'}:
        raise RuntimeError('Unexpected metadata entered blind AnnData.')


In [ ]:
if RUN_PIPELINE:
    from pathlib import Path

    from datetime import datetime, timezone

    import hashlib

    import json

    import gc

    import numpy as np

    import pandas as pd

    import scipy.sparse as sp

    import scanpy as sc

    required_objects = ['adata', 'obs', 'eligible_indices', 'PLAN_LOCK_PATH', 'AMENDMENT_PATH', 'FEATURE_ELIGIBILITY_PATH', 'HVG_TABLE_PATH', 'HVG_RAW_MATRIX_PATH', 'BLIND_OBS_PATH', 'RESULT_LOCK_PATH', 'runtime_versions', 'TARGET_PANEL', 'N_HVG']

    missing_objects = [name for name in required_objects if name not in globals()]

    if missing_objects:
        raise RuntimeError(f'Required STEP 15B.1 objects are no longer in memory:\n{missing_objects}\n\nDo NOT improvise. Rerun STEP 15B.1 through the LOESS failure, then run this recovery cell.')

    if adata.shape != (159144, len(eligible_indices)):
        raise RuntimeError(f'Unexpected AnnData shape: {adata.shape}')

    if len(eligible_indices) != 26598:
        raise RuntimeError('Eligible-gene count no longer matches the completed blind eligibility analysis.')

    if set(adata.obs.columns) != {'GSM'}:
        raise RuntimeError('Diagnosis-blind AnnData contains unexpected metadata.')

    if RESULT_LOCK_PATH.exists():
        raise RuntimeError('STEP 15B.1 result lock already exists. Do not overwrite it.')

    LOESS_RULE_PATH = Path(PLAN_LOCK_PATH).parent / 'GSE157827_STEP15A_AMENDMENT02_LOESS_NUMERICAL_STABILITY_RULE_v1.json'

    SPAN_SEQUENCE = [0.3, 0.4, 0.5, 0.6, 0.75, 1.0]

    def sha256_file(path):
        digest = hashlib.sha256()
        with open(path, 'rb') as handle:
            for block in iter(lambda: handle.read(1024 * 1024), b''):
                digest.update(block)
        return digest.hexdigest()

    loess_rule = {'dataset': 'GSE157827', 'parent_step': '15A', 'amendment': '15A-AMENDMENT-02', 'version': 'v1', 'created_utc': datetime.now(timezone.utc).isoformat(), 'created_before_successful_STEP15B1_HVG_result': True, 'trigger': 'Scanpy seurat_v3_paper HVG selection failed at the documented default LOESS span 0.3 because scikit-misc raised a near-singularity ValueError.', 'observed_error': 'ValueError: There are other near singularities as well. 0.090619', 'numerical_resolution_rule': 'Attempt the frozen Seurat-v3-paper analysis using spans in the fixed ordered sequence [0.30, 0.40, 0.50, 0.60, 0.75, 1.00]. Select the first span that completes without numerical error. No choice may depend on diagnosis, target identity, HVG identities, or downstream results.', 'span_sequence': SPAN_SEQUENCE, 'selection_rule': 'smallest numerically successful span', 'unchanged': {'input_population': '159144 frozen singlet nuclei', 'eligible_gene_set': '26598 genes from completed STEP 15B.1 eligibility', 'HVG_framework': 'Scanpy Seurat-v3-paper normalized variance', 'HVG_number': 3000, 'batch_key': 'GSM', 'input': 'raw integer UMI counts', 'diagnosis_blinding': True, 'target_blinding': True, 'target_panel_excluded': TARGET_PANEL}, 'parameter_amended': 'LOESS smoothing span only', 'reason_parameter_amended': 'numerical stability', 'parent_STEP15A_sha256': sha256_file(PLAN_LOCK_PATH), 'software_amendment01_sha256': sha256_file(AMENDMENT_PATH)}

    if LOESS_RULE_PATH.exists():
        with open(LOESS_RULE_PATH, 'r', encoding='utf-8') as handle:
            existing_rule = json.load(handle)
        if existing_rule.get('span_sequence') != SPAN_SEQUENCE:
            raise RuntimeError('A different LOESS recovery rule already exists.')
        print('Existing LOESS stability rule verified.')
    else:
        with open(LOESS_RULE_PATH, 'w', encoding='utf-8') as handle:
            json.dump(loess_rule, handle, indent=2)
        print('STEP 15A-AMENDMENT-02 stability rule CREATED')

    print('\nFrozen span sequence:', SPAN_SEQUENCE)

    print('Selection criterion: smallest numerically successful span')

    print('Diagnosis/targets used to choose span: NO')

    attempt_records = []

    hvg_df = None

    selected_span = None

    print('\n' + '=' * 78)

    print('LOESS STABILITY ATTEMPTS')

    print('=' * 78)


In [ ]:
if RUN_PIPELINE:
    for span in SPAN_SEQUENCE:
        print(f'\nTrying span={span:.2f} ...')
        try:
            candidate_hvg_df = sc.pp.highly_variable_genes(adata, flavor='seurat_v3_paper', n_top_genes=N_HVG, batch_key='GSM', span=span, subset=False, inplace=False, check_values=True)
            n_this = int(candidate_hvg_df['highly_variable'].sum())
            if n_this != N_HVG:
                raise RuntimeError(f'span={span:.2f} completed but selected {n_this} HVGs instead of {N_HVG}.')
            attempt_records.append({'span': span, 'status': 'SUCCESS', 'error': None})
            hvg_df = candidate_hvg_df
            selected_span = span
            print(f'SUCCESS at span={span:.2f}')
            print(f'Selected HVGs: {n_this:,}')
            break
        except ValueError as exc:
            attempt_records.append({'span': span, 'status': 'NUMERICAL_FAILURE', 'error': str(exc)})
            print(f'Numerical failure at span={span:.2f}')
            print(str(exc))

    if 'candidate_hvg_df' in locals():
        del candidate_hvg_df

    gc.collect()

    if hvg_df is None:
        raise RuntimeError('No frozen candidate span produced a stable Seurat-v3-paper fit. STOP here; do not change the method or normalize the counts.')

    print('\n' + '=' * 78)

    print(f'FROZEN SUCCESSFUL LOESS SPAN: {selected_span:.2f}')

    print('=' * 78)

    positions = np.flatnonzero(hvg_df['highly_variable'].to_numpy())

    ranks = hvg_df.iloc[positions]['highly_variable_rank'].to_numpy(dtype=float)

    if len(positions) != N_HVG:
        raise RuntimeError(f'Expected {N_HVG} selected genes; found {len(positions)}.')

    if not np.isfinite(ranks).all():
        raise RuntimeError('Selected HVGs contain non-finite ranks.')

    positions = positions[np.argsort(ranks, kind='stable')]

    X_hvg_raw = adata.X[:, positions].copy().tocsr()

    selected = adata.var.iloc[positions].copy().reset_index(drop=True)

    selected_metrics = hvg_df.iloc[positions].reset_index(drop=True)

    for metric in ['means', 'variances', 'variances_norm', 'highly_variable_rank', 'highly_variable_nbatches', 'highly_variable_intersection']:
        if metric in selected_metrics.columns:
            selected[metric] = selected_metrics[metric].to_numpy()

    selected.insert(0, 'matrix_column', np.arange(N_HVG, dtype=np.int32))

    selected_upper = selected['gene_symbol_upper'].astype(str).str.upper()

    if selected_upper.isin(TARGET_PANEL).any():
        raise RuntimeError('Locked target entered HVGs.')

    if selected_upper.str.startswith('MT-').any():
        raise RuntimeError('Mitochondrial gene entered HVGs.')

    if selected_upper.str.match('^RP[SL][0-9]').any():
        raise RuntimeError('Ribosomal gene entered HVGs.')

    if selected_upper.str.match('^HB[ABDEGQMZ][0-9]').any():
        raise RuntimeError('Hemoglobin gene entered HVGs.')

    sp.save_npz(HVG_RAW_MATRIX_PATH, X_hvg_raw, compressed=True)

    selected.to_csv(HVG_TABLE_PATH, index=False)

    def sha256_list(values):
        payload = ('\n'.join((str(v) for v in values)) + '\n').encode('utf-8')
        return hashlib.sha256(payload).hexdigest()

    result_lock = {'dataset': 'GSE157827', 'step': '15B.1', 'version': 'v1', 'created_utc': datetime.now(timezone.utc).isoformat(), 'analysis_role': 'Diagnosis-blind donor-aware Seurat-v3-paper HVG selection', 'input_nuclei': 159144, 'input_donors': 21, 'eligible_genes': int(len(eligible_indices)), 'selected_HVGs': N_HVG, 'HVG_framework': 'scanpy Seurat-v3-paper normalized variance', 'HVG_batch_key': 'GSM', 'HVG_input': 'raw integer UMI counts', 'diagnosis_loaded': False, 'sample_label_loaded': False, 'target_expression_inspected': False, 'target_panel_excluded': TARGET_PANEL, 'software_environment': runtime_versions, 'software_amendment01': str(AMENDMENT_PATH), 'loess_stability_amendment02': str(LOESS_RULE_PATH), 'loess_stability_amendment02_sha256': sha256_file(LOESS_RULE_PATH), 'default_scanpy_span': 0.3, 'span_selection_rule': 'smallest numerically successful frozen candidate', 'span_attempts': attempt_records, 'selected_LOESS_span': selected_span, 'feature_eligibility_file': str(FEATURE_ELIGIBILITY_PATH), 'blind_obs_file': str(BLIND_OBS_PATH), 'ordered_HVG_table': str(HVG_TABLE_PATH), 'raw_HVG_matrix': str(HVG_RAW_MATRIX_PATH), 'raw_HVG_matrix_shape': [int(X_hvg_raw.shape[0]), int(X_hvg_raw.shape[1])], 'raw_HVG_matrix_nnz': int(X_hvg_raw.nnz), 'nucleus_id_sha256': sha256_list(obs['nucleus_id'].astype(str)), 'HVG_gene_id_sha256': sha256_list(selected['gene_id'].astype(str)), 'HVG_symbol_sha256': sha256_list(selected['gene_symbol_upper'].astype(str)), 'normalization_performed': False, 'PCA_performed': False, 'Harmony_performed': False, 'Leiden_performed': False, 'annotation_performed': False, 'DE_performed': False, 'next_step': 'STEP 15B.2: CP10K/log1p -> PCA50 -> GSM-only Harmony -> 15-neighbor graph using 30 Harmony PCs -> Leiden 0.8 plus 0.5/1.2 audits.'}


In [ ]:
if RUN_PIPELINE:
    with open(RESULT_LOCK_PATH, 'w', encoding='utf-8') as handle:
        json.dump(result_lock, handle, indent=2)

    print('\n' + '=' * 78)

    print('GSE157827 STEP 15B.1 COMPLETE')

    print('=' * 78)

    print(f'\nEligible genes: {len(eligible_indices):,}')

    print(f'Selected HVGs: {N_HVG:,}')

    print(f'LOESS span selected by frozen stability rule: {selected_span:.2f}')

    print('\nSpan attempts:')

    for record in attempt_records:
        print(f"  {record['span']:.2f}: {record['status']}")

    print('\nFrozen raw-HVG matrix:')

    print(f'  nuclei: {X_hvg_raw.shape[0]:,}')

    print(f'  genes: {X_hvg_raw.shape[1]:,}')

    print(f'  nonzeros: {X_hvg_raw.nnz:,}')

    print('\nLocked targets among HVGs: 0')

    print('Diagnosis used: NO')

    print('Target expression inspected: NO')

    print('Normalization performed: NO')

    print('PCA/Harmony/Leiden performed: NO')

    print(f'\nSTEP 15B.1 lock:\n{RESULT_LOCK_PATH}')

    print('\nNEXT: STEP 15B.2 — CP10K/log1p -> PCA50 -> Harmony -> neighbors -> Leiden')


### PCA, donor correction, and Leiden clustering

The frozen 3,000-HVG matrix is normalized by full-library CP10K/log1p, reduced to 50 PCs, corrected with harmonypy 2.0.0 using GSM only, and clustered from the first 30 corrected PCs with a 15-neighbor graph. Resolution 0.8 is primary; 0.5 and 1.2 are audit resolutions.


In [ ]:
if RUN_PIPELINE:
    from pathlib import Path

    from datetime import datetime, timezone

    from importlib.metadata import version

    import gc

    import hashlib

    import json

    import anndata as ad

    import harmonypy as hm

    import numpy as np

    import pandas as pd

    import scanpy as sc

    import scipy.sparse as sp

    from sklearn.utils.sparsefuncs import inplace_csr_row_scale

    expected_versions = {'numpy': '2.1.3', 'pandas': '2.2.3', 'anndata': '0.12.6', 'scanpy': '1.11.5', 'harmonypy': '2.0.0', 'igraph': '1.0.0', 'leidenalg': '0.12.0'}

    observed_versions = {package: version(package) for package in expected_versions}

    for package, expected in expected_versions.items():
        observed = observed_versions[package]
        if observed != expected:
            raise RuntimeError(f'{package}: expected {expected}, found {observed}.\nDo not continue with a changed runtime.')

    print('=' * 78)

    print('GSE157827 STEP 15B.2 — DIRECT HARMONY2')

    print('=' * 78)

    print('\nEnvironment verified:')

    for package, v in observed_versions.items():
        print(f'  {package}: {v}')

    BASE = Path('/content/drive/MyDrive/AD_Astrocyte_Paper_01')

    if not BASE.exists():
        raise FileNotFoundError(f'Project directory missing:\n{BASE}')

    validation_dirs = sorted(BASE.glob('*GSE157827*confirmatory*validation*'))

    if len(validation_dirs) != 1:
        raise RuntimeError(f'Expected exactly one validation directory; found {len(validation_dirs)}:\n' + '\n'.join((str(x) for x in validation_dirs)))

    VALIDATION_DIR = validation_dirs[0]

    def resolve_one(pattern, label):
        matches = sorted(VALIDATION_DIR.glob(pattern))
        if len(matches) != 1:
            raise RuntimeError(f'{label}: expected exactly one match for {pattern}; found {len(matches)}:\n' + '\n'.join((str(x) for x in matches)))
        return matches[0]

    PLAN_LOCK_PATH = resolve_one('*STEP15A*ANNOTATION*SENSITIVITY*PLAN*LOCK*.json', 'STEP 15A plan lock')

    AMENDMENT01_PATH = resolve_one('*STEP15A*AMENDMENT01*SOFTWARE*.json', 'AMENDMENT-01')

    AMENDMENT02_PATH = resolve_one('*STEP15A*AMENDMENT02*LOESS*.json', 'AMENDMENT-02')

    HVG_LOCK_PATH = resolve_one('*STEP15B1*HVG*SELECTION*LOCK*.json', 'STEP 15B.1 result lock')

    PARTITION_PATH = resolve_one('*STEP8_matrix_partition_manifest_v1_1.csv', 'STEP 8 partition manifest')

    print('\nValidation directory:')

    print(VALIDATION_DIR)

    EXPECTED_NUCLEI = 159144

    EXPECTED_DONORS = 21

    EXPECTED_FEATURES = 33538

    EXPECTED_HVGS = 3000

    SEED = 20260912

    TARGET_PANEL = {'CREB5', 'CSRP1', 'SHC1', 'KCNH2', 'MRGPRF', 'AJAP1', 'CCDC3'}

    AMENDMENT03_PATH = VALIDATION_DIR / 'GSE157827_STEP15A_AMENDMENT03_DIRECT_HARMONYPY2_IMPLEMENTATION_v1.json'

    PCA_PATH = VALIDATION_DIR / 'GSE157827_STEP15B2_PCA50_v1.npy'

    HARMONY_PATH = VALIDATION_DIR / 'GSE157827_STEP15B2_Harmony2_50_v1.npy'

    DISTANCE_PATH = VALIDATION_DIR / 'GSE157827_STEP15B2_neighbor_distances15_v1.npz'

    CONNECTIVITY_PATH = VALIDATION_DIR / 'GSE157827_STEP15B2_neighbor_connectivities15_v1.npz'


In [ ]:
if RUN_PIPELINE:
    CLUSTER_PATH = VALIDATION_DIR / 'GSE157827_STEP15B2_Leiden_assignments_v1.csv.gz'

    CLUSTER_AUDIT_PATH = VALIDATION_DIR / 'GSE157827_STEP15B2_Leiden_cluster_audit_v1.csv'

    LOCK_PATH = VALIDATION_DIR / 'GSE157827_STEP15B2_EMBEDDING_CLUSTERING_LOCK_v1.json'

    if LOCK_PATH.exists():
        raise RuntimeError(f'STEP 15B.2 already has a completed result lock:\n{LOCK_PATH}')

    with open(HVG_LOCK_PATH, 'r', encoding='utf-8') as handle:
        hvg_lock = json.load(handle)

    if int(hvg_lock.get('selected_HVGs', -1)) != EXPECTED_HVGS:
        raise RuntimeError('STEP 15B.1 HVG count changed.')

    if float(hvg_lock.get('selected_LOESS_span', -1)) != 0.4:
        raise RuntimeError('STEP 15B.1 LOESS span changed.')

    if hvg_lock.get('diagnosis_loaded') is not False:
        raise RuntimeError('STEP 15B.1 diagnosis audit failed.')

    if hvg_lock.get('target_expression_inspected') is not False:
        raise RuntimeError('STEP 15B.1 target audit failed.')

    print('\nSTEP 15B.1 verified:')

    print('  Frozen HVGs: 3,000')

    print('  LOESS span: 0.40')

    print('  Diagnosis used: NO')

    print('  Targets inspected: NO')

    HVG_MATRIX_PATH = Path(hvg_lock['raw_HVG_matrix'])

    HVG_TABLE_PATH = Path(hvg_lock['ordered_HVG_table'])

    BLIND_OBS_PATH = Path(hvg_lock['blind_obs_file'])

    for path in [HVG_MATRIX_PATH, HVG_TABLE_PATH, BLIND_OBS_PATH]:
        if not path.exists():
            raise FileNotFoundError(f'Frozen STEP 15B.1 artifact missing:\n{path}')

    X_raw = sp.load_npz(HVG_MATRIX_PATH).tocsr()

    hvg_table = pd.read_csv(HVG_TABLE_PATH)

    obs = pd.read_csv(BLIND_OBS_PATH, dtype={'nucleus_id': str, 'GSM': str, 'barcode': str, 'matrix_partition': str})

    obs['global_row'] = pd.to_numeric(obs['global_row'], errors='raise').astype(np.int64)

    obs = obs.sort_values('global_row').reset_index(drop=True)

    if X_raw.shape != (EXPECTED_NUCLEI, EXPECTED_HVGS):
        raise RuntimeError(f'Frozen HVG matrix shape changed: {X_raw.shape}')

    if X_raw.nnz != int(hvg_lock['raw_HVG_matrix_nnz']):
        raise RuntimeError('Frozen HVG matrix NNZ changed.')

    if len(hvg_table) != EXPECTED_HVGS:
        raise RuntimeError('HVG table does not contain 3,000 genes.')

    if len(obs) != EXPECTED_NUCLEI:
        raise RuntimeError('Frozen nucleus count changed.')

    if obs['nucleus_id'].duplicated().any():
        raise RuntimeError('Duplicate nucleus IDs.')

    if not np.array_equal(obs['global_row'].to_numpy(), np.arange(EXPECTED_NUCLEI, dtype=np.int64)):
        raise RuntimeError('Frozen nucleus ordering changed.')

    if obs['GSM'].nunique() != EXPECTED_DONORS:
        raise RuntimeError('Frozen donor count changed.')

    for forbidden in ['diagnosis', 'sample_label']:
        if forbidden in obs.columns:
            raise RuntimeError(f'Forbidden metadata entered STEP 15B.2: {forbidden}')

    symbol_column = 'gene_symbol_upper' if 'gene_symbol_upper' in hvg_table.columns else 'gene_symbol'

    symbols = hvg_table[symbol_column].fillna('').astype(str).str.upper()

    target_overlap = sorted(set(symbols).intersection(TARGET_PANEL))

    if target_overlap:
        raise RuntimeError(f'Locked target(s) entered HVG matrix: {target_overlap}')


In [ ]:
if RUN_PIPELINE:
    def sha256_list(values):
        payload = ('\n'.join((str(value) for value in values)) + '\n').encode('utf-8')
        return hashlib.sha256(payload).hexdigest()

    if sha256_list(obs['nucleus_id'].astype(str)) != hvg_lock['nucleus_id_sha256']:
        raise RuntimeError('Frozen nucleus-order hash mismatch.')

    def sha256_file(path):
        digest = hashlib.sha256()
        with open(path, 'rb') as handle:
            for block in iter(lambda: handle.read(1024 * 1024), b''):
                digest.update(block)
        return digest.hexdigest()

    amendment03 = {'dataset': 'GSE157827', 'parent_step': '15A', 'amendment': '15A-AMENDMENT-03', 'version': 'v1', 'created_utc': datetime.now(timezone.utc).isoformat(), 'created_before_STEP15B2_results': True, 'reason': 'The frozen STEP 15 workflow specifies Harmony2 donor correction. The current prospectively verified environment already contains harmonypy 2.0.0, which is the rewritten Harmony2 implementation matching the R harmony2 reference algorithm. Direct harmonypy 2.0.0 is therefore used instead of introducing an incompatible Scanpy prerelease solely as a wrapper.', 'scientific_method_changed': False, 'Harmony_method': 'Harmony2', 'Harmony_implementation': 'harmonypy 2.0.0 direct API', 'planned_wrapper': 'Scanpy Harmony2', 'wrapper_changed': True, 'batch_key': 'GSM donor only', 'STEP15B1_recomputed': False, 'HVGs_changed': False, 'normalization_changed': False, 'PCA_parameters_changed': False, 'neighbor_parameters_changed': False, 'Leiden_resolutions_changed': False, 'diagnosis_blinding_changed': False, 'target_blinding_changed': False, 'software': observed_versions, 'STEP15A_sha256': sha256_file(PLAN_LOCK_PATH), 'amendment01_sha256': sha256_file(AMENDMENT01_PATH), 'amendment02_sha256': sha256_file(AMENDMENT02_PATH), 'STEP15B1_lock_sha256': sha256_file(HVG_LOCK_PATH)}

    if AMENDMENT03_PATH.exists():
        with open(AMENDMENT03_PATH, 'r', encoding='utf-8') as handle:
            existing_amendment = json.load(handle)
        if existing_amendment.get('Harmony_implementation') != 'harmonypy 2.0.0 direct API':
            raise RuntimeError('Conflicting AMENDMENT-03 exists.')
        print('\nExisting AMENDMENT-03 verified.')
    else:
        with open(AMENDMENT03_PATH, 'w', encoding='utf-8') as handle:
            json.dump(amendment03, handle, indent=2)
        print('\nSTEP 15A-AMENDMENT-03 CREATED')

    print('Harmony2 scientific method changed: NO')

    print('Implementation wrapper changed: YES')

    print('Diagnosis/target blinding changed: NO')

    partitions = pd.read_csv(PARTITION_PATH, dtype={'GSM': str, 'partition_file': str})

    for column in ['global_row_start', 'global_row_end_exclusive']:
        partitions[column] = pd.to_numeric(partitions[column], errors='raise').astype(np.int64)

    partitions = partitions.sort_values('global_row_start').reset_index(drop=True)

    if len(partitions) != EXPECTED_DONORS:
        raise RuntimeError('Expected 21 raw partitions.')

    if int(partitions.iloc[0]['global_row_start']) != 0:
        raise RuntimeError('First raw partition does not start at 0.')

    if int(partitions.iloc[-1]['global_row_end_exclusive']) != EXPECTED_NUCLEI:
        raise RuntimeError('Final raw partition endpoint changed.')

    if not np.array_equal(partitions['global_row_start'].to_numpy()[1:], partitions['global_row_end_exclusive'].to_numpy()[:-1]):
        raise RuntimeError('Raw partitions are not contiguous.')

    library_size = np.empty(EXPECTED_NUCLEI, dtype=np.float32)

    print('\nComputing CP10K denominators from all 33,538 raw genes...')


In [ ]:
if RUN_PIPELINE:
    for donor_number, partition in enumerate(partitions.itertuples(index=False), start=1):
        gsm = str(partition.GSM)
        start = int(partition.global_row_start)
        end = int(partition.global_row_end_exclusive)
        matrix_path = Path(partition.partition_file)
        if not matrix_path.exists():
            raise FileNotFoundError(f'{gsm}: missing raw matrix:\n{matrix_path}')
        X_full = sp.load_npz(matrix_path).tocsr()
        expected_shape = (end - start, EXPECTED_FEATURES)
        if X_full.shape != expected_shape:
            raise RuntimeError(f'{gsm}: expected shape {expected_shape}, found {X_full.shape}')
        donor_obs = obs.iloc[start:end]
        if not donor_obs['GSM'].astype(str).eq(gsm).all():
            raise RuntimeError(f'{gsm}: raw-matrix / observation row mismatch.')
        donor_library_size = np.asarray(X_full.sum(axis=1)).ravel().astype(np.float32)
        if np.any(donor_library_size <= 0):
            raise RuntimeError(f'{gsm}: zero-count nucleus.')
        library_size[start:end] = donor_library_size
        print(f'  [{donor_number:02d}/21] {gsm}: {end - start:,} nuclei')
        del X_full
        del donor_library_size
        gc.collect()

    if not np.isfinite(library_size).all():
        raise RuntimeError('Library size contains invalid values.')

    print('\nApplying log1p(CP10K) to frozen HVGs...')

    X_norm = X_raw.astype(np.float32, copy=True).tocsr()

    row_scale = (10000.0 / library_size).astype(np.float32)

    inplace_csr_row_scale(X_norm, row_scale)

    np.log1p(X_norm.data, out=X_norm.data)

    if not np.isfinite(X_norm.data).all():
        raise RuntimeError('Normalized matrix contains non-finite values.')

    if 'gene_id' in hvg_table.columns:
        var_names = hvg_table['gene_id'].astype(str).to_numpy()
    else:
        var_names = np.asarray([f'HVG_{i:04d}' for i in range(EXPECTED_HVGS)], dtype=object)

    if len(np.unique(var_names)) != EXPECTED_HVGS:
        raise RuntimeError('HVG identifiers are duplicated.')

    adata = ad.AnnData(X=X_norm, obs=obs[['nucleus_id', 'GSM', 'barcode']].copy(), var=hvg_table.copy())

    adata.obs_names = pd.Index(adata.obs['nucleus_id'].astype(str))

    adata.var_names = pd.Index(var_names.astype(str))

    print('Running centered PCA50...')

    sc.pp.pca(adata, n_comps=50, zero_center=True, svd_solver='arpack', random_state=SEED)

    pca = np.asarray(adata.obsm['X_pca'], dtype=np.float64)

    if pca.shape != (EXPECTED_NUCLEI, 50):
        raise RuntimeError(f'Unexpected PCA shape: {pca.shape}')

    if not np.isfinite(pca).all():
        raise RuntimeError('PCA contains non-finite values.')

    variance_ratio = np.asarray(adata.uns['pca']['variance_ratio'], dtype=np.float64)

    np.save(PCA_PATH, pca, allow_pickle=False)

    latent_obs = adata.obs.copy()

    del adata

    del X_norm

    del X_raw

    del library_size

    del row_scale


In [ ]:
if RUN_PIPELINE:
    gc.collect()

    print('\nRunning Harmony2 using GSM donor only...')

    harmony_result = hm.run_harmony(data_mat=pca, meta_data=latent_obs[['GSM']].copy(), vars_use='GSM', theta=2.0, lamb=None, sigma=0.1, nclust=None, tau=0, block_size=0.05, max_iter_harmony=10, max_iter_kmeans=4, epsilon_cluster=0.001, epsilon_harmony=0.01, alpha=0.2, batch_prop_cutoff=1e-05, verbose=True, random_state=SEED, ncores=1)

    harmony = np.asarray(harmony_result.Z_corr, dtype=np.float64)

    if harmony.shape != (EXPECTED_NUCLEI, 50):
        raise RuntimeError(f'Unexpected Harmony2 shape: {harmony.shape}')

    if not np.isfinite(harmony).all():
        raise RuntimeError('Harmony2 contains non-finite values.')

    np.save(HARMONY_PATH, harmony, allow_pickle=False)

    latent = ad.AnnData(X=np.zeros((EXPECTED_NUCLEI, 1), dtype=np.float32), obs=latent_obs)

    latent.obsm['X_harmony30'] = harmony[:, :30]

    print('\nBuilding 15-neighbor Euclidean graph from first 30 Harmony2 components...')

    sc.pp.neighbors(latent, n_neighbors=15, use_rep='X_harmony30', metric='euclidean', random_state=SEED, key_added='harmony15')

    distances = latent.obsp['harmony15_distances'].tocsr()

    connectivities = latent.obsp['harmony15_connectivities'].tocsr()

    if distances.shape != (EXPECTED_NUCLEI, EXPECTED_NUCLEI):
        raise RuntimeError('Neighbor distance matrix shape mismatch.')

    if connectivities.shape != distances.shape:
        raise RuntimeError('Neighbor connectivity matrix shape mismatch.')

    sp.save_npz(DISTANCE_PATH, distances, compressed=True)

    sp.save_npz(CONNECTIVITY_PATH, connectivities, compressed=True)

    donor_codes = pd.Categorical(latent.obs['GSM']).codes

    coo = distances.tocoo()

    different_donor = (donor_codes[coo.row] != donor_codes[coo.col]).astype(np.float64)

    different_counts = np.bincount(coo.row, weights=different_donor, minlength=EXPECTED_NUCLEI)

    neighbor_counts = np.bincount(coo.row, minlength=EXPECTED_NUCLEI)

    different_fraction = np.divide(different_counts, neighbor_counts, out=np.zeros(EXPECTED_NUCLEI, dtype=np.float64), where=neighbor_counts > 0)

    resolutions = {'leiden_r05': 0.5, 'leiden_r08': 0.8, 'leiden_r12': 1.2}

    for key, resolution in resolutions.items():
        print(f'Running Leiden resolution {resolution:.1f}...')
        sc.tl.leiden(latent, resolution=resolution, random_state=SEED, key_added=key, neighbors_key='harmony15', flavor='igraph', directed=False)
        n_clusters = int(latent.obs[key].nunique())
        if n_clusters < 2:
            raise RuntimeError(f'{key} produced fewer than two clusters.')

    assignments = latent.obs[['nucleus_id', 'GSM', 'barcode', 'leiden_r05', 'leiden_r08', 'leiden_r12']].copy()

    assignments.to_csv(CLUSTER_PATH, index=False, compression={'method': 'gzip', 'compresslevel': 6})

    audit_frames = []

    for key, resolution in resolutions.items():
        frame = assignments.groupby(key, observed=True).agg(nuclei=('nucleus_id', 'size'), donors=('GSM', 'nunique')).reset_index().rename(columns={key: 'cluster'})
        frame.insert(0, 'resolution', resolution)
        frame['percent_of_nuclei'] = 100.0 * frame['nuclei'] / EXPECTED_NUCLEI
        audit_frames.append(frame)

    cluster_audit = pd.concat(audit_frames, ignore_index=True)

    cluster_audit.to_csv(CLUSTER_AUDIT_PATH, index=False)

    run_lock = {'dataset': 'GSE157827', 'step': '15B.2', 'version': 'v1', 'created_utc': datetime.now(timezone.utc).isoformat(), 'analysis_role': 'Diagnosis-blind independent annotation sensitivity', 'plan_lock': str(PLAN_LOCK_PATH), 'software_amendment01': str(AMENDMENT01_PATH), 'loess_amendment02': str(AMENDMENT02_PATH), 'Harmony2_implementation_amendment03': str(AMENDMENT03_PATH), 'HVG_lock': str(HVG_LOCK_PATH), 'nuclei': EXPECTED_NUCLEI, 'donors': EXPECTED_DONORS, 'HVGs': EXPECTED_HVGS, 'normalization': {'method': 'log1p(CP10K)', 'denominator': 'Total raw UMI counts across all 33,538 genes per nucleus', 'target_sum': 10000.0}, 'PCA': {'components': 50, 'zero_center': True, 'solver': 'arpack', 'random_seed': SEED, 'total_explained_variance_ratio': float(variance_ratio.sum())}, 'Harmony': {'method': 'Harmony2', 'implementation': 'harmonypy 2.0.0', 'batch_key': 'GSM', 'input_components': 50, 'theta': 2.0, 'sigma': 0.1, 'lambda': 'dynamic', 'alpha': 0.2, 'max_iter_harmony': 10, 'max_iter_kmeans': 4, 'epsilon_cluster': 0.001, 'epsilon_harmony': 0.01, 'batch_prop_cutoff': 1e-05, 'random_seed': SEED, 'ncores': 1, 'diagnosis_used': False}, 'neighbors': {'representation': 'first 30 Harmony2 components', 'n_neighbors': 15, 'metric': 'euclidean', 'random_seed': SEED, 'median_different_donor_neighbor_fraction': float(np.median(different_fraction)), 'q10_different_donor_neighbor_fraction': float(np.quantile(different_fraction, 0.1)), 'q90_different_donor_neighbor_fraction': float(np.quantile(different_fraction, 0.9))}, 'Leiden': {'primary_resolution': 0.8, 'audit_resolutions': [0.5, 1.2], 'random_seed': SEED, 'cluster_number_forced': False, 'resolution_cluster_counts': {str(resolution): int(assignments[key].nunique()) for key, resolution in resolutions.items()}}, 'software': observed_versions, 'diagnosis_loaded': False, 'sample_label_loaded': False, 'target_expression_inspected': False, 'locked_targets_in_HVGs': 0, 'cell_types_assigned': False, 'differential_expression_performed': False, 'outputs': {'PCA50': str(PCA_PATH), 'Harmony2_50': str(HARMONY_PATH), 'neighbor_distances': str(DISTANCE_PATH), 'neighbor_connectivities': str(CONNECTIVITY_PATH), 'Leiden_assignments': str(CLUSTER_PATH), 'cluster_audit': str(CLUSTER_AUDIT_PATH)}}

    with open(LOCK_PATH, 'w', encoding='utf-8') as handle:
        json.dump(run_lock, handle, indent=2)

    print('\n' + '=' * 78)

    print('GSE157827 STEP 15B.2 COMPLETE')

    print('=' * 78)

    print(f'\nNuclei: {EXPECTED_NUCLEI:,}')

    print(f'Frozen HVGs: {EXPECTED_HVGS:,}')

    print('CP10K denominator: all 33,538 raw genes')

    print(f'PCA50 explained variance ratio: {variance_ratio.sum():.4f}')

    print('Harmony method: Harmony2')


In [ ]:
if RUN_PIPELINE:
    print('Harmony implementation: harmonypy 2.0.0')

    print('Harmony batch variable: GSM donor only')

    print(f'Median different-donor neighbor fraction: {np.median(different_fraction):.4f}')

    for key, resolution in resolutions.items():
        print(f'Leiden resolution {resolution:.1f}: {assignments[key].nunique()} clusters')

    print('Cluster number forced: NO')

    print('Diagnosis used: NO')

    print('Target expression inspected: NO')

    print('Cell types assigned: NO')

    print('Differential expression performed: NO')

    print('\nSTEP 15B.2 lock:')

    print(LOCK_PATH)

    print('\nNEXT: STEP 15C — frozen marker-based annotation of resolution-0.8 clusters.')


### Alternative cluster annotation, pseudobulk, and DE


In [ ]:
if RUN_PIPELINE:
    from pathlib import Path

    from datetime import datetime, timezone

    from importlib.metadata import version

    import gc

    import hashlib

    import json

    import numpy as np

    import pandas as pd

    import scipy.sparse as sp

    BASE = Path('/content/drive/MyDrive/AD_Astrocyte_Paper_01')

    validation_dirs = sorted(BASE.glob('*GSE157827*confirmatory*validation*'))

    if len(validation_dirs) != 1:
        raise RuntimeError(f'Expected exactly one validation directory; found {len(validation_dirs)}:\n' + '\n'.join((str(x) for x in validation_dirs)))

    VALIDATION_DIR = validation_dirs[0]

    def resolve_one(pattern, label):
        matches = sorted(VALIDATION_DIR.glob(pattern))
        if len(matches) != 1:
            raise RuntimeError(f'{label}: expected exactly one match for {pattern}; found {len(matches)}:\n' + '\n'.join((str(x) for x in matches)))
        return matches[0]

    PLAN15A_PATH = resolve_one('*STEP15A*ANNOTATION*SENSITIVITY*PLAN*LOCK*.json', 'STEP 15A plan')

    STEP15B1_LOCK_PATH = resolve_one('*STEP15B1*HVG*SELECTION*LOCK*.json', 'STEP 15B.1 lock')

    STEP15B2_LOCK_PATH = resolve_one('*STEP15B2*EMBEDDING_CLUSTERING_LOCK*.json', 'STEP 15B.2 lock')

    CLUSTER_PATH = resolve_one('*STEP15B2*Leiden_assignments*.csv.gz', 'STEP 15B.2 Leiden assignments')

    STEP9D_PATH = resolve_one('*STEP9D_CLUSTERING_PLAN_LOCK*.json', 'STEP 9D marker-plan lock')

    VAR_PATH = resolve_one('*STEP8_feature_metadata_v1_1.csv.gz', 'STEP 8 feature metadata')

    OBS_PATH = resolve_one('*STEP8_singlet_obs_v1_1.csv.gz', 'STEP 8 structural observations')

    PARTITION_PATH = resolve_one('*STEP8_matrix_partition_manifest_v1_1.csv', 'STEP 8 partition manifest')

    CLUSTER_LABEL_PATH = VALIDATION_DIR / 'GSE157827_STEP15C_cluster_annotations_r08_v1.csv'

    MARKER_PROFILE_PATH = VALIDATION_DIR / 'GSE157827_STEP15C_cluster_marker_profiles_r08_v1.csv'

    ASTROCYTE_PATH = VALIDATION_DIR / 'GSE157827_STEP15C_alternative_astrocyte_manifest_v1.csv.gz'

    DONOR_SUMMARY_PATH = VALIDATION_DIR / 'GSE157827_STEP15C_alternative_astrocytes_per_donor_v1.csv'

    COMPOSITION_PATH = VALIDATION_DIR / 'GSE157827_STEP15C_annotation_composition_v1.csv'

    LOCK_PATH = VALIDATION_DIR / 'GSE157827_STEP15C_ALTERNATIVE_ASTROCYTE_IDENTITY_LOCK_v1.json'

    if LOCK_PATH.exists():
        raise RuntimeError(f'STEP 15C already has a completed lock. Do not overwrite it:\n{LOCK_PATH}')

    EXPECTED_NUCLEI = 159144

    EXPECTED_DONORS = 21

    EXPECTED_FEATURES = 33538

    EXPECTED_PRIMARY_CLUSTERS = 26

    PRIMARY_RESOLUTION = 0.8

    TARGET_PANEL = {'CREB5', 'CSRP1', 'SHC1', 'KCNH2', 'MRGPRF', 'AJAP1', 'CCDC3'}

    with open(PLAN15A_PATH, 'r', encoding='utf-8') as handle:
        plan15a = json.load(handle)

    with open(STEP15B1_LOCK_PATH, 'r', encoding='utf-8') as handle:
        step15b1 = json.load(handle)

    with open(STEP15B2_LOCK_PATH, 'r', encoding='utf-8') as handle:
        step15b2 = json.load(handle)

    with open(STEP9D_PATH, 'r', encoding='utf-8') as handle:
        step9d = json.load(handle)

    if int(step15b1.get('selected_HVGs', -1)) != 3000:
        raise RuntimeError('STEP 15B.1 HVG count changed.')

    if float(step15b1.get('selected_LOESS_span', -1)) != 0.4:
        raise RuntimeError('STEP 15B.1 LOESS span changed.')

    if step15b1.get('diagnosis_loaded') is not False:
        raise RuntimeError('STEP 15B.1 diagnosis audit failed.')


In [ ]:
if RUN_PIPELINE:
    if step15b1.get('target_expression_inspected') is not False:
        raise RuntimeError('STEP 15B.1 target audit failed.')

    if step15b2.get('diagnosis_loaded') is not False:
        raise RuntimeError('STEP 15B.2 diagnosis audit failed.')

    if step15b2.get('target_expression_inspected') is not False:
        raise RuntimeError('STEP 15B.2 target audit failed.')

    cluster_counts = step15b2.get('Leiden', {}).get('resolution_cluster_counts', {})

    if int(cluster_counts.get('0.8', -1)) != EXPECTED_PRIMARY_CLUSTERS:
        raise RuntimeError('STEP 15B.2 resolution 0.8 does not contain 26 clusters.')

    leiden_plan = plan15a.get('Leiden', {})

    if float(leiden_plan.get('primary_sensitivity_resolution', -1)) != PRIMARY_RESOLUTION:
        raise RuntimeError('STEP 15A primary Leiden resolution is not 0.8.')

    audit_resolutions = leiden_plan.get('additional_resolution_audits')

    if audit_resolutions != [0.5, 1.2]:
        raise RuntimeError('STEP 15A audit resolutions changed.')

    if leiden_plan.get('cluster_number_forced') is not False:
        raise RuntimeError('STEP 15A cluster-number rule changed.')

    if plan15a.get('diagnosis_prohibited_until_alternative_astrocytes_are_frozen') is not True:
        raise RuntimeError('STEP 15A diagnosis-blinding rule changed.')

    if plan15a.get('targets_prohibited_from_HVGs_PCA_Harmony_neighbors_Leiden_annotation') is not True:
        raise RuntimeError('STEP 15A target-blinding rule changed.')

    if int(plan15a.get('minimum_astrocytes_per_donor', -1)) != 20:
        raise RuntimeError('STEP 15A minimum donor astrocyte threshold changed.')

    marker_panels = step9d.get('cluster_marker_panels')

    if not isinstance(marker_panels, dict):
        raise RuntimeError('STEP 9D marker panels unavailable.')

    required_lineages = {'Astrocyte', 'Endothelial', 'Excitatory_neuron', 'Inhibitory_neuron', 'Microglia', 'Oligodendrocyte'}

    if set(marker_panels.keys()) != required_lineages:
        raise RuntimeError('STEP 9D lineage panels changed.')

    plan_marker_panels = plan15a.get('annotation_marker_panels')

    if plan_marker_panels != marker_panels:
        raise RuntimeError('STEP 15A marker panels differ from the frozen STEP 9D panels.')

    all_marker_genes = {str(gene).strip().upper() for genes in marker_panels.values() for gene in genes}

    target_overlap = all_marker_genes & TARGET_PANEL

    if target_overlap:
        raise RuntimeError(f'Locked target occurs in annotation marker panels:\n{sorted(target_overlap)}')

    print('=' * 78)

    print('GSE157827 STEP 15C — DIAGNOSIS-BLIND ALTERNATIVE ANNOTATION')

    print('=' * 78)

    print('\nSTEP 15A primary resolution: 0.8')

    print('Observed resolution-0.8 clusters: 26')

    print('Diagnosis loaded: NO')

    print('sample_label loaded: NO')

    print('Locked targets queried: NO')

    print('\nFrozen marker panels:')

    for lineage, genes in marker_panels.items():
        print(f'  {lineage:20s}: ' + ', '.join(genes))

    clusters = pd.read_csv(CLUSTER_PATH, dtype={'nucleus_id': str, 'GSM': str, 'barcode': str, 'leiden_r05': str, 'leiden_r08': str, 'leiden_r12': str})

    obs = pd.read_csv(OBS_PATH, usecols=['nucleus_id', 'GSM', 'barcode', 'matrix_partition', 'partition_row', 'global_row'], dtype={'nucleus_id': str, 'GSM': str, 'barcode': str, 'matrix_partition': str})

    obs['partition_row'] = pd.to_numeric(obs['partition_row'], errors='raise').astype(np.int64)

    obs['global_row'] = pd.to_numeric(obs['global_row'], errors='raise').astype(np.int64)

    obs = obs.sort_values('global_row').reset_index(drop=True)

    var = pd.read_csv(VAR_PATH, dtype=str)

    partitions = pd.read_csv(PARTITION_PATH, dtype={'GSM': str, 'partition_file': str})


In [ ]:
if RUN_PIPELINE:
    for column in ['global_row_start', 'global_row_end_exclusive']:
        partitions[column] = pd.to_numeric(partitions[column], errors='raise').astype(np.int64)

    partitions = partitions.sort_values('global_row_start').reset_index(drop=True)

    if len(clusters) != EXPECTED_NUCLEI:
        raise RuntimeError(f'Expected {EXPECTED_NUCLEI:,} cluster rows; found {len(clusters):,}.')

    if len(obs) != EXPECTED_NUCLEI:
        raise RuntimeError('STEP 8 observation count changed.')

    if len(var) != EXPECTED_FEATURES:
        raise RuntimeError('STEP 8 feature count changed.')

    if len(partitions) != EXPECTED_DONORS:
        raise RuntimeError('STEP 8 donor count changed.')

    if not np.array_equal(obs['global_row'].to_numpy(), np.arange(EXPECTED_NUCLEI, dtype=np.int64)):
        raise RuntimeError('STEP 8 global-row ordering changed.')

    if not np.array_equal(clusters['nucleus_id'].astype(str).to_numpy(), obs['nucleus_id'].astype(str).to_numpy()):
        raise RuntimeError('Cluster and STEP 8 nucleus order differ.')

    if not np.array_equal(clusters['GSM'].astype(str).to_numpy(), obs['GSM'].astype(str).to_numpy()):
        raise RuntimeError('Cluster and STEP 8 donor order differ.')

    if clusters['leiden_r08'].nunique() != EXPECTED_PRIMARY_CLUSTERS:
        raise RuntimeError('Resolution 0.8 no longer contains 26 clusters.')

    cluster_labels_ordered = sorted(clusters['leiden_r08'].astype(str).unique().tolist(), key=lambda x: int(x))

    if len(cluster_labels_ordered) != EXPECTED_PRIMARY_CLUSTERS:
        raise RuntimeError('Unexpected number of cluster labels.')

    cluster_to_index = {label: index for index, label in enumerate(cluster_labels_ordered)}

    cluster_index_all = np.asarray([cluster_to_index[label] for label in clusters['leiden_r08'].astype(str)], dtype=np.int16)

    if 'gene_symbol' not in var.columns:
        raise RuntimeError('gene_symbol absent from STEP 8 metadata.')

    symbol_to_indices = {}

    for index, symbol in enumerate(var['gene_symbol'].fillna('').astype(str)):
        symbol = symbol.strip().upper()
        if symbol:
            symbol_to_indices.setdefault(symbol, []).append(index)

    marker_genes = sorted(all_marker_genes)

    missing_markers = [gene for gene in marker_genes if gene not in symbol_to_indices]

    if missing_markers:
        raise RuntimeError('Frozen annotation marker(s) missing from the STEP 8 feature table:\n' + '\n'.join(missing_markers))

    print(f'\nFrozen marker genes available: {len(marker_genes)}/{len(marker_genes)}')

    N_CLUSTERS = EXPECTED_PRIMARY_CLUSTERS

    cluster_n = np.zeros(N_CLUSTERS, dtype=np.int64)

    marker_log_sums = {gene: np.zeros(N_CLUSTERS, dtype=np.float64) for gene in marker_genes}

    print('\nReading diagnosis-blind marker expression...')


In [ ]:
if RUN_PIPELINE:
    for donor_number, partition in enumerate(partitions.itertuples(index=False), start=1):
        gsm = str(partition.GSM)
        start = int(partition.global_row_start)
        end = int(partition.global_row_end_exclusive)
        matrix_path = Path(partition.partition_file)
        if not matrix_path.exists():
            raise FileNotFoundError(f'{gsm}: matrix partition missing:\n{matrix_path}')
        donor_obs = obs.iloc[start:end]
        if not donor_obs['GSM'].astype(str).eq(gsm).all():
            raise RuntimeError(f'{gsm}: donor/global-row mismatch.')
        expected_partition_rows = np.arange(end - start, dtype=np.int64)
        if not np.array_equal(donor_obs['partition_row'].to_numpy(), expected_partition_rows):
            raise RuntimeError(f'{gsm}: partition row order invalid.')
        X = sp.load_npz(matrix_path).tocsr()
        if X.shape != (end - start, EXPECTED_FEATURES):
            raise RuntimeError(f'{gsm}: unexpected matrix shape {X.shape}.')
        library_size = np.asarray(X.sum(axis=1)).ravel().astype(np.float64)
        if np.any(library_size <= 0):
            raise RuntimeError(f'{gsm}: zero-count nucleus.')
        donor_cluster_index = cluster_index_all[start:end]
        cluster_n += np.bincount(donor_cluster_index, minlength=N_CLUSTERS)
        for gene in marker_genes:
            indices = symbol_to_indices[gene]
            if len(indices) == 1:
                raw = X[:, indices[0]].toarray().ravel()
            else:
                raw = np.asarray(X[:, indices].sum(axis=1)).ravel()
            raw = raw.astype(np.float64, copy=False)
            log_cp10k = np.log1p(raw / library_size * 10000.0)
            marker_log_sums[gene] += np.bincount(donor_cluster_index, weights=log_cp10k, minlength=N_CLUSTERS)
        print(f'  [{donor_number:02d}/21] {gsm}: {end - start:,} nuclei')
        del X
        del library_size
        del donor_cluster_index
        gc.collect()

    if int(cluster_n.sum()) != EXPECTED_NUCLEI:
        raise RuntimeError('Cluster counts do not sum to 159,144.')

    if np.any(cluster_n <= 0):
        raise RuntimeError('At least one cluster is empty.')

    marker_mean = {gene: marker_log_sums[gene] / cluster_n for gene in marker_genes}

    marker_z = {}

    for gene in marker_genes:
        values = marker_mean[gene]
        sd = values.std(ddof=0)
        if sd == 0:
            marker_z[gene] = np.zeros(N_CLUSTERS, dtype=np.float64)
        else:
            marker_z[gene] = (values - values.mean()) / sd

    lineages = list(marker_panels.keys())

    lineage_scores = np.zeros((N_CLUSTERS, len(lineages)), dtype=np.float64)

    lineage_positive_markers = np.zeros((N_CLUSTERS, len(lineages)), dtype=np.int16)


In [ ]:
if RUN_PIPELINE:
    for lineage_index, lineage in enumerate(lineages):
        genes = [str(gene).strip().upper() for gene in marker_panels[lineage]]
        Z = np.column_stack([marker_z[gene] for gene in genes])
        lineage_scores[:, lineage_index] = Z.mean(axis=1)
        lineage_positive_markers[:, lineage_index] = (Z > 0).sum(axis=1)

    top_index = np.argmax(lineage_scores, axis=1)

    top_score = lineage_scores[np.arange(N_CLUSTERS), top_index]

    sorted_scores = np.sort(lineage_scores, axis=1)

    second_score = sorted_scores[:, -2]

    score_margin = top_score - second_score

    number_at_top = np.sum(np.isclose(lineage_scores, top_score[:, None], rtol=0, atol=1e-12), axis=1)

    unique_top = number_at_top == 1

    top_positive_markers = lineage_positive_markers[np.arange(N_CLUSTERS), top_index]

    accepted = unique_top & (top_score > 0) & (top_positive_markers >= 2)

    top_lineage = np.asarray([lineages[index] for index in top_index], dtype=object)

    assigned_lineage = np.where(accepted, top_lineage, 'Unknown')

    cluster_annotation = pd.DataFrame({'cluster': cluster_labels_ordered, 'nuclei': cluster_n, 'top_lineage': top_lineage, 'assigned_lineage': assigned_lineage, 'top_score': top_score, 'second_score': second_score, 'score_margin': score_margin, 'unique_top': unique_top, 'top_positive_z_markers': top_positive_markers, 'accepted': accepted})

    for lineage_index, lineage in enumerate(lineages):
        cluster_annotation[f'score_{lineage}'] = lineage_scores[:, lineage_index]
        cluster_annotation[f'positive_z_{lineage}'] = lineage_positive_markers[:, lineage_index]

    cluster_annotation.to_csv(CLUSTER_LABEL_PATH, index=False)

    gene_to_lineages = {}

    for lineage, genes in marker_panels.items():
        for gene in genes:
            gene = str(gene).strip().upper()
            gene_to_lineages.setdefault(gene, []).append(lineage)

    marker_rows = []

    for cluster_index, cluster_label in enumerate(cluster_labels_ordered):
        for gene in marker_genes:
            marker_rows.append({'cluster': cluster_label, 'marker': gene, 'lineage_panels': ';'.join(gene_to_lineages[gene]), 'mean_log1p_CP10K': float(marker_mean[gene][cluster_index]), 'z_across_clusters': float(marker_z[gene][cluster_index])})

    marker_profile = pd.DataFrame(marker_rows)

    marker_profile.to_csv(MARKER_PROFILE_PATH, index=False)

    astrocyte_clusters = set(cluster_annotation.loc[cluster_annotation['assigned_lineage'] == 'Astrocyte', 'cluster'].astype(str))

    if len(astrocyte_clusters) == 0:
        raise RuntimeError('No resolution-0.8 cluster was assigned Astrocyte.')

    astrocytes = clusters.loc[clusters['leiden_r08'].astype(str).isin(astrocyte_clusters), ['nucleus_id', 'GSM', 'barcode', 'leiden_r08']].copy()

    astrocytes['alternative_cell_type'] = 'Astrocyte'

    astrocytes.to_csv(ASTROCYTE_PATH, index=False, compression={'method': 'gzip', 'compresslevel': 6})

    donor_order = obs['GSM'].drop_duplicates().astype(str).tolist()

    all_donors = pd.DataFrame({'GSM': donor_order})

    astro_counts = astrocytes['GSM'].value_counts().rename_axis('GSM').rename('alternative_astrocyte_nuclei').reset_index()

    donor_summary = all_donors.merge(astro_counts, on='GSM', how='left')

    donor_summary['alternative_astrocyte_nuclei'] = donor_summary['alternative_astrocyte_nuclei'].fillna(0).astype(np.int64)

    donor_summary['eligible_ge20'] = donor_summary['alternative_astrocyte_nuclei'] >= 20

    donor_summary.to_csv(DONOR_SUMMARY_PATH, index=False)

    composition = cluster_annotation.groupby('assigned_lineage', observed=True)['nuclei'].sum().rename('nuclei').reset_index()

    composition['percent_of_all_nuclei'] = 100.0 * composition['nuclei'] / EXPECTED_NUCLEI

    composition = composition.sort_values('nuclei', ascending=False).reset_index(drop=True)

    composition.to_csv(COMPOSITION_PATH, index=False)

    def sha256_file(path):
        digest = hashlib.sha256()
        with open(path, 'rb') as handle:
            for block in iter(lambda: handle.read(1024 * 1024), b''):
                digest.update(block)
        return digest.hexdigest()


In [ ]:
if RUN_PIPELINE:
    def sha256_list(values):
        payload = ('\n'.join((str(x) for x in values)) + '\n').encode('utf-8')
        return hashlib.sha256(payload).hexdigest()

    all_donors_ge20 = bool(donor_summary['eligible_ge20'].all())

    lock = {'dataset': 'GSE157827', 'step': '15C', 'version': 'v1', 'created_utc': datetime.now(timezone.utc).isoformat(), 'analysis_role': 'Diagnosis-blind alternative astrocyte identity freeze', 'STEP15A_plan': str(PLAN15A_PATH), 'STEP15B1_lock': str(STEP15B1_LOCK_PATH), 'STEP15B2_lock': str(STEP15B2_LOCK_PATH), 'STEP15B2_lock_sha256': sha256_file(STEP15B2_LOCK_PATH), 'STEP9D_marker_plan': str(STEP9D_PATH), 'STEP9D_marker_plan_sha256': sha256_file(STEP9D_PATH), 'primary_resolution': 0.8, 'primary_clusters': EXPECTED_PRIMARY_CLUSTERS, 'marker_panels': marker_panels, 'annotation_rule': {'marker_expression': 'mean nucleus-level log1p(CP10K) within resolution-0.8 cluster', 'CP10K_denominator': 'all 33,538 raw gene counts per nucleus', 'marker_standardization': 'z-score each marker across the 26 cluster means; ddof=0', 'lineage_score': 'mean marker z-score within lineage', 'unique_top_required': True, 'minimum_top_score_exclusive': 0.0, 'minimum_positive_z_markers': 2, 'tie_atol': 1e-12, 'unknown_if_rule_fails': True, 'published_composition_used': False, 'STEP9B_labels_used': False}, 'diagnosis_loaded': False, 'sample_label_loaded': False, 'target_expression_inspected': False, 'differential_expression_performed': False, 'astrocyte_clusters': sorted(astrocyte_clusters, key=int), 'alternative_astrocyte_nuclei': int(len(astrocytes)), 'alternative_astrocyte_nucleus_id_sha256': sha256_list(astrocytes['nucleus_id'].astype(str)), 'donors': EXPECTED_DONORS, 'minimum_required_astrocytes_per_donor': 20, 'minimum_observed_astrocytes_per_donor': int(donor_summary['alternative_astrocyte_nuclei'].min()), 'all_21_donors_ge20': all_donors_ge20, 'ready_for_alternative_pseudobulk': all_donors_ge20, 'outputs': {'cluster_annotations': str(CLUSTER_LABEL_PATH), 'marker_profile_audit': str(MARKER_PROFILE_PATH), 'alternative_astrocyte_manifest': str(ASTROCYTE_PATH), 'donor_astrocyte_counts': str(DONOR_SUMMARY_PATH), 'global_composition_audit': str(COMPOSITION_PATH)}, 'software': {'numpy': version('numpy'), 'pandas': version('pandas'), 'scipy': version('scipy')}}

    with open(LOCK_PATH, 'w', encoding='utf-8') as handle:
        json.dump(lock, handle, indent=2)

    print('\n' + '=' * 78)

    print('GSE157827 STEP 15C COMPLETE')

    print('=' * 78)

    print('\nCluster annotations:')

    print(cluster_annotation[['cluster', 'nuclei', 'assigned_lineage', 'top_score', 'top_positive_z_markers', 'score_margin']].to_string(index=False, float_format=lambda x: f'{x:.4f}'))

    print('\nGlobal diagnosis-blind composition:')

    print(composition.to_string(index=False, float_format=lambda x: f'{x:.3f}'))

    print('\nAlternative astrocyte clusters:', sorted(astrocyte_clusters, key=int))

    print('Alternative astrocyte nuclei:', f'{len(astrocytes):,}')

    print('\nAlternative astrocytes per donor:')

    print(donor_summary.to_string(index=False))

    print('\nMinimum astrocytes in any donor:', int(donor_summary['alternative_astrocyte_nuclei'].min()))

    print('All 21 donors have >=20 astrocytes:', all_donors_ge20)

    print('\nDiagnosis loaded: NO')

    print('sample_label loaded: NO')

    print('Locked targets queried: NO')

    print('Differential expression performed: NO')

    print('\nSTEP 15C lock:')

    print(LOCK_PATH)

    if all_donors_ge20:
        print('\nNEXT: STEP 15D — alternative astrocyte pseudobulk on the exact frozen 16,871-gene universe.')
    else:
        print('\nSTOP: one or more donors have <20 alternative astrocytes. Do not reveal diagnosis or run DE.')


In [ ]:
if RUN_PIPELINE:
    from pathlib import Path

    from datetime import datetime, timezone

    from importlib.metadata import version

    import gc

    import hashlib

    import json

    import numpy as np

    import pandas as pd

    import scipy.sparse as sp

    BASE = Path('/content/drive/MyDrive/AD_Astrocyte_Paper_01')

    validation_dirs = sorted(BASE.glob('*GSE157827*confirmatory*validation*'))

    if len(validation_dirs) != 1:
        raise RuntimeError(f'Expected exactly one validation directory; found {len(validation_dirs)}:\n' + '\n'.join((str(x) for x in validation_dirs)))

    VALIDATION_DIR = validation_dirs[0]

    def resolve_one(pattern, label):
        matches = sorted(VALIDATION_DIR.glob(pattern))
        if len(matches) != 1:
            raise RuntimeError(f'{label}: expected exactly one match for {pattern}; found {len(matches)}:\n' + '\n'.join((str(x) for x in matches)))
        return matches[0]

    STEP15C_LOCK_PATH = resolve_one('*STEP15C_ALTERNATIVE_ASTROCYTE_IDENTITY_LOCK*.json', 'STEP 15C identity lock')

    ASTROCYTE_PATH = resolve_one('*STEP15C_alternative_astrocyte_manifest*.csv.gz', 'STEP 15C astrocyte manifest')

    OBS_PATH = resolve_one('*STEP8_singlet_obs_v1_1.csv.gz', 'STEP 8 observation table')

    VAR_PATH = resolve_one('*STEP8_feature_metadata_v1_1.csv.gz', 'STEP 8 feature metadata')

    PARTITION_PATH = resolve_one('*STEP8_matrix_partition_manifest_v1_1.csv', 'STEP 8 partition manifest')

    STEP12_FEATURES_PATH = resolve_one('*STEP12_primary_DE_features_v1.csv.gz', 'STEP 12 frozen DE features')

    STEP12_LOCK_PATH = resolve_one('*STEP12_GENE_UNIVERSE_LOCK_v1.json', 'STEP 12 gene-universe lock')

    PSEUDOBULK_PATH = VALIDATION_DIR / 'GSE157827_STEP15C2_alternative_astrocyte_pseudobulk_16871_raw_counts_v1.npz'

    COUNT_TABLE_PATH = VALIDATION_DIR / 'GSE157827_STEP15C2_alternative_astrocyte_pseudobulk_16871_genes_x_donors_v1.csv.gz'

    DONOR_MANIFEST_PATH = VALIDATION_DIR / 'GSE157827_STEP15C2_alternative_astrocyte_pseudobulk_donor_manifest_v1.csv'

    LOCK_PATH = VALIDATION_DIR / 'GSE157827_STEP15C2_ALTERNATIVE_PSEUDOBULK_LOCK_v1.json'

    if LOCK_PATH.exists():
        raise RuntimeError(f'STEP 15C.2 already has a completed lock.\nDo not overwrite it:\n{LOCK_PATH}')

    EXPECTED_NUCLEI = 159144

    EXPECTED_ASTROCYTES = 17157

    EXPECTED_DONORS = 21

    EXPECTED_FEATURES = 33538

    EXPECTED_GENES = 16871

    EXPECTED_GENE_HASH = 'b2cd51ebe6557d79be9d7d4b9f7d514a77778bcc4857c7371252388827e8466a'

    with open(STEP15C_LOCK_PATH, 'r', encoding='utf-8') as handle:
        step15c = json.load(handle)

    if step15c.get('diagnosis_loaded') is not False:
        raise RuntimeError('STEP 15C diagnosis-blinding audit failed.')

    if step15c.get('target_expression_inspected') is not False:
        raise RuntimeError('STEP 15C target-blinding audit failed.')

    if int(step15c.get('alternative_astrocyte_nuclei', -1)) != EXPECTED_ASTROCYTES:
        raise RuntimeError('STEP 15C astrocyte count changed.')

    if step15c.get('all_21_donors_ge20') is not True:
        raise RuntimeError('STEP 15C donor eligibility failed.')

    if step15c.get('ready_for_alternative_pseudobulk') is not True:
        raise RuntimeError('STEP 15C is not marked ready for alternative pseudobulk.')

    astrocytes = pd.read_csv(ASTROCYTE_PATH, dtype={'nucleus_id': str, 'GSM': str, 'barcode': str, 'leiden_r08': str})

    if len(astrocytes) != EXPECTED_ASTROCYTES:
        raise RuntimeError(f'Expected {EXPECTED_ASTROCYTES:,} alternative astrocytes; found {len(astrocytes):,}.')

    if astrocytes['nucleus_id'].duplicated().any():
        raise RuntimeError('Duplicate nuclei in STEP 15C astrocyte manifest.')


In [ ]:
if RUN_PIPELINE:
    if astrocytes['GSM'].nunique() != EXPECTED_DONORS:
        raise RuntimeError('Alternative astrocytes no longer represent all 21 donors.')

    def sha256_list(values):
        payload = ('\n'.join((str(x) for x in values)) + '\n').encode('utf-8')
        return hashlib.sha256(payload).hexdigest()

    observed_astro_hash = sha256_list(astrocytes['nucleus_id'].astype(str))

    if observed_astro_hash != step15c['alternative_astrocyte_nucleus_id_sha256']:
        raise RuntimeError('STEP 15C astrocyte identity hash mismatch.')

    obs = pd.read_csv(OBS_PATH, usecols=['nucleus_id', 'GSM', 'barcode', 'partition_row', 'global_row'], dtype={'nucleus_id': str, 'GSM': str, 'barcode': str})

    obs['partition_row'] = pd.to_numeric(obs['partition_row'], errors='raise').astype(np.int64)

    obs['global_row'] = pd.to_numeric(obs['global_row'], errors='raise').astype(np.int64)

    if len(obs) != EXPECTED_NUCLEI:
        raise RuntimeError('STEP 8 structural observation count changed.')

    if obs['nucleus_id'].duplicated().any():
        raise RuntimeError('Duplicate nucleus IDs in STEP 8 observations.')

    astrocytes = astrocytes.merge(obs[['nucleus_id', 'GSM', 'barcode', 'partition_row', 'global_row']], on=['nucleus_id', 'GSM', 'barcode'], how='left', validate='one_to_one')

    if astrocytes[['partition_row', 'global_row']].isna().any().any():
        raise RuntimeError('Could not map all alternative astrocytes to STEP 8 matrix rows.')

    var = pd.read_csv(VAR_PATH, dtype=str)

    if len(var) != EXPECTED_FEATURES:
        raise RuntimeError('STEP 8 feature count changed.')

    if 'gene_id' not in var.columns:
        raise RuntimeError('gene_id absent from STEP 8 features.')

    if var['gene_id'].duplicated().any():
        raise RuntimeError('STEP 8 gene IDs are not unique.')

    features = pd.read_csv(STEP12_FEATURES_PATH, dtype={'gene_id': str, 'gene_symbol': str})

    if len(features) != EXPECTED_GENES:
        raise RuntimeError(f'Frozen STEP 12 universe contains {len(features):,} genes rather than {EXPECTED_GENES:,}.')

    if features['gene_id'].duplicated().any():
        raise RuntimeError('Duplicate gene IDs in frozen STEP 12 universe.')

    feature_ids = features['gene_id'].astype(str).tolist()

    observed_gene_hash = hashlib.sha256(('\n'.join(feature_ids) + '\n').encode('utf-8')).hexdigest()

    if observed_gene_hash != EXPECTED_GENE_HASH:
        raise RuntimeError('Frozen STEP 12 gene-universe SHA-256 mismatch.')

    gene_id_to_index = {str(gene_id): index for index, gene_id in enumerate(var['gene_id'].astype(str))}

    missing_gene_ids = [gene_id for gene_id in feature_ids if gene_id not in gene_id_to_index]

    if missing_gene_ids:
        raise RuntimeError(f'{len(missing_gene_ids)} frozen STEP 12 genes are absent from STEP 8 features.')

    frozen_feature_indices = np.asarray([gene_id_to_index[gene_id] for gene_id in feature_ids], dtype=np.int64)

    if len(np.unique(frozen_feature_indices)) != EXPECTED_GENES:
        raise RuntimeError('Frozen feature-index mapping is not one-to-one.')

    print('=' * 78)

    print('GSE157827 STEP 15C.2 — DIAGNOSIS-BLIND ALTERNATIVE PSEUDOBULK')

    print('=' * 78)

    print(f'\nFrozen alternative astrocytes: {len(astrocytes):,}')

    print(f"Donors: {astrocytes['GSM'].nunique()}")

    print(f'Frozen gene universe: {len(features):,}')

    print('Frozen universe SHA-256 verified: YES')

    print('\nDiagnosis loaded: NO')

    print('age/sex loaded: NO')

    print('Target expression queried: NO')

    print('Gene universe refiltered: NO')

    partitions = pd.read_csv(PARTITION_PATH, usecols=['GSM', 'partition_file', 'global_row_start', 'global_row_end_exclusive'], dtype={'GSM': str, 'partition_file': str})

    for column in ['global_row_start', 'global_row_end_exclusive']:
        partitions[column] = pd.to_numeric(partitions[column], errors='raise').astype(np.int64)


In [ ]:
if RUN_PIPELINE:
    partitions = partitions.sort_values('global_row_start').reset_index(drop=True)

    if len(partitions) != EXPECTED_DONORS:
        raise RuntimeError('STEP 8 partition manifest does not contain 21 donors.')

    if partitions['GSM'].duplicated().any():
        raise RuntimeError('Duplicate GSM in partition manifest.')

    donor_order = partitions['GSM'].astype(str).tolist()

    pseudobulk = np.zeros((EXPECTED_DONORS, EXPECTED_GENES), dtype=np.int64)

    donor_rows = []

    for donor_index, partition in enumerate(partitions.itertuples(index=False)):
        gsm = str(partition.GSM)
        matrix_path = Path(partition.partition_file)
        if not matrix_path.exists():
            raise FileNotFoundError(f'{gsm}: raw-count partition missing:\n{matrix_path}')
        donor_astro = astrocytes.loc[astrocytes['GSM'] == gsm].sort_values('partition_row').copy()
        n_astro = len(donor_astro)
        if n_astro < 20:
            raise RuntimeError(f'{gsm}: only {n_astro} alternative astrocytes; frozen donor threshold failed.')
        rows = donor_astro['partition_row'].to_numpy(dtype=np.int64)
        if len(np.unique(rows)) != len(rows):
            raise RuntimeError(f'{gsm}: duplicated partition rows in frozen astrocyte manifest.')
        X = sp.load_npz(matrix_path).tocsr()
        if X.shape[1] != EXPECTED_FEATURES:
            raise RuntimeError(f'{gsm}: expected {EXPECTED_FEATURES:,} raw features, found {X.shape[1]:,}.')
        if np.any(rows < 0) or np.any(rows >= X.shape[0]):
            raise RuntimeError(f'{gsm}: invalid partition row.')
        total_all_gene_umi = int(X[rows, :].sum())
        donor_counts = np.asarray(X[rows, :][:, frozen_feature_indices].sum(axis=0)).ravel()
        if np.any(donor_counts < 0):
            raise RuntimeError(f'{gsm}: negative pseudobulk count.')
        if not np.all(donor_counts == np.rint(donor_counts)):
            raise RuntimeError(f'{gsm}: non-integer pseudobulk count.')
        donor_counts = donor_counts.astype(np.int64, copy=False)
        pseudobulk[donor_index, :] = donor_counts
        total_frozen_gene_umi = int(donor_counts.sum())
        detected_frozen_genes = int(np.count_nonzero(donor_counts))
        donor_rows.append({'matrix_row': int(donor_index), 'GSM': gsm, 'alternative_astrocyte_nuclei': int(n_astro), 'total_raw_UMI_all_33538_features': total_all_gene_umi, 'total_raw_UMI_frozen_16871_genes': total_frozen_gene_umi, 'detected_frozen_genes': detected_frozen_genes, 'eligible_ge20': bool(n_astro >= 20)})
        print(f'[{donor_index + 1:02d}/21] {gsm}: {n_astro:,} astrocytes, {total_frozen_gene_umi:,} UMIs in frozen universe')
        del X
        del donor_counts
        del donor_astro
        del rows
        gc.collect()

    donor_manifest = pd.DataFrame(donor_rows)

    if donor_manifest['alternative_astrocyte_nuclei'].sum() != EXPECTED_ASTROCYTES:
        raise RuntimeError('Per-donor alternative astrocytes do not sum to 17,157.')

    if not donor_manifest['eligible_ge20'].all():
        raise RuntimeError('At least one donor fails the frozen >=20 rule.')

    if pseudobulk.shape != (EXPECTED_DONORS, EXPECTED_GENES):
        raise RuntimeError(f'Unexpected pseudobulk shape: {pseudobulk.shape}')

    if not np.issubdtype(pseudobulk.dtype, np.integer):
        raise RuntimeError('Pseudobulk matrix is not integer.')

    if np.any(pseudobulk < 0):
        raise RuntimeError('Negative pseudobulk count detected.')

    if np.any(pseudobulk.sum(axis=1) <= 0):
        raise RuntimeError('At least one donor pseudobulk has zero total count.')


In [ ]:
if RUN_PIPELINE:
    if np.any(pseudobulk.sum(axis=0) < 0):
        raise RuntimeError('Invalid gene count total.')

    pseudobulk_sparse = sp.csr_matrix(pseudobulk)

    sp.save_npz(PSEUDOBULK_PATH, pseudobulk_sparse, compressed=True)

    saved_check = sp.load_npz(PSEUDOBULK_PATH).tocsr()

    if saved_check.shape != (EXPECTED_DONORS, EXPECTED_GENES):
        raise RuntimeError('Saved pseudobulk shape verification failed.')

    if not np.array_equal(saved_check.toarray(), pseudobulk):
        raise RuntimeError('Saved pseudobulk count verification failed.')

    del saved_check

    count_table = features[['gene_id', 'gene_symbol']].copy()

    for donor_index, gsm in enumerate(donor_order):
        count_table[gsm] = pseudobulk[donor_index, :]

    count_table.to_csv(COUNT_TABLE_PATH, index=False, compression={'method': 'gzip', 'compresslevel': 6})

    donor_manifest.to_csv(DONOR_MANIFEST_PATH, index=False)

    def sha256_file(path):
        digest = hashlib.sha256()
        with open(path, 'rb') as handle:
            for block in iter(lambda: handle.read(1024 * 1024), b''):
                digest.update(block)
        return digest.hexdigest()

    lock = {'dataset': 'GSE157827', 'step': '15C.2', 'version': 'v1', 'created_utc': datetime.now(timezone.utc).isoformat(), 'analysis_role': 'Diagnosis-blind alternative astrocyte donor-level pseudobulk freeze', 'STEP15C_identity_lock': str(STEP15C_LOCK_PATH), 'STEP15C_identity_lock_sha256': sha256_file(STEP15C_LOCK_PATH), 'STEP12_gene_universe_lock': str(STEP12_LOCK_PATH), 'STEP12_features': str(STEP12_FEATURES_PATH), 'frozen_gene_universe_size': EXPECTED_GENES, 'frozen_gene_universe_sha256': observed_gene_hash, 'gene_universe_refiltered': False, 'alternative_astrocyte_nuclei': EXPECTED_ASTROCYTES, 'donors': EXPECTED_DONORS, 'minimum_astrocytes_per_donor': int(donor_manifest['alternative_astrocyte_nuclei'].min()), 'all_donors_ge20': bool(donor_manifest['eligible_ge20'].all()), 'matrix_shape': [EXPECTED_DONORS, EXPECTED_GENES], 'matrix_dtype': str(pseudobulk.dtype), 'raw_integer_counts': True, 'normalization_performed': False, 'diagnosis_loaded': False, 'age_loaded': False, 'sex_loaded': False, 'target_expression_queried': False, 'differential_expression_performed': False, 'donor_order': donor_order, 'outputs': {'raw_pseudobulk_npz': str(PSEUDOBULK_PATH), 'genes_x_donors_raw_counts': str(COUNT_TABLE_PATH), 'diagnosis_blind_donor_manifest': str(DONOR_MANIFEST_PATH)}, 'software': {'numpy': version('numpy'), 'pandas': version('pandas'), 'scipy': version('scipy')}}

    with open(LOCK_PATH, 'w', encoding='utf-8') as handle:
        json.dump(lock, handle, indent=2)

    print('\n' + '=' * 78)

    print('GSE157827 STEP 15C.2 COMPLETE')

    print('=' * 78)

    print(f'\nAlternative astrocytes: {EXPECTED_ASTROCYTES:,}')

    print(f'Donors: {EXPECTED_DONORS}')

    print(f'Frozen genes: {EXPECTED_GENES:,}')

    print('Gene universe refiltered: NO')

    print('Frozen gene-universe SHA-256:')

    print(observed_gene_hash)

    print('\nRaw pseudobulk shape:', pseudobulk.shape)

    print('Raw pseudobulk dtype:', pseudobulk.dtype)

    print('Minimum astrocytes per donor:', int(donor_manifest['alternative_astrocyte_nuclei'].min()))

    print('\nDiagnosis loaded: NO')

    print('age loaded: NO')

    print('sex loaded: NO')

    print('Target expression queried: NO')

    print('Differential expression performed: NO')

    print('\nDiagnosis-blind donor manifest:')

    print(donor_manifest.to_string(index=False))

    print('\nSTEP 15C.2 lock:')

    print(LOCK_PATH)

    print('\nNEXT: STEP 15D — load the already-frozen donor design (age + sex + diagnosis), fit the exact ~ age + sex + diagnosis PyDESeq2 model, and ONLY THEN reveal CREB5.')


In [ ]:
if RUN_PIPELINE:
    from pathlib import Path

    import importlib.metadata as md

    import json

    import os

    import subprocess

    import sys

    import textwrap

    BASE = Path('/content/drive/MyDrive/AD_Astrocyte_Paper_01')

    dirs = sorted(BASE.glob('*GSE157827*confirmatory*validation*'))

    if len(dirs) != 1:
        raise RuntimeError(f'Expected one validation directory; found {len(dirs)}:\n' + '\n'.join((str(x) for x in dirs)))

    VALIDATION_DIR = dirs[0]

    print('=' * 78)

    print('GSE157827 STEP 15D — PREPARATION')

    print('=' * 78)

    print('\nValidation directory:')

    print(VALIDATION_DIR)

    REQUIRED = {'numpy': '2.1.3', 'pandas': '2.2.3', 'scipy': '1.16.3', 'scikit-learn': '1.6.1', 'anndata': '0.12.6', 'pydeseq2': '0.5.4', 'formulaic': '1.2.2', 'formulaic-contrasts': '1.0.0'}

    def installed_version(package):
        try:
            return md.version(package)
        except md.PackageNotFoundError:
            return None

    observed_before = {package: installed_version(package) for package in REQUIRED}

    needs_restore = any((observed_before[package] != expected for package, expected in REQUIRED.items()))

    if needs_restore:
        print('\nRestoring exact locked PyDESeq2 environment...')
        packages = [f'{package}=={expected}' for package, expected in REQUIRED.items()]
        installation = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', '--upgrade-strategy', 'only-if-needed', *packages], text=True, capture_output=True)
        if installation.returncode != 0:
            print('\nPIP STDOUT:')
            print(installation.stdout)
            print('\nPIP STDERR:')
            print(installation.stderr)
            raise RuntimeError('Exact PyDESeq2 environment restoration failed. STEP 15D was NOT run.')
    else:
        print('\nExact PyDESeq2 environment already present.')

    verify_code = '\nimport json\nimport sys\nfrom importlib.metadata import version\n\nfrom pydeseq2.dds import DeseqDataSet\nfrom pydeseq2.ds import DeseqStats\n\npackages = [\n    "numpy",\n    "pandas",\n    "scipy",\n    "scikit-learn",\n    "anndata",\n    "pydeseq2",\n    "formulaic",\n    "formulaic-contrasts",\n]\n\nprint(\n    json.dumps(\n        {\n            "python":\n                sys.version.split()[0],\n\n            "versions":\n                {\n                    package:\n                        version(package)\n                    for package\n                    in packages\n                },\n\n            "pydeseq2_imports":\n                True,\n        }\n    )\n)\n'

    verification_run = subprocess.run([sys.executable, '-c', verify_code], text=True, capture_output=True)

    if verification_run.returncode != 0:
        print('\nVERIFY STDOUT:')
        print(verification_run.stdout)
        print('\nVERIFY STDERR:')
        print(verification_run.stderr)
        raise RuntimeError('Fresh PyDESeq2 verification failed. STEP 15D was NOT run.')

    verification = json.loads(verification_run.stdout.strip())

    for package, expected in REQUIRED.items():
        observed = verification['versions'][package]
        if observed != expected:
            raise RuntimeError(f'{package}: expected {expected}, found {observed}.')

    print('\nFresh subprocess environment verified:')

    print('  Python:', verification['python'])

    for package, value in verification['versions'].items():
        print(f'  {package}: {value}')

    RUNNER_PATH = Path('/content/GSE157827_STEP15D_runner.py')


In [ ]:
if RUN_PIPELINE:
    runner = '\nfrom pathlib import Path\nfrom datetime import datetime, timezone\nimport hashlib\nimport importlib.metadata as md\nimport json\nimport os\n\nimport numpy as np\nimport pandas as pd\nimport scipy.sparse as sp\n\nfrom pydeseq2.dds import DeseqDataSet\nfrom pydeseq2.ds import DeseqStats\n\n\n# ============================================================\n# A. Constants\n# ============================================================\n\nEXPECTED_DONORS = 21\nEXPECTED_AD = 12\nEXPECTED_CONTROL = 9\nEXPECTED_GENES = 16_871\nEXPECTED_ASTROCYTES = 17_157\n\nEXPECTED_UNIVERSE_SHA256 = (\n    "b2cd51ebe6557d79be9d7d4b9f7d514"\n    "a77778bcc4857c7371252388827e8466a"\n)\n\nPRIMARY_GENE = "CREB5"\n\nSECONDARY_GENES = [\n    "CSRP1",\n    "SHC1",\n    "KCNH2",\n    "MRGPRF",\n    "AJAP1",\n    "CCDC3",\n]\n\n\nVALIDATION_DIR = Path(\n    os.environ[\n        "GSE_VALIDATION_DIR"\n    ]\n)\n\n\n# ============================================================\n# B. Helpers\n# ============================================================\n\ndef resolve_one(pattern, label):\n\n    matches = sorted(\n        VALIDATION_DIR.glob(pattern)\n    )\n\n    if len(matches) != 1:\n\n        raise RuntimeError(\n            f"{label}: expected exactly one "\n            f"match for {pattern}; "\n            f"found {len(matches)}:\\n"\n            +\n            "\\n".join(\n                str(x)\n                for x in matches\n            )\n        )\n\n    return matches[0]\n\n\ndef sha256_file(path):\n\n    digest = hashlib.sha256()\n\n    with open(\n        path,\n        "rb",\n    ) as handle:\n\n        for block in iter(\n            lambda: handle.read(\n                1024 * 1024\n            ),\n            b"",\n        ):\n\n            digest.update(block)\n\n    return digest.hexdigest()\n\n\ndef json_number(value):\n\n    try:\n        value = float(value)\n\n    except (\n        TypeError,\n        ValueError,\n    ):\n        return None\n\n    if not np.isfinite(value):\n        return None\n\n    return value\n\n\n# ============================================================\n# C. Resolve frozen inputs\n# ============================================================\n\nSTEP15C_LOCK_PATH = resolve_one(\n    "*STEP15C_ALTERNATIVE_ASTROCYTE_IDENTITY_LOCK*.json",\n    "STEP 15C identity lock",\n)\n\nSTEP15C2_LOCK_PATH = resolve_one(\n    "*STEP15C2_ALTERNATIVE_PSEUDOBULK_LOCK*.json",\n    "STEP 15C.2 pseudobulk lock",\n)\n\nCOUNTS_PATH = resolve_one(\n    "*STEP15C2_alternative_astrocyte_"\n    "pseudobulk_16871_raw_counts*.npz",\n    "STEP 15C.2 pseudobulk counts",\n)\n\nALT_DONOR_PATH = resolve_one(\n    "*STEP15C2_alternative_astrocyte_"\n    "pseudobulk_donor_manifest*.csv",\n    "STEP 15C.2 donor manifest",\n)\n\nFEATURES_PATH = resolve_one(\n    "*STEP12_primary_DE_features_v1.csv.gz",\n    "STEP 12 frozen features",\n)\n\nSTEP12_LOCK_PATH = resolve_one(\n    "*STEP12_GENE_UNIVERSE_LOCK*.json",\n    "STEP 12 gene-universe lock",\n)\n\nDESIGN_PATH = resolve_one(\n    "*STEP11B_primary_design_manifest_v1.csv",\n    "STEP 11B design manifest",\n)\n\nDESIGN_LOCK_PATH = resolve_one(\n    "*STEP11B_PRIMARY_DESIGN_LOCK*.json",\n    "STEP 11B design lock",\n)\n\nSTEP13A_LOCK_PATH = resolve_one(\n    "*STEP13A_PYDESEQ2_ENVIRONMENT_LOCK*.json",\n    "STEP 13A environment lock",\n)\n\nPRIMARY_RESULT_LOCK_PATH = resolve_one(\n    "*STEP13B_PRIMARY_CONFIRMATORY_RESULT_LOCK*.json",\n    "STEP 13B primary result lock",\n)\n\n\n# ============================================================\n# D. Output paths\n# ============================================================\n\nRESULTS_PATH = (\n    VALIDATION_DIR /\n    "GSE157827_STEP15D_DESeq2_"\n    "alternative_annotation_age_sex_adjusted_v1.csv.gz"\n)\n\nSIGNIFICANT_PATH = (\n    VALIDATION_DIR /\n    "GSE157827_STEP15D_DESeq2_"\n    "alternative_annotation_FDR05_v1.csv.gz"\n)\n\nTARGET_PATH = (\n    VALIDATION_DIR /\n    "GSE157827_STEP15D_"\n    "locked_target_results_v1.csv"\n)\n\nSIZE_FACTOR_PATH = (\n    VALIDATION_DIR /\n    "GSE157827_STEP15D_"\n    "DESeq2_size_factors_v1.csv"\n)\n\nLOCK_PATH = (\n    VALIDATION_DIR /\n    "GSE157827_STEP15D_"\n    "ALTERNATIVE_ANNOTATION_SENSITIVITY_LOCK_v1.json"\n)\n\n\nif LOCK_PATH.exists():\n\n    raise RuntimeError(\n        "STEP 15D already has a completed result lock.\\n"\n        "Do not overwrite it:\\n"\n        f"{LOCK_PATH}"\n    )\n\n\n# ============================================================\n# E. Verify exact software\n# ============================================================\n\nREQUIRED = {\n    "numpy": "2.1.3",\n    "pandas": "2.2.3",\n    "scipy": "1.16.3",\n    "scikit-learn": "1.6.1",\n    "anndata": "0.12.6",\n    "pydeseq2": "0.5.4",\n    "formulaic": "1.2.2",\n    "formulaic-contrasts": "1.0.0",\n}\n\n\nobserved_versions = {\n    package:\n        md.version(package)\n    for package\n    in REQUIRED\n}\n\n\nif observed_versions != REQUIRED:\n\n    raise RuntimeError(\n        "STEP 15D software mismatch.\\n"\n        f"Observed:\\n{observed_versions}"\n    )\n\n\n# ============================================================\n# F. Verify all frozen upstream locks BEFORE diagnosis reveal\n# ============================================================\n\nwith open(\n    STEP15C_LOCK_PATH,\n    "r",\n    encoding="utf-8",\n) as handle:\n\n    step15c = json.load(handle)\n\n\nwith open(\n    STEP15C2_LOCK_PATH,\n    "r",\n    encoding="utf-8",\n) as handle:\n\n    step15c2 = json.load(handle)\n\n\nwith open(\n    STEP12_LOCK_PATH,\n    "r",\n    encoding="utf-8",\n) as handle:\n\n    step12 = json.load(handle)\n\n\nwith open(\n    DESIGN_LOCK_PATH,\n    "r",\n    encoding="utf-8",\n) as handle:\n\n    design_lock = json.load(handle)\n\n\nwith open(\n    STEP13A_LOCK_PATH,\n    "r",\n    encoding="utf-8",\n) as handle:\n\n    step13a = json.load(handle)\n\n\nif (\n    step15c.get(\n        "alternative_astrocyte_nuclei"\n    )\n    != EXPECTED_ASTROCYTES\n):\n\n    raise RuntimeError(\n        "STEP 15C astrocyte count changed."\n    )\n\n\nif step15c.get(\n    "all_21_donors_ge20"\n) is not True:\n\n    raise RuntimeError(\n        "STEP 15C donor threshold failed."\n    )\n\n\nif step15c2.get(\n    "diagnosis_loaded"\n) is not False:\n\n    raise RuntimeError(\n        "Diagnosis was already loaded in STEP 15C.2."\n    )\n\n\nif step15c2.get(\n    "target_expression_queried"\n) is not False:\n\n    raise RuntimeError(\n        "Target expression was already queried "\n        "in STEP 15C.2."\n    )\n\n\nif step15c2.get(\n    "differential_expression_performed"\n) is not False:\n\n    raise RuntimeError(\n        "Unexpected prior DE in STEP 15C.2."\n    )\n\n\nif int(\n    step15c2.get(\n        "frozen_gene_universe_size",\n        -1,\n    )\n) != EXPECTED_GENES:\n\n    raise RuntimeError(\n        "STEP 15C.2 gene-universe size changed."\n    )\n\n\nif step15c2.get(\n    "frozen_gene_universe_sha256"\n) != EXPECTED_UNIVERSE_SHA256:\n\n    raise RuntimeError(\n        "STEP 15C.2 gene-universe hash changed."\n    )\n\n\nif step15c2.get(\n    "gene_universe_refiltered"\n) is not False:\n\n    raise RuntimeError(\n        "STEP 15C.2 gene universe was refiltered."\n    )\n\n\nif not step12.get(\n    "primary_gene_universe_frozen",\n    False,\n):\n\n    raise RuntimeError(\n        "STEP 12 gene universe is not frozen."\n    )\n\n\nif int(\n    step12.get(\n        "primary_DE_features",\n        -1,\n    )\n) != EXPECTED_GENES:\n\n    raise RuntimeError(\n        "STEP 12 gene-universe size mismatch."\n    )\n\n\nif step12.get(\n    "primary_universe_gene_id_sha256"\n) != EXPECTED_UNIVERSE_SHA256:\n\n    raise RuntimeError(\n        "STEP 12 universe hash mismatch."\n    )\n\n\nif design_lock.get(\n    "primary_design"\n) != "~ age + sex + diagnosis":\n\n    raise RuntimeError(\n        "STEP 11B primary design changed."\n    )\n\n\nif design_lock.get(\n    "primary_contrast"\n) != "AD versus Control":\n\n    raise RuntimeError(\n        "STEP 11B contrast changed."\n    )\n\n\nif design_lock.get(\n    "design_full_rank"\n) is not True:\n\n    raise RuntimeError(\n        "STEP 11B design is not locked full-rank."\n    )\n\n\nif step13a.get(\n    "DE_engine_version"\n) != "0.5.4":\n\n    raise RuntimeError(\n        "STEP 13A PyDESeq2 version changed."\n    )\n\n\nprint(\n    "\\n"\n    +\n    "=" * 78\n)\n\nprint(\n    "GSE157827 STEP 15D — "\n    "INDEPENDENT ANNOTATION SENSITIVITY DE"\n)\n\nprint(\n    "=" * 78\n)\n\nprint(\n    "\\nPre-DE frozen-state verification: PASS"\n)\n\nprint(\n    f"Alternative astrocytes: "\n    f"{EXPECTED_ASTROCYTES:,}"\n)\n\nprint(\n    f"Frozen genes: "\n    f"{EXPECTED_GENES:,}"\n)\n\nprint(\n    "Gene universe refiltered: NO"\n)\n\nprint(\n    "Target expression queried before DE: NO"\n)\n\n\n# ============================================================\n# G. Load alternative raw pseudobulk\n# ============================================================\n\nX = sp.load_npz(\n    COUNTS_PATH\n).tocsr()\n\n\nif X.shape != (\n    EXPECTED_DONORS,\n    EXPECTED_GENES,\n):\n\n    raise RuntimeError(\n        f"Unexpected alternative count shape: "\n        f"{X.shape}"\n    )\n\n\nif not np.issubdtype(\n    X.dtype,\n    np.integer,\n):\n\n    raise RuntimeError(\n        "Alternative pseudobulk counts "\n        "are not raw integers."\n    )\n\n\nif X.data.size > 0 and X.data.min() < 0:\n\n    raise RuntimeError(\n        "Negative count detected."\n    )\n\n\nif np.any(\n    np.asarray(\n        X.sum(\n            axis=1\n        )\n    ).ravel() <= 0\n):\n\n    raise RuntimeError(\n        "At least one donor has zero "\n        "pseudobulk library size."\n    )\n\n\n# ============================================================\n# H. Verify exact 16,871-gene feature order\n# ============================================================\n\nfeatures = pd.read_csv(\n    FEATURES_PATH,\n    dtype={\n        "gene_id": str,\n        "gene_symbol": str,\n    },\n)\n\n\nif len(\n    features\n) != EXPECTED_GENES:\n\n    raise RuntimeError(\n        "Frozen feature table is not 16,871 genes."\n    )\n\n\nif features[\n    "gene_id"\n].duplicated().any():\n\n    raise RuntimeError(\n        "Duplicate frozen gene IDs."\n    )\n\n\nfeature_ids = (\n    features[\n        "gene_id"\n    ]\n    .astype(str)\n    .tolist()\n)\n\n\nobserved_hash = hashlib.sha256(\n    (\n        "\\n".join(\n            feature_ids\n        )\n        +\n        "\\n"\n    ).encode(\n        "utf-8"\n    )\n).hexdigest()\n\n\nif observed_hash != EXPECTED_UNIVERSE_SHA256:\n\n    raise RuntimeError(\n        "Frozen feature SHA-256 mismatch."\n    )\n\n\n# ============================================================\n# I. LAST BLIND CHECK: alternative donor order\n# ============================================================\n\nalt_donors = (\n    pd.read_csv(\n        ALT_DONOR_PATH,\n        dtype={\n            "GSM": str,\n        },\n    )\n    .sort_values(\n        "matrix_row"\n    )\n    .reset_index(\n        drop=True\n    )\n)\n\n\nif len(\n    alt_donors\n) != EXPECTED_DONORS:\n\n    raise RuntimeError(\n        "Alternative donor manifest "\n        "does not contain 21 donors."\n    )\n\n\nif not np.array_equal(\n    alt_donors[\n        "matrix_row"\n    ].to_numpy(),\n    np.arange(\n        EXPECTED_DONORS\n    ),\n):\n\n    raise RuntimeError(\n        "Alternative matrix_row is not 0..20."\n    )\n\n\nif not alt_donors[\n    "eligible_ge20"\n].all():\n\n    raise RuntimeError(\n        "At least one alternative donor "\n        "fails >=20 astrocytes."\n    )\n\n\n# ============================================================\n# J. DIAGNOSIS REVEAL IS NOW ALLOWED\n#\n# Alternative astrocyte identity, raw pseudobulk counts,\n# donor order, and 16,871-gene universe are all frozen.\n# ============================================================\n\ndesign = (\n    pd.read_csv(\n        DESIGN_PATH,\n        dtype={\n            "GSM": str,\n            "sample_label": str,\n            "diagnosis": str,\n            "sex": str,\n        },\n    )\n    .sort_values(\n        "matrix_row"\n    )\n    .reset_index(\n        drop=True\n    )\n)\n\n\nif len(\n    design\n) != EXPECTED_DONORS:\n\n    raise RuntimeError(\n        "STEP 11B design does not contain 21 donors."\n    )\n\n\nif not np.array_equal(\n    design[\n        "matrix_row"\n    ].to_numpy(),\n    np.arange(\n        EXPECTED_DONORS\n    ),\n):\n\n    raise RuntimeError(\n        "STEP 11B matrix_row is not 0..20."\n    )\n\n\nif not np.array_equal(\n    design[\n        "GSM"\n    ].astype(str).to_numpy(),\n    alt_donors[\n        "GSM"\n    ].astype(str).to_numpy(),\n):\n\n    raise RuntimeError(\n        "Alternative pseudobulk donor order "\n        "does not match frozen STEP 11B design."\n    )\n\n\ngroup_counts = (\n    design[\n        "diagnosis"\n    ]\n    .astype(str)\n    .value_counts()\n    .to_dict()\n)\n\n\nif group_counts != {\n    "AD": EXPECTED_AD,\n    "Control": EXPECTED_CONTROL,\n}:\n\n    raise RuntimeError(\n        f"Unexpected diagnosis counts: "\n        f"{group_counts}"\n    )\n\n\nprint(\n    "\\nDiagnosis reveal after pseudobulk freeze: YES"\n)\n\nprint(\n    "AD donors: 12"\n)\n\nprint(\n    "Control donors: 9"\n)\n\n\n# ============================================================\n# K. Construct donor x gene counts DataFrame\n# ============================================================\n\nsample_ids = (\n    design[\n        "GSM"\n    ]\n    .astype(str)\n    .tolist()\n)\n\n\ncounts = pd.DataFrame(\n\n    X.toarray().astype(\n        np.int64,\n        copy=False,\n    ),\n\n    index=pd.Index(\n        sample_ids,\n        dtype=object,\n    ),\n\n    columns=pd.Index(\n        feature_ids,\n        dtype=object,\n    ),\n)\n\n\nif counts.shape != (\n    EXPECTED_DONORS,\n    EXPECTED_GENES,\n):\n\n    raise RuntimeError(\n        "DESeq2 count DataFrame shape mismatch."\n    )\n\n\nif counts.to_numpy().min() < 0:\n\n    raise RuntimeError(\n        "Negative DESeq2 count."\n    )\n\n\n# ============================================================\n# L. Exact frozen model metadata\n# ============================================================\n\nmetadata = design[\n    [\n        "age",\n        "sex",\n        "diagnosis",\n    ]\n].copy()\n\n\nmetadata[\n    "age"\n] = pd.to_numeric(\n    metadata[\n        "age"\n    ],\n    errors="raise",\n)\n\n\nmetadata[\n    "sex"\n] = pd.Categorical(\n    metadata[\n        "sex"\n    ].astype(str),\n\n    categories=[\n        "F",\n        "M",\n    ],\n\n    ordered=True,\n)\n\n\nmetadata[\n    "diagnosis"\n] = pd.Categorical(\n    metadata[\n        "diagnosis"\n    ].astype(str),\n\n    categories=[\n        "Control",\n        "AD",\n    ],\n\n    ordered=True,\n)\n\n\nmetadata.index = pd.Index(\n    sample_ids,\n    dtype=object,\n)\n\n\nmetadata.columns = pd.Index(\n    metadata.columns.astype(\n        str\n    ).tolist(),\n    dtype=object,\n)\n\n\nif not counts.index.equals(\n    metadata.index\n):\n\n    raise RuntimeError(\n        "Count/metadata donor-order mismatch."\n    )\n\n\nif metadata.isna().any().any():\n\n    raise RuntimeError(\n        "Missing frozen-model metadata."\n    )\n\n\n# ============================================================\n# M. PRE-FIT REPORT\n# ============================================================\n\nprint(\n    "\\n"\n    +\n    "=" * 78\n)\n\nprint(\n    "STEP 15D PRE-FIT LOCKED MODEL"\n)\n\nprint(\n    "=" * 78\n)\n\nprint(\n    "\\nBiological replicates: 21"\n)\n\nprint(\n    "Frozen genes: 16,871"\n)\n\nprint(\n    "Astrocyte definition: "\n    "independent Harmony2/Leiden annotation"\n)\n\nprint(\n    "Model: ~ age + sex + diagnosis"\n)\n\nprint(\n    "Contrast: AD versus Control"\n)\n\nprint(\n    "Reference diagnosis: Control"\n)\n\nprint(\n    "Reference sex: F"\n)\n\nprint(\n    "Positive log2FC: higher expression in AD"\n)\n\nprint(\n    "\\nFrozen universe hash verified: YES"\n)\n\nprint(\n    "Raw integer counts verified: YES"\n)\n\nprint(\n    "Donor ordering verified: YES"\n)\n\nprint(\n    "\\nTarget genes have NOT yet been extracted."\n)\n\n\n# ============================================================\n# N. Construct exact DESeq2 model\n# ============================================================\n\ndds = DeseqDataSet(\n\n    counts=counts,\n\n    metadata=metadata,\n\n    design=\n        "~ age + sex + diagnosis",\n\n    refit_cooks=True,\n\n    n_cpus=2,\n\n    quiet=False,\n)\n\n\ndesign_matrix = dds.obsm[\n    "design_matrix"\n]\n\n\ndesign_rank = int(\n    np.linalg.matrix_rank(\n        design_matrix.to_numpy()\n    )\n)\n\n\nprint(\n    "\\nPyDESeq2 design columns:"\n)\n\nprint(\n    design_matrix.columns.tolist()\n)\n\nprint(\n    "Design rank:",\n    design_rank,\n    "/",\n    design_matrix.shape[1],\n)\n\n\nif (\n    design_rank\n    !=\n    design_matrix.shape[1]\n):\n\n    raise RuntimeError(\n        "STEP 15D design is not full rank."\n    )\n\n\nif design_matrix.shape[1] != 4:\n\n    raise RuntimeError(\n        "Expected exactly four "\n        "model parameters."\n    )\n\n\n# ============================================================\n# O. Fit DESeq2\n# ============================================================\n\nprint(\n    "\\nRunning STEP 15D DESeq2 model..."\n)\n\n\ndds.deseq2()\n\n\nprint(\n    "\\nDESEQ2 FIT COMPLETE"\n)\n\n\n# ============================================================\n# P. Save size factors\n# ============================================================\n\nif (\n    "size_factors"\n    not in\n    dds.obs.columns\n):\n\n    raise RuntimeError(\n        "PyDESeq2 size factors missing."\n    )\n\n\nsize_factors = (\n    dds.obs[\n        "size_factors"\n    ]\n    .to_numpy(\n        dtype=float\n    )\n)\n\n\nif len(\n    size_factors\n) != EXPECTED_DONORS:\n\n    raise RuntimeError(\n        "Unexpected size-factor count."\n    )\n\n\nif (\n    not np.isfinite(\n        size_factors\n    ).all()\n    or\n    (\n        size_factors <= 0\n    ).any()\n):\n\n    raise RuntimeError(\n        "Invalid DESeq2 size factors."\n    )\n\n\npd.DataFrame(\n    {\n        "GSM":\n            sample_ids,\n\n        "size_factor":\n            size_factors,\n    }\n).to_csv(\n    SIZE_FACTOR_PATH,\n    index=False,\n)\n\n\n# ============================================================\n# Q. AD vs Control Wald statistics\n#\n# Default Wald p-value = two-sided.\n# ============================================================\n\nstats = DeseqStats(\n\n    dds,\n\n    contrast=[\n        "diagnosis",\n        "AD",\n        "Control",\n    ],\n\n    alpha=0.05,\n\n    cooks_filter=True,\n\n    independent_filter=True,\n\n    n_cpus=2,\n\n    quiet=False,\n)\n\n\nstats.summary()\n\n\nresults = (\n    stats.results_df\n    .copy()\n)\n\n\n# ============================================================\n# R. Construct complete genome-wide result table\n# ============================================================\n\nresults.index = pd.Index(\n    results.index.astype(\n        str\n    ).tolist(),\n    dtype=object,\n)\n\n\nresults.index.name = "gene_id"\n\n\nresults = (\n    results\n    .reset_index()\n)\n\n\ngene_map = (\n    features[\n        [\n            "gene_id",\n            "gene_symbol",\n        ]\n    ]\n    .copy()\n)\n\n\nresults = results.merge(\n    gene_map,\n    on="gene_id",\n    how="left",\n    validate="one_to_one",\n)\n\n\nif len(\n    results\n) != EXPECTED_GENES:\n\n    raise RuntimeError(\n        "DE result does not contain "\n        "all 16,871 genes."\n    )\n\n\nrequired_result_columns = [\n    "baseMean",\n    "log2FoldChange",\n    "lfcSE",\n    "stat",\n    "pvalue",\n    "padj",\n]\n\n\nmissing = [\n    column\n    for column in required_result_columns\n    if column not in results.columns\n]\n\n\nif missing:\n\n    raise RuntimeError(\n        "Missing DESeq2 result columns: "\n        f"{missing}"\n    )\n\n\nresults = results[\n    [\n        "gene_id",\n        "gene_symbol",\n        "baseMean",\n        "log2FoldChange",\n        "lfcSE",\n        "stat",\n        "pvalue",\n        "padj",\n    ]\n]\n\n\n# ============================================================\n# S. SAVE FULL RESULTS BEFORE TARGET REVEAL\n# ============================================================\n\nresults.to_csv(\n\n    RESULTS_PATH,\n\n    index=False,\n\n    compression={\n        "method":\n            "gzip",\n\n        "compresslevel":\n            6,\n    },\n)\n\n\nfinite_p = np.isfinite(\n    results[\n        "pvalue"\n    ]\n)\n\n\nfinite_padj = np.isfinite(\n    results[\n        "padj"\n    ]\n)\n\n\ngenome_sig = (\n    finite_padj\n    &\n    (\n        results[\n            "padj"\n        ]\n        <\n        0.05\n    )\n)\n\n\nsignificant = (\n    results.loc[\n        genome_sig\n    ]\n    .copy()\n    .sort_values(\n        [\n            "padj",\n            "pvalue",\n        ]\n    )\n)\n\n\nsignificant.to_csv(\n\n    SIGNIFICANT_PATH,\n\n    index=False,\n\n    compression={\n        "method":\n            "gzip",\n\n        "compresslevel":\n            6,\n    },\n)\n\n\nprint(\n    "\\nFULL GENOME-WIDE RESULTS SAVED."\n)\n\nprint(\n    "Target reveal is now permitted."\n)\n\n\n# ============================================================\n# T. FIRST STEP-15 TARGET REVEAL\n# ============================================================\n\nlocked_targets = [\n    PRIMARY_GENE,\n    *SECONDARY_GENES,\n]\n\n\ntarget_rows = []\n\n\nfor gene in locked_targets:\n\n    matches = results.loc[\n        results[\n            "gene_symbol"\n        ].astype(str)\n        ==\n        gene\n    ].copy()\n\n\n    if len(matches) == 0:\n\n        target_rows.append(\n            {\n                "gene":\n                    gene,\n\n                "status":\n                    "not_in_frozen_primary_universe",\n\n                "gene_id":\n                    None,\n\n                "baseMean":\n                    np.nan,\n\n                "log2FoldChange":\n                    np.nan,\n\n                "lfcSE":\n                    np.nan,\n\n                "stat":\n                    np.nan,\n\n                "pvalue":\n                    np.nan,\n\n                "genome_wide_padj":\n                    np.nan,\n            }\n        )\n\n\n    elif len(matches) == 1:\n\n        row = matches.iloc[0]\n\n        target_rows.append(\n            {\n                "gene":\n                    gene,\n\n                "status":\n                    "tested",\n\n                "gene_id":\n                    str(\n                        row[\n                            "gene_id"\n                        ]\n                    ),\n\n                "baseMean":\n                    float(\n                        row[\n                            "baseMean"\n                        ]\n                    ),\n\n                "log2FoldChange":\n                    float(\n                        row[\n                            "log2FoldChange"\n                        ]\n                    ),\n\n                "lfcSE":\n                    float(\n                        row[\n                            "lfcSE"\n                        ]\n                    ),\n\n                "stat":\n                    float(\n                        row[\n                            "stat"\n                        ]\n                    ),\n\n                "pvalue":\n                    float(\n                        row[\n                            "pvalue"\n                        ]\n                    ),\n\n                "genome_wide_padj":\n                    (\n                        float(\n                            row[\n                                "padj"\n                            ]\n                        )\n                        if np.isfinite(\n                            row[\n                                "padj"\n                            ]\n                        )\n                        else np.nan\n                    ),\n            }\n        )\n\n\n    else:\n\n        target_rows.append(\n            {\n                "gene":\n                    gene,\n\n                "status":\n                    (\n                        "ambiguous_multiple_features:"\n                        f"{len(matches)}"\n                    ),\n\n                "gene_id":\n                    None,\n\n                "baseMean":\n                    np.nan,\n\n                "log2FoldChange":\n                    np.nan,\n\n                "lfcSE":\n                    np.nan,\n\n                "stat":\n                    np.nan,\n\n                "pvalue":\n                    np.nan,\n\n                "genome_wide_padj":\n                    np.nan,\n            }\n        )\n\n\ntargets = pd.DataFrame(\n    target_rows\n)\n\n\n# ============================================================\n# U. Secondary six-gene BH family\n# ============================================================\n\ntargets[\n    "secondary_BH_qvalue"\n] = np.nan\n\n\nsecondary_mask = (\n    targets[\n        "gene"\n    ].isin(\n        SECONDARY_GENES\n    )\n)\n\n\nsecondary = targets.loc[\n    secondary_mask\n].copy()\n\n\nsecondary_family_complete = bool(\n    (\n        secondary[\n            "status"\n        ]\n        ==\n        "tested"\n    ).all()\n    and\n    np.isfinite(\n        secondary[\n            "pvalue"\n        ]\n    ).all()\n)\n\n\ndef benjamini_hochberg(pvalues):\n\n    p = np.asarray(\n        pvalues,\n        dtype=float,\n    )\n\n    m = len(p)\n\n    order = np.argsort(p)\n\n    ranked = p[order]\n\n    adjusted = (\n        ranked\n        *\n        m\n        /\n        np.arange(\n            1,\n            m + 1\n        )\n    )\n\n    adjusted = np.minimum.accumulate(\n        adjusted[\n            ::-1\n        ]\n    )[\n        ::-1\n    ]\n\n    adjusted = np.minimum(\n        adjusted,\n        1.0,\n    )\n\n    output = np.empty(\n        m,\n        dtype=float,\n    )\n\n    output[\n        order\n    ] = adjusted\n\n    return output\n\n\nif secondary_family_complete:\n\n    qvalues = benjamini_hochberg(\n        secondary[\n            "pvalue"\n        ].to_numpy(\n            dtype=float\n        )\n    )\n\n    for index, qvalue in zip(\n        secondary.index,\n        qvalues,\n    ):\n\n        targets.loc[\n            index,\n            "secondary_BH_qvalue"\n        ] = qvalue\n\n\ntargets.to_csv(\n    TARGET_PATH,\n    index=False,\n)\n\n\n# ============================================================\n# V. Evaluate frozen CREB5 STEP-15 success rule\n# ============================================================\n\ncreb5_rows = targets.loc[\n    targets[\n        "gene"\n    ]\n    ==\n    PRIMARY_GENE\n]\n\n\nif len(\n    creb5_rows\n) != 1:\n\n    raise RuntimeError(\n        "Unexpected CREB5 target structure."\n    )\n\n\ncreb5 = creb5_rows.iloc[0]\n\n\nCREB5_present = (\n    creb5[\n        "status"\n    ]\n    ==\n    "tested"\n)\n\n\nCREB5_positive = bool(\n    CREB5_present\n    and\n    np.isfinite(\n        creb5[\n            "log2FoldChange"\n        ]\n    )\n    and\n    creb5[\n        "log2FoldChange"\n    ]\n    >\n    0\n)\n\n\nCREB5_nominal_significant = bool(\n    CREB5_present\n    and\n    np.isfinite(\n        creb5[\n            "pvalue"\n        ]\n    )\n    and\n    creb5[\n        "pvalue"\n    ]\n    <\n    0.05\n)\n\n\nCREB5_sensitivity_support = bool(\n    CREB5_present\n    and\n    CREB5_positive\n    and\n    CREB5_nominal_significant\n)\n\n\n# ============================================================\n# W. Genome-wide result summaries\n# ============================================================\n\nn_genome_sig = int(\n    genome_sig.sum()\n)\n\n\nn_genome_up = int(\n    (\n        genome_sig\n        &\n        (\n            results[\n                "log2FoldChange"\n            ]\n            >\n            0\n        )\n    ).sum()\n)\n\n\nn_genome_down = int(\n    (\n        genome_sig\n        &\n        (\n            results[\n                "log2FoldChange"\n            ]\n            <\n            0\n        )\n    ).sum()\n)\n\n\n# ============================================================\n# X. Compare with already-known STEP 13B primary result\n# ============================================================\n\nwith open(\n    PRIMARY_RESULT_LOCK_PATH,\n    "r",\n    encoding="utf-8",\n) as handle:\n\n    primary_result = json.load(\n        handle\n    )\n\n\nprimary_lfc = primary_result.get(\n    "CREB5_log2FoldChange"\n)\n\n\nprimary_p = primary_result.get(\n    "CREB5_nominal_pvalue"\n)\n\n\nsame_direction_as_primary = bool(\n    primary_lfc is not None\n    and\n    np.isfinite(\n        float(\n            primary_lfc\n        )\n    )\n    and\n    float(\n        primary_lfc\n    )\n    >\n    0\n    and\n    CREB5_positive\n)\n\n\n# ============================================================\n# Y. Formal STEP 15D result lock\n# ============================================================\n\nresult_lock = {\n\n    "dataset":\n        "GSE157827",\n\n    "step":\n        "15D",\n\n    "version":\n        "v1",\n\n    "created_utc":\n        datetime.now(\n            timezone.utc\n        ).isoformat(),\n\n    "analysis_role":\n        (\n            "Independent astrocyte-annotation "\n            "sensitivity differential expression"\n        ),\n\n    "STEP15C_identity_lock":\n        str(\n            STEP15C_LOCK_PATH\n        ),\n\n    "STEP15C_identity_lock_sha256":\n        sha256_file(\n            STEP15C_LOCK_PATH\n        ),\n\n    "STEP15C2_pseudobulk_lock":\n        str(\n            STEP15C2_LOCK_PATH\n        ),\n\n    "STEP15C2_pseudobulk_lock_sha256":\n        sha256_file(\n            STEP15C2_LOCK_PATH\n        ),\n\n    "STEP11B_design_lock":\n        str(\n            DESIGN_LOCK_PATH\n        ),\n\n    "STEP11B_design_manifest":\n        str(\n            DESIGN_PATH\n        ),\n\n    "STEP12_gene_universe_lock":\n        str(\n            STEP12_LOCK_PATH\n        ),\n\n    "alternative_astrocyte_nuclei":\n        EXPECTED_ASTROCYTES,\n\n    "number_of_donors":\n        EXPECTED_DONORS,\n\n    "AD_donors":\n        EXPECTED_AD,\n\n    "Control_donors":\n        EXPECTED_CONTROL,\n\n    "frozen_gene_universe":\n        EXPECTED_GENES,\n\n    "frozen_gene_universe_sha256":\n        EXPECTED_UNIVERSE_SHA256,\n\n    "gene_universe_refiltered":\n        False,\n\n    "DE_engine":\n        "PyDESeq2 0.5.4",\n\n    "design":\n        "~ age + sex + diagnosis",\n\n    "contrast":\n        "AD versus Control",\n\n    "reference_diagnosis":\n        "Control",\n\n    "reference_sex":\n        "F",\n\n    "positive_log2FC_meaning":\n        "higher expression in AD",\n\n    "refit_cooks":\n        True,\n\n    "cooks_filter":\n        True,\n\n    "independent_filter":\n        True,\n\n    "genome_wide_alpha":\n        0.05,\n\n    "genome_wide_FDR_significant":\n        n_genome_sig,\n\n    "genome_wide_FDR_up_in_AD":\n        n_genome_up,\n\n    "genome_wide_FDR_down_in_AD":\n        n_genome_down,\n\n    "finite_nominal_pvalues":\n        int(\n            finite_p.sum()\n        ),\n\n    "finite_genome_wide_adjusted_pvalues":\n        int(\n            finite_padj.sum()\n        ),\n\n    "CREB5_sensitivity_support_rule":\n        (\n            "log2FoldChange > 0 AND "\n            "two-sided nominal p < 0.05"\n        ),\n\n    "CREB5_present":\n        bool(\n            CREB5_present\n        ),\n\n    "CREB5_gene_id":\n        (\n            str(\n                creb5[\n                    "gene_id"\n                ]\n            )\n            if CREB5_present\n            else None\n        ),\n\n    "CREB5_baseMean":\n        json_number(\n            creb5[\n                "baseMean"\n            ]\n        ),\n\n    "CREB5_log2FoldChange":\n        json_number(\n            creb5[\n                "log2FoldChange"\n            ]\n        ),\n\n    "CREB5_lfcSE":\n        json_number(\n            creb5[\n                "lfcSE"\n            ]\n        ),\n\n    "CREB5_Wald_stat":\n        json_number(\n            creb5[\n                "stat"\n            ]\n        ),\n\n    "CREB5_nominal_pvalue":\n        json_number(\n            creb5[\n                "pvalue"\n            ]\n        ),\n\n    "CREB5_genome_wide_padj":\n        json_number(\n            creb5[\n                "genome_wide_padj"\n            ]\n        ),\n\n    "CREB5_positive_direction":\n        bool(\n            CREB5_positive\n        ),\n\n    "CREB5_nominal_p_below_0_05":\n        bool(\n            CREB5_nominal_significant\n        ),\n\n    "CREB5_SENSITIVITY_SUPPORT":\n        bool(\n            CREB5_sensitivity_support\n        ),\n\n    "primary_STEP13B_CREB5_log2FoldChange":\n        json_number(\n            primary_lfc\n        ),\n\n    "primary_STEP13B_CREB5_nominal_pvalue":\n        json_number(\n            primary_p\n        ),\n\n    "same_positive_direction_as_primary":\n        same_direction_as_primary,\n\n    "secondary_locked_genes":\n        SECONDARY_GENES,\n\n    "secondary_testing_rule":\n        (\n            "Benjamini-Hochberg correction "\n            "across the six locked secondary genes"\n        ),\n\n    "secondary_family_complete":\n        bool(\n            secondary_family_complete\n        ),\n\n    "software":\n        observed_versions,\n\n    "full_results":\n        str(\n            RESULTS_PATH\n        ),\n\n    "FDR05_results":\n        str(\n            SIGNIFICANT_PATH\n        ),\n\n    "locked_target_results":\n        str(\n            TARGET_PATH\n        ),\n\n    "size_factors":\n        str(\n            SIZE_FACTOR_PATH\n        ),\n}\n\n\nwith open(\n    LOCK_PATH,\n    "w",\n    encoding="utf-8",\n) as handle:\n\n    json.dump(\n        result_lock,\n        handle,\n        indent=2,\n    )\n\n\n# ============================================================\n# Z. FINAL REPORT\n# ============================================================\n\nprint(\n    "\\n"\n    +\n    "=" * 78\n)\n\nprint(\n    "GSE157827 STEP 15D COMPLETE"\n)\n\nprint(\n    "=" * 78\n)\n\n\nprint(\n    f"\\nGenes tested: "\n    f"{len(results):,}"\n)\n\nprint(\n    f"Finite nominal p-values: "\n    f"{int(finite_p.sum()):,}"\n)\n\nprint(\n    f"Finite genome-wide adjusted p-values: "\n    f"{int(finite_padj.sum()):,}"\n)\n\n\nprint(\n    "\\nGenome-wide FDR < 0.05:"\n)\n\nprint(\n    f"    Total: {n_genome_sig}"\n)\n\nprint(\n    f"    Up in AD: {n_genome_up}"\n)\n\nprint(\n    f"    Down in AD: {n_genome_down}"\n)\n\n\nprint(\n    "\\nDESeq2 size-factor range:"\n)\n\nprint(\n    "    min / median / max =",\n    f"{size_factors.min():.4f}",\n    "/",\n    f"{np.median(size_factors):.4f}",\n    "/",\n    f"{size_factors.max():.4f}",\n)\n\n\nprint(\n    "\\n"\n    +\n    "=" * 78\n)\n\nprint(\n    "CREB5 — INDEPENDENT ANNOTATION SENSITIVITY"\n)\n\nprint(\n    "=" * 78\n)\n\n\nif CREB5_present:\n\n    print(\n        "Gene ID:",\n        creb5[\n            "gene_id"\n        ]\n    )\n\n    print(\n        "baseMean:",\n        f"{creb5[\'baseMean\']:.6f}"\n    )\n\n    print(\n        "log2FoldChange (AD vs Control):",\n        f"{creb5[\'log2FoldChange\']:.6f}"\n    )\n\n    print(\n        "lfcSE:",\n        f"{creb5[\'lfcSE\']:.6f}"\n    )\n\n    print(\n        "Wald statistic:",\n        f"{creb5[\'stat\']:.6f}"\n    )\n\n    print(\n        "Nominal two-sided p-value:",\n        f"{creb5[\'pvalue\']:.8g}"\n    )\n\n    print(\n        "Genome-wide padj:",\n        (\n            f"{creb5[\'genome_wide_padj\']:.8g}"\n            if np.isfinite(\n                creb5[\n                    "genome_wide_padj"\n                ]\n            )\n            else\n            "NA"\n        )\n    )\n\n\nprint(\n    "\\nDirection positive:",\n    "YES"\n    if CREB5_positive\n    else "NO"\n)\n\nprint(\n    "Nominal p < 0.05:",\n    "YES"\n    if CREB5_nominal_significant\n    else "NO"\n)\n\n\nprint(\n    "\\nCREB5 INDEPENDENT-ANNOTATION "\n    "SENSITIVITY SUPPORT:",\n    "YES"\n    if CREB5_sensitivity_support\n    else "NO"\n)\n\n\nprint(\n    "\\nPrimary STEP 13B:"\n)\n\nprint(\n    "    log2FC:",\n    primary_lfc\n)\n\nprint(\n    "    nominal p:",\n    primary_p\n)\n\nprint(\n    "Same positive direction as primary:",\n    "YES"\n    if same_direction_as_primary\n    else "NO"\n)\n\n\nprint(\n    "\\nSecondary locked genes:"\n)\n\n\nsecondary_display = targets.loc[\n    targets[\n        "gene"\n    ].isin(\n        SECONDARY_GENES\n    ),\n    [\n        "gene",\n        "status",\n        "gene_id",\n        "baseMean",\n        "log2FoldChange",\n        "pvalue",\n        "secondary_BH_qvalue",\n        "genome_wide_padj",\n    ],\n]\n\n\nprint(\n    secondary_display.to_string(\n        index=False,\n        float_format=lambda x:\n            f"{x:.6g}",\n    )\n)\n\n\nprint(\n    "\\nFull genome-wide results:"\n)\n\nprint(\n    RESULTS_PATH\n)\n\n\nprint(\n    "\\nSTEP 15D sensitivity lock:"\n)\n\nprint(\n    LOCK_PATH\n)\n'

    RUNNER_PATH.write_text(textwrap.dedent(runner), encoding='utf-8')

    env = os.environ.copy()

    env['GSE_VALIDATION_DIR'] = str(VALIDATION_DIR)

    env['PYTHONHASHSEED'] = '0'

    env['OMP_NUM_THREADS'] = '1'

    env['OPENBLAS_NUM_THREADS'] = '1'

    env['MKL_NUM_THREADS'] = '1'

    env['PYTHONUNBUFFERED'] = '1'

    print('\n' + '=' * 78)

    print('LAUNCHING STEP 15D')

    print('=' * 78)

    process = subprocess.Popen([sys.executable, '-u', str(RUNNER_PATH)], env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)

    for line in process.stdout:
        print(line, end='', flush=True)

    return_code = process.wait()

    if return_code != 0:
        raise RuntimeError('STEP 15D failed. The actual error is printed immediately above.')


In [ ]:
if RUN_PIPELINE:
    from pathlib import Path

    from datetime import datetime, timezone

    import hashlib

    import json

    import numpy as np

    import pandas as pd

    BASE = Path('/content/drive/MyDrive/AD_Astrocyte_Paper_01')

    dirs = sorted(BASE.glob('*GSE157827*confirmatory*validation*'))

    if len(dirs) != 1:
        raise RuntimeError(f'Expected one validation directory; found {len(dirs)}.')

    VALIDATION_DIR = dirs[0]

    def resolve_one(pattern, label):
        matches = sorted(VALIDATION_DIR.glob(pattern))
        if len(matches) != 1:
            raise RuntimeError(f'{label}: expected one match for {pattern}; found {len(matches)}:\n' + '\n'.join((str(x) for x in matches)))
        return matches[0]

    PLAN_PATH = resolve_one('*STEP15A*ANNOTATION*SENSITIVITY*PLAN*LOCK*.json', 'STEP 15A plan')

    PRIMARY_LOCK_PATH = resolve_one('*STEP13B_PRIMARY_CONFIRMATORY_RESULT_LOCK*.json', 'STEP 13B primary lock')

    PRIMARY_RESULTS_PATH = resolve_one('*STEP13B_DESeq2_primary_age_sex_adjusted_v1.csv.gz', 'STEP 13B results')

    ALT_LOCK_PATH = resolve_one('*STEP15D_ALTERNATIVE_ANNOTATION_SENSITIVITY_LOCK*.json', 'STEP 15D lock')

    ALT_RESULTS_PATH = resolve_one('*STEP15D_DESeq2_alternative_annotation_age_sex_adjusted_v1.csv.gz', 'STEP 15D results')

    PRIMARY_TARGET_PATH = resolve_one('*STEP13B_locked_target_results_v1.csv', 'STEP 13B target table')

    ALT_TARGET_PATH = resolve_one('*STEP15D_locked_target_results_v1.csv', 'STEP 15D target table')

    TARGET_COMPARISON_PATH = VALIDATION_DIR / 'GSE157827_STEP15E_locked_target_concordance_v1.csv'

    LOCK_PATH = VALIDATION_DIR / 'GSE157827_STEP15E_PRIMARY_VS_ALTERNATIVE_CONCORDANCE_LOCK_v1.json'

    if LOCK_PATH.exists():
        raise RuntimeError(f'STEP 15E already completed. Do not overwrite:\n{LOCK_PATH}')

    with open(PLAN_PATH, 'r', encoding='utf-8') as handle:
        plan = json.load(handle)

    with open(PRIMARY_LOCK_PATH, 'r', encoding='utf-8') as handle:
        primary_lock = json.load(handle)

    with open(ALT_LOCK_PATH, 'r', encoding='utf-8') as handle:
        alt_lock = json.load(handle)

    if plan.get('CREB5_sensitivity_support_rule') != 'positive log2FoldChange and nominal two-sided p < 0.05':
        raise RuntimeError('STEP 15A CREB5 rule changed.')

    if alt_lock.get('CREB5_SENSITIVITY_SUPPORT') is not True:
        raise RuntimeError('STEP 15D did not satisfy the frozen CREB5 sensitivity rule.')

    primary = pd.read_csv(PRIMARY_RESULTS_PATH, dtype={'gene_id': str, 'gene_symbol': str})

    alternative = pd.read_csv(ALT_RESULTS_PATH, dtype={'gene_id': str, 'gene_symbol': str})

    if len(primary) != 16871:
        raise RuntimeError('Primary result table is not 16,871 genes.')

    if len(alternative) != 16871:
        raise RuntimeError('Alternative result table is not 16,871 genes.')

    if primary['gene_id'].duplicated().any():
        raise RuntimeError('Duplicate primary gene IDs.')

    if alternative['gene_id'].duplicated().any():
        raise RuntimeError('Duplicate alternative gene IDs.')

    comparison = primary.merge(alternative, on=['gene_id', 'gene_symbol'], how='inner', suffixes=('_primary', '_alternative'), validate='one_to_one')

    if len(comparison) != 16871:
        raise RuntimeError('Primary/alternative gene universes differ.')

    finite_lfc = np.isfinite(comparison['log2FoldChange_primary']) & np.isfinite(comparison['log2FoldChange_alternative'])

    lfc_comparison = comparison.loc[finite_lfc].copy()

    if len(lfc_comparison) != 16871:
        raise RuntimeError('Not all genes have finite LFCs.')

    pearson_lfc = float(lfc_comparison[['log2FoldChange_primary', 'log2FoldChange_alternative']].corr(method='pearson').iloc[0, 1])


In [ ]:
if RUN_PIPELINE:
    spearman_lfc = float(lfc_comparison[['log2FoldChange_primary', 'log2FoldChange_alternative']].corr(method='spearman').iloc[0, 1])

    primary_sign = np.sign(lfc_comparison['log2FoldChange_primary'].to_numpy(dtype=float))

    alternative_sign = np.sign(lfc_comparison['log2FoldChange_alternative'].to_numpy(dtype=float))

    nonzero_both = (primary_sign != 0) & (alternative_sign != 0)

    sign_concordance = float(np.mean(primary_sign[nonzero_both] == alternative_sign[nonzero_both]))

    absolute_lfc_difference = np.abs(lfc_comparison['log2FoldChange_alternative'].to_numpy(dtype=float) - lfc_comparison['log2FoldChange_primary'].to_numpy(dtype=float))

    median_absolute_lfc_difference = float(np.median(absolute_lfc_difference))

    primary_targets = pd.read_csv(PRIMARY_TARGET_PATH, dtype={'gene': str, 'gene_id': str})

    alt_targets = pd.read_csv(ALT_TARGET_PATH, dtype={'gene': str, 'gene_id': str})

    targets = primary_targets.merge(alt_targets, on=['gene', 'status', 'gene_id'], suffixes=('_primary', '_alternative'), validate='one_to_one')

    targets['delta_log2FC_alternative_minus_primary'] = targets['log2FoldChange_alternative'] - targets['log2FoldChange_primary']

    targets['same_direction'] = np.sign(targets['log2FoldChange_primary']) == np.sign(targets['log2FoldChange_alternative'])

    targets.to_csv(TARGET_COMPARISON_PATH, index=False)

    creb5 = targets.loc[targets['gene'] == 'CREB5']

    if len(creb5) != 1:
        raise RuntimeError('Expected exactly one CREB5 comparison.')

    creb5 = creb5.iloc[0]

    primary_lfc = float(creb5['log2FoldChange_primary'])

    alt_lfc = float(creb5['log2FoldChange_alternative'])

    primary_p = float(creb5['pvalue_primary'])

    alt_p = float(creb5['pvalue_alternative'])

    creb5_delta_lfc = alt_lfc - primary_lfc

    creb5_percent_lfc_change = 100.0 * creb5_delta_lfc / abs(primary_lfc)

    def sha256_file(path):
        digest = hashlib.sha256()
        with open(path, 'rb') as handle:
            for block in iter(lambda: handle.read(1024 * 1024), b''):
                digest.update(block)
        return digest.hexdigest()

    lock = {'dataset': 'GSE157827', 'step': '15E', 'version': 'v1', 'created_utc': datetime.now(timezone.utc).isoformat(), 'analysis_role': 'Descriptive concordance audit between frozen primary and independent-annotation sensitivity analyses', 'new_differential_expression_performed': False, 'new_filtering_performed': False, 'new_significance_testing_performed': False, 'parameter_tuning_performed': False, 'primary_result_lock': str(PRIMARY_LOCK_PATH), 'primary_result_lock_sha256': sha256_file(PRIMARY_LOCK_PATH), 'alternative_result_lock': str(ALT_LOCK_PATH), 'alternative_result_lock_sha256': sha256_file(ALT_LOCK_PATH), 'genes_compared': int(len(comparison)), 'genome_wide_LFC_Pearson_r': pearson_lfc, 'genome_wide_LFC_Spearman_rho': spearman_lfc, 'genome_wide_direction_concordance': sign_concordance, 'median_absolute_log2FC_difference': median_absolute_lfc_difference, 'CREB5': {'gene_id': str(creb5['gene_id']), 'primary_log2FoldChange': primary_lfc, 'alternative_log2FoldChange': alt_lfc, 'delta_log2FoldChange': creb5_delta_lfc, 'percent_effect_change': creb5_percent_lfc_change, 'primary_nominal_pvalue': primary_p, 'alternative_nominal_pvalue': alt_p, 'same_positive_direction': bool(primary_lfc > 0 and alt_lfc > 0), 'primary_nominal_p_below_0_05': bool(primary_p < 0.05), 'alternative_nominal_p_below_0_05': bool(alt_p < 0.05), 'frozen_STEP15_support_rule_met': bool(alt_lfc > 0 and alt_p < 0.05)}, 'locked_target_comparison': str(TARGET_COMPARISON_PATH), 'STEP15_CONCLUSION': 'CREB5 association is robust to the predeclared independent astrocyte-annotation sensitivity analysis.'}

    with open(LOCK_PATH, 'w', encoding='utf-8') as handle:
        json.dump(lock, handle, indent=2)

    print('=' * 78)

    print('GSE157827 STEP 15E COMPLETE')

    print('=' * 78)

    print(f'\nGenes compared: {len(comparison):,}')

    print('Genome-wide log2FC Pearson r:', f'{pearson_lfc:.6f}')

    print('Genome-wide log2FC Spearman rho:', f'{spearman_lfc:.6f}')

    print('Genome-wide direction concordance:', f'{100 * sign_concordance:.2f}%')

    print('Median absolute log2FC difference:', f'{median_absolute_lfc_difference:.6f}')

    print('\n' + '=' * 78)

    print('CREB5 PRIMARY vs INDEPENDENT ANNOTATION')

    print('=' * 78)

    print('\nPrimary log2FC:', f'{primary_lfc:.9f}')

    print('Alternative log2FC:', f'{alt_lfc:.9f}')

    print('Difference:', f'{creb5_delta_lfc:+.9f}')

    print('Relative effect change:', f'{creb5_percent_lfc_change:+.3f}%')

    print('\nPrimary nominal p:', f'{primary_p:.8g}')

    print('Alternative nominal p:', f'{alt_p:.8g}')

    print('\nSame positive direction:', 'YES' if primary_lfc > 0 and alt_lfc > 0 else 'NO')

    print('Primary p < 0.05:', 'YES' if primary_p < 0.05 else 'NO')

    print('Alternative p < 0.05:', 'YES' if alt_p < 0.05 else 'NO')

    print('\nSTEP 15 INDEPENDENT-ANNOTATION ROBUSTNESS: PASS')

    print('\nLocked-target comparison:')

    print(targets[['gene', 'log2FoldChange_primary', 'log2FoldChange_alternative', 'delta_log2FC_alternative_minus_primary', 'pvalue_primary', 'pvalue_alternative', 'same_direction']].to_string(index=False, float_format=lambda x: f'{x:.6g}'))


In [ ]:
if RUN_PIPELINE:
    print('\nSTEP 15E lock:')

    print(LOCK_PATH)


## Frozen-result check

This cell is the fast test when `RUN_PIPELINE = False`. It does not fit a model. It confirms that the primary result and four robustness outputs required by the manuscript are present and displays the CREB5 rows.


In [ ]:

import pandas as pd

result_files = {
    "Primary adjusted model": PROJECT / "GSE157827_STEP13B_locked_target_results_v1.csv",
    "Diagnosis-only model": PROJECT / "GSE157827_STEP14A_CREB5_sensitivity_diagnosis_only_v1.csv",
    "Exclude severe-QC-loss donors": PROJECT / "GSE157827_STEP14B_CREB5_sensitivity_exclude_severe_QC_loss_v1.csv",
    "Exclude Scrublet-rescue donor": PROJECT / "GSE157827_STEP14C_CREB5_sensitivity_exclude_scrublet_rescue_donor_v1.csv",
    "Independent astrocyte annotation": PROJECT / "GSE157827_STEP15D_locked_target_results_v1.csv",
}

status = pd.DataFrame(
    [{"analysis": name, "exists": path.exists(), "path": str(path)}
     for name, path in result_files.items()]
)
display(status)

def creb5_row(path):
    table = pd.read_csv(path)
    for column in table.columns:
        values = table[column].astype(str).str.upper()
        hit = table.loc[values.eq("CREB5")]
        if len(hit):
            return hit.iloc[[0]].copy()
    if len(table) == 1:
        return table.iloc[[0]].copy()
    raise RuntimeError(f"CREB5 row not found in {path.name}")

if status["exists"].all():
    rows = []
    for name, path in result_files.items():
        row = creb5_row(path)
        row.insert(0, "analysis", name)
        rows.append(row)
    creb5_summary = pd.concat(rows, ignore_index=True, sort=False)
    display(creb5_summary)
else:
    print("One or more frozen outputs are missing. Set RUN_PIPELINE=True in a clean reconstruction folder.")


### Expected primary result

The frozen primary GSE157827 estimate is CREB5 log2FC ≈ 0.857 with nominal two-sided P ≈ 3.78 × 10⁻⁵. The sensitivity analyses are robustness checks on the same cohort and are not independent studies or pooled estimates.
